In [ ]:
PROJECT_NAME = 'meshqc_train_full'                                         
N_WORKERS = 4
FOLDS = '0,1,2,3,4'

from pathlib import Path
import os, sys, json, time, shutil, subprocess, signal

os.environ['CUBLAS_WORKSPACE_CONFIG'] = ':4096:8'                     
os.environ['PYTHONHASHSEED'] = '42'

BUNDLE = Path('/content/meshqc_code'); BUNDLE.mkdir(parents=True, exist_ok=True)
PROJECT_DIR = Path('/content/drive/MyDrive') / PROJECT_NAME

TARGETS = ['abstract','artifacts','intersection','lowpoly','noisy',
           'open','partial','scale','set','simple','quality']
NAMES = ['vitb336', 'dino448_s2', 'v5_224']
print('исходники:', BUNDLE, '\nпрогон:   ', PROJECT_DIR)


In [ ]:
subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q',
                       'kagglehub', 'timm', 'open_clip_torch', 'lightgbm',
                       'iterative-stratification', 'pyarrow'])

from google.colab import drive
if not Path('/content/drive/MyDrive').exists():
    drive.mount('/content/drive')
PROJECT_DIR.mkdir(parents=True, exist_ok=True)

import kagglehub
DATA_ROOT = Path(kagglehub.dataset_download('daniilantonov5/3d-mesh-quality-control'))
print('данные:', DATA_ROOT)


In [ ]:
%%writefile /content/meshqc_code/runtime.py
"""Runtime for one isolated GPU branch. Executed by worker.py, never in notebook globals."""
import os, json, time, hashlib, shutil, subprocess, sys
from pathlib import Path
import joblib
import numpy as np
import pandas as pd
import torch


def digest_json(x):
    return hashlib.sha256(json.dumps(x, sort_keys=True, ensure_ascii=False).encode()).hexdigest()


def _atomic(path, writer, mirror=True):
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    tmp = path.with_name(path.name + '.tmp')
    writer(tmp)
    os.replace(tmp, path)


def record_status(state, **kw):
    _atomic(BRANCH_DIR / 'status.json', lambda p: p.write_text(
        json.dumps(dict(branch=BRANCH, state=state, time=time.time(), **kw), ensure_ascii=False), encoding='utf-8'))


def art_path(name): return ART / (name + '.joblib')
def rt(name): return CFG.run_tag + '_' + name
def has_artifact(name): return art_path(name).exists() and name not in CFG.force_recompute
def load_artifact(name): return joblib.load(art_path(name))


def save_artifact(name, obj, compress=3):
    if name.endswith('_preds') and isinstance(obj, dict) and 'oof' in obj:
        for k in ('oof', 'te'):
            a = np.asarray(obj[k])
            if a.ndim != 2 or a.shape[1] != 11 or not np.isfinite(a).all() or np.any((a < 0) | (a > 1)):
                raise ValueError(f'{name}: invalid {k}; fold predictions NOT saved')
        obj = dict(obj, contract=CONTRACT, cfg=current_cfg())
    _atomic(art_path(name), lambda p: joblib.dump(obj, p, compress=compress))
    return obj


def cached(name, fn, compress=3):
    if CFG.resume and has_artifact(name):
        print('[cache]', name, flush=True)
        return load_artifact(name)
    obj = fn()
    return save_artifact(name, obj, compress)


def current_cfg():
    out = {}
    for k in dir(CFG):
        if k.startswith('_'): continue
        v = getattr(CFG, k)
        if isinstance(v, (str, bool, int, float, list, tuple, dict)) or v is None:
            out[k] = v
    out['feat_ver'] = globals().get('FEAT_VER')
    out['feature_columns'] = list(globals().get('FEAT_COLS') or [])
    return out


def run_cfg(): return current_cfg()


def _cfg_mismatch(saved):
                                                                               
    return []


def rng_state():
    return dict(py=random.getstate(), np=np.random.get_state(), torch=torch.get_rng_state(),
                cuda=torch.cuda.get_rng_state_all() if torch.cuda.is_available() else [])


def set_rng_state(st):
    random.setstate(st['py']); np.random.set_state(st['np']); torch.set_rng_state(st['torch'].cpu())
    if torch.cuda.is_available() and st.get('cuda'):
        torch.cuda.set_rng_state_all([s.cpu() for s in st['cuda']])


def _to_fp16(sd):
    return {k: v.half() if torch.is_tensor(v) and v.is_floating_point() else v for k, v in sd.items()}


def _to_fp32(sd):
    return {k: v.float() if torch.is_tensor(v) and v.is_floating_point() else v for k, v in sd.items()}


def save_ckpt(fold, payload):
    payload['contract'] = CONTRACT
    _atomic(CKPT / f'{CFG.run_tag}_fold{fold}.pt', lambda p: torch.save(payload, p))


def load_ckpt(fold):
    p = CKPT / f'{CFG.run_tag}_fold{fold}.pt'
    if CFG.resume and p.exists():
        d = torch.load(p, map_location='cpu', weights_only=False)
        if d.get('contract') != CONTRACT: raise ValueError('Checkpoint contract mismatch')
        return d
    return None


def drop_ckpt(fold):
    if not CFG.keep_fold_ckpt:
        (CKPT / f'{CFG.run_tag}_fold{fold}.pt').unlink(missing_ok=True)


def log_epoch(row):
    row['run'] = BRANCH
    old = pd.read_csv(LOG_CSV) if LOG_CSV.exists() else pd.DataFrame()
    df = pd.concat([old, pd.DataFrame([row])], ignore_index=True)
    df = df.drop_duplicates(['run', 'fold', 'epoch'], keep='last')
    _atomic(LOG_CSV, lambda p: df.to_csv(p, index=False))
    record_status('training', fold=int(row['fold']), epoch=int(row['epoch']), metric=float(row['val_metric']))


def keep_snapshots(fold, snaps):
                                                                             
    payload = dict(contract=CONTRACT, cfg=current_cfg(), feature_columns=list(FEAT_COLS),
                   snapshots=[(float(s), _to_fp32(st)) for s, st in snaps])
    _atomic(BRANCH_DIR / 'models' / f'fold{fold}_snapshots.pt', lambda p: torch.save(payload, p))


def mirror_feature_cache():
    dest = BRANCH_DIR / 'feature_cache'; dest.mkdir(exist_ok=True)
    for p in CFG.cache.glob('*.parquet'):
        q = dest / p.name
        _atomic(q, lambda tmp, p=p: shutil.copyfile(p, tmp))


def guard_data():
    for df in (train_df, test_df):
        if df.item_id.isna().any() or not df.item_id.is_unique: raise ValueError('Duplicate/missing item_id')
    y = train_df[CFG.target_cols].to_numpy()
    if not np.isin(y, [0, 1]).all(): raise ValueError('Labels must be binary')
    for df, directory in [(train_df, TRAIN_DIR), (test_df, TEST_DIR)]:
        missing = [f'{i}{ext}' for i in df.item_id for ext in ('.png', '.npz')
                   if not (Path(directory) / f'{i}{ext}').is_file()]
        if missing: raise FileNotFoundError(f'Missing inputs: {missing[:10]} ({len(missing)} total)')
    data = dict(train_ids=train_df.item_id.tolist(), test_ids=test_df.item_id.tolist(), targets=CFG.target_cols,
                labels=y.tolist())
    data_hash = digest_json(data)
    p = BRANCH_DIR / 'data_contract.json'
    if p.exists() and json.loads(p.read_text())['hash'] != data_hash: raise ValueError('Dataset changed: use a new project directory')
    _atomic(p, lambda p: p.write_text(json.dumps(dict(hash=data_hash, **data)), encoding='utf-8'))


def prepare_training():
    if CFG.tile != EXPECTED_TILE: raise ValueError('Source changed tile unexpectedly; refusing to train')
    if BRANCH == 'vitb336':
        globals()['feat_tr'], globals()['feat_te'] = assemble_features(IMG_TRAIN, IMG_TEST, CFG.tile)
        check_view_layout()
    for frame in (feat_tr, feat_te):
        if not frame.item_id.is_unique: raise ValueError('Duplicate feature rows')
                                                                           
    for df, imgs, feats, labels in [(train_df, IMG_TRAIN, feat_tr, True), (test_df, IMG_TEST, feat_te, False)]:
        ds = MeshViewDataset(df, imgs, feats, train=False, labels=labels)
        if not np.isfinite(ds.feats).all(): raise ValueError('Nonfinite branch input features; no silent replacement')
    meta = dict(contract=CONTRACT, cfg=current_cfg(), feature_columns=list(FEAT_COLS),
                feature_hash=digest_json(list(FEAT_COLS)), fold_ids=train_df.fold.tolist())
    p = BRANCH_DIR / 'feature_schema.json'
    if p.exists() and json.loads(p.read_text()) != json.loads(json.dumps(meta)):
        raise ValueError('Feature/fold schema changed: use a new project directory')
    _atomic(p, lambda p: p.write_text(json.dumps(meta, ensure_ascii=False, indent=2), encoding='utf-8'))
    train_df[['item_id', 'fold'] + CFG.target_cols].to_csv(BRANCH_DIR / 'train_manifest.csv', index=False)
    test_df[['item_id']].to_csv(BRANCH_DIR / 'test_manifest.csv', index=False)
    mirror_feature_cache()


def train_and_export():
    oof = pd.DataFrame(np.nan, index=train_df.item_id, columns=CFG.target_cols)
    tests = []
    for fold in CFG.folds_to_run:
        record_status('training', fold=fold)
        nm = rt(f'fold{fold}_preds')
        if has_artifact(nm) and load_artifact(nm).get('contract') != CONTRACT:
            raise ValueError('Prediction contract mismatch')
        (oi, op), (ti, tp) = train_fold(fold)
        oi, ti = list(map(str, oi)), list(map(str, ti))
        if len(set(oi)) != len(oi) or set(oi) != set(train_df.loc[train_df.fold == fold, 'item_id']):
            raise ValueError('OOF IDs do not match held-out fold')
        if len(set(ti)) != len(ti) or set(ti) != set(test_df.item_id): raise ValueError('Test IDs mismatch')
        for a in (op, tp):
            if not np.isfinite(a).all() or np.any((a < 0) | (a > 1)): raise ValueError('Invalid predictions')
        oof.loc[oi] = op
        tests.append(pd.DataFrame(tp, index=ti, columns=CFG.target_cols).loc[test_df.item_id].to_numpy())
    _atomic(BRANCH_DIR / 'oof.csv', lambda p: oof.rename_axis('item_id').reset_index().to_csv(p, index=False))
    te = pd.DataFrame(np.mean(tests, axis=0), columns=CFG.target_cols)
    te.insert(0, 'item_id', test_df.item_id.to_numpy())
    _atomic(BRANCH_DIR / 'test_probabilities.csv', lambda p: te.to_csv(p, index=False))
    coverage = int(np.isfinite(oof.to_numpy()).all(1).sum())
    record_status('complete', folds=len(tests), fold_ids=list(map(int, CFG.folds_to_run)),
                  coverage=coverage, features=len(FEAT_COLS), train=len(oof), test=len(te))


def infer_from_snapshots():
    """Regenerate fold probabilities from the exact saved snapshot ensembles."""
    from torch.utils.data import DataLoader
    for fold in CFG.folds_to_run:
        path = BRANCH_DIR / 'models' / f'fold{fold}_snapshots.pt'
        if not path.exists(): raise FileNotFoundError(f'Missing model artifact: {path}')
        saved = torch.load(path, map_location='cpu', weights_only=False)
        if saved.get('contract') != CONTRACT: raise ValueError(f'fold {fold}: model contract mismatch')
        if saved.get('feature_columns') != list(FEAT_COLS): raise ValueError(f'fold {fold}: feature order mismatch')
        va = train_df[train_df.fold == fold].reset_index(drop=True)
        lva = DataLoader(MeshViewDataset(va, IMG_TRAIN, feat_tr, train=False),
                         batch_size=CFG.batch_size*2, shuffle=False,
                         num_workers=CFG.num_workers, pin_memory=True)
        lte = DataLoader(MeshViewDataset(test_df.assign(**{c:0 for c in CFG.target_cols}),
                         IMG_TEST, feat_te, train=False, labels=False),
                         batch_size=CFG.batch_size*2, shuffle=False,
                         num_workers=CFG.num_workers, pin_memory=True)
        model = MultiViewNet(len(FEAT_COLS)).to(CFG.device).eval()
        oa=ta=0.0
        for _, state in saved['snapshots']:
            model.load_state_dict(_to_fp32(state))
            if BRANCH == 'vitb336':
                op,oi=predict(model,lva,desc=f'OOF fold{fold}')
                tp,ti=predict(model,lte,desc=f'test fold{fold}')
            else:
                op,oi=predict(model,lva,tta=CFG.tta)
                tp,ti=predict(model,lte,tta=CFG.tta)
            oa=oa+op;ta=ta+tp
        op,tp=oa/len(saved['snapshots']),ta/len(saved['snapshots'])
        save_artifact(rt(f'fold{fold}_preds'),dict(oof=op,oof_ids=list(oi),te=tp,te_ids=list(ti),
                      best=float(max(x[0] for x in saved['snapshots'])),
                      ens=float('nan'),snap_scores=[float(x[0]) for x in saved['snapshots']]))
        del model,lva,lte;gc.collect();torch.cuda.empty_cache()


In [ ]:
%%writefile /content/meshqc_code/worker.py
"""Run each branch in a fresh Python process (Linux/Colab CUDA)."""
import os, sys, json, hashlib, argparse, traceback, shutil, subprocess
from pathlib import Path

if __name__ == '__main__':
    parser=argparse.ArgumentParser()
    parser.add_argument('--branch',required=True,choices=['vitb336','v5_224','dino448_s2'])
    parser.add_argument('--root',required=True)
    parser.add_argument('--workers',type=int,default=4)
    parser.add_argument('--folds',default='0,1,2,3,4')
    parser.add_argument('--inference-only',action='store_true')
    parser.add_argument('--validate-only',action='store_true')
    args=parser.parse_args()
    BUNDLE=Path(__file__).resolve().parent
    stages=json.loads((BUNDLE/f'{args.branch}_stages.json').read_text(encoding='utf-8'))
    for stage in stages: compile(stage['source'],str(BUNDLE/f"cell_{stage['source_cell']}"),'exec')
    if args.validate_only:
        print(args.branch, 'syntax OK:',len(stages),'stages'); sys.exit(0)
    if not sys.platform.startswith('linux'): raise RuntimeError('Training requires Linux/Colab')
    import multiprocessing as mp
    mp.set_start_method('fork',force=True)
    import torch
    if not torch.cuda.is_available(): raise RuntimeError('Enable a GPU runtime in Colab')
    import matplotlib
    matplotlib.use('Agg')
    import matplotlib.pyplot as plt
    from IPython.display import display
    BRANCH=args.branch
    PROJECT=Path(args.root)
    BRANCH_DIR=PROJECT/'branches'/BRANCH
    EXPECTED_TILE={'vitb336':336,'v5_224':224,'dino448_s2':448}[BRANCH]
                                                                                         
    import importlib.metadata
    versions={x:importlib.metadata.version(x) for x in
              ['torch','numpy','pandas','scipy','Pillow','timm','open_clip_torch','scikit-learn','iterative-stratification']}
    selected_folds=[int(x) for x in args.folds.split(',') if x.strip()]
    if not selected_folds or any(x not in range(5) for x in selected_folds):
        raise ValueError('--folds must be a comma-separated subset of 0,1,2,3,4')
    identity=dict(branch=BRANCH,workers=args.workers,folds=selected_folds,versions=versions,
                  code={p.name:hashlib.sha256(p.read_bytes()).hexdigest() for p in
                        [Path(__file__),BUNDLE/'runtime.py',BUNDLE/f'{BRANCH}_stages.json']})
    CONTRACT=hashlib.sha256(json.dumps(identity,sort_keys=True).encode()).hexdigest()
    LOCAL_DIR=Path('/content/meshqc_local')/CONTRACT[:16]/BRANCH
    for p in (LOCAL_DIR,BRANCH_DIR): p.mkdir(parents=True,exist_ok=True)
    cp=BRANCH_DIR/'contract.json'
    if cp.exists():
        if json.loads(cp.read_text())['hash']!=CONTRACT:
            raise RuntimeError('Code/config/dependencies changed. Use a new PROJECT_DIR; no old cache was loaded.')
    else:
        cp.write_text(json.dumps(dict(hash=CONTRACT,identity=identity),indent=2),encoding='utf-8')
    exec(compile(stages[0]['source'],'config','exec'),globals())
    CFG.drive_dir=str(BRANCH_DIR); CFG.work=BRANCH_DIR; CFG.cache=LOCAL_DIR/'cache'
    CFG.use_drive=False; CFG.resume=True; CFG.folds_to_run=selected_folds; CFG.n_folds=5
    CFG.num_workers=args.workers; CFG.force_recompute=[]
    CFG.reuse_runs={}; CFG.ensemble_tags=[]; CFG.use_dino_frozen=False
    ART=BRANCH_DIR/'artifacts'; CKPT=BRANCH_DIR/'ckpt'; LOG_CSV=BRANCH_DIR/'training_log.csv'
    for p in (CFG.cache,ART,CKPT,BRANCH_DIR/'models'): p.mkdir(parents=True,exist_ok=True)
    exec(compile((BUNDLE/'runtime.py').read_text(encoding='utf-8'),'runtime','exec'),globals())
                                                                                     
    import random, time
    def seed_everything(seed=42):
        random.seed(seed); np.random.seed(seed); os.environ['PYTHONHASHSEED']=str(seed)
        torch.manual_seed(seed); torch.cuda.manual_seed_all(seed)
        torch.backends.cudnn.deterministic=True; torch.backends.cudnn.benchmark=False
        torch.use_deterministic_algorithms(True, warn_only=True)
    seed_everything(CFG.seed)
    IS_VIT=True; RUN_T0=time.time()
                                                                    
    if BRANCH=='vitb336': R=Reporter
    for p in (BRANCH_DIR/'feature_cache').glob('*.parquet'):
        dst=CFG.cache/p.name
        if not dst.exists(): shutil.copyfile(p,dst)
    fig_counter=[0]
    def save_figures(*a,**kw):
        figdir=BRANCH_DIR/'figures';figdir.mkdir(exist_ok=True)
        for num in plt.get_fignums():
            fig_counter[0]+=1
            plt.figure(num).savefig(figdir/f'plot_{fig_counter[0]:04}.png',dpi=110,bbox_inches='tight')
        plt.close('all')
    plt.show=save_figures
    try:
        for j,stage in enumerate(stages[1:],1):
            record_status('preparing',source_cell=stage['source_cell'],stage=j,total=len(stages)-1)
            print(f"\n[{BRANCH}] source cell {stage['source_cell']} ({j}/{len(stages)-1})",flush=True)
            exec(compile(stage['source'],f"{BRANCH}/cell_{stage['source_cell']}",'exec'),globals())
            if j==1: guard_data()
            mirror_feature_cache()
        prepare_training()
        if args.inference_only:
            infer_from_snapshots()
        train_and_export()
    except BaseException as exc:
        record_status('failed',error=f'{type(exc).__name__}: {exc}')
        traceback.print_exc(); raise


In [ ]:
%%writefile /content/meshqc_code/vitb336_source.py
                    
import os, gc, json, math, random, re, time, copy, warnings, shutil, itertools
from pathlib import Path
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
from tqdm.auto import tqdm
import matplotlib.pyplot as plt
import matplotlib as mpl
from sklearn.metrics import f1_score

warnings.filterwarnings('ignore')
mpl.rcParams.update({
    'figure.facecolor': 'white', 'axes.facecolor': '#fbfbfd', 'axes.grid': True,
    'grid.alpha': .25, 'grid.linestyle': '--', 'axes.spines.top': False,
    'axes.spines.right': False, 'font.size': 10, 'axes.titlesize': 11,
    'axes.titleweight': 'bold'})
PAL = {'good': '#2e9e5b', 'bad': '#d1495b', 'warn': '#e8a33d', 'main': '#2d6cdf',
       'alt': '#8a4fbd', 'grey': '#9aa0a6', 'dark': '#1f2933'}


class Reporter:
    W = 86
    _t = {}
    @classmethod
    def section(cls, t, s=''):
        print('\n' + '=' * cls.W); print(f'  {t}')
        if s: print(f'  {s}')
        print('=' * cls.W); cls._t[t] = time.time()
    @classmethod
    def done(cls, t, extra=''):
        print(f'  [OK] {t} — {(time.time() - cls._t.get(t, time.time())) / 60:.1f} мин'
              + (f' | {extra}' if extra else ''))
    @staticmethod
    def kv(**kw):
        for k, v in kw.items(): print(f'    {k:<32s} {v}')
    @staticmethod
    def ok(m): print(f'  [+] {m}')
    @staticmethod
    def warn(m): print(f'  [!] {m}')
    @staticmethod
    def fail(m): print(f'  [x] {m}')
    @staticmethod
    def bar(v, lo=12, hi=17, w=36, label=''):
        f = 0. if hi <= lo else max(0., min(1., (v - lo) / (hi - lo)))
        n = int(round(f * w))
        print(f'    {label:<22s} |{"█" * n}{"·" * (w - n)}| {v:.3f}')


R = Reporter


class CFG:
    seed = 42
    dataset = 'daniilantonov5/3d-mesh-quality-control'

                                            
                                                                                  
                                                                
    tile = 336
    view_mode = 'multiview'
    grid_rows, grid_cols = 2, 3

    backbone    = 'vit_base_patch14_reg4_dinov2.lvd142m'
    backbone_lr = 2e-5
    layer_decay = 0.70
    batch_size  = 2
    accum       = 12                               
    run_tag     = 'vitb336'

    drop_rate = 0.2
                                                                                    
                                                         
    epochs = 12
    mixup = 0.4
    asl_weight = 0.65                                                       
    lr = 3e-4
    head_lr_mult = 10.0
    weight_decay = 0.05
    warmup_frac = 0.1
    label_smooth = 0.01
    amp = True
    num_workers = 4
    ema_decay = 0.999

                          
    use_mldecoder = True
    mld_tokens = 8
    mld_dim = 512
    mld_layers = 2

                                
    sym_aug = True
    azimuth_idx = [0, 1, 2, 3]
    pole_idx = [4, 5]
    view_dropout = 0.0                                               
    pixel_noise = 0.0                                                  

    use_clip = True
    clip_model = ('ViT-B-32', 'laion2b_s34b_b79k')
    clip_pca = 64

    n_folds = 5
    folds_to_run = [0, 1, 2, 3, 4]

    n_snapshots = 3

    use_drive = True
    drive_dir = '/content/drive/MyDrive/sber_meshqc_v9'
    resume = True
    ckpt_every = 1
    ckpt_fp16 = True
    keep_fold_ckpt = False
    force_recompute = []

                                                                           
                                                              
    reuse_runs = {
        'v5_224': '/content/drive/MyDrive/sber_meshqc_v5/artifacts/dinos224_geo2_v5_fold{k}_preds.joblib',
    }

    artifact_cols = ['abstract','artifacts','intersection','lowpoly','noisy',
                     'open','partial','scale','set','simple']
    target_cols = artifact_cols + ['quality']
    n_targets = 11
    device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

                     
import kagglehub

root = Path(kagglehub.dataset_download(CFG.dataset))
print('dataset root:', root)

def _pick_dir(mode):
    """Папка с максимальным числом .npz, в пути которой встречается train/test."""
    best, best_n = None, -1
    for p in root.rglob('*'):
        if not p.is_dir():
            continue
        if mode not in str(p).lower():
            continue
        n = sum(1 for _ in p.glob('*.npz'))
        if n > best_n:
            best, best_n = p, n
    if best is None or best_n == 0:
        raise FileNotFoundError(f'не найдена папка с .npz для {mode}')
    return best

def _pick_csv(mode):
    cands = list(root.rglob('*.csv'))
    def score(p):
        n = p.name.lower(); s = 0
        if mode == 'train':
            s += 10 * ('train' in n) - 20 * ('submission' in n) - 20 * ('test' in n)
        else:
            s += 10 * ('test' in n) - 5 * ('submission' in n) - 20 * ('train' in n)
        return s - 0.001 * len(n)
    return sorted(cands, key=score, reverse=True)[0]

TRAIN_DIR, TEST_DIR = _pick_dir('train'), _pick_dir('test')
TRAIN_CSV, TEST_CSV = _pick_csv('train'), _pick_csv('test')
print('train dir :', TRAIN_DIR, len(list(TRAIN_DIR.glob('*.npz'))), 'npz /',
      len(list(TRAIN_DIR.glob('*.png'))), 'png')
print('test  dir :', TEST_DIR,  len(list(TEST_DIR.glob('*.npz'))),  'npz /',
      len(list(TEST_DIR.glob('*.png'))),  'png')
print('train csv :', TRAIN_CSV)
print('test  csv :', TEST_CSV)

train_df = pd.read_csv(TRAIN_CSV)
test_df  = pd.read_csv(TEST_CSV)[['item_id']].copy()
train_df['item_id'] = train_df['item_id'].astype(str)
test_df['item_id']  = test_df['item_id'].astype(str)

if 'quality' not in train_df.columns:
    train_df['quality'] = (train_df[CFG.artifact_cols].sum(1) == 0).astype(int)

print(train_df.shape, test_df.shape)
train_df.head()

                     
from sklearn.metrics import f1_score

def competition_metric(y_true, y_pred):
    """10*F1(quality) + 10*F1_weighted(10 дефектов) — точная формула организаторов."""
    f1_q = f1_score(y_true[:, 10], y_pred[:, 10], zero_division=0)
    f1_a = f1_score(y_true[:, :10], y_pred[:, :10], average='weighted', zero_division=0)
    return 10 * f1_q + 10 * f1_a, f1_q, f1_a

                                                                                          
Y = train_df[CFG.target_cols].values
for err in [0.02, 0.05, 0.10]:
    rng = np.random.default_rng(0)
    fake = Y.copy()
    flip = rng.random(fake[:, :10].shape) < err
    fake[:, :10] = np.abs(fake[:, :10] - flip)
    fake[:, 10] = (fake[:, :10].sum(1) == 0).astype(int)
    s, fq, fa = competition_metric(Y, fake)
    print(f'FPR/FNR по дефектам {err:.0%} -> метрика {s:5.2f} (quality F1={fq:.3f}, artefacts F1={fa:.3f})')
print('\n>>> Ошибка 5% по дефектам уже срезает quality F1 до ~0.7. '
      'Поэтому quality предсказывается ОТДЕЛЬНОЙ головой, а не только правилом.')

                     
from PIL import Image
from concurrent.futures import ProcessPoolExecutor

Image.MAX_IMAGE_PIXELS = None

def infer_grid(w, h):
    """(rows, cols) для 6 видов по соотношению сторон."""
    ar = w / h
    cands = {(2, 3): 3 / 2, (3, 2): 2 / 3, (1, 6): 6.0, (6, 1): 1 / 6}
    return min(cands, key=lambda k: abs(math.log(ar / cands[k])))

def split_views(img):
    """PIL.Image -> список из 6 PIL.Image в каноническом порядке чтения."""
    w, h = img.size
    r, c = infer_grid(w, h)
    tw, th = w // c, h // r
    return [img.crop((j * tw, i * th, (j + 1) * tw, (i + 1) * th)) for i in range(r) for j in range(c)]

def make_cache(args):
    src, dst, tile = args
    try:
        img = Image.open(src).convert('RGB')
        views = [v.resize((tile, tile), Image.BILINEAR) for v in split_views(img)]
        out = Image.new('RGB', (3 * tile, 2 * tile))
        for k, v in enumerate(views):
            out.paste(v, ((k % 3) * tile, (k // 3) * tile))
        out.save(dst, format='PNG', optimize=False, compress_level=1)
        return 1
    except Exception:
        return 0

def build_image_cache(ids, src_dir, split):
    out_dir = CFG.cache / f'img_{split}_{CFG.tile}'
    out_dir.mkdir(parents=True, exist_ok=True)
    jobs = [(str(Path(src_dir) / f'{i}.png'), str(out_dir / f'{i}.png'), CFG.tile)
            for i in ids if not (out_dir / f'{i}.png').exists()
            and (Path(src_dir) / f'{i}.png').exists()]
    if jobs:
        with ProcessPoolExecutor(max_workers=os.cpu_count()) as ex:
            ok = list(tqdm(ex.map(make_cache, jobs, chunksize=32), total=len(jobs),
                           desc=f'cache {split}'))
        print(f'{split}: закэшировано {sum(ok)}/{len(jobs)}')
    missing = [i for i in ids if not (out_dir / f'{i}.png').exists()]
    print(f'{split}: всего в кэше {len(ids) - len(missing)}/{len(ids)}, нет картинки у {len(missing)}')
    return out_dir, set(missing)

                                                                                       
_probe = [Image.open(Path(TRAIN_DIR) / f'{i}.png').size
          for i in train_df['item_id'].head(40) if (Path(TRAIN_DIR) / f'{i}.png').exists()]
if _probe:
    _sizes = pd.Series([f'{w}x{h}' for w, h in _probe]).value_counts()
    print('исходные размеры листа с 6 видами:'); print(_sizes.head(5).to_string())
    _w, _h = _probe[0]
    _r, _c = infer_grid(_w, _h)
    NATIVE_TILE = int(min(_w // _c, _h // _r))
    R.kv(**{'раскладка': f'{_r}x{_c}', 'нативный размер вида': f'{NATIVE_TILE} px',
            'используем': f'{CFG.tile} ({100 * (CFG.tile / NATIVE_TILE) ** 2:.0f}% площади кадра)'})
    if CFG.tile > NATIVE_TILE:
        R.warn(f'tile {CFG.tile} > нативного — это апскейл, снижаю')
        CFG.tile = NATIVE_TILE

IMG_TRAIN, MISS_TRAIN = build_image_cache(train_df['item_id'].tolist(), TRAIN_DIR, 'train')
IMG_TEST,  MISS_TEST  = build_image_cache(test_df['item_id'].tolist(),  TEST_DIR,  'test')

                     
import os, gc, math, numpy as np, pandas as pd
from pathlib import Path
from PIL import Image
from scipy import ndimage
from scipy.sparse import coo_matrix
from scipy.sparse.csgraph import connected_components
from scipy.spatial import ConvexHull, QhullError
from concurrent.futures import ProcessPoolExecutor
from concurrent.futures.process import BrokenProcessPool
import multiprocessing as mp

FACE_CAP  = 300_000
VERT_CAP  = 200_000
HULL_CAP  = 20_000
COMP_CAP  = 3_000_000
WELD_TOL  = 1e-6                              
FEAT_VER  = 'v4selfint'                                                                
                                                                                          
                                                                                              
                                                                                
N_WORKERS = max(1, (os.cpu_count() or 4) - 1)
CHUNK     = 512                                                     

def _pack21(q):
    """Три целых из [0, 2**21) -> один int64. Точно, без коллизий."""
    return (q[:, 0] << 42) | (q[:, 1] << 21) | q[:, 2]

def _hash3(q):
    q = q.astype(np.int64, copy=False)
    return (q[:, 0] * 73856093) ^ (q[:, 1] * 19349663) ^ (q[:, 2] * 83492791)

def _stats(x, pref):
    keys = ['mean', 'std', 'p10', 'p50', 'p90', 'max', 'cv']
    if len(x) == 0:
        return {f'{pref}_{k}': 0.0 for k in keys}
    x = np.asarray(x, dtype=np.float32)
    m, s = float(x.mean()), float(x.std())
    q10, q50, q90 = np.percentile(x, [10, 50, 90])
    return {f'{pref}_mean': m, f'{pref}_std': s, f'{pref}_p10': float(q10),
            f'{pref}_p50': float(q50), f'{pref}_p90': float(q90),
            f'{pref}_max': float(x.max()), f'{pref}_cv': float(s / (abs(m) + 1e-9))}

def _edge_table(Fi, nfw):
    """Рёбра -> (номер группы, число граней на ребро, отсортированные id граней)."""
    E = np.concatenate([Fi[:, [0, 1]], Fi[:, [1, 2]], Fi[:, [2, 0]]], 0)
    E.sort(axis=1)
    fid = np.tile(np.arange(nfw, dtype=np.int32), 3)
    order = np.lexsort((E[:, 1], E[:, 0]))
    Es, fids = E[order], fid[order]
    new = np.ones(len(Es), dtype=bool)
    new[1:] = (Es[1:] != Es[:-1]).any(1)
    grp = np.cumsum(new) - 1
    counts = np.bincount(grp)
    return E, Es, fids, grp, counts, new

def mesh_features(npz_path):
    f = {'geo_ok': 0.0, 'geo_subsampled': 0.0}
    try:
        f['npz_mb'] = os.path.getsize(npz_path) / 1e6
    except Exception:
        return f
    try:
        with np.load(npz_path, allow_pickle=False) as d:
            keys = list(d.keys())
            V = np.asarray(d['vertices' if 'vertices' in keys else keys[0]], dtype=np.float32)
            Fc = None
            for k in ('faces', 'triangles', 'f'):
                if k in keys:
                    Fc = np.asarray(d[k], dtype=np.int64); break
            if Fc is None and len(keys) > 1:
                Fc = np.asarray(d[keys[1]], dtype=np.int64)
    except Exception:
        return f
    if V.ndim != 2 or V.shape[1] != 3 or len(V) == 0:
        return f
    if Fc is None or Fc.ndim != 2 or Fc.shape[1] != 3 or len(Fc) == 0:
        Fc = np.zeros((0, 3), dtype=np.int64)
    else:
        Fc = Fc[(Fc >= 0).all(1) & (Fc < len(V)).all(1)]

    f['geo_ok'] = 1.0
    nv_raw, nf_raw = len(V), len(Fc)
    vmin, vmax = V.min(0), V.max(0)
    ext = (vmax - vmin).astype(np.float64)
    ext_s = np.sort(ext)[::-1]
    diag = float(np.linalg.norm(ext)) + 1e-12
    center = ((vmin + vmax) / 2).astype(np.float32)
    inv_s = np.float32(1.0 / diag)
    f.update({'n_verts_raw': float(nv_raw), 'n_faces': float(nf_raw),
              'log_verts': math.log1p(nv_raw), 'log_faces': math.log1p(nf_raw),
              'ext_max': float(ext_s[0]), 'ext_mid': float(ext_s[1]), 'ext_min': float(ext_s[2]),
              'ext_ratio_min': float(ext_s[2] / (ext_s[0] + 1e-12)),
              'ext_ratio_mid': float(ext_s[1] / (ext_s[0] + 1e-12)),
              'bbox_diag': diag,
              'bbox_fill': float(nv_raw / (ext.prod() + 1e-12)) if ext.prod() > 0 else 0.0})

                                                         
                                                                                           
                                                                                         
                                                                                                
    step = np.float32(diag * WELD_TOL)
    q = np.clip(np.rint((V - vmin) / step), 0, (1 << 21) - 1).astype(np.int64)
    code = _pack21(q)
    del q
    ucode, first, invmap = np.unique(code, return_index=True, return_inverse=True)
    del code, ucode
    nv = len(first)
    f['n_verts'] = float(nv)
    f['weld_ratio'] = float(1.0 - nv / max(nv_raw, 1))                                                         
    f['f_per_v'] = nf_raw / max(nv, 1)
    Vw_full = V[first]
    del first

    if nf_raw:
        Fw = invmap[Fc]
        ok = (Fw[:, 0] != Fw[:, 1]) & (Fw[:, 1] != Fw[:, 2]) & (Fw[:, 0] != Fw[:, 2])
        f['degen_after_weld'] = float(1.0 - ok.mean())
        Fw = Fw[ok]
        hf = _hash3(np.sort(Fw, axis=1))
        _, uidx = np.unique(hf, return_index=True)
        f['dup_face_ratio'] = float(1.0 - len(uidx) / max(len(Fw), 1))
        Fw = Fw[np.sort(uidx)]
        del hf, uidx, ok
    else:
        Fw = np.zeros((0, 3), dtype=np.int64)
        f['degen_after_weld'] = 0.0; f['dup_face_ratio'] = 0.0
    del invmap, Fc
    nf = len(Fw)

                                                 
    Vs = Vw_full[::max(1, int(np.ceil(nv / VERT_CAP)))]
    if nv > VERT_CAP:
        f['geo_subsampled'] = 1.0
    Vs = (Vs - center) * inv_s
    try:
        if len(Vs) < 3:
            raise ValueError('too few vertices')
        ev = np.clip(np.linalg.eigvalsh(np.cov(Vs.T.astype(np.float64)))[::-1], 1e-16, None)
        f.update({'pca_1': float(ev[0]), 'pca_2': float(ev[1]), 'pca_3': float(ev[2]),
                  'pca_flat': float(ev[2] / ev[0]), 'pca_lin': float(ev[1] / ev[0]),
                  'pca_aniso': float((ev[0] - ev[2]) / ev.sum())})
    except Exception:
        f.update({k: 0.0 for k in ['pca_1', 'pca_2', 'pca_3', 'pca_flat', 'pca_lin', 'pca_aniso']})

                                                                       
    f.update({'hull_ok': 0.0, 'solidity': 0.0, 'solidity_signed': 0.0, 'hull_area_ratio': 0.0,
              'hull_pts_frac': 0.0, 'sphericity': 0.0, 'hull_vol': 0.0, 'hull_area': 0.0})
    try:
        Vh = Vs[::max(1, int(np.ceil(len(Vs) / HULL_CAP)))].astype(np.float64)
        if len(Vh) >= 8:
            hull = ConvexHull(Vh, qhull_options='QJ')
            f['hull_ok'] = 1.0
            f['hull_vol'] = float(hull.volume)
            f['hull_pts_frac'] = len(hull.vertices) / len(Vh)
            f['hull_area'] = float(hull.area)
    except (QhullError, ValueError, MemoryError):
        pass

    if nf == 0:
        f['no_faces'] = 1.0
        f.update({'area_total': 0.0, 'volume': 0.0, 'abs_volume': 0.0, 'vol_ratio': 0.0,
                  'vol_over_area': 0.0, 'absvol_over_area': 0.0, 'vol_over_bbox': 0.0,
                  'absvol_over_bbox': 0.0})
        for pref in ('dihed', 'dihedf', 'farea', 'elen', 'aspect', 'valence', 'defect'):
            f.update(_stats([], pref))
        return f
    f['no_faces'] = 0.0

                                                               
    total_area, vol6, absvol6 = 0.0, 0.0, 0.0
    for s0 in range(0, nf, 500_000):
        Pc = ((Vw_full[Fw[s0:s0 + 500_000]] - center) * inv_s).astype(np.float64)
        cr_c = np.cross(Pc[:, 1] - Pc[:, 0], Pc[:, 2] - Pc[:, 0])
        total_area += 0.5 * float(np.linalg.norm(cr_c, axis=1).sum())
        contrib = (Pc[:, 0] * np.cross(Pc[:, 1], Pc[:, 2])).sum(1)
        vol6 += float(contrib.sum()); absvol6 += float(np.abs(contrib).sum())
        del Pc, cr_c, contrib
    total_area += 1e-12
    vol = abs(vol6 / 6.0)
    absvol = absvol6 / 6.0
                                                                                            
                                                                                            
    f.update({'area_total': total_area, 'volume': vol, 'abs_volume': absvol,
              'vol_ratio': vol / (absvol + 1e-12),
              'vol_over_area': vol / (total_area ** 1.5 + 1e-12),
              'absvol_over_area': absvol / (total_area ** 1.5 + 1e-12),
              'vol_over_bbox': vol / (float(np.prod(ext * inv_s)) + 1e-12),
              'absvol_over_bbox': absvol / (float(np.prod(ext * inv_s)) + 1e-12)})
    if f['hull_ok']:
        f['solidity'] = absvol / (f['hull_vol'] + 1e-12)
        f['solidity_signed'] = vol / (f['hull_vol'] + 1e-12)
        f['hull_area_ratio'] = total_area / (f.get('hull_area', 0.0) + 1e-12)
        f['sphericity'] = (math.pi ** (1 / 3)) * (6 * absvol) ** (2 / 3) / (total_area + 1e-12)

                                                             
    f.update({'n_comp': 1.0, 'n_comp_big': 1.0, 'comp_top1': 1.0, 'comp_top2': 0.0,
              'comp_entropy': 0.0, 'bbox_iou_max': 0.0, 'bbox_iou_mean': 0.0,
              'bbox_iou_frac_pos': 0.0, 'comp_sep_max': 0.0, 'comp_sep_mean': 0.0,
              'inside_frac_max': 0.0, 'comp_exact': 1.0})
    if nf <= COMP_CAP:
        try:
            Ef = np.concatenate([Fw[:, [0, 1]], Fw[:, [1, 2]], Fw[:, [2, 0]]], 0).astype(np.int32)
            A = coo_matrix((np.ones(len(Ef), dtype=np.int8), (Ef[:, 0], Ef[:, 1])), shape=(nv, nv))
            del Ef
            ncomp, lab = connected_components(A, directed=False)
            del A
            used = np.unique(Fw)
            sizes_all = np.bincount(lab[used], minlength=ncomp).astype(np.float64)
            sizes = np.sort(sizes_all[sizes_all > 0])[::-1]
            tot = sizes.sum()
            big_idx = np.where(sizes_all > 0.005 * tot)[0]
            f.update({'n_comp': float(len(sizes)), 'n_comp_big': float(len(big_idx)),
                      'comp_top1': float(sizes[0] / tot),
                      'comp_top2': float(sizes[1] / tot) if len(sizes) > 1 else 0.0,
                      'comp_entropy': float(-((sizes / tot) * np.log(sizes / tot + 1e-12)).sum())})
            if len(big_idx) > 1:
                top = big_idx[np.argsort(-sizes_all[big_idx])][:8]
                pts_l, boxes = [], []
                for c in top:
                    ic = used[lab[used] == c]
                    if len(ic) > 20_000:
                        ic = ic[::len(ic) // 20_000 + 1]
                    p = (Vw_full[ic] - center) * inv_s
                    pts_l.append(p); boxes.append((p.min(0), p.max(0)))
                ious, seps, insides = [], [], []
                for i in range(len(boxes)):
                    for j in range(len(boxes)):
                        if i == j:
                            continue
                        lo, hi = boxes[j]
                        insides.append(float(((pts_l[i] >= lo) & (pts_l[i] <= hi)).all(1).mean()))
                        if j <= i:
                            continue
                        lo2 = np.maximum(boxes[i][0], boxes[j][0])
                        hi2 = np.minimum(boxes[i][1], boxes[j][1])
                        it = float(np.prod(np.clip(hi2 - lo2, 0, None)))
                        vi = float(np.prod(np.clip(boxes[i][1] - boxes[i][0], 1e-6, None)))
                        vj = float(np.prod(np.clip(boxes[j][1] - boxes[j][0], 1e-6, None)))
                        ious.append(it / (vi + vj - it + 1e-12))
                        seps.append(float(np.linalg.norm(
                            (boxes[i][0] + boxes[i][1]) / 2 - (boxes[j][0] + boxes[j][1]) / 2)))
                f.update({'bbox_iou_max': float(max(ious)), 'bbox_iou_mean': float(np.mean(ious)),
                          'bbox_iou_frac_pos': float(np.mean(np.asarray(ious) > 1e-6)),
                          'comp_sep_max': float(max(seps)), 'comp_sep_mean': float(np.mean(seps)),
                          'inside_frac_max': float(max(insides))})
            del lab, sizes_all, sizes, used
        except Exception:
            f['comp_exact'] = 0.0
    else:
        f['comp_exact'] = 0.0

                                                            
    if nf > FACE_CAP:
        f['geo_subsampled'] = 1.0
        anchor = (Vw_full[Fw[:, 0]] - vmin) / (ext.astype(np.float32) + 1e-9)
        Fp, g = None, 1
        while g < 40:
            g += 1
            cid = np.minimum((anchor * g).astype(np.int32), g - 1)
            key = (cid[:, 0] * g + cid[:, 1]) * g + cid[:, 2]
            cnt = np.bincount(key, minlength=g ** 3)
            best = int(np.argmax(cnt))
            if cnt[best] <= FACE_CAP:
                Fp = Fw[key == best]; break
        if Fp is None or len(Fp) < 1000:
            Fp = Fw[:FACE_CAP]
        f['patch_grid'] = float(g)
        del anchor
    else:
        Fp = Fw
        f['patch_grid'] = 1.0
    f['patch_face_frac'] = len(Fp) / max(nf, 1)
    del Fw
    nfw = len(Fp)

    uvp, Fi = np.unique(Fp, return_inverse=True)
    Fi = Fi.reshape(nfw, 3).astype(np.int32)
    Vp = ((Vw_full[uvp] - center) * inv_s).astype(np.float32)
    nvp = len(uvp)
    del uvp, Fp, Vw_full, V

    P = Vp[Fi]
    cr = np.cross(P[:, 1] - P[:, 0], P[:, 2] - P[:, 0])
    area2 = np.linalg.norm(cr, axis=1)
    area = 0.5 * area2
    nrm = cr / (area2[:, None] + 1e-12)
    del cr
    f.update(_stats(area / (area.mean() + 1e-12), 'farea'))
    f['degenerate_ratio'] = float((area < 1e-10).mean())

    L = np.stack([np.linalg.norm(P[:, 1] - P[:, 0], axis=1),
                  np.linalg.norm(P[:, 2] - P[:, 1], axis=1),
                  np.linalg.norm(P[:, 0] - P[:, 2], axis=1)], 1)
    f.update(_stats(L.ravel() / (L.mean() + 1e-12), 'elen'))
    aspect = L.max(1) / (L.min(1) + 1e-12)
    f.update(_stats(np.log1p(aspect), 'aspect'))
    f['sliver_ratio'] = float((aspect > 20).mean())

                                                                                  
                                                                                      
                                                                                     
                                                                                          
                                                                                   
                                      
    L0, L1, L2 = L[:, 0], L[:, 1], L[:, 2]
    with np.errstate(divide='ignore', invalid='ignore'):
        cA = np.clip((L1 ** 2 + L2 ** 2 - L0 ** 2) / (2 * L1 * L2 + 1e-12), -1, 1)
        cB = np.clip((L0 ** 2 + L2 ** 2 - L1 ** 2) / (2 * L0 * L2 + 1e-12), -1, 1)
        cC = np.clip((L0 ** 2 + L1 ** 2 - L2 ** 2) / (2 * L0 * L1 + 1e-12), -1, 1)
    tri_ang = np.degrees(np.arccos(np.stack([cA, cB, cC], 1)))
    min_ang, max_ang = tri_ang.min(1), tri_ang.max(1)
    f.update(_stats(min_ang, 'minang')); f.update(_stats(max_ang, 'maxang'))
    f['sliver_angle_ratio'] = float((min_ang < 5).mean())
    tri_quality = np.clip((4 * math.sqrt(3) * area) / (L0 ** 2 + L1 ** 2 + L2 ** 2 + 1e-12), 0, 1)
    f.update(_stats(tri_quality, 'triq'))
    del L0, L1, L2, cA, cB, cC, tri_ang, min_ang, max_ang, tri_quality

    for qd in (1, 2):
        f[f'uniq_normals_q{qd}'] = len(np.unique(_hash3(np.rint(nrm * 10 ** qd)))) / nfw
                                                                               
    hn = _hash3(np.rint(nrm * 20))
    _, ninv = np.unique(hn, return_inverse=True)
    mass = np.bincount(ninv, weights=area)
    mass = np.sort(mass)[::-1]
    f['normal_top6_mass'] = float(mass[:6].sum() / (mass.sum() + 1e-12))
    f['normal_top1_mass'] = float(mass[0] / (mass.sum() + 1e-12))
    del hn, ninv, mass

                                                                                    
                                                                                 
                                                                                  
                                                                                   
                                                                                       
                                                                          
                                                                                          
                                                                                       
    f.update({'selfint_ok': 0.0, 'selfint_pair_ratio': 0.0, 'selfint_face_ratio': 0.0,
              'selfint_any': 0.0, 'selfint_capped': 0.0})
                                                                            
                                                                                    
    SELFINT_FACE_CAP = 40_000
    SELFINT_PAIR_CAP = 400_000
    if 0 < nfw <= SELFINT_FACE_CAP:
        try:
            from scipy.spatial import cKDTree
            cen = Vp[Fi].mean(1)
            elen_med = float(np.median(L)) if len(L) else 1e-6
            r = max(elen_med * 2.5, 1e-6)
            pairs = cKDTree(cen).query_pairs(r=r, output_type='ndarray')
            if len(pairs):
                share = (Fi[pairs[:, 0], :, None] == Fi[pairs[:, 1], None, :]).any((1, 2))
                pairs = pairs[~share]
            if len(pairs) > SELFINT_PAIR_CAP:
                sel = np.random.default_rng(0).choice(len(pairs), SELFINT_PAIR_CAP, replace=False)
                pairs = pairs[sel]
                f['selfint_capped'] = 1.0
            f['selfint_ok'] = 1.0
            if len(pairs):
                ii, jj = pairs[:, 0], pairs[:, 1]
                PA, PB = Vp[Fi[ii]], Vp[Fi[jj]]
                nA, nB = nrm[ii], nrm[jj]

                def _edge_hits(a, b, p0, p1, p2, n):
                    d = np.einsum('ij,ij->i', n, p0)
                    ta = np.einsum('ij,ij->i', n, a) - d
                    tb = np.einsum('ij,ij->i', n, b) - d
                    denom = ta - tb
                    safe = np.abs(denom) > 1e-12
                    t = np.clip(np.divide(ta, denom, out=np.zeros_like(ta), where=safe), 0.0, 1.0)
                    pt = a + t[:, None] * (b - a)
                    e0 = np.cross(p1 - p0, pt - p0); e1 = np.cross(p2 - p1, pt - p1)
                    e2 = np.cross(p0 - p2, pt - p2)
                    s0 = np.einsum('ij,ij->i', e0, n); s1 = np.einsum('ij,ij->i', e1, n)
                    s2 = np.einsum('ij,ij->i', e2, n)
                    inside = (((s0 >= -1e-9) & (s1 >= -1e-9) & (s2 >= -1e-9)) |
                              ((s0 <= 1e-9) & (s1 <= 1e-9) & (s2 <= 1e-9)))
                    return (ta * tb <= 0) & safe & inside

                hit = np.zeros(len(pairs), dtype=bool)
                for e in range(3):
                    hit |= _edge_hits(PA[:, e], PA[:, (e + 1) % 3], PB[:, 0], PB[:, 1], PB[:, 2], nB)
                for e in range(3):
                    hit |= _edge_hits(PB[:, e], PB[:, (e + 1) % 3], PA[:, 0], PA[:, 1], PA[:, 2], nA)
                n_hit = int(hit.sum())
                f['selfint_pair_ratio'] = float(n_hit / max(nfw, 1))
                f['selfint_any'] = float(n_hit > 0)
                if n_hit:
                    f['selfint_face_ratio'] = float(
                        len(np.unique(np.concatenate([ii[hit], jj[hit]]))) / max(nfw, 1))
                del PA, PB, nA, nB, hit
            del cen, pairs
        except Exception:
            pass
    else:
        f['selfint_capped'] = 1.0

                                                                              
    E, Es, fids, grp, counts, _ = _edge_table(Fi, nfw)
    n_edges = len(counts)
    f.update({'n_edges': float(n_edges),
              'boundary_edge_ratio': float((counts == 1).mean()),
              'nonmanifold_edge_ratio': float((counts >= 3).mean()),
              'euler_norm': float((nvp - n_edges + nfw) / max(nfw, 1)),
              'euler_char': float(nvp - n_edges + nfw),
              'watertight': float((counts == 1).sum() == 0 and (counts >= 3).sum() == 0),
              'manifold_edge_ratio': float((counts == 2).mean())})

                                                                  
    bnd = np.where(counts == 1)[0]
    f['nonmanifold_vert_ratio'] = 0.0
    if len(bnd):
        st = np.searchsorted(grp, bnd)
        be = Es[st]
        try:
            Ab = coo_matrix((np.ones(len(be), dtype=np.int8), (be[:, 0], be[:, 1])), shape=(nvp, nvp))
            nb_comp, lb = connected_components(Ab, directed=False)
            used_b = np.unique(be)
            f['n_boundary_loops'] = float(len(np.unique(lb[used_b])))
            f['boundary_vert_frac'] = float(len(used_b) / nvp)
            del Ab, lb, used_b
        except Exception:
            f['n_boundary_loops'] = 0.0; f['boundary_vert_frac'] = 0.0
                                                                                   
                                                                               
                                                                                 
                                                                                
                                                                                 
                                                                                   
        deg_b = np.bincount(be.ravel(), minlength=nvp)
        f['nonmanifold_vert_ratio'] = float((deg_b > 2).sum() / max(nvp, 1))
        del be, st, deg_b
    else:
        f['n_boundary_loops'] = 0.0; f['boundary_vert_frac'] = 0.0

    two = np.where(counts == 2)[0]
    if len(two):
        st = np.searchsorted(grp, two)
        fa, fb = fids[st], fids[st + 1]
        cosang = np.clip((nrm[fa] * nrm[fb]).sum(1), -1, 1)
        ang = np.degrees(np.arccos(cosang))
        angf = np.degrees(np.arccos(np.abs(cosang)))
        f.update(_stats(ang, 'dihed')); f.update(_stats(angf, 'dihedf'))
        for t in (10, 30, 60, 90):
            f[f'dihed_gt{t}'] = float((ang > t).mean())
        for t in (5, 15, 30, 60):
            f[f'dihedf_gt{t}'] = float((angf > t).mean())
        f['flip_ratio'] = float((cosang < 0).mean())
        hist = np.bincount((angf / 5).astype(np.int32).clip(0, 17), minlength=18).astype(np.float64)
        p = hist / max(hist.sum(), 1)
        f['dihed_entropy'] = float(-(p[p > 0] * np.log(p[p > 0])).sum())
        f['smoothness'] = float(angf.mean() * math.sqrt(nfw) / 100.0)
                                                                           
        el = np.linalg.norm(Vp[Es[st][:, 0]] - Vp[Es[st][:, 1]], axis=1)
        f['dihedf_areaw'] = float((angf * el).sum() / (el.sum() + 1e-12))
        del fa, fb, cosang, ang, angf, el
    else:
        f.update(_stats([], 'dihed')); f.update(_stats([], 'dihedf'))
        for t in (10, 30, 60, 90):
            f[f'dihed_gt{t}'] = 0.0
        for t in (5, 15, 30, 60):
            f[f'dihedf_gt{t}'] = 0.0
        f.update({'flip_ratio': 0.0, 'dihed_entropy': 0.0, 'smoothness': 0.0, 'dihedf_areaw': 0.0})

                                                                       
    val = np.bincount(Fi.ravel(), minlength=nvp).astype(np.float32)
    f.update(_stats(val, 'valence'))
    f['valence_le3'] = float((val <= 3).mean())
    f['valence_ge8'] = float((val >= 8).mean())
    e01 = P[:, 1] - P[:, 0]; e12 = P[:, 2] - P[:, 1]; e20 = P[:, 0] - P[:, 2]
    def _ang(u, v):
        cu = (u * v).sum(1) / (np.linalg.norm(u, axis=1) * np.linalg.norm(v, axis=1) + 1e-12)
        return np.arccos(np.clip(cu, -1, 1))
    a0 = _ang(e01, -e20); a1 = _ang(e12, -e01); a2 = _ang(e20, -e12)
    defect = np.full(nvp, 2 * np.pi, dtype=np.float64)
    np.subtract.at(defect, Fi[:, 0], a0)
    np.subtract.at(defect, Fi[:, 1], a1)
    np.subtract.at(defect, Fi[:, 2], a2)
    interior = val > 0
    f.update(_stats(np.abs(defect[interior]), 'defect'))
    f['defect_sum'] = float(defect[interior].sum() / (2 * np.pi))                                
    f['defect_gt05'] = float((np.abs(defect[interior]) > 0.5).mean())
    del defect, val, a0, a1, a2, e01, e12, e20

    del E, Es, fids, grp, counts, two, nrm, P, Vp, Fi, L, aspect, area, area2
    return f

                     
def image_features(png_path, tile):
    """Силуэтные признаки кадра. Зависят от tile (мозаика режется по нему),
    поэтому кэшируются отдельно от геометрии."""
    f = {'img_ok': 0.0}
    try:
        img = Image.open(png_path).convert('L')
    except Exception:
        return f
    a = np.asarray(img, dtype=np.float32) / 255.0
    t = tile
    f['img_ok'] = 1.0
    covs, comps, edens, stds, bws, bhs = [], [], [], [], [], []
    for k in range(6):
        v = a[(k // 3) * t:(k // 3 + 1) * t, (k % 3) * t:(k % 3 + 1) * t]
        bg = np.median(np.concatenate([v[0], v[-1], v[:, 0], v[:, -1]]))
        mask = np.abs(v - bg) > 0.06
        cov = float(mask.mean()); covs.append(cov)
        if mask.any():
            ys, xs = np.where(mask)
            bws.append((xs.max() - xs.min() + 1) / t)
            bhs.append((ys.max() - ys.min() + 1) / t)
            lab, ncc = ndimage.label(mask)
            if ncc:
                sz = np.bincount(lab.ravel())[1:]
                comps.append(float((sz > 0.01 * sz.sum()).sum()))
            else:
                comps.append(0.0)
        else:
            bws.append(0.0); bhs.append(0.0); comps.append(0.0)
        edens.append(float(np.abs(np.diff(v, axis=1)).mean() + np.abs(np.diff(v, axis=0)).mean()))
        stds.append(float(v.std()))

    def agg(vals, pref):
        v = np.asarray(vals, dtype=np.float64)
        return {f'{pref}_mean': v.mean(), f'{pref}_min': v.min(), f'{pref}_max': v.max(),
                f'{pref}_std': v.std(), f'{pref}_range': v.max() - v.min()}

    f.update(agg(covs, 'cov')); f.update(agg(comps, 'ncc')); f.update(agg(edens, 'edge'))
    f.update(agg(stds, 'pxstd')); f.update(agg(bws, 'bw')); f.update(agg(bhs, 'bh'))
    f['cov_empty_views'] = float((np.asarray(covs) < 0.01).sum())
    f['bbox_fill_view'] = float(np.mean(np.asarray(covs) /
                                        (np.asarray(bws) * np.asarray(bhs) + 1e-6)))
    return f


                                                                                       
                   
 
                                                                                   
                                                                                  
                                                                                      
                                                                                       
                                                                                      
                                                
                                                                                       

def _mesh_one(args):
    iid, npz = args
    d = {'item_id': iid}
    try:
        d.update(mesh_features(npz))
    except Exception:
        d['geo_ok'] = 0.0
    return d


def _img_one(args):
    iid, png, tile = args
    d = {'item_id': iid}
    try:
        d.update(image_features(png, tile))
    except Exception:
        d['img_ok'] = 0.0
    return d


def _chunked_build(jobs, worker, final, ckpt, desc):
    """Общий каркас: чанками, с чекпоинтом, с резюмированием после обрыва."""
    if final.exists():
        df = pd.read_parquet(final)
        R.ok(f'{desc}: из кэша {df.shape}')
        return df
    done = pd.read_parquet(ckpt) if ckpt.exists() else pd.DataFrame(columns=['item_id'])
    have = set(done['item_id'].astype(str)) if len(done) else set()
    todo = [j for j in jobs if j[0] not in have]
    R.kv(**{desc: f'готово {len(have)}, осталось {len(todo)}, воркеров {N_WORKERS}'})
    rows = [done] if len(done) else []
    for s in range(0, len(todo), CHUNK):
        part = todo[s:s + CHUNK]
        try:
            ctx = mp.get_context('fork')
        except ValueError:
            ctx = None
        try:
            with ProcessPoolExecutor(max_workers=N_WORKERS, mp_context=ctx) as ex:
                got = list(tqdm(ex.map(worker, part, chunksize=4), total=len(part),
                                desc=f'{desc} {s + len(part)}/{len(todo)}', leave=False))
        except (BrokenProcessPool, OSError, MemoryError) as e:
            R.warn(f'пул упал ({type(e).__name__}), чанк считается последовательно')
            got = [worker(j) for j in tqdm(part, desc='serial', leave=False)]
        rows.append(pd.DataFrame(got))
        _atomic(ckpt, lambda p: pd.concat(rows, ignore_index=True).to_parquet(p, index=False))
        gc.collect()
    df = pd.concat(rows, ignore_index=True).drop_duplicates('item_id') if rows else \
        pd.DataFrame({'item_id': [j[0] for j in jobs]})
    _atomic(final, lambda p: df.to_parquet(p, index=False))
    if ckpt.exists():
        ckpt.unlink()
    return df


def build_mesh_features(ids, npz_dir, split):
    """Кэш НЕ содержит tile: геометрия одна на все прогоны."""
    jobs = [(str(i), str(Path(npz_dir) / f'{i}.npz')) for i in ids]
    df = _chunked_build(jobs, _mesh_one,
                        CFG.cache / f'mesh_{split}_{FEAT_VER}.parquet',
                        CFG.cache / f'mesh_{split}_{FEAT_VER}_partial.parquet',
                        f'геометрия {split}')
    return df.set_index('item_id').reindex([str(i) for i in ids]).reset_index().fillna(0.0)


def build_image_features(ids, img_dir, split, tile):
    jobs = [(str(i), str(Path(img_dir) / f'{i}.png'), tile) for i in ids]
    df = _chunked_build(jobs, _img_one,
                        CFG.cache / f'imgf_{split}_{tile}.parquet',
                        CFG.cache / f'imgf_{split}_{tile}_partial.parquet',
                        f'силуэты {split} @{tile}')
    return df.set_index('item_id').reindex([str(i) for i in ids]).reset_index().fillna(0.0)


R.section('Геометрические признаки', f'версия {FEAT_VER}, считаются один раз на оба прогона')
mesh_tr = build_mesh_features(train_df['item_id'].tolist(), TRAIN_DIR, 'train')
mesh_te = build_mesh_features(test_df['item_id'].tolist(), TEST_DIR, 'test')
R.kv(**{'геометрия train': mesh_tr.shape, 'геометрия test': mesh_te.shape})


                     
CLIP_PROMPTS = {
    'abs_text':     ['3D text lettering', 'a sign with written words', 'an extruded logo'],
    'abs_chart':    ['a bar chart', 'a pie chart', 'a graph plot', 'a diagram', 'a flowchart',
                     'a table with data', 'an infographic'],
    'abs_support':  ['3D printing support structure', 'scaffolding lattice'],
    'abs_voxel':    ['minecraft voxel blocks', 'blocky pixelated cubes'],
    'obj_single':   ['a 3D model of a single object', 'a product render on white background'],
    'obj_semantic': ['a chair', 'a car', 'a building', 'a plant', 'a character figure',
                     'a tool', 'furniture'],
    'p_lowpoly':    ['a low-poly 3D model with visible flat facets'],
    'p_noisy':      ['a noisy 3D scan with rough surface', 'a point cloud scan'],
    'p_broken':     ['a broken mesh with holes and artifacts'],
    'p_hollow':     ['a hollow thin shell', 'a flat plane'],
    'p_multi':      ['several separate objects scattered apart', 'a collection of many objects'],
    'p_small':      ['a tiny object in the middle of a large empty frame'],
    'p_smooth':     ['a smooth clean 3D render of one object'],
}


class _ViewsDataset(torch.utils.data.Dataset):
    """Читает ИСХОДНЫЙ рендер и режет на 6 квадратов нужного размера.

    Источник — оригинальный PNG, а не кэш под конкретный tile, поэтому CLIP-признаки
    не зависят от конфигурации прогона и считаются один раз на оба.
    """
    def __init__(self, ids, src_dir, size):
        self.ids, self.dir, self.size = [str(i) for i in ids], Path(src_dir), size

    def __len__(self):
        return len(self.ids)

    def __getitem__(self, i):
        p = self.dir / f'{self.ids[i]}.png'
        s = self.size
        if p.exists():
            vs = [np.asarray(v.resize((s, s), Image.BILINEAR), dtype=np.uint8)
                  for v in split_views(Image.open(p).convert('RGB'))]
        else:
            vs = [np.full((s, s, 3), 255, np.uint8)] * 6
        return torch.from_numpy(np.stack(vs)), self.ids[i]


def compute_clip_features(ids, src_dir, split):
    """Узкое место прошлого прогона: декодирование PNG шло в один поток в теле цикла,
    и 8964 объекта заняли 83 минуты при простаивающем GPU. Здесь чтение вынесено
    в DataLoader с num_workers — то же самое считается в разы быстрее."""
    import open_clip
    name, pretrained = CFG.clip_model
    model, _, _ = open_clip.create_model_and_transforms(name, pretrained=pretrained)
    tokenizer = open_clip.get_tokenizer(name)
    model = model.to(CFG.device).eval()

    groups = list(CLIP_PROMPTS)
    flat, owner = [], []
    for g in groups:
        flat += CLIP_PROMPTS[g]; owner += [g] * len(CLIP_PROMPTS[g])
    with torch.no_grad():
        tf = model.encode_text(tokenizer(flat).to(CFG.device)).float()
        tf = tf / tf.norm(dim=-1, keepdim=True)
    owner = np.array(owner)

    size = model.visual.image_size
    size = size[0] if isinstance(size, (tuple, list)) else int(size)
    mean = torch.tensor([0.48145466, 0.4578275, 0.40821073], device=CFG.device).view(1, 3, 1, 1)
    std = torch.tensor([0.26862954, 0.26130258, 0.27577711], device=CFG.device).view(1, 3, 1, 1)

    ds = _ViewsDataset(ids, src_dir, size)
    dl = torch.utils.data.DataLoader(ds, batch_size=32, shuffle=False,
                                     num_workers=CFG.num_workers, pin_memory=True)
    rows = []
    for x, iids in tqdm(dl, desc=f'CLIP {split}'):
        x = x.to(CFG.device, non_blocking=True)
        B = x.shape[0]
        x = x.permute(0, 1, 4, 2, 3).reshape(B * 6, 3, size, size).float() / 255.0
        x = (x - mean) / std
        with torch.no_grad():
            im = model.encode_image(x).float()
        im = im / im.norm(dim=-1, keepdim=True)
        sim = (im @ tf.T).view(B, 6, -1).cpu().numpy()
        gsim = np.stack([sim[:, :, owner == g].max(-1) for g in groups], -1)
        feat = {}
        for j, g in enumerate(groups):
            v = gsim[:, :, j]
            feat[f'clip_{g}_mean'] = v.mean(1)
            feat[f'clip_{g}_max'] = v.max(1)
            feat[f'clip_{g}_std'] = v.std(1)
        abs_max = np.stack([feat[f'clip_{g}_max'] for g in groups if g.startswith('abs_')], 1).max(1)
        obj_max = np.stack([feat[f'clip_{g}_max'] for g in groups if g.startswith('obj_')], 1).max(1)
        def_max = np.stack([feat[f'clip_{g}_max'] for g in groups
                            if g.startswith('p_') and g != 'p_smooth'], 1).max(1)
        feat['clip_abstract_margin'] = abs_max - obj_max
        feat['clip_abstract_prob'] = 1 / (1 + np.exp(-100 * (abs_max - obj_max)))
        feat['clip_defect_margin'] = def_max - feat['clip_p_smooth_max']
        rows.append(pd.DataFrame(feat, index=list(iids)))
    del model
    gc.collect(); torch.cuda.empty_cache()
    return pd.concat(rows).rename_axis('item_id').reset_index()


R.section('CLIP zero-shot', 'считается один раз, от разрешения прогона не зависит')
if CFG.use_clip:
    clip_tr = cached('clip_train_orig',
                     lambda: compute_clip_features(train_df['item_id'].tolist(), TRAIN_DIR, 'train'))
    clip_te = cached('clip_test_orig',
                     lambda: compute_clip_features(test_df['item_id'].tolist(), TEST_DIR, 'test'))
    R.kv(**{'CLIP-признаков': clip_tr.shape[1] - 1})
    _m = train_df.merge(clip_tr, on='item_id', how='left').fillna(0)
    cc = pd.Series({c: np.corrcoef(_m[c], _m['abstract'])[0, 1]
                    for c in clip_tr.columns if c != 'item_id'}).sort_values(key=abs, ascending=False)
    print('  корреляция с abstract (топ-5):')
    print('  ' + cc.head(5).round(3).to_string().replace('\n', '\n  '))
else:
    clip_tr = clip_te = None
    R.warn('CLIP выключен')


                                                                                          
FEAT_COLS = None                                                                   


def assemble_features(img_train, img_test, tile):
    """Геометрия (одна на все прогоны) + силуэты под этот tile + CLIP."""
    global FEAT_COLS
    ftr = mesh_tr.merge(build_image_features(train_df['item_id'].tolist(), img_train, 'train', tile),
                        on='item_id', how='left')
    fte = mesh_te.merge(build_image_features(test_df['item_id'].tolist(), img_test, 'test', tile),
                        on='item_id', how='left')
    if clip_tr is not None:
        ftr = ftr.merge(clip_tr, on='item_id', how='left')
        fte = fte.merge(clip_te, on='item_id', how='left')

    cols = [c for c in ftr.columns if c != 'item_id' and c in fte.columns]
    ftr[cols] = ftr[cols].replace([np.inf, -np.inf], 0).fillna(0).astype(np.float32)
    fte[cols] = fte[cols].replace([np.inf, -np.inf], 0).fillna(0).astype(np.float32)

    if FEAT_COLS is None:
                                                                                    
                                                                             
        import hashlib
        keep, seen, const, dup = [], {}, [], []
        for c in cols:
            v = np.nan_to_num(ftr[c].values.astype(np.float64))
            if v.std() == 0:
                const.append(c); continue
            key = hashlib.md5(np.ascontiguousarray(np.round(v, 9)).tobytes()).hexdigest()
            if key in seen:
                dup.append(c); continue
            seen[key] = c; keep.append(c)
        FEAT_COLS = cached(f'feat_cols_{FEAT_VER}', lambda: keep)
        R.kv(**{'признаков всего': len(cols), 'после дедупликации': len(FEAT_COLS),
                'спектральных': sum(c.startswith('spec') for c in FEAT_COLS),
                'CLIP': sum(c.startswith('clip') for c in FEAT_COLS)})
    for df_ in (ftr, fte):
        for c in FEAT_COLS:
            if c not in df_.columns:
                df_[c] = 0.0
    return ftr, fte


                     
from iterstrat.ml_stratifiers import MultilabelStratifiedKFold

mskf = MultilabelStratifiedKFold(n_splits=CFG.n_folds, shuffle=True, random_state=CFG.seed)
train_df['fold'] = -1
for k, (_, vi) in enumerate(mskf.split(train_df, train_df[CFG.target_cols].values)):
    train_df.loc[train_df.index[vi], 'fold'] = k
display(train_df.groupby('fold')[CFG.target_cols].mean().round(4))

                     
from torch.utils.data import Dataset, DataLoader

MEAN = np.array([0.485, 0.456, 0.406], dtype=np.float32)
STD  = np.array([0.229, 0.224, 0.225], dtype=np.float32)


def load_views(path, tile):
    a = np.asarray(Image.open(path).convert('RGB'), dtype=np.uint8)
    t = tile
    return np.stack([a[(k // 3) * t:(k // 3 + 1) * t, (k % 3) * t:(k % 3 + 1) * t]
                     for k in range(6)])


def symmetry(v, rot=0, mirror=False, flip_ud=False):
    """Точная симметрия съёмочного стенда: 4 азимута с шагом 90° + верх + низ.

    Множество из шести кадров не меняется — меняются только их порядок и ориентация,
    поэтому ВСЕ 11 меток инвариантны. Это 4*2*2 = 16 преобразований без риска испортить
    разметку, в отличие от зума (ломает `scale`), пиксельного шума (ломает `noisy`,
    вес 0.274) и обнуления вида (создаёт условие `partial` при метке 0).

    Прежний флип `v[:, :, ::-1]` отражал кадры, но НЕ разворачивал порядок азимутов —
    получалась конфигурация, которой не соответствует ни один реальный объект.
    """
    az, po = list(CFG.azimuth_idx), list(CFG.pole_idx)
    out = [v[az[(k + rot) % 4]] for k in range(4)]
    top, bot = po
    pt = np.rot90(v[top], -rot, axes=(0, 1))
    pb = np.rot90(v[bot],  rot, axes=(0, 1))
    if mirror:
        out = [out[0]] + out[1:][::-1]
        out = [x[:, ::-1] for x in out]
        pt, pb = pt[:, ::-1], pb[:, ::-1]
    if flip_ud:
        out = [x[::-1] for x in out]
        pt, pb = pb[::-1], pt[::-1]
    return np.ascontiguousarray(np.stack(out + [pt, pb]))


SYM_ALL = [(r, m, u) for r in range(4) for m in (False, True) for u in (False, True)]
                                                                                
                                                                                  
CFG.tta_syms = [(r, m, False) for r in range(4) for m in (False, True)]


class MeshViewDataset(Dataset):
    def __init__(self, df, img_dir, feats, train=True, labels=True):
        self.ids = df['item_id'].tolist()
        self.dir = Path(img_dir)
        self.train = train
        self.labels = df[CFG.target_cols].values.astype(np.float32) if labels else None
        self.feats = feats.set_index('item_id').reindex(self.ids)[FEAT_COLS] \
                          .fillna(0.0).values.astype(np.float32)

    def __len__(self):
        return len(self.ids)

    def _aug(self, v):
        rng = np.random
        if CFG.sym_aug:
            v = symmetry(v, *SYM_ALL[rng.randint(len(SYM_ALL))])
        elif rng.rand() < 0.5:
            v = symmetry(v, 0, True, False)
        if rng.rand() < 0.3:
                                                                                 
                                                           
            ys, xs = (int(z) for z in rng.randint(-8, 9, size=2))
            out = np.full_like(v, 255)
            H, W = v.shape[1], v.shape[2]
            y0, y1 = max(0, ys), min(H, H + ys)
            x0, x1 = max(0, xs), min(W, W + xs)
            out[:, y0:y1, x0:x1] = v[:, y0 - ys:y1 - ys, x0 - xs:x1 - xs]
            v = out
        if CFG.view_dropout and rng.rand() < CFG.view_dropout:
            v = v.copy(); v[rng.randint(6)] = 255
        v = v.astype(np.float32)
        if rng.rand() < 0.3:
            v = np.clip(v * rng.uniform(0.85, 1.15) + rng.uniform(-15, 15), 0, 255)
        if CFG.pixel_noise and rng.rand() < CFG.pixel_noise:
            v = np.clip(v + rng.normal(0, 4, v.shape), 0, 255)
        return v

    def __getitem__(self, i):
        p = self.dir / f'{self.ids[i]}.png'
        v = load_views(p, CFG.tile) if p.exists() else \
            np.full((6, CFG.tile, CFG.tile, 3), 255, np.uint8)
        v = self._aug(v) if self.train else v.astype(np.float32)
        v = ((v / 255.0 - MEAN) / STD).astype(np.float32)
        out = {'views': torch.from_numpy(np.ascontiguousarray(v)).permute(0, 3, 1, 2),
               'feats': torch.from_numpy(self.feats[i]), 'item_id': self.ids[i]}
        if self.labels is not None:
            out['y'] = torch.from_numpy(self.labels[i])
        return out


def _sym_torch(v, rot, mirror, flip_ud=False):
    az, po = list(CFG.azimuth_idx), list(CFG.pole_idx)
    v = v[:, [az[(k + rot) % 4] for k in range(4)] + po]
    if rot:
        v = torch.cat([v[:, :4], torch.rot90(v[:, 4:5], -rot, dims=(-2, -1)),
                       torch.rot90(v[:, 5:6], rot, dims=(-2, -1))], 1)
    if mirror:
        v = torch.cat([v[:, :1], v[:, 1:4].flip(1), v[:, 4:]], 1).flip(-1)
    if flip_ud:
        v = torch.cat([v[:, :4], v[:, 5:6], v[:, 4:5]], 1).flip(-2)
    return v


@torch.no_grad()
def predict(model, loader, tta=None, desc='predict'):
    """tta=1 — один проход (мониторинг эпох), иначе TTA по симметриям стенда."""
    model.eval()
    syms = [(0, False, False)] if tta == 1 else CFG.tta_syms
    probs, ids = [], []
    for b in tqdm(loader, desc=f'{desc} (TTA x{len(syms)})', leave=False):
        v = b['views'].to(CFG.device, non_blocking=True)
        f = b['feats'].to(CFG.device, non_blocking=True)
        acc = 0
        with torch.cuda.amp.autocast(enabled=CFG.amp):
            for s in syms:
                acc = acc + torch.sigmoid(model(_sym_torch(v, *s), f)).float()
        probs.append((acc / len(syms)).cpu().numpy()); ids.extend(b['item_id'])
    return np.vstack(probs), ids


def check_view_layout(n=200):
    """Симметрии осмысленны, только если виды 0-3 — азимуты, 4-5 — полюса.
    Проверяем данными: соседние азимуты должны быть похожи сильнее, чем азимут и полюс."""
    acc, cnt = np.zeros((6, 6)), 0
    for iid in train_df['item_id'].head(n):
        p = Path(IMG_TRAIN) / f'{iid}.png'
        if not p.exists():
            continue
        x = load_views(p, CFG.tile).astype(np.float32).mean(-1)
        m = (np.abs(x - np.median(x)) > 15).reshape(6, -1).astype(np.float32)
        m -= m.mean(1, keepdims=True)
        nrm = np.linalg.norm(m, axis=1) + 1e-9
        acc += (m @ m.T) / np.outer(nrm, nrm); cnt += 1
    S = acc / max(cnt, 1)
    ring = np.mean([S[i, (i + 1) % 4] for i in range(4)])
    cross = np.mean([S[i, j] for i in CFG.azimuth_idx for j in CFG.pole_idx])
    fig, ax = plt.subplots(1, 2, figsize=(12, 4))
    im = ax[0].imshow(S, cmap='viridis'); ax[0].grid(False)
    ax[0].set_title('Сходство силуэтов между видами')
    for i in range(6):
        for j in range(6):
            ax[0].text(j, i, f'{S[i, j]:.2f}', ha='center', va='center', fontsize=7,
                       color='w' if S[i, j] < S.max() * .6 else 'k')
    plt.colorbar(im, ax=ax[0], fraction=.046)
    ax[1].bar(['соседние\nазимуты', 'азимут-полюс'], [ring, cross],
              color=[PAL['good'], PAL['bad']])
    ax[1].set_title('Кольцевая структура')
    plt.tight_layout(); plt.show()
    if ring > cross + 0.02:
        R.ok(f'раскладка подтверждена (кольцо {ring:.3f} > крест {cross:.3f}) — '
             f'{len(SYM_ALL)} симметрий включены, TTA x{len(CFG.tta_syms)}')
    else:
        CFG.sym_aug = False
        CFG.tta_syms = [(0, False, False), (0, True, False)]
        R.warn('кольцевая структура не подтверждена — симметрии выключены')
    return S


_t = np.random.randint(0, 255, (6, 64, 64, 3), dtype=np.uint8)
assert np.array_equal(symmetry(_t, 0, False, False), _t)
assert np.array_equal(symmetry(symmetry(_t, 1), 3), _t)
assert np.array_equal(symmetry(symmetry(_t, 0, True), 0, True), _t)
assert np.array_equal(symmetry(symmetry(_t, 0, False, True), 0, False, True), _t)
assert len({tuple(sorted(int(x.sum()) for x in symmetry(_t, *s))) for s in SYM_ALL}) == 1
R.ok(f'{len(SYM_ALL)} симметрий: множество кадров сохраняется, преобразования обратимы')


                     
import timm


class MLDecoder(nn.Module):
    """Голова классификации с запросом на класс (Ridnik et al., WACV 2023, arXiv 2111.12933).

    Зачем именно здесь. До сих пор каждый вид сжимался глобальным пулингом в один
    вектор, и только потом решалось, какие метки поставить. Для `noisy` или `simple`
    это нормально — они про объект целиком. Но `open` (F1 0.30), `artifacts` (0.33)
    и `intersection` (0.15) — дефекты **локальные**: дыра занимает малую долю кадра,
    и при усреднении по всем патчам её сигнал разбавляется в сотни раз.

    ML-Decoder заводит обучаемый запрос на каждый класс и даёт ему через cross-attention
    самому выбрать, на какие участки каких видов смотреть. Запрос `open` может
    сосредоточиться на краях силуэта, запрос `set` — на разнесённых кусках. Self-attention
    между запросами в оригинальной статье убран как избыточный, поэтому стоимость линейна
    по числу токенов: 11 запросов почти ничего не стоят.

    Токены: патч-сетка каждого вида ужимается адаптивным пулингом до
    `mld_tokens` x `mld_tokens`, к ней прибавляется кодирование номера вида, и все шесть
    видов склеиваются в одну последовательность. При 8x8 это 384 токена независимо от
    разрешения — то есть 448 не удорожает голову, только бэкбон.
    """

    def __init__(self, d_in, n_out, dim=384, layers=2, heads=8):
        super().__init__()
        self.proj = nn.Linear(d_in, dim) if d_in != dim else nn.Identity()
        self.query = nn.Parameter(torch.zeros(1, n_out, dim))
        nn.init.trunc_normal_(self.query, std=0.02)
        self.view_emb = nn.Parameter(torch.zeros(1, 6, 1, dim))
        nn.init.trunc_normal_(self.view_emb, std=0.02)
        self.blocks = nn.ModuleList()
        for _ in range(layers):
            self.blocks.append(nn.ModuleDict({
                'norm_q': nn.LayerNorm(dim), 'norm_k': nn.LayerNorm(dim),
                'attn': nn.MultiheadAttention(dim, heads, dropout=0.1, batch_first=True),
                'norm_f': nn.LayerNorm(dim),
                'ffn': nn.Sequential(nn.Linear(dim, 2 * dim), nn.GELU(),
                                     nn.Dropout(0.1), nn.Linear(2 * dim, dim))}))
        self.norm_out = nn.LayerNorm(dim)

    def forward(self, tokens, extra=None):
        """tokens: (B, 6, T, d_in) — патч-токены каждого вида.
        extra:  (B, d)  — гео-вектор, добавляется как ещё один ключ."""
        B, V, T, _ = tokens.shape
        x = self.proj(tokens) + self.view_emb[:, :V]
        x = x.reshape(B, V * T, -1)
        if extra is not None:
            x = torch.cat([x, extra.unsqueeze(1)], 1)
        q = self.query.expand(B, -1, -1)
        att_last = None
        for blk in self.blocks:
            a, w = blk['attn'](blk['norm_q'](q), blk['norm_k'](x), blk['norm_k'](x),
                               need_weights=True, average_attn_weights=True)
            q = q + a
            q = q + blk['ffn'](blk['norm_f'](q))
            att_last = w
        return self.norm_out(q), att_last                                          


class MultiViewNet(nn.Module):
    def __init__(self, n_feats, n_out=CFG.n_targets):
        super().__init__()
        kw = dict(pretrained=True, num_classes=0, drop_rate=CFG.drop_rate)
        if IS_VIT:
            kw['img_size'] = CFG.tile
        self.backbone = timm.create_model(CFG.backbone, **kw)
        d = self.backbone.num_features
        self.d = d
        self.geo = nn.Sequential(
            nn.BatchNorm1d(n_feats), nn.Linear(n_feats, 256), nn.SiLU(), nn.Dropout(0.2),
            nn.Linear(256, 128), nn.SiLU())

        if CFG.use_mldecoder:
            self.dec = MLDecoder(d, n_out, dim=CFG.mld_dim, layers=CFG.mld_layers)
            self.geo_to_tok = nn.Linear(128, CFG.mld_dim)
                                                                                        
            self.cls_w = nn.Parameter(torch.zeros(n_out, CFG.mld_dim))
            self.cls_b = nn.Parameter(torch.zeros(n_out))
            nn.init.trunc_normal_(self.cls_w, std=0.02)
        else:
            self.att_v = nn.Sequential(nn.Linear(d, 256), nn.Tanh())
            self.att_u = nn.Sequential(nn.Linear(d, 256), nn.Sigmoid())
            self.att_w = nn.Linear(256, 1)
            self.head = nn.Sequential(nn.Linear(2 * d + 128, 512), nn.SiLU(),
                                      nn.Dropout(0.3), nn.Linear(512, n_out))

    def _patch_tokens(self, x):
        """(B*V, 3, H, W) -> (B*V, T, d): патч-токены, ужатые до mld_tokens^2."""
        n = CFG.mld_tokens
        if IS_VIT:
            f = self.backbone.forward_features(x)                         
            npre = f.shape[1] - int(math.isqrt(f.shape[1])) ** 2
            g = f[:, npre:]
            s = int(math.isqrt(g.shape[1]))
            g = g[:, :s * s].transpose(1, 2).reshape(g.shape[0], -1, s, s)
        else:
            g = self.backbone.forward_features(x)                          
        g = F.adaptive_avg_pool2d(g, (n, n))
        return g.flatten(2).transpose(1, 2)                             

    def embed(self, views, feats):
        B, V = views.shape[:2]
        flat = views.flatten(0, 1)
        gv = self.geo(feats)
        if CFG.use_mldecoder:
            tok = self._patch_tokens(flat).view(B, V, -1, self.d)
            q, att = self.dec(tok, self.geo_to_tok(gv))
            logits = (q * self.cls_w.unsqueeze(0)).sum(-1) + self.cls_b
            return logits, att
        z = self.backbone(flat).view(B, V, -1)
        a = self.att_w(self.att_v(z) * self.att_u(z))
        w = torch.softmax(a, 1)
        pooled = torch.cat([(z * w).sum(1), z.max(1).values], -1)
        return self.head(torch.cat([pooled, gv], 1)), w.squeeze(-1)

    def forward(self, views, feats, return_att=False):
        logits, att = self.embed(views, feats)
        return (logits, att) if return_att else logits


class AsymmetricLoss(nn.Module):
    """Ridnik et al., ICCV 2021 — как в прогоне, давшем 14.987."""
    def __init__(self, gamma_neg=4.0, gamma_pos=0.0, clip=0.05, eps=1e-8):
        super().__init__()
        self.gn, self.gp, self.clip, self.eps = gamma_neg, gamma_pos, clip, eps

    def forward(self, logits, y):
        p = torch.sigmoid(logits)
        p_neg = (1 - p + self.clip).clamp(max=1.0)
        loss = y * torch.log(p.clamp(min=self.eps)) + (1 - y) * torch.log(p_neg.clamp(min=self.eps))
        pt = p * y + (1 - p) * (1 - y)
        gamma = self.gp * y + self.gn * (1 - y)
        return -(loss * (1 - pt).pow(gamma)).mean()


class ModelEMA:
    def __init__(self, model, decay=0.999):
        self.ema = copy.deepcopy(model).eval()
        for p in self.ema.parameters():
            p.requires_grad_(False)
        self.decay = decay

    @torch.no_grad()
    def update(self, model, step=None):
        d = self.decay if step is None else min(self.decay, (1 + step) / (10 + step))
        msd = model.state_dict()
        for k, v in self.ema.state_dict().items():
            if v.dtype.is_floating_point:
                v.mul_(d).add_(msd[k].detach(), alpha=1 - d)
            else:
                v.copy_(msd[k])

    def state_dict(self): return self.ema.state_dict()
    def load_state_dict(self, sd): self.ema.load_state_dict(sd)


def _selftest_model():
    m = MultiViewNet(n_feats=16).to(CFG.device).eval()
    v = torch.randn(2, 6, 3, CFG.tile, CFG.tile, device=CFG.device)
    f = torch.randn(2, 16, device=CFG.device)
    with torch.no_grad():
        o, a = m(v, f, return_att=True)
    R.ok(f'модель собрана: logits {tuple(o.shape)}, attention {tuple(a.shape)}, '
         f'параметров {sum(p.numel() for p in m.parameters()) / 1e6:.1f}M')
    if CFG.use_mldecoder:
        R.kv(**{'токенов в декодере': f'6 видов x {CFG.mld_tokens}^2 = '
                                      f'{6 * CFG.mld_tokens ** 2} + 1 гео'})
    del m, v, f; gc.collect(); torch.cuda.empty_cache()


                                                                               


                     
from torch.cuda.amp import autocast, GradScaler


def per_class_f1(y, p):
    return pd.Series([f1_score(y[:, i], p[:, i], zero_division=0) for i in range(11)],
                     index=CFG.target_cols)


def exact_f1_threshold(y, p, plateau=0.995):
    """Точный argmax F1 по порогу за один проход сортировки.

    Разрез допустим только там, где значение p МЕНЯЕТСЯ: иначе порог попадает между
    двумя одинаковыми вероятностями и `p > th` выбрасывает оба объекта вместо одного.
    При насыщенных сигмоидах и усреднении по фолдам совпадения массовые.
    """
    y = np.asarray(y).astype(np.int32); p = np.asarray(p, np.float64)
    P = int(y.sum())
    if P == 0:
        return 0.5, 0.0
    o = np.argsort(-p, kind='stable'); ys, ps = y[o], p[o]
    f1 = 2 * np.cumsum(ys) / (np.arange(1, len(y) + 1) + P)
    fb = float(f1.max())
    cut = np.empty(len(ps), bool); cut[-1] = True; cut[:-1] = ps[:-1] > ps[1:]
    ok = np.where((f1 >= plateau * fb) & cut)[0]
    if not len(ok):
        ok = np.where(cut)[0][[int(np.argmax(f1[cut]))]]
    i = int(ok[len(ok) // 2])
    return float((ps[i] + ps[i + 1]) / 2 if i + 1 < len(ps) else ps[i] - 1e-6), fb


def quick_thresholds(y, prob):
    return np.array([exact_f1_threshold(y[:, i], prob[:, i])[0] for i in range(11)])


def _fmt_eta(sec):
    sec = int(max(sec, 0))
    return f'{sec // 3600}ч {sec % 3600 // 60:02d}м' if sec >= 3600 else f'{sec // 60}м {sec % 60:02d}с'


def live_dashboard(hist, fold):
    if not hist:
        return
    h = pd.DataFrame(hist)
    fig, ax = plt.subplots(1, 3, figsize=(16, 4))
    ax[0].plot(h['epoch'], h['val_metric'], '-o', ms=4, color=PAL['main'], label='raw')
    if h['val_metric_ema'].notna().any():
        ax[0].plot(h['epoch'], h['val_metric_ema'], '-s', ms=4, color=PAL['alt'], label='EMA')
    ax[0].axhline(14.25, color=PAL['grey'], ls=':', lw=1, label='CNN в v5')
    ax[0].set_xlabel('эпоха'); ax[0].legend(fontsize=8)
    ax[0].set_title(f'{CFG.run_tag} fold {fold}: метрика')
    ax[1].plot(h['epoch'], 10 * h['f1_quality'], '-o', ms=4, color=PAL['bad'], label='quality')
    ax[1].plot(h['epoch'], 10 * h['f1_artefacts'], '-o', ms=4, color=PAL['warn'], label='artefacts')
    ax[1].set_xlabel('эпоха'); ax[1].legend(fontsize=8); ax[1].set_title('Половины метрики')
    f1c = [c for c in h.columns if c.startswith('f1_') and c not in ('f1_quality', 'f1_artefacts')]
    last = h.iloc[-1][f1c].astype(float).sort_values()
    ax[2].barh([c[3:] for c in last.index], last.values,
               color=[PAL['bad'] if v < .4 else PAL['warn'] if v < .6 else PAL['good']
                      for v in last.values])
    ax[2].set_xlim(0, 1); ax[2].set_title(f'per-class F1, эпоха {int(h["epoch"].iloc[-1])}')
    plt.tight_layout(); plt.show()


def train_fold(fold):
    if CFG.resume and has_artifact(rt(f'fold{fold}_preds')):
        d = load_artifact(rt(f'fold{fold}_preds'))
        R.ok(f'фолд {fold} взят из кэша (метрика {d["ens"]:.3f}) — обучение пропускаем')
        return (d['oof_ids'], d['oof']), (d['te_ids'], d['te'])

    R.section(f'{CFG.run_tag}: фолд {fold} / {CFG.n_folds}',
              f'{CFG.backbone.split(".")[0]} @ {CFG.tile} | эпох {CFG.epochs}')
    seed_everything(CFG.seed + fold)
    tr = train_df[train_df.fold != fold].reset_index(drop=True)
    va = train_df[train_df.fold == fold].reset_index(drop=True)
    ltr = DataLoader(MeshViewDataset(tr, IMG_TRAIN, feat_tr, train=True),
                     batch_size=CFG.batch_size, shuffle=True, drop_last=True,
                     num_workers=CFG.num_workers, pin_memory=True, persistent_workers=True)
    lva = DataLoader(MeshViewDataset(va, IMG_TRAIN, feat_tr, train=False),
                     batch_size=CFG.batch_size * 2, num_workers=CFG.num_workers, pin_memory=True)
    R.kv(**{'объектов train / val': f'{len(tr)} / {len(va)}',
            'шагов на эпоху': f'{len(ltr)} (оптимизаторских {len(ltr) // CFG.accum})'})

    model = MultiViewNet(len(FEAT_COLS)).to(CFG.device)
    ema = ModelEMA(model, CFG.ema_decay)
    head = [p for n_, p in model.named_parameters() if not n_.startswith('backbone')]
    head_ids = {id(p) for p in head}

    if IS_VIT and CFG.layer_decay < 1.0:
        blocks = getattr(model.backbone, 'blocks', [])
        nb_ = len(blocks)
        buckets = {}
        for name, p in model.backbone.named_parameters():
            m_ = re.search(r'blocks\.(\d+)\.', name)
            depth = (int(m_.group(1)) + 1) if m_ else (0 if any(
                k in name for k in ('patch_embed', 'pos_embed', 'cls_token', 'reg_token')) else nb_)
            buckets.setdefault(round(CFG.layer_decay ** (nb_ - depth), 5), []).append(p)
        groups = [{'params': ps, 'lr': CFG.backbone_lr * s} for s, ps in buckets.items()]
        R.kv(**{'LR бэкбона': f'{CFG.backbone_lr * min(buckets):.2e} … {CFG.backbone_lr:.2e} '
                              f'(затухание {CFG.layer_decay}, блоков {nb_})'})
    else:
        groups = [{'params': [p for p in model.backbone.parameters()], 'lr': CFG.backbone_lr}]
    groups.append({'params': head, 'lr': CFG.lr * CFG.head_lr_mult})
    opt = torch.optim.AdamW(groups, weight_decay=CFG.weight_decay)

    steps = CFG.epochs * max(len(ltr) // CFG.accum, 1)
    warm = int(CFG.warmup_frac * steps)
    sched = torch.optim.lr_scheduler.LambdaLR(
        opt, lambda s: s / max(warm, 1) if s < warm
        else 0.5 * (1 + math.cos(math.pi * (s - warm) / max(steps - warm, 1))))

    pos = tr[CFG.target_cols].sum(0).values.astype(np.float32)
    pw = np.clip((len(tr) - pos) / np.clip(pos, 1, None), 1.0, 20.0)
    bce = nn.BCEWithLogitsLoss(pos_weight=torch.tensor(pw, device=CFG.device))
    asl = AsymmetricLoss()
    scaler = GradScaler(enabled=CFG.amp)

    snaps, best, start_ep, hist, gstep = [], -1, 0, [], 0
    ck = load_ckpt(fold)
    if ck is not None:
        model.load_state_dict(_to_fp32(ck['model']))
        if ck.get('ema') is not None:
            ema.load_state_dict(_to_fp32(ck['ema']))
        try:
            opt.load_state_dict(ck['opt']); sched.load_state_dict(ck['sched'])
            scaler.load_state_dict(ck['scaler'])
        except Exception as e:
            R.warn(f'состояние оптимизатора не восстановлено: {e}')
        snaps = [(s, _to_fp32(sd)) for s, sd in ck['snaps']]
        best, start_ep, hist = ck['best'], ck['epoch'] + 1, ck.get('hist', [])
        set_rng_state(ck['rng'])
        R.ok(f'продолжаем с эпохи {start_ep + 1}/{CFG.epochs} (лучшая пока {best:.3f})')

    yv = va[CFG.target_cols].values
    pred, ep_times = None, []

    for ep in range(start_ep, CFG.epochs):
        t0 = time.time(); model.train()
        tot, nb_i = 0.0, 0
        opt.zero_grad(set_to_none=True)
        pbar = tqdm(ltr, desc=f'{CFG.run_tag} f{fold} ep{ep + 1:>2}/{CFG.epochs}', leave=False)
        for it, b in enumerate(pbar):
            v = b['views'].to(CFG.device, non_blocking=True)
            f = b['feats'].to(CFG.device, non_blocking=True)
            y = b['y'].to(CFG.device, non_blocking=True)
            y = y * (1 - CFG.label_smooth) + 0.5 * CFG.label_smooth
            if CFG.mixup > 0 and random.random() < 0.5:
                lam = float(np.random.beta(CFG.mixup, CFG.mixup))
                pm = torch.randperm(v.size(0), device=v.device)
                v = lam * v + (1 - lam) * v[pm]
                f = lam * f + (1 - lam) * f[pm]
                y = lam * y + (1 - lam) * y[pm]
            with autocast(enabled=CFG.amp):
                logits = model(v, f)
                loss = (1 - CFG.asl_weight) * bce(logits, y) + CFG.asl_weight * asl(logits, y)
            scaler.scale(loss / CFG.accum).backward()
            if (it + 1) % CFG.accum == 0 or it + 1 == len(ltr):
                scaler.unscale_(opt)
                torch.nn.utils.clip_grad_norm_(model.parameters(), 5.0)
                scaler.step(opt); scaler.update(); opt.zero_grad(set_to_none=True)
                sched.step(); gstep += 1; ema.update(model, gstep)
            tot += float(loss.detach()); nb_i += 1
            if it % 20 == 0:
                mem = torch.cuda.max_memory_allocated() / 1e9 if torch.cuda.is_available() else 0
                pbar.set_postfix_str(f'L={tot / nb_i:.3f} lr={opt.param_groups[-1]["lr"]:.1e} '
                                     f'{mem:.1f}G')

        prob, _ = predict(model, lva, tta=1, desc=f'val ep{ep + 1}')
        pred = (prob > quick_thresholds(yv, prob)).astype(int)
        sc, fq, fa = competition_metric(yv, pred)
        sc_ema = np.nan
        if ep >= 1:
            pe, _ = predict(ema.ema, lva, tta=1, desc=f'val-EMA ep{ep + 1}')
            prd = (pe > quick_thresholds(yv, pe)).astype(int)
            sc_ema, fqe, fae = competition_metric(yv, prd)
            if sc_ema > sc:
                prob, pred, sc, fq, fa = pe, prd, sc_ema, fqe, fae

        dt = time.time() - t0; ep_times.append(dt)
        left = (CFG.epochs - ep - 1) * np.median(ep_times)
        src = 'EMA' if (not np.isnan(sc_ema) and sc_ema == sc) else 'raw'
        print(f'  ep{ep + 1:>2}/{CFG.epochs} | метрика {sc:6.3f} ({src}) = quality {10 * fq:5.2f} '
              f'+ artefacts {10 * fa:5.2f} | loss {tot / nb_i:.3f} | {dt / 60:.1f} мин | '
              f'ост. {_fmt_eta(left)}' + (' <-- best' if sc > best else ''))
        pcf = per_class_f1(yv, pred)
        print('        ' + '  '.join(f'{c[:5]}={pcf[c]:.2f}' for c in CFG.target_cols))

        row = {'run': CFG.run_tag, 'fold': fold, 'epoch': ep + 1, 'train_loss': tot / nb_i,
               'val_metric': sc, 'val_metric_ema': sc_ema, 'f1_quality': fq,
               'f1_artefacts': fa, 'sec': dt, 'time': time.strftime('%Y-%m-%d %H:%M:%S'),
               **{f'f1_{c}': float(v_) for c, v_ in pcf.items()}}
        log_epoch(row); hist.append(row)

        if ep >= 1:
            keep = ema.ema if src == 'EMA' else model
            snaps.append((sc, {k: v_.detach().cpu().clone() for k, v_ in keep.state_dict().items()}))
            snaps.sort(key=lambda x: -x[0]); del snaps[CFG.n_snapshots:]
        best = max(best, sc)

        if (ep + 1) % CFG.ckpt_every == 0 or ep == CFG.epochs - 1:
            conv = _to_fp16 if CFG.ckpt_fp16 else (lambda x: x)
            save_ckpt(fold, {'epoch': ep, 'best': best, 'rng': rng_state(), 'hist': hist,
                             'model': conv(model.state_dict()), 'ema': conv(ema.state_dict()),
                             'opt': opt.state_dict(), 'sched': sched.state_dict(),
                             'scaler': scaler.state_dict(),
                             'snaps': [(s, conv(sd)) for s, sd in snaps],
                             'cfg': {'tile': CFG.tile, 'backbone': CFG.backbone,
                                     'epochs': CFG.epochs, 'feat_ver': FEAT_VER}})

    live_dashboard(hist, fold)
    if not snaps:
        snaps = [(best, {k: v_.detach().cpu().clone() for k, v_ in model.state_dict().items()})]

    lte = DataLoader(MeshViewDataset(test_df.assign(**{c: 0 for c in CFG.target_cols}),
                                     IMG_TEST, feat_te, train=False, labels=False),
                     batch_size=CFG.batch_size * 2, num_workers=CFG.num_workers, pin_memory=True)
    oa, ta = 0.0, 0.0
    for k, (s_, st) in enumerate(snaps):
        model.load_state_dict(st)
        op, oof_ids = predict(model, lva, desc=f'OOF snap{k + 1}/{len(snaps)}')
        tp, te_ids = predict(model, lte, desc=f'test snap{k + 1}/{len(snaps)}')
        oa = oa + op; ta = ta + tp
    oof_prob, te_prob = oa / len(snaps), ta / len(snaps)
    ens = competition_metric(yv, (oof_prob > quick_thresholds(yv, oof_prob)).astype(int))[0]
    print(f'  лучшая эпоха {best:.3f} -> снапшот-ансамбль + TTA {ens:.3f} ({ens - best:+.3f})')
    R.bar(ens, label=f'фолд {fold}')

    keep_snapshots(fold, snaps)
    save_artifact(rt(f'fold{fold}_preds'),
                  {'oof': oof_prob, 'oof_ids': list(oof_ids), 'te': te_prob,
                   'te_ids': list(te_ids), 'best': best, 'ens': ens})
    drop_ckpt(fold)
    del model, ema, ltr, lva, snaps; gc.collect(); torch.cuda.empty_cache()
    return (oof_ids, oof_prob), (te_ids, te_prob)


In [ ]:
%%writefile /content/meshqc_code/vitb336_stages.json
[
  {
    "source_cell": 6,
    "source": "import os, gc, json, math, random, re, time, copy, warnings, shutil, itertools\nfrom pathlib import Path\nimport numpy as np\nimport pandas as pd\nimport torch\nimport torch.nn as nn\nimport torch.nn.functional as F\nfrom tqdm.auto import tqdm\nimport matplotlib.pyplot as plt\nimport matplotlib as mpl\nfrom sklearn.metrics import f1_score\n\nwarnings.filterwarnings('ignore')\nmpl.rcParams.update({\n    'figure.facecolor': 'white', 'axes.facecolor': '#fbfbfd', 'axes.grid': True,\n    'grid.alpha': .25, 'grid.linestyle': '--', 'axes.spines.top': False,\n    'axes.spines.right': False, 'font.size': 10, 'axes.titlesize': 11,\n    'axes.titleweight': 'bold'})\nPAL = {'good': '#2e9e5b', 'bad': '#d1495b', 'warn': '#e8a33d', 'main': '#2d6cdf',\n       'alt': '#8a4fbd', 'grey': '#9aa0a6', 'dark': '#1f2933'}\n\n\nclass Reporter:\n    W = 86\n    _t = {}\n    @classmethod\n    def section(cls, t, s=''):\n        print('\\n' + '=' * cls.W); print(f'  {t}')\n        if s: print(f'  {s}')\n        print('=' * cls.W); cls._t[t] = time.time()\n    @classmethod\n    def done(cls, t, extra=''):\n        print(f'  [OK] {t} — {(time.time() - cls._t.get(t, time.time())) / 60:.1f} мин'\n              + (f' | {extra}' if extra else ''))\n    @staticmethod\n    def kv(**kw):\n        for k, v in kw.items(): print(f'    {k:<32s} {v}')\n    @staticmethod\n    def ok(m): print(f'  [+] {m}')\n    @staticmethod\n    def warn(m): print(f'  [!] {m}')\n    @staticmethod\n    def fail(m): print(f'  [x] {m}')\n    @staticmethod\n    def bar(v, lo=12, hi=17, w=36, label=''):\n        f = 0. if hi <= lo else max(0., min(1., (v - lo) / (hi - lo)))\n        n = int(round(f * w))\n        print(f'    {label:<22s} |{\"█\" * n}{\"·\" * (w - n)}| {v:.3f}')\n\n\nR = Reporter\n\n\nclass CFG:\n    seed = 42\n    dataset = 'daniilantonov5/3d-mesh-quality-control'\n\n    # ---- единственное разрешение: 336 ----\n    # Нативный размер вида 512 px. 336 даёт 576 токенов против 256 при 224 и стоит\n    # втрое дешевле, чем 448. По отдаче на час это лучшая точка.\n    tile = 336\n    view_mode = 'multiview'\n    grid_rows, grid_cols = 2, 3\n\n    backbone    = 'vit_base_patch14_reg4_dinov2.lvd142m'\n    backbone_lr = 2e-5\n    layer_decay = 0.70\n    batch_size  = 2\n    accum       = 12          # эффективный батч 24\n    run_tag     = 'vitb336'\n\n    drop_rate = 0.2\n    # Лучшая эпоха во всех прошлых прогонах приходилась на 4-9 из 16, дальше метрика\n    # падала и не восстанавливалась. 12 эпох — с запасом.\n    epochs = 12\n    mixup = 0.4\n    asl_weight = 0.65         # пропорция BCE/ASL из прогона, давшего 14.987\n    lr = 3e-4\n    head_lr_mult = 10.0\n    weight_decay = 0.05\n    warmup_frac = 0.1\n    label_smooth = 0.01\n    amp = True\n    num_workers = 4\n    ema_decay = 0.999\n\n    # ---- ML-Decoder ----\n    use_mldecoder = True\n    mld_tokens = 8\n    mld_dim = 512\n    mld_layers = 2\n\n    # ---- симметрии стенда ----\n    sym_aug = True\n    azimuth_idx = [0, 1, 2, 3]\n    pole_idx = [4, 5]\n    view_dropout = 0.0        # создаёт условие `partial` при метке 0\n    pixel_noise = 0.0         # противоречит классу `noisy` (вес 0.274)\n\n    use_clip = True\n    clip_model = ('ViT-B-32', 'laion2b_s34b_b79k')\n    clip_pca = 64\n\n    n_folds = 5\n    folds_to_run = [0, 1, 2, 3, 4]\n\n    n_snapshots = 3\n\n    use_drive = True\n    drive_dir = '/content/drive/MyDrive/sber_meshqc_v9'\n    resume = True\n    ckpt_every = 1\n    ckpt_fp16 = True\n    keep_fold_ckpt = False\n    force_recompute = []\n\n    # Готовый прогон v5 (14.987 на лидерборде) входит в ансамбль бесплатно:\n    # его fold-предсказания весят 0.1 МБ и уже лежат на Drive.\n    reuse_runs = {\n        'v5_224': '/content/drive/MyDrive/sber_meshqc_v5/artifacts/dinos224_geo2_v5_fold{k}_preds.joblib',\n    }\n\n    artifact_cols = ['abstract','artifacts','intersection','lowpoly','noisy',\n                     'open','partial','scale','set','simple']\n    target_cols = artifact_cols + ['quality']\n    n_targets = 11\n    device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')"
  },
  {
    "source_cell": 12,
    "source": "import kagglehub\n\nroot = Path(kagglehub.dataset_download(CFG.dataset))\nprint('dataset root:', root)\n\ndef _pick_dir(mode):\n    \"\"\"Папка с максимальным числом .npz, в пути которой встречается train/test.\"\"\"\n    best, best_n = None, -1\n    for p in root.rglob('*'):\n        if not p.is_dir():\n            continue\n        if mode not in str(p).lower():\n            continue\n        n = sum(1 for _ in p.glob('*.npz'))\n        if n > best_n:\n            best, best_n = p, n\n    if best is None or best_n == 0:\n        raise FileNotFoundError(f'не найдена папка с .npz для {mode}')\n    return best\n\ndef _pick_csv(mode):\n    cands = list(root.rglob('*.csv'))\n    def score(p):\n        n = p.name.lower(); s = 0\n        if mode == 'train':\n            s += 10 * ('train' in n) - 20 * ('submission' in n) - 20 * ('test' in n)\n        else:\n            s += 10 * ('test' in n) - 5 * ('submission' in n) - 20 * ('train' in n)\n        return s - 0.001 * len(n)\n    return sorted(cands, key=score, reverse=True)[0]\n\nTRAIN_DIR, TEST_DIR = _pick_dir('train'), _pick_dir('test')\nTRAIN_CSV, TEST_CSV = _pick_csv('train'), _pick_csv('test')\nprint('train dir :', TRAIN_DIR, len(list(TRAIN_DIR.glob('*.npz'))), 'npz /',\n      len(list(TRAIN_DIR.glob('*.png'))), 'png')\nprint('test  dir :', TEST_DIR,  len(list(TEST_DIR.glob('*.npz'))),  'npz /',\n      len(list(TEST_DIR.glob('*.png'))),  'png')\nprint('train csv :', TRAIN_CSV)\nprint('test  csv :', TEST_CSV)\n\ntrain_df = pd.read_csv(TRAIN_CSV)\ntest_df  = pd.read_csv(TEST_CSV)[['item_id']].copy()\ntrain_df['item_id'] = train_df['item_id'].astype(str)\ntest_df['item_id']  = test_df['item_id'].astype(str)\n\nif 'quality' not in train_df.columns:\n    train_df['quality'] = (train_df[CFG.artifact_cols].sum(1) == 0).astype(int)\n\nprint(train_df.shape, test_df.shape)\ntrain_df.head()"
  },
  {
    "source_cell": 17,
    "source": "from sklearn.metrics import f1_score\n\ndef competition_metric(y_true, y_pred):\n    \"\"\"10*F1(quality) + 10*F1_weighted(10 дефектов) — точная формула организаторов.\"\"\"\n    f1_q = f1_score(y_true[:, 10], y_pred[:, 10], zero_division=0)\n    f1_a = f1_score(y_true[:, :10], y_pred[:, :10], average='weighted', zero_division=0)\n    return 10 * f1_q + 10 * f1_a, f1_q, f1_a\n\n# сколько стоит \"OR-правило\" quality = нет дефектов, если по дефектам ошибаться независимо\nY = train_df[CFG.target_cols].values\nfor err in [0.02, 0.05, 0.10]:\n    rng = np.random.default_rng(0)\n    fake = Y.copy()\n    flip = rng.random(fake[:, :10].shape) < err\n    fake[:, :10] = np.abs(fake[:, :10] - flip)\n    fake[:, 10] = (fake[:, :10].sum(1) == 0).astype(int)\n    s, fq, fa = competition_metric(Y, fake)\n    print(f'FPR/FNR по дефектам {err:.0%} -> метрика {s:5.2f} (quality F1={fq:.3f}, artefacts F1={fa:.3f})')\nprint('\\n>>> Ошибка 5% по дефектам уже срезает quality F1 до ~0.7. '\n      'Поэтому quality предсказывается ОТДЕЛЬНОЙ головой, а не только правилом.')"
  },
  {
    "source_cell": 19,
    "source": "from PIL import Image\nfrom concurrent.futures import ProcessPoolExecutor\n\nImage.MAX_IMAGE_PIXELS = None\n\ndef infer_grid(w, h):\n    \"\"\"(rows, cols) для 6 видов по соотношению сторон.\"\"\"\n    ar = w / h\n    cands = {(2, 3): 3 / 2, (3, 2): 2 / 3, (1, 6): 6.0, (6, 1): 1 / 6}\n    return min(cands, key=lambda k: abs(math.log(ar / cands[k])))\n\ndef split_views(img):\n    \"\"\"PIL.Image -> список из 6 PIL.Image в каноническом порядке чтения.\"\"\"\n    w, h = img.size\n    r, c = infer_grid(w, h)\n    tw, th = w // c, h // r\n    return [img.crop((j * tw, i * th, (j + 1) * tw, (i + 1) * th)) for i in range(r) for j in range(c)]\n\ndef make_cache(args):\n    src, dst, tile = args\n    try:\n        img = Image.open(src).convert('RGB')\n        views = [v.resize((tile, tile), Image.BILINEAR) for v in split_views(img)]\n        out = Image.new('RGB', (3 * tile, 2 * tile))\n        for k, v in enumerate(views):\n            out.paste(v, ((k % 3) * tile, (k // 3) * tile))\n        out.save(dst, format='PNG', optimize=False, compress_level=1)\n        return 1\n    except Exception:\n        return 0\n\ndef build_image_cache(ids, src_dir, split):\n    out_dir = CFG.cache / f'img_{split}_{CFG.tile}'\n    out_dir.mkdir(parents=True, exist_ok=True)\n    jobs = [(str(Path(src_dir) / f'{i}.png'), str(out_dir / f'{i}.png'), CFG.tile)\n            for i in ids if not (out_dir / f'{i}.png').exists()\n            and (Path(src_dir) / f'{i}.png').exists()]\n    if jobs:\n        with ProcessPoolExecutor(max_workers=os.cpu_count()) as ex:\n            ok = list(tqdm(ex.map(make_cache, jobs, chunksize=32), total=len(jobs),\n                           desc=f'cache {split}'))\n        print(f'{split}: закэшировано {sum(ok)}/{len(jobs)}')\n    missing = [i for i in ids if not (out_dir / f'{i}.png').exists()]\n    print(f'{split}: всего в кэше {len(ids) - len(missing)}/{len(ids)}, нет картинки у {len(missing)}')\n    return out_dir, set(missing)\n\n# --- сколько пикселей на вид даёт исходный рендер: апскейл выше этого бессмысленно ---\n_probe = [Image.open(Path(TRAIN_DIR) / f'{i}.png').size\n          for i in train_df['item_id'].head(40) if (Path(TRAIN_DIR) / f'{i}.png').exists()]\nif _probe:\n    _sizes = pd.Series([f'{w}x{h}' for w, h in _probe]).value_counts()\n    print('исходные размеры листа с 6 видами:'); print(_sizes.head(5).to_string())\n    _w, _h = _probe[0]\n    _r, _c = infer_grid(_w, _h)\n    NATIVE_TILE = int(min(_w // _c, _h // _r))\n    R.kv(**{'раскладка': f'{_r}x{_c}', 'нативный размер вида': f'{NATIVE_TILE} px',\n            'используем': f'{CFG.tile} ({100 * (CFG.tile / NATIVE_TILE) ** 2:.0f}% площади кадра)'})\n    if CFG.tile > NATIVE_TILE:\n        R.warn(f'tile {CFG.tile} > нативного — это апскейл, снижаю')\n        CFG.tile = NATIVE_TILE\n\nIMG_TRAIN, MISS_TRAIN = build_image_cache(train_df['item_id'].tolist(), TRAIN_DIR, 'train')\nIMG_TEST,  MISS_TEST  = build_image_cache(test_df['item_id'].tolist(),  TEST_DIR,  'test')"
  },
  {
    "source_cell": 22,
    "source": "import os, gc, math, numpy as np, pandas as pd\nfrom pathlib import Path\nfrom PIL import Image\nfrom scipy import ndimage\nfrom scipy.sparse import coo_matrix\nfrom scipy.sparse.csgraph import connected_components\nfrom scipy.spatial import ConvexHull, QhullError\nfrom concurrent.futures import ProcessPoolExecutor\nfrom concurrent.futures.process import BrokenProcessPool\nimport multiprocessing as mp\n\nFACE_CAP  = 300_000\nVERT_CAP  = 200_000\nHULL_CAP  = 20_000\nCOMP_CAP  = 3_000_000\nWELD_TOL  = 1e-6         # доля диагонали bbox\nFEAT_VER  = 'v4selfint'  # версия набора фич: входит в имя кэша. Поднята с v3weld, т.к.\n                         # добавлены §4.5.1-4.5.3 (самопересечения/качество треугольников/\n                         # non-manifold вершины) — старый parquet без них не подхватится молча\n# Было min(4, ...) — на машине с 8-12 ядрами это втрое резало самый дорогой шаг.\nN_WORKERS = max(1, (os.cpu_count() or 4) - 1)\nCHUNK     = 512          # чаще чекпоинт: при обрыве теряется меньше\n\ndef _pack21(q):\n    \"\"\"Три целых из [0, 2**21) -> один int64. Точно, без коллизий.\"\"\"\n    return (q[:, 0] << 42) | (q[:, 1] << 21) | q[:, 2]\n\ndef _hash3(q):\n    q = q.astype(np.int64, copy=False)\n    return (q[:, 0] * 73856093) ^ (q[:, 1] * 19349663) ^ (q[:, 2] * 83492791)\n\ndef _stats(x, pref):\n    keys = ['mean', 'std', 'p10', 'p50', 'p90', 'max', 'cv']\n    if len(x) == 0:\n        return {f'{pref}_{k}': 0.0 for k in keys}\n    x = np.asarray(x, dtype=np.float32)\n    m, s = float(x.mean()), float(x.std())\n    q10, q50, q90 = np.percentile(x, [10, 50, 90])\n    return {f'{pref}_mean': m, f'{pref}_std': s, f'{pref}_p10': float(q10),\n            f'{pref}_p50': float(q50), f'{pref}_p90': float(q90),\n            f'{pref}_max': float(x.max()), f'{pref}_cv': float(s / (abs(m) + 1e-9))}\n\ndef _edge_table(Fi, nfw):\n    \"\"\"Рёбра -> (номер группы, число граней на ребро, отсортированные id граней).\"\"\"\n    E = np.concatenate([Fi[:, [0, 1]], Fi[:, [1, 2]], Fi[:, [2, 0]]], 0)\n    E.sort(axis=1)\n    fid = np.tile(np.arange(nfw, dtype=np.int32), 3)\n    order = np.lexsort((E[:, 1], E[:, 0]))\n    Es, fids = E[order], fid[order]\n    new = np.ones(len(Es), dtype=bool)\n    new[1:] = (Es[1:] != Es[:-1]).any(1)\n    grp = np.cumsum(new) - 1\n    counts = np.bincount(grp)\n    return E, Es, fids, grp, counts, new\n\ndef mesh_features(npz_path):\n    f = {'geo_ok': 0.0, 'geo_subsampled': 0.0}\n    try:\n        f['npz_mb'] = os.path.getsize(npz_path) / 1e6\n    except Exception:\n        return f\n    try:\n        with np.load(npz_path, allow_pickle=False) as d:\n            keys = list(d.keys())\n            V = np.asarray(d['vertices' if 'vertices' in keys else keys[0]], dtype=np.float32)\n            Fc = None\n            for k in ('faces', 'triangles', 'f'):\n                if k in keys:\n                    Fc = np.asarray(d[k], dtype=np.int64); break\n            if Fc is None and len(keys) > 1:\n                Fc = np.asarray(d[keys[1]], dtype=np.int64)\n    except Exception:\n        return f\n    if V.ndim != 2 or V.shape[1] != 3 or len(V) == 0:\n        return f\n    if Fc is None or Fc.ndim != 2 or Fc.shape[1] != 3 or len(Fc) == 0:\n        Fc = np.zeros((0, 3), dtype=np.int64)\n    else:\n        Fc = Fc[(Fc >= 0).all(1) & (Fc < len(V)).all(1)]\n\n    f['geo_ok'] = 1.0\n    nv_raw, nf_raw = len(V), len(Fc)\n    vmin, vmax = V.min(0), V.max(0)\n    ext = (vmax - vmin).astype(np.float64)\n    ext_s = np.sort(ext)[::-1]\n    diag = float(np.linalg.norm(ext)) + 1e-12\n    center = ((vmin + vmax) / 2).astype(np.float32)\n    inv_s = np.float32(1.0 / diag)\n    f.update({'n_verts_raw': float(nv_raw), 'n_faces': float(nf_raw),\n              'log_verts': math.log1p(nv_raw), 'log_faces': math.log1p(nf_raw),\n              'ext_max': float(ext_s[0]), 'ext_mid': float(ext_s[1]), 'ext_min': float(ext_s[2]),\n              'ext_ratio_min': float(ext_s[2] / (ext_s[0] + 1e-12)),\n              'ext_ratio_mid': float(ext_s[1] / (ext_s[0] + 1e-12)),\n              'bbox_diag': diag,\n              'bbox_fill': float(nv_raw / (ext.prod() + 1e-12)) if ext.prod() > 0 else 0.0})\n\n    # ================== СВАРКА ВЕРШИН ==================\n    # Экспортёры (glTF/OBJ с раздельными нормалями и UV) дублируют вершины на каждую грань.\n    # Без сварки НИ ОДНО ребро не имеет двух граней: двугранные углы пусты, boundary=1.0,\n    # компонент столько же, сколько треугольников. Вся топология обязана считаться после сварки.\n    step = np.float32(diag * WELD_TOL)\n    q = np.clip(np.rint((V - vmin) / step), 0, (1 << 21) - 1).astype(np.int64)\n    code = _pack21(q)\n    del q\n    ucode, first, invmap = np.unique(code, return_index=True, return_inverse=True)\n    del code, ucode\n    nv = len(first)\n    f['n_verts'] = float(nv)\n    f['weld_ratio'] = float(1.0 - nv / max(nv_raw, 1))          # 0 = уже сварен, ~0.83 = по 3 вершины на грань\n    f['f_per_v'] = nf_raw / max(nv, 1)\n    Vw_full = V[first]\n    del first\n\n    if nf_raw:\n        Fw = invmap[Fc]\n        ok = (Fw[:, 0] != Fw[:, 1]) & (Fw[:, 1] != Fw[:, 2]) & (Fw[:, 0] != Fw[:, 2])\n        f['degen_after_weld'] = float(1.0 - ok.mean())\n        Fw = Fw[ok]\n        hf = _hash3(np.sort(Fw, axis=1))\n        _, uidx = np.unique(hf, return_index=True)\n        f['dup_face_ratio'] = float(1.0 - len(uidx) / max(len(Fw), 1))\n        Fw = Fw[np.sort(uidx)]\n        del hf, uidx, ok\n    else:\n        Fw = np.zeros((0, 3), dtype=np.int64)\n        f['degen_after_weld'] = 0.0; f['dup_face_ratio'] = 0.0\n    del invmap, Fc\n    nf = len(Fw)\n\n    # ---- вершинные статистики (подвыборка) ----\n    Vs = Vw_full[::max(1, int(np.ceil(nv / VERT_CAP)))]\n    if nv > VERT_CAP:\n        f['geo_subsampled'] = 1.0\n    Vs = (Vs - center) * inv_s\n    try:\n        if len(Vs) < 3:\n            raise ValueError('too few vertices')\n        ev = np.clip(np.linalg.eigvalsh(np.cov(Vs.T.astype(np.float64)))[::-1], 1e-16, None)\n        f.update({'pca_1': float(ev[0]), 'pca_2': float(ev[1]), 'pca_3': float(ev[2]),\n                  'pca_flat': float(ev[2] / ev[0]), 'pca_lin': float(ev[1] / ev[0]),\n                  'pca_aniso': float((ev[0] - ev[2]) / ev.sum())})\n    except Exception:\n        f.update({k: 0.0 for k in ['pca_1', 'pca_2', 'pca_3', 'pca_flat', 'pca_lin', 'pca_aniso']})\n\n    # ---- выпуклая оболочка: solidity/сферичность -> simple, open ----\n    f.update({'hull_ok': 0.0, 'solidity': 0.0, 'solidity_signed': 0.0, 'hull_area_ratio': 0.0,\n              'hull_pts_frac': 0.0, 'sphericity': 0.0, 'hull_vol': 0.0, 'hull_area': 0.0})\n    try:\n        Vh = Vs[::max(1, int(np.ceil(len(Vs) / HULL_CAP)))].astype(np.float64)\n        if len(Vh) >= 8:\n            hull = ConvexHull(Vh, qhull_options='QJ')\n            f['hull_ok'] = 1.0\n            f['hull_vol'] = float(hull.volume)\n            f['hull_pts_frac'] = len(hull.vertices) / len(Vh)\n            f['hull_area'] = float(hull.area)\n    except (QhullError, ValueError, MemoryError):\n        pass\n\n    if nf == 0:\n        f['no_faces'] = 1.0\n        f.update({'area_total': 0.0, 'volume': 0.0, 'abs_volume': 0.0, 'vol_ratio': 0.0,\n                  'vol_over_area': 0.0, 'absvol_over_area': 0.0, 'vol_over_bbox': 0.0,\n                  'absvol_over_bbox': 0.0})\n        for pref in ('dihed', 'dihedf', 'farea', 'elen', 'aspect', 'valence', 'defect'):\n            f.update(_stats([], pref))\n        return f\n    f['no_faces'] = 0.0\n\n    # ---- точные площадь/объём по полному сваренному мешу ----\n    total_area, vol6, absvol6 = 0.0, 0.0, 0.0\n    for s0 in range(0, nf, 500_000):\n        Pc = ((Vw_full[Fw[s0:s0 + 500_000]] - center) * inv_s).astype(np.float64)\n        cr_c = np.cross(Pc[:, 1] - Pc[:, 0], Pc[:, 2] - Pc[:, 0])\n        total_area += 0.5 * float(np.linalg.norm(cr_c, axis=1).sum())\n        contrib = (Pc[:, 0] * np.cross(Pc[:, 1], Pc[:, 2])).sum(1)\n        vol6 += float(contrib.sum()); absvol6 += float(np.abs(contrib).sum())\n        del Pc, cr_c, contrib\n    total_area += 1e-12\n    vol = abs(vol6 / 6.0)\n    absvol = absvol6 / 6.0\n    # vol_ratio ~1 у корректно ориентированного меша и ~0 при несогласованной намотке граней\n    # (сама по себе сильная улика для artifacts/noisy). Объём для solidity берём устойчивый.\n    f.update({'area_total': total_area, 'volume': vol, 'abs_volume': absvol,\n              'vol_ratio': vol / (absvol + 1e-12),\n              'vol_over_area': vol / (total_area ** 1.5 + 1e-12),\n              'absvol_over_area': absvol / (total_area ** 1.5 + 1e-12),\n              'vol_over_bbox': vol / (float(np.prod(ext * inv_s)) + 1e-12),\n              'absvol_over_bbox': absvol / (float(np.prod(ext * inv_s)) + 1e-12)})\n    if f['hull_ok']:\n        f['solidity'] = absvol / (f['hull_vol'] + 1e-12)\n        f['solidity_signed'] = vol / (f['hull_vol'] + 1e-12)\n        f['hull_area_ratio'] = total_area / (f.get('hull_area', 0.0) + 1e-12)\n        f['sphericity'] = (math.pi ** (1 / 3)) * (6 * absvol) ** (2 / 3) / (total_area + 1e-12)\n\n    # ---- компоненты связности на полном СВАРЕННОМ меше ----\n    f.update({'n_comp': 1.0, 'n_comp_big': 1.0, 'comp_top1': 1.0, 'comp_top2': 0.0,\n              'comp_entropy': 0.0, 'bbox_iou_max': 0.0, 'bbox_iou_mean': 0.0,\n              'bbox_iou_frac_pos': 0.0, 'comp_sep_max': 0.0, 'comp_sep_mean': 0.0,\n              'inside_frac_max': 0.0, 'comp_exact': 1.0})\n    if nf <= COMP_CAP:\n        try:\n            Ef = np.concatenate([Fw[:, [0, 1]], Fw[:, [1, 2]], Fw[:, [2, 0]]], 0).astype(np.int32)\n            A = coo_matrix((np.ones(len(Ef), dtype=np.int8), (Ef[:, 0], Ef[:, 1])), shape=(nv, nv))\n            del Ef\n            ncomp, lab = connected_components(A, directed=False)\n            del A\n            used = np.unique(Fw)\n            sizes_all = np.bincount(lab[used], minlength=ncomp).astype(np.float64)\n            sizes = np.sort(sizes_all[sizes_all > 0])[::-1]\n            tot = sizes.sum()\n            big_idx = np.where(sizes_all > 0.005 * tot)[0]\n            f.update({'n_comp': float(len(sizes)), 'n_comp_big': float(len(big_idx)),\n                      'comp_top1': float(sizes[0] / tot),\n                      'comp_top2': float(sizes[1] / tot) if len(sizes) > 1 else 0.0,\n                      'comp_entropy': float(-((sizes / tot) * np.log(sizes / tot + 1e-12)).sum())})\n            if len(big_idx) > 1:\n                top = big_idx[np.argsort(-sizes_all[big_idx])][:8]\n                pts_l, boxes = [], []\n                for c in top:\n                    ic = used[lab[used] == c]\n                    if len(ic) > 20_000:\n                        ic = ic[::len(ic) // 20_000 + 1]\n                    p = (Vw_full[ic] - center) * inv_s\n                    pts_l.append(p); boxes.append((p.min(0), p.max(0)))\n                ious, seps, insides = [], [], []\n                for i in range(len(boxes)):\n                    for j in range(len(boxes)):\n                        if i == j:\n                            continue\n                        lo, hi = boxes[j]\n                        insides.append(float(((pts_l[i] >= lo) & (pts_l[i] <= hi)).all(1).mean()))\n                        if j <= i:\n                            continue\n                        lo2 = np.maximum(boxes[i][0], boxes[j][0])\n                        hi2 = np.minimum(boxes[i][1], boxes[j][1])\n                        it = float(np.prod(np.clip(hi2 - lo2, 0, None)))\n                        vi = float(np.prod(np.clip(boxes[i][1] - boxes[i][0], 1e-6, None)))\n                        vj = float(np.prod(np.clip(boxes[j][1] - boxes[j][0], 1e-6, None)))\n                        ious.append(it / (vi + vj - it + 1e-12))\n                        seps.append(float(np.linalg.norm(\n                            (boxes[i][0] + boxes[i][1]) / 2 - (boxes[j][0] + boxes[j][1]) / 2)))\n                f.update({'bbox_iou_max': float(max(ious)), 'bbox_iou_mean': float(np.mean(ious)),\n                          'bbox_iou_frac_pos': float(np.mean(np.asarray(ious) > 1e-6)),\n                          'comp_sep_max': float(max(seps)), 'comp_sep_mean': float(np.mean(seps)),\n                          'inside_frac_max': float(max(insides))})\n            del lab, sizes_all, sizes, used\n        except Exception:\n            f['comp_exact'] = 0.0\n    else:\n        f['comp_exact'] = 0.0\n\n    # ---- пространственный патч для рёберной топологии ----\n    if nf > FACE_CAP:\n        f['geo_subsampled'] = 1.0\n        anchor = (Vw_full[Fw[:, 0]] - vmin) / (ext.astype(np.float32) + 1e-9)\n        Fp, g = None, 1\n        while g < 40:\n            g += 1\n            cid = np.minimum((anchor * g).astype(np.int32), g - 1)\n            key = (cid[:, 0] * g + cid[:, 1]) * g + cid[:, 2]\n            cnt = np.bincount(key, minlength=g ** 3)\n            best = int(np.argmax(cnt))\n            if cnt[best] <= FACE_CAP:\n                Fp = Fw[key == best]; break\n        if Fp is None or len(Fp) < 1000:\n            Fp = Fw[:FACE_CAP]\n        f['patch_grid'] = float(g)\n        del anchor\n    else:\n        Fp = Fw\n        f['patch_grid'] = 1.0\n    f['patch_face_frac'] = len(Fp) / max(nf, 1)\n    del Fw\n    nfw = len(Fp)\n\n    uvp, Fi = np.unique(Fp, return_inverse=True)\n    Fi = Fi.reshape(nfw, 3).astype(np.int32)\n    Vp = ((Vw_full[uvp] - center) * inv_s).astype(np.float32)\n    nvp = len(uvp)\n    del uvp, Fp, Vw_full, V\n\n    P = Vp[Fi]\n    cr = np.cross(P[:, 1] - P[:, 0], P[:, 2] - P[:, 0])\n    area2 = np.linalg.norm(cr, axis=1)\n    area = 0.5 * area2\n    nrm = cr / (area2[:, None] + 1e-12)\n    del cr\n    f.update(_stats(area / (area.mean() + 1e-12), 'farea'))\n    f['degenerate_ratio'] = float((area < 1e-10).mean())\n\n    L = np.stack([np.linalg.norm(P[:, 1] - P[:, 0], axis=1),\n                  np.linalg.norm(P[:, 2] - P[:, 1], axis=1),\n                  np.linalg.norm(P[:, 0] - P[:, 2], axis=1)], 1)\n    f.update(_stats(L.ravel() / (L.mean() + 1e-12), 'elen'))\n    aspect = L.max(1) / (L.min(1) + 1e-12)\n    f.update(_stats(np.log1p(aspect), 'aspect'))\n    f['sliver_ratio'] = float((aspect > 20).mean())\n\n    # ---- §4.5.2 качество треугольников: худший угол + нормализованная форма ----\n    # min-угол — классический индикатор \"плохого\" элемента (aspect ratio его не всегда\n    # ловит: длинный тупоугольный треугольник может иметь умеренный aspect, но острый\n    # min-угол). tri_quality = 4*sqrt(3)*area/(l0^2+l1^2+l2^2) -> 1.0 для равностороннего,\n    # ->0 для вырожденного; оба считаются векторно из уже посчитанных L и area, без\n    # дополнительного прохода по мешу.\n    L0, L1, L2 = L[:, 0], L[:, 1], L[:, 2]\n    with np.errstate(divide='ignore', invalid='ignore'):\n        cA = np.clip((L1 ** 2 + L2 ** 2 - L0 ** 2) / (2 * L1 * L2 + 1e-12), -1, 1)\n        cB = np.clip((L0 ** 2 + L2 ** 2 - L1 ** 2) / (2 * L0 * L2 + 1e-12), -1, 1)\n        cC = np.clip((L0 ** 2 + L1 ** 2 - L2 ** 2) / (2 * L0 * L1 + 1e-12), -1, 1)\n    tri_ang = np.degrees(np.arccos(np.stack([cA, cB, cC], 1)))\n    min_ang, max_ang = tri_ang.min(1), tri_ang.max(1)\n    f.update(_stats(min_ang, 'minang')); f.update(_stats(max_ang, 'maxang'))\n    f['sliver_angle_ratio'] = float((min_ang < 5).mean())\n    tri_quality = np.clip((4 * math.sqrt(3) * area) / (L0 ** 2 + L1 ** 2 + L2 ** 2 + 1e-12), 0, 1)\n    f.update(_stats(tri_quality, 'triq'))\n    del L0, L1, L2, cA, cB, cC, tri_ang, min_ang, max_ang, tri_quality\n\n    for qd in (1, 2):\n        f[f'uniq_normals_q{qd}'] = len(np.unique(_hash3(np.rint(nrm * 10 ** qd)))) / nfw\n    # концентрация нормалей: доля площади в 6 крупнейших кластерах -> примитивы\n    hn = _hash3(np.rint(nrm * 20))\n    _, ninv = np.unique(hn, return_inverse=True)\n    mass = np.bincount(ninv, weights=area)\n    mass = np.sort(mass)[::-1]\n    f['normal_top6_mass'] = float(mass[:6].sum() / (mass.sum() + 1e-12))\n    f['normal_top1_mass'] = float(mass[0] / (mass.sum() + 1e-12))\n    del hn, ninv, mass\n\n    # ---- §4.5.1 самопересечения: близкие по центроиду грани без общей вершины ----\n    # intersection сейчас ловится только CNN с рендеров; хуже того, его confusion\n    # портит quality (10 из 16 баллов), даже когда сам класс почти ничего не весит\n    # (см. §14, \"потолок по классам\": intersection+scale = 0.18 балла при идеальном\n    # предсказании) — поэтому цель этих фич не \"поднять F1(intersection)\" сама по себе,\n    # а снизить количество ложных срабатываний, которые размывают quality.\n    # cKDTree.query_pairs даёт кандидатов почти даром; точный тест — 6 рёберно-плоскостных\n    # проверок на кандидата (Möller-style), полностью векторизован по всем парам разом.\n    f.update({'selfint_ok': 0.0, 'selfint_pair_ratio': 0.0, 'selfint_face_ratio': 0.0,\n              'selfint_any': 0.0, 'selfint_capped': 0.0})\n    # 40k вместо 120k: самопересечения нужны для intersection, который стоит\n    # 0.115 балла ДАЖЕ при идеальном предсказании — точность там не окупает времени.\n    SELFINT_FACE_CAP = 40_000\n    SELFINT_PAIR_CAP = 400_000\n    if 0 < nfw <= SELFINT_FACE_CAP:\n        try:\n            from scipy.spatial import cKDTree\n            cen = Vp[Fi].mean(1)\n            elen_med = float(np.median(L)) if len(L) else 1e-6\n            r = max(elen_med * 2.5, 1e-6)\n            pairs = cKDTree(cen).query_pairs(r=r, output_type='ndarray')\n            if len(pairs):\n                share = (Fi[pairs[:, 0], :, None] == Fi[pairs[:, 1], None, :]).any((1, 2))\n                pairs = pairs[~share]\n            if len(pairs) > SELFINT_PAIR_CAP:\n                sel = np.random.default_rng(0).choice(len(pairs), SELFINT_PAIR_CAP, replace=False)\n                pairs = pairs[sel]\n                f['selfint_capped'] = 1.0\n            f['selfint_ok'] = 1.0\n            if len(pairs):\n                ii, jj = pairs[:, 0], pairs[:, 1]\n                PA, PB = Vp[Fi[ii]], Vp[Fi[jj]]\n                nA, nB = nrm[ii], nrm[jj]\n\n                def _edge_hits(a, b, p0, p1, p2, n):\n                    d = np.einsum('ij,ij->i', n, p0)\n                    ta = np.einsum('ij,ij->i', n, a) - d\n                    tb = np.einsum('ij,ij->i', n, b) - d\n                    denom = ta - tb\n                    safe = np.abs(denom) > 1e-12\n                    t = np.clip(np.divide(ta, denom, out=np.zeros_like(ta), where=safe), 0.0, 1.0)\n                    pt = a + t[:, None] * (b - a)\n                    e0 = np.cross(p1 - p0, pt - p0); e1 = np.cross(p2 - p1, pt - p1)\n                    e2 = np.cross(p0 - p2, pt - p2)\n                    s0 = np.einsum('ij,ij->i', e0, n); s1 = np.einsum('ij,ij->i', e1, n)\n                    s2 = np.einsum('ij,ij->i', e2, n)\n                    inside = (((s0 >= -1e-9) & (s1 >= -1e-9) & (s2 >= -1e-9)) |\n                              ((s0 <= 1e-9) & (s1 <= 1e-9) & (s2 <= 1e-9)))\n                    return (ta * tb <= 0) & safe & inside\n\n                hit = np.zeros(len(pairs), dtype=bool)\n                for e in range(3):\n                    hit |= _edge_hits(PA[:, e], PA[:, (e + 1) % 3], PB[:, 0], PB[:, 1], PB[:, 2], nB)\n                for e in range(3):\n                    hit |= _edge_hits(PB[:, e], PB[:, (e + 1) % 3], PA[:, 0], PA[:, 1], PA[:, 2], nA)\n                n_hit = int(hit.sum())\n                f['selfint_pair_ratio'] = float(n_hit / max(nfw, 1))\n                f['selfint_any'] = float(n_hit > 0)\n                if n_hit:\n                    f['selfint_face_ratio'] = float(\n                        len(np.unique(np.concatenate([ii[hit], jj[hit]]))) / max(nfw, 1))\n                del PA, PB, nA, nB, hit\n            del cen, pairs\n        except Exception:\n            pass\n    else:\n        f['selfint_capped'] = 1.0\n\n    # ---- рёбра/углы ПОСЛЕ сварки + диагностика \"как было бы БЕЗ сварки\" ----\n    E, Es, fids, grp, counts, _ = _edge_table(Fi, nfw)\n    n_edges = len(counts)\n    f.update({'n_edges': float(n_edges),\n              'boundary_edge_ratio': float((counts == 1).mean()),\n              'nonmanifold_edge_ratio': float((counts >= 3).mean()),\n              'euler_norm': float((nvp - n_edges + nfw) / max(nfw, 1)),\n              'euler_char': float(nvp - n_edges + nfw),\n              'watertight': float((counts == 1).sum() == 0 and (counts >= 3).sum() == 0),\n              'manifold_edge_ratio': float((counts == 2).mean())})\n\n    # число граничных петель -> сколько «дыр» в поверхности (open)\n    bnd = np.where(counts == 1)[0]\n    f['nonmanifold_vert_ratio'] = 0.0\n    if len(bnd):\n        st = np.searchsorted(grp, bnd)\n        be = Es[st]\n        try:\n            Ab = coo_matrix((np.ones(len(be), dtype=np.int8), (be[:, 0], be[:, 1])), shape=(nvp, nvp))\n            nb_comp, lb = connected_components(Ab, directed=False)\n            used_b = np.unique(be)\n            f['n_boundary_loops'] = float(len(np.unique(lb[used_b])))\n            f['boundary_vert_frac'] = float(len(used_b) / nvp)\n            del Ab, lb, used_b\n        except Exception:\n            f['n_boundary_loops'] = 0.0; f['boundary_vert_frac'] = 0.0\n        # ---- §4.5.3 non-manifold вершины: степень >2 в графе граничных рёбер ----\n        # у нормальной граничной петли каждая вершина имеет ровно 2 инцидентных\n        # граничных ребра; степень >2 -> вершина, где встречаются несколько \"дыр\"\n        # или веток границы (типичная примета artifacts/open после плохого шва).\n        # Это дополняет уже имеющиеся nonmanifold_edge_ratio/watertight на уровне\n        # вершин, а не рёбер, и считается одним bincount без дополнительного графа.\n        deg_b = np.bincount(be.ravel(), minlength=nvp)\n        f['nonmanifold_vert_ratio'] = float((deg_b > 2).sum() / max(nvp, 1))\n        del be, st, deg_b\n    else:\n        f['n_boundary_loops'] = 0.0; f['boundary_vert_frac'] = 0.0\n\n    two = np.where(counts == 2)[0]\n    if len(two):\n        st = np.searchsorted(grp, two)\n        fa, fb = fids[st], fids[st + 1]\n        cosang = np.clip((nrm[fa] * nrm[fb]).sum(1), -1, 1)\n        ang = np.degrees(np.arccos(cosang))\n        angf = np.degrees(np.arccos(np.abs(cosang)))\n        f.update(_stats(ang, 'dihed')); f.update(_stats(angf, 'dihedf'))\n        for t in (10, 30, 60, 90):\n            f[f'dihed_gt{t}'] = float((ang > t).mean())\n        for t in (5, 15, 30, 60):\n            f[f'dihedf_gt{t}'] = float((angf > t).mean())\n        f['flip_ratio'] = float((cosang < 0).mean())\n        hist = np.bincount((angf / 5).astype(np.int32).clip(0, 17), minlength=18).astype(np.float64)\n        p = hist / max(hist.sum(), 1)\n        f['dihed_entropy'] = float(-(p[p > 0] * np.log(p[p > 0])).sum())\n        f['smoothness'] = float(angf.mean() * math.sqrt(nfw) / 100.0)\n        # взвешенный по длине ребра угол: устойчивее к мелким треугольникам\n        el = np.linalg.norm(Vp[Es[st][:, 0]] - Vp[Es[st][:, 1]], axis=1)\n        f['dihedf_areaw'] = float((angf * el).sum() / (el.sum() + 1e-12))\n        del fa, fb, cosang, ang, angf, el\n    else:\n        f.update(_stats([], 'dihed')); f.update(_stats([], 'dihedf'))\n        for t in (10, 30, 60, 90):\n            f[f'dihed_gt{t}'] = 0.0\n        for t in (5, 15, 30, 60):\n            f[f'dihedf_gt{t}'] = 0.0\n        f.update({'flip_ratio': 0.0, 'dihed_entropy': 0.0, 'smoothness': 0.0, 'dihedf_areaw': 0.0})\n\n    # ---- валентность вершин и угловой дефект (гауссова кривизна) ----\n    val = np.bincount(Fi.ravel(), minlength=nvp).astype(np.float32)\n    f.update(_stats(val, 'valence'))\n    f['valence_le3'] = float((val <= 3).mean())\n    f['valence_ge8'] = float((val >= 8).mean())\n    e01 = P[:, 1] - P[:, 0]; e12 = P[:, 2] - P[:, 1]; e20 = P[:, 0] - P[:, 2]\n    def _ang(u, v):\n        cu = (u * v).sum(1) / (np.linalg.norm(u, axis=1) * np.linalg.norm(v, axis=1) + 1e-12)\n        return np.arccos(np.clip(cu, -1, 1))\n    a0 = _ang(e01, -e20); a1 = _ang(e12, -e01); a2 = _ang(e20, -e12)\n    defect = np.full(nvp, 2 * np.pi, dtype=np.float64)\n    np.subtract.at(defect, Fi[:, 0], a0)\n    np.subtract.at(defect, Fi[:, 1], a1)\n    np.subtract.at(defect, Fi[:, 2], a2)\n    interior = val > 0\n    f.update(_stats(np.abs(defect[interior]), 'defect'))\n    f['defect_sum'] = float(defect[interior].sum() / (2 * np.pi))     # ~ эйлерова характеристика\n    f['defect_gt05'] = float((np.abs(defect[interior]) > 0.5).mean())\n    del defect, val, a0, a1, a2, e01, e12, e20\n\n    del E, Es, fids, grp, counts, two, nrm, P, Vp, Fi, L, aspect, area, area2\n    return f"
  },
  {
    "source_cell": 24,
    "source": "def image_features(png_path, tile):\n    \"\"\"Силуэтные признаки кадра. Зависят от tile (мозаика режется по нему),\n    поэтому кэшируются отдельно от геометрии.\"\"\"\n    f = {'img_ok': 0.0}\n    try:\n        img = Image.open(png_path).convert('L')\n    except Exception:\n        return f\n    a = np.asarray(img, dtype=np.float32) / 255.0\n    t = tile\n    f['img_ok'] = 1.0\n    covs, comps, edens, stds, bws, bhs = [], [], [], [], [], []\n    for k in range(6):\n        v = a[(k // 3) * t:(k // 3 + 1) * t, (k % 3) * t:(k % 3 + 1) * t]\n        bg = np.median(np.concatenate([v[0], v[-1], v[:, 0], v[:, -1]]))\n        mask = np.abs(v - bg) > 0.06\n        cov = float(mask.mean()); covs.append(cov)\n        if mask.any():\n            ys, xs = np.where(mask)\n            bws.append((xs.max() - xs.min() + 1) / t)\n            bhs.append((ys.max() - ys.min() + 1) / t)\n            lab, ncc = ndimage.label(mask)\n            if ncc:\n                sz = np.bincount(lab.ravel())[1:]\n                comps.append(float((sz > 0.01 * sz.sum()).sum()))\n            else:\n                comps.append(0.0)\n        else:\n            bws.append(0.0); bhs.append(0.0); comps.append(0.0)\n        edens.append(float(np.abs(np.diff(v, axis=1)).mean() + np.abs(np.diff(v, axis=0)).mean()))\n        stds.append(float(v.std()))\n\n    def agg(vals, pref):\n        v = np.asarray(vals, dtype=np.float64)\n        return {f'{pref}_mean': v.mean(), f'{pref}_min': v.min(), f'{pref}_max': v.max(),\n                f'{pref}_std': v.std(), f'{pref}_range': v.max() - v.min()}\n\n    f.update(agg(covs, 'cov')); f.update(agg(comps, 'ncc')); f.update(agg(edens, 'edge'))\n    f.update(agg(stds, 'pxstd')); f.update(agg(bws, 'bw')); f.update(agg(bhs, 'bh'))\n    f['cov_empty_views'] = float((np.asarray(covs) < 0.01).sum())\n    f['bbox_fill_view'] = float(np.mean(np.asarray(covs) /\n                                        (np.asarray(bws) * np.asarray(bhs) + 1e-6)))\n    return f\n\n\n# =====================================================================================\n#  Раздельные кэши.\n#\n#  В прошлой версии геометрия и силуэты считались одним проходом и кэшировались под\n#  именем с `tile`. Из-за этого второй прогон с другим разрешением пересчитывал бы\n#  ГЕОМЕТРИЮ — а это самая дорогая часть (сварка вершин, топология, спектр лапласиана,\n#  самопересечения на 9633 мешах). Между тем геометрия от разрешения рендера не зависит\n#  вообще. Здесь она считается ОДИН раз и переиспользуется обоими прогонами; по `tile`\n#  кэшируются только дешёвые силуэтные признаки.\n# =====================================================================================\n\ndef _mesh_one(args):\n    iid, npz = args\n    d = {'item_id': iid}\n    try:\n        d.update(mesh_features(npz))\n    except Exception:\n        d['geo_ok'] = 0.0\n    return d\n\n\ndef _img_one(args):\n    iid, png, tile = args\n    d = {'item_id': iid}\n    try:\n        d.update(image_features(png, tile))\n    except Exception:\n        d['img_ok'] = 0.0\n    return d\n\n\ndef _chunked_build(jobs, worker, final, ckpt, desc):\n    \"\"\"Общий каркас: чанками, с чекпоинтом, с резюмированием после обрыва.\"\"\"\n    if final.exists():\n        df = pd.read_parquet(final)\n        R.ok(f'{desc}: из кэша {df.shape}')\n        return df\n    done = pd.read_parquet(ckpt) if ckpt.exists() else pd.DataFrame(columns=['item_id'])\n    have = set(done['item_id'].astype(str)) if len(done) else set()\n    todo = [j for j in jobs if j[0] not in have]\n    R.kv(**{desc: f'готово {len(have)}, осталось {len(todo)}, воркеров {N_WORKERS}'})\n    rows = [done] if len(done) else []\n    for s in range(0, len(todo), CHUNK):\n        part = todo[s:s + CHUNK]\n        try:\n            ctx = mp.get_context('fork')\n        except ValueError:\n            ctx = None\n        try:\n            with ProcessPoolExecutor(max_workers=N_WORKERS, mp_context=ctx) as ex:\n                got = list(tqdm(ex.map(worker, part, chunksize=4), total=len(part),\n                                desc=f'{desc} {s + len(part)}/{len(todo)}', leave=False))\n        except (BrokenProcessPool, OSError, MemoryError) as e:\n            R.warn(f'пул упал ({type(e).__name__}), чанк считается последовательно')\n            got = [worker(j) for j in tqdm(part, desc='serial', leave=False)]\n        rows.append(pd.DataFrame(got))\n        _atomic(ckpt, lambda p: pd.concat(rows, ignore_index=True).to_parquet(p, index=False))\n        gc.collect()\n    df = pd.concat(rows, ignore_index=True).drop_duplicates('item_id') if rows else \\\n        pd.DataFrame({'item_id': [j[0] for j in jobs]})\n    _atomic(final, lambda p: df.to_parquet(p, index=False))\n    if ckpt.exists():\n        ckpt.unlink()\n    return df\n\n\ndef build_mesh_features(ids, npz_dir, split):\n    \"\"\"Кэш НЕ содержит tile: геометрия одна на все прогоны.\"\"\"\n    jobs = [(str(i), str(Path(npz_dir) / f'{i}.npz')) for i in ids]\n    df = _chunked_build(jobs, _mesh_one,\n                        CFG.cache / f'mesh_{split}_{FEAT_VER}.parquet',\n                        CFG.cache / f'mesh_{split}_{FEAT_VER}_partial.parquet',\n                        f'геометрия {split}')\n    return df.set_index('item_id').reindex([str(i) for i in ids]).reset_index().fillna(0.0)\n\n\ndef build_image_features(ids, img_dir, split, tile):\n    jobs = [(str(i), str(Path(img_dir) / f'{i}.png'), tile) for i in ids]\n    df = _chunked_build(jobs, _img_one,\n                        CFG.cache / f'imgf_{split}_{tile}.parquet',\n                        CFG.cache / f'imgf_{split}_{tile}_partial.parquet',\n                        f'силуэты {split} @{tile}')\n    return df.set_index('item_id').reindex([str(i) for i in ids]).reset_index().fillna(0.0)\n\n\nR.section('Геометрические признаки', f'версия {FEAT_VER}, считаются один раз на оба прогона')\nmesh_tr = build_mesh_features(train_df['item_id'].tolist(), TRAIN_DIR, 'train')\nmesh_te = build_mesh_features(test_df['item_id'].tolist(), TEST_DIR, 'test')\nR.kv(**{'геометрия train': mesh_tr.shape, 'геометрия test': mesh_te.shape})\n"
  },
  {
    "source_cell": 26,
    "source": "CLIP_PROMPTS = {\n    'abs_text':     ['3D text lettering', 'a sign with written words', 'an extruded logo'],\n    'abs_chart':    ['a bar chart', 'a pie chart', 'a graph plot', 'a diagram', 'a flowchart',\n                     'a table with data', 'an infographic'],\n    'abs_support':  ['3D printing support structure', 'scaffolding lattice'],\n    'abs_voxel':    ['minecraft voxel blocks', 'blocky pixelated cubes'],\n    'obj_single':   ['a 3D model of a single object', 'a product render on white background'],\n    'obj_semantic': ['a chair', 'a car', 'a building', 'a plant', 'a character figure',\n                     'a tool', 'furniture'],\n    'p_lowpoly':    ['a low-poly 3D model with visible flat facets'],\n    'p_noisy':      ['a noisy 3D scan with rough surface', 'a point cloud scan'],\n    'p_broken':     ['a broken mesh with holes and artifacts'],\n    'p_hollow':     ['a hollow thin shell', 'a flat plane'],\n    'p_multi':      ['several separate objects scattered apart', 'a collection of many objects'],\n    'p_small':      ['a tiny object in the middle of a large empty frame'],\n    'p_smooth':     ['a smooth clean 3D render of one object'],\n}\n\n\nclass _ViewsDataset(torch.utils.data.Dataset):\n    \"\"\"Читает ИСХОДНЫЙ рендер и режет на 6 квадратов нужного размера.\n\n    Источник — оригинальный PNG, а не кэш под конкретный tile, поэтому CLIP-признаки\n    не зависят от конфигурации прогона и считаются один раз на оба.\n    \"\"\"\n    def __init__(self, ids, src_dir, size):\n        self.ids, self.dir, self.size = [str(i) for i in ids], Path(src_dir), size\n\n    def __len__(self):\n        return len(self.ids)\n\n    def __getitem__(self, i):\n        p = self.dir / f'{self.ids[i]}.png'\n        s = self.size\n        if p.exists():\n            vs = [np.asarray(v.resize((s, s), Image.BILINEAR), dtype=np.uint8)\n                  for v in split_views(Image.open(p).convert('RGB'))]\n        else:\n            vs = [np.full((s, s, 3), 255, np.uint8)] * 6\n        return torch.from_numpy(np.stack(vs)), self.ids[i]\n\n\ndef compute_clip_features(ids, src_dir, split):\n    \"\"\"Узкое место прошлого прогона: декодирование PNG шло в один поток в теле цикла,\n    и 8964 объекта заняли 83 минуты при простаивающем GPU. Здесь чтение вынесено\n    в DataLoader с num_workers — то же самое считается в разы быстрее.\"\"\"\n    import open_clip\n    name, pretrained = CFG.clip_model\n    model, _, _ = open_clip.create_model_and_transforms(name, pretrained=pretrained)\n    tokenizer = open_clip.get_tokenizer(name)\n    model = model.to(CFG.device).eval()\n\n    groups = list(CLIP_PROMPTS)\n    flat, owner = [], []\n    for g in groups:\n        flat += CLIP_PROMPTS[g]; owner += [g] * len(CLIP_PROMPTS[g])\n    with torch.no_grad():\n        tf = model.encode_text(tokenizer(flat).to(CFG.device)).float()\n        tf = tf / tf.norm(dim=-1, keepdim=True)\n    owner = np.array(owner)\n\n    size = model.visual.image_size\n    size = size[0] if isinstance(size, (tuple, list)) else int(size)\n    mean = torch.tensor([0.48145466, 0.4578275, 0.40821073], device=CFG.device).view(1, 3, 1, 1)\n    std = torch.tensor([0.26862954, 0.26130258, 0.27577711], device=CFG.device).view(1, 3, 1, 1)\n\n    ds = _ViewsDataset(ids, src_dir, size)\n    dl = torch.utils.data.DataLoader(ds, batch_size=32, shuffle=False,\n                                     num_workers=CFG.num_workers, pin_memory=True)\n    rows = []\n    for x, iids in tqdm(dl, desc=f'CLIP {split}'):\n        x = x.to(CFG.device, non_blocking=True)\n        B = x.shape[0]\n        x = x.permute(0, 1, 4, 2, 3).reshape(B * 6, 3, size, size).float() / 255.0\n        x = (x - mean) / std\n        with torch.no_grad():\n            im = model.encode_image(x).float()\n        im = im / im.norm(dim=-1, keepdim=True)\n        sim = (im @ tf.T).view(B, 6, -1).cpu().numpy()\n        gsim = np.stack([sim[:, :, owner == g].max(-1) for g in groups], -1)\n        feat = {}\n        for j, g in enumerate(groups):\n            v = gsim[:, :, j]\n            feat[f'clip_{g}_mean'] = v.mean(1)\n            feat[f'clip_{g}_max'] = v.max(1)\n            feat[f'clip_{g}_std'] = v.std(1)\n        abs_max = np.stack([feat[f'clip_{g}_max'] for g in groups if g.startswith('abs_')], 1).max(1)\n        obj_max = np.stack([feat[f'clip_{g}_max'] for g in groups if g.startswith('obj_')], 1).max(1)\n        def_max = np.stack([feat[f'clip_{g}_max'] for g in groups\n                            if g.startswith('p_') and g != 'p_smooth'], 1).max(1)\n        feat['clip_abstract_margin'] = abs_max - obj_max\n        feat['clip_abstract_prob'] = 1 / (1 + np.exp(-100 * (abs_max - obj_max)))\n        feat['clip_defect_margin'] = def_max - feat['clip_p_smooth_max']\n        rows.append(pd.DataFrame(feat, index=list(iids)))\n    del model\n    gc.collect(); torch.cuda.empty_cache()\n    return pd.concat(rows).rename_axis('item_id').reset_index()\n\n\nR.section('CLIP zero-shot', 'считается один раз, от разрешения прогона не зависит')\nif CFG.use_clip:\n    clip_tr = cached('clip_train_orig',\n                     lambda: compute_clip_features(train_df['item_id'].tolist(), TRAIN_DIR, 'train'))\n    clip_te = cached('clip_test_orig',\n                     lambda: compute_clip_features(test_df['item_id'].tolist(), TEST_DIR, 'test'))\n    R.kv(**{'CLIP-признаков': clip_tr.shape[1] - 1})\n    _m = train_df.merge(clip_tr, on='item_id', how='left').fillna(0)\n    cc = pd.Series({c: np.corrcoef(_m[c], _m['abstract'])[0, 1]\n                    for c in clip_tr.columns if c != 'item_id'}).sort_values(key=abs, ascending=False)\n    print('  корреляция с abstract (топ-5):')\n    print('  ' + cc.head(5).round(3).to_string().replace('\\n', '\\n  '))\nelse:\n    clip_tr = clip_te = None\n    R.warn('CLIP выключен')\n\n\n# =============================== сборка таблицы признаков ===============================\nFEAT_COLS = None          # фиксируется на первом прогоне и переиспользуется вторым\n\n\ndef assemble_features(img_train, img_test, tile):\n    \"\"\"Геометрия (одна на все прогоны) + силуэты под этот tile + CLIP.\"\"\"\n    global FEAT_COLS\n    ftr = mesh_tr.merge(build_image_features(train_df['item_id'].tolist(), img_train, 'train', tile),\n                        on='item_id', how='left')\n    fte = mesh_te.merge(build_image_features(test_df['item_id'].tolist(), img_test, 'test', tile),\n                        on='item_id', how='left')\n    if clip_tr is not None:\n        ftr = ftr.merge(clip_tr, on='item_id', how='left')\n        fte = fte.merge(clip_te, on='item_id', how='left')\n\n    cols = [c for c in ftr.columns if c != 'item_id' and c in fte.columns]\n    ftr[cols] = ftr[cols].replace([np.inf, -np.inf], 0).fillna(0).astype(np.float32)\n    fte[cols] = fte[cols].replace([np.inf, -np.inf], 0).fillna(0).astype(np.float32)\n\n    if FEAT_COLS is None:\n        # Список фич фиксируется ОДИН раз. Иначе у второго прогона размерность входа\n        # гео-ветки окажется другой, и общий стэкинг поверх обоих развалится.\n        import hashlib\n        keep, seen, const, dup = [], {}, [], []\n        for c in cols:\n            v = np.nan_to_num(ftr[c].values.astype(np.float64))\n            if v.std() == 0:\n                const.append(c); continue\n            key = hashlib.md5(np.ascontiguousarray(np.round(v, 9)).tobytes()).hexdigest()\n            if key in seen:\n                dup.append(c); continue\n            seen[key] = c; keep.append(c)\n        FEAT_COLS = cached(f'feat_cols_{FEAT_VER}', lambda: keep)\n        R.kv(**{'признаков всего': len(cols), 'после дедупликации': len(FEAT_COLS),\n                'спектральных': sum(c.startswith('spec') for c in FEAT_COLS),\n                'CLIP': sum(c.startswith('clip') for c in FEAT_COLS)})\n    for df_ in (ftr, fte):\n        for c in FEAT_COLS:\n            if c not in df_.columns:\n                df_[c] = 0.0\n    return ftr, fte\n"
  },
  {
    "source_cell": 32,
    "source": "from iterstrat.ml_stratifiers import MultilabelStratifiedKFold\n\nmskf = MultilabelStratifiedKFold(n_splits=CFG.n_folds, shuffle=True, random_state=CFG.seed)\ntrain_df['fold'] = -1\nfor k, (_, vi) in enumerate(mskf.split(train_df, train_df[CFG.target_cols].values)):\n    train_df.loc[train_df.index[vi], 'fold'] = k\ndisplay(train_df.groupby('fold')[CFG.target_cols].mean().round(4))"
  },
  {
    "source_cell": 34,
    "source": "from torch.utils.data import Dataset, DataLoader\n\nMEAN = np.array([0.485, 0.456, 0.406], dtype=np.float32)\nSTD  = np.array([0.229, 0.224, 0.225], dtype=np.float32)\n\n\ndef load_views(path, tile):\n    a = np.asarray(Image.open(path).convert('RGB'), dtype=np.uint8)\n    t = tile\n    return np.stack([a[(k // 3) * t:(k // 3 + 1) * t, (k % 3) * t:(k % 3 + 1) * t]\n                     for k in range(6)])\n\n\ndef symmetry(v, rot=0, mirror=False, flip_ud=False):\n    \"\"\"Точная симметрия съёмочного стенда: 4 азимута с шагом 90° + верх + низ.\n\n    Множество из шести кадров не меняется — меняются только их порядок и ориентация,\n    поэтому ВСЕ 11 меток инвариантны. Это 4*2*2 = 16 преобразований без риска испортить\n    разметку, в отличие от зума (ломает `scale`), пиксельного шума (ломает `noisy`,\n    вес 0.274) и обнуления вида (создаёт условие `partial` при метке 0).\n\n    Прежний флип `v[:, :, ::-1]` отражал кадры, но НЕ разворачивал порядок азимутов —\n    получалась конфигурация, которой не соответствует ни один реальный объект.\n    \"\"\"\n    az, po = list(CFG.azimuth_idx), list(CFG.pole_idx)\n    out = [v[az[(k + rot) % 4]] for k in range(4)]\n    top, bot = po\n    pt = np.rot90(v[top], -rot, axes=(0, 1))\n    pb = np.rot90(v[bot],  rot, axes=(0, 1))\n    if mirror:\n        out = [out[0]] + out[1:][::-1]\n        out = [x[:, ::-1] for x in out]\n        pt, pb = pt[:, ::-1], pb[:, ::-1]\n    if flip_ud:\n        out = [x[::-1] for x in out]\n        pt, pb = pb[::-1], pt[::-1]\n    return np.ascontiguousarray(np.stack(out + [pt, pb]))\n\n\nSYM_ALL = [(r, m, u) for r in range(4) for m in (False, True) for u in (False, True)]\n# В TTA переворот верх-низ не берём: тест снят в одной ориентации, и подмешивать\n# перевёрнутые кадры на инференсе — это сдвиг распределения, а не усреднение шума.\nCFG.tta_syms = [(r, m, False) for r in range(4) for m in (False, True)]\n\n\nclass MeshViewDataset(Dataset):\n    def __init__(self, df, img_dir, feats, train=True, labels=True):\n        self.ids = df['item_id'].tolist()\n        self.dir = Path(img_dir)\n        self.train = train\n        self.labels = df[CFG.target_cols].values.astype(np.float32) if labels else None\n        self.feats = feats.set_index('item_id').reindex(self.ids)[FEAT_COLS] \\\n                          .fillna(0.0).values.astype(np.float32)\n\n    def __len__(self):\n        return len(self.ids)\n\n    def _aug(self, v):\n        rng = np.random\n        if CFG.sym_aug:\n            v = symmetry(v, *SYM_ALL[rng.randint(len(SYM_ALL))])\n        elif rng.rand() < 0.5:\n            v = symmetry(v, 0, True, False)\n        if rng.rand() < 0.3:\n            # сдвиг с заливкой фоном, а не np.roll: заворачивание краёв вклеивает\n            # кусок объекта с противоположной стороны кадра\n            ys, xs = (int(z) for z in rng.randint(-8, 9, size=2))\n            out = np.full_like(v, 255)\n            H, W = v.shape[1], v.shape[2]\n            y0, y1 = max(0, ys), min(H, H + ys)\n            x0, x1 = max(0, xs), min(W, W + xs)\n            out[:, y0:y1, x0:x1] = v[:, y0 - ys:y1 - ys, x0 - xs:x1 - xs]\n            v = out\n        if CFG.view_dropout and rng.rand() < CFG.view_dropout:\n            v = v.copy(); v[rng.randint(6)] = 255\n        v = v.astype(np.float32)\n        if rng.rand() < 0.3:\n            v = np.clip(v * rng.uniform(0.85, 1.15) + rng.uniform(-15, 15), 0, 255)\n        if CFG.pixel_noise and rng.rand() < CFG.pixel_noise:\n            v = np.clip(v + rng.normal(0, 4, v.shape), 0, 255)\n        return v\n\n    def __getitem__(self, i):\n        p = self.dir / f'{self.ids[i]}.png'\n        v = load_views(p, CFG.tile) if p.exists() else \\\n            np.full((6, CFG.tile, CFG.tile, 3), 255, np.uint8)\n        v = self._aug(v) if self.train else v.astype(np.float32)\n        v = ((v / 255.0 - MEAN) / STD).astype(np.float32)\n        out = {'views': torch.from_numpy(np.ascontiguousarray(v)).permute(0, 3, 1, 2),\n               'feats': torch.from_numpy(self.feats[i]), 'item_id': self.ids[i]}\n        if self.labels is not None:\n            out['y'] = torch.from_numpy(self.labels[i])\n        return out\n\n\ndef _sym_torch(v, rot, mirror, flip_ud=False):\n    az, po = list(CFG.azimuth_idx), list(CFG.pole_idx)\n    v = v[:, [az[(k + rot) % 4] for k in range(4)] + po]\n    if rot:\n        v = torch.cat([v[:, :4], torch.rot90(v[:, 4:5], -rot, dims=(-2, -1)),\n                       torch.rot90(v[:, 5:6], rot, dims=(-2, -1))], 1)\n    if mirror:\n        v = torch.cat([v[:, :1], v[:, 1:4].flip(1), v[:, 4:]], 1).flip(-1)\n    if flip_ud:\n        v = torch.cat([v[:, :4], v[:, 5:6], v[:, 4:5]], 1).flip(-2)\n    return v\n\n\n@torch.no_grad()\ndef predict(model, loader, tta=None, desc='predict'):\n    \"\"\"tta=1 — один проход (мониторинг эпох), иначе TTA по симметриям стенда.\"\"\"\n    model.eval()\n    syms = [(0, False, False)] if tta == 1 else CFG.tta_syms\n    probs, ids = [], []\n    for b in tqdm(loader, desc=f'{desc} (TTA x{len(syms)})', leave=False):\n        v = b['views'].to(CFG.device, non_blocking=True)\n        f = b['feats'].to(CFG.device, non_blocking=True)\n        acc = 0\n        with torch.cuda.amp.autocast(enabled=CFG.amp):\n            for s in syms:\n                acc = acc + torch.sigmoid(model(_sym_torch(v, *s), f)).float()\n        probs.append((acc / len(syms)).cpu().numpy()); ids.extend(b['item_id'])\n    return np.vstack(probs), ids\n\n\ndef check_view_layout(n=200):\n    \"\"\"Симметрии осмысленны, только если виды 0-3 — азимуты, 4-5 — полюса.\n    Проверяем данными: соседние азимуты должны быть похожи сильнее, чем азимут и полюс.\"\"\"\n    acc, cnt = np.zeros((6, 6)), 0\n    for iid in train_df['item_id'].head(n):\n        p = Path(IMG_TRAIN) / f'{iid}.png'\n        if not p.exists():\n            continue\n        x = load_views(p, CFG.tile).astype(np.float32).mean(-1)\n        m = (np.abs(x - np.median(x)) > 15).reshape(6, -1).astype(np.float32)\n        m -= m.mean(1, keepdims=True)\n        nrm = np.linalg.norm(m, axis=1) + 1e-9\n        acc += (m @ m.T) / np.outer(nrm, nrm); cnt += 1\n    S = acc / max(cnt, 1)\n    ring = np.mean([S[i, (i + 1) % 4] for i in range(4)])\n    cross = np.mean([S[i, j] for i in CFG.azimuth_idx for j in CFG.pole_idx])\n    fig, ax = plt.subplots(1, 2, figsize=(12, 4))\n    im = ax[0].imshow(S, cmap='viridis'); ax[0].grid(False)\n    ax[0].set_title('Сходство силуэтов между видами')\n    for i in range(6):\n        for j in range(6):\n            ax[0].text(j, i, f'{S[i, j]:.2f}', ha='center', va='center', fontsize=7,\n                       color='w' if S[i, j] < S.max() * .6 else 'k')\n    plt.colorbar(im, ax=ax[0], fraction=.046)\n    ax[1].bar(['соседние\\nазимуты', 'азимут-полюс'], [ring, cross],\n              color=[PAL['good'], PAL['bad']])\n    ax[1].set_title('Кольцевая структура')\n    plt.tight_layout(); plt.show()\n    if ring > cross + 0.02:\n        R.ok(f'раскладка подтверждена (кольцо {ring:.3f} > крест {cross:.3f}) — '\n             f'{len(SYM_ALL)} симметрий включены, TTA x{len(CFG.tta_syms)}')\n    else:\n        CFG.sym_aug = False\n        CFG.tta_syms = [(0, False, False), (0, True, False)]\n        R.warn('кольцевая структура не подтверждена — симметрии выключены')\n    return S\n\n\n_t = np.random.randint(0, 255, (6, 64, 64, 3), dtype=np.uint8)\nassert np.array_equal(symmetry(_t, 0, False, False), _t)\nassert np.array_equal(symmetry(symmetry(_t, 1), 3), _t)\nassert np.array_equal(symmetry(symmetry(_t, 0, True), 0, True), _t)\nassert np.array_equal(symmetry(symmetry(_t, 0, False, True), 0, False, True), _t)\nassert len({tuple(sorted(int(x.sum()) for x in symmetry(_t, *s))) for s in SYM_ALL}) == 1\nR.ok(f'{len(SYM_ALL)} симметрий: множество кадров сохраняется, преобразования обратимы')\n"
  },
  {
    "source_cell": 36,
    "source": "import timm\n\n\nclass MLDecoder(nn.Module):\n    \"\"\"Голова классификации с запросом на класс (Ridnik et al., WACV 2023, arXiv 2111.12933).\n\n    Зачем именно здесь. До сих пор каждый вид сжимался глобальным пулингом в один\n    вектор, и только потом решалось, какие метки поставить. Для `noisy` или `simple`\n    это нормально — они про объект целиком. Но `open` (F1 0.30), `artifacts` (0.33)\n    и `intersection` (0.15) — дефекты **локальные**: дыра занимает малую долю кадра,\n    и при усреднении по всем патчам её сигнал разбавляется в сотни раз.\n\n    ML-Decoder заводит обучаемый запрос на каждый класс и даёт ему через cross-attention\n    самому выбрать, на какие участки каких видов смотреть. Запрос `open` может\n    сосредоточиться на краях силуэта, запрос `set` — на разнесённых кусках. Self-attention\n    между запросами в оригинальной статье убран как избыточный, поэтому стоимость линейна\n    по числу токенов: 11 запросов почти ничего не стоят.\n\n    Токены: патч-сетка каждого вида ужимается адаптивным пулингом до\n    `mld_tokens` x `mld_tokens`, к ней прибавляется кодирование номера вида, и все шесть\n    видов склеиваются в одну последовательность. При 8x8 это 384 токена независимо от\n    разрешения — то есть 448 не удорожает голову, только бэкбон.\n    \"\"\"\n\n    def __init__(self, d_in, n_out, dim=384, layers=2, heads=8):\n        super().__init__()\n        self.proj = nn.Linear(d_in, dim) if d_in != dim else nn.Identity()\n        self.query = nn.Parameter(torch.zeros(1, n_out, dim))\n        nn.init.trunc_normal_(self.query, std=0.02)\n        self.view_emb = nn.Parameter(torch.zeros(1, 6, 1, dim))\n        nn.init.trunc_normal_(self.view_emb, std=0.02)\n        self.blocks = nn.ModuleList()\n        for _ in range(layers):\n            self.blocks.append(nn.ModuleDict({\n                'norm_q': nn.LayerNorm(dim), 'norm_k': nn.LayerNorm(dim),\n                'attn': nn.MultiheadAttention(dim, heads, dropout=0.1, batch_first=True),\n                'norm_f': nn.LayerNorm(dim),\n                'ffn': nn.Sequential(nn.Linear(dim, 2 * dim), nn.GELU(),\n                                     nn.Dropout(0.1), nn.Linear(2 * dim, dim))}))\n        self.norm_out = nn.LayerNorm(dim)\n\n    def forward(self, tokens, extra=None):\n        \"\"\"tokens: (B, 6, T, d_in) — патч-токены каждого вида.\n        extra:  (B, d)  — гео-вектор, добавляется как ещё один ключ.\"\"\"\n        B, V, T, _ = tokens.shape\n        x = self.proj(tokens) + self.view_emb[:, :V]\n        x = x.reshape(B, V * T, -1)\n        if extra is not None:\n            x = torch.cat([x, extra.unsqueeze(1)], 1)\n        q = self.query.expand(B, -1, -1)\n        att_last = None\n        for blk in self.blocks:\n            a, w = blk['attn'](blk['norm_q'](q), blk['norm_k'](x), blk['norm_k'](x),\n                               need_weights=True, average_attn_weights=True)\n            q = q + a\n            q = q + blk['ffn'](blk['norm_f'](q))\n            att_last = w\n        return self.norm_out(q), att_last          # (B, n_out, dim), (B, n_out, L)\n\n\nclass MultiViewNet(nn.Module):\n    def __init__(self, n_feats, n_out=CFG.n_targets):\n        super().__init__()\n        kw = dict(pretrained=True, num_classes=0, drop_rate=CFG.drop_rate)\n        if IS_VIT:\n            kw['img_size'] = CFG.tile\n        self.backbone = timm.create_model(CFG.backbone, **kw)\n        d = self.backbone.num_features\n        self.d = d\n        self.geo = nn.Sequential(\n            nn.BatchNorm1d(n_feats), nn.Linear(n_feats, 256), nn.SiLU(), nn.Dropout(0.2),\n            nn.Linear(256, 128), nn.SiLU())\n\n        if CFG.use_mldecoder:\n            self.dec = MLDecoder(d, n_out, dim=CFG.mld_dim, layers=CFG.mld_layers)\n            self.geo_to_tok = nn.Linear(128, CFG.mld_dim)\n            # голова: по одному линейному выходу на класс, применённому к своему запросу\n            self.cls_w = nn.Parameter(torch.zeros(n_out, CFG.mld_dim))\n            self.cls_b = nn.Parameter(torch.zeros(n_out))\n            nn.init.trunc_normal_(self.cls_w, std=0.02)\n        else:\n            self.att_v = nn.Sequential(nn.Linear(d, 256), nn.Tanh())\n            self.att_u = nn.Sequential(nn.Linear(d, 256), nn.Sigmoid())\n            self.att_w = nn.Linear(256, 1)\n            self.head = nn.Sequential(nn.Linear(2 * d + 128, 512), nn.SiLU(),\n                                      nn.Dropout(0.3), nn.Linear(512, n_out))\n\n    def _patch_tokens(self, x):\n        \"\"\"(B*V, 3, H, W) -> (B*V, T, d): патч-токены, ужатые до mld_tokens^2.\"\"\"\n        n = CFG.mld_tokens\n        if IS_VIT:\n            f = self.backbone.forward_features(x)          # (N, 1+r+P, d)\n            npre = f.shape[1] - int(math.isqrt(f.shape[1])) ** 2\n            g = f[:, npre:]\n            s = int(math.isqrt(g.shape[1]))\n            g = g[:, :s * s].transpose(1, 2).reshape(g.shape[0], -1, s, s)\n        else:\n            g = self.backbone.forward_features(x)          # (N, d, H', W')\n        g = F.adaptive_avg_pool2d(g, (n, n))\n        return g.flatten(2).transpose(1, 2)                # (N, n*n, d)\n\n    def embed(self, views, feats):\n        B, V = views.shape[:2]\n        flat = views.flatten(0, 1)\n        gv = self.geo(feats)\n        if CFG.use_mldecoder:\n            tok = self._patch_tokens(flat).view(B, V, -1, self.d)\n            q, att = self.dec(tok, self.geo_to_tok(gv))\n            logits = (q * self.cls_w.unsqueeze(0)).sum(-1) + self.cls_b\n            return logits, att\n        z = self.backbone(flat).view(B, V, -1)\n        a = self.att_w(self.att_v(z) * self.att_u(z))\n        w = torch.softmax(a, 1)\n        pooled = torch.cat([(z * w).sum(1), z.max(1).values], -1)\n        return self.head(torch.cat([pooled, gv], 1)), w.squeeze(-1)\n\n    def forward(self, views, feats, return_att=False):\n        logits, att = self.embed(views, feats)\n        return (logits, att) if return_att else logits\n\n\nclass AsymmetricLoss(nn.Module):\n    \"\"\"Ridnik et al., ICCV 2021 — как в прогоне, давшем 14.987.\"\"\"\n    def __init__(self, gamma_neg=4.0, gamma_pos=0.0, clip=0.05, eps=1e-8):\n        super().__init__()\n        self.gn, self.gp, self.clip, self.eps = gamma_neg, gamma_pos, clip, eps\n\n    def forward(self, logits, y):\n        p = torch.sigmoid(logits)\n        p_neg = (1 - p + self.clip).clamp(max=1.0)\n        loss = y * torch.log(p.clamp(min=self.eps)) + (1 - y) * torch.log(p_neg.clamp(min=self.eps))\n        pt = p * y + (1 - p) * (1 - y)\n        gamma = self.gp * y + self.gn * (1 - y)\n        return -(loss * (1 - pt).pow(gamma)).mean()\n\n\nclass ModelEMA:\n    def __init__(self, model, decay=0.999):\n        self.ema = copy.deepcopy(model).eval()\n        for p in self.ema.parameters():\n            p.requires_grad_(False)\n        self.decay = decay\n\n    @torch.no_grad()\n    def update(self, model, step=None):\n        d = self.decay if step is None else min(self.decay, (1 + step) / (10 + step))\n        msd = model.state_dict()\n        for k, v in self.ema.state_dict().items():\n            if v.dtype.is_floating_point:\n                v.mul_(d).add_(msd[k].detach(), alpha=1 - d)\n            else:\n                v.copy_(msd[k])\n\n    def state_dict(self): return self.ema.state_dict()\n    def load_state_dict(self, sd): self.ema.load_state_dict(sd)\n\n\ndef _selftest_model():\n    m = MultiViewNet(n_feats=16).to(CFG.device).eval()\n    v = torch.randn(2, 6, 3, CFG.tile, CFG.tile, device=CFG.device)\n    f = torch.randn(2, 16, device=CFG.device)\n    with torch.no_grad():\n        o, a = m(v, f, return_att=True)\n    R.ok(f'модель собрана: logits {tuple(o.shape)}, attention {tuple(a.shape)}, '\n         f'параметров {sum(p.numel() for p in m.parameters()) / 1e6:.1f}M')\n    if CFG.use_mldecoder:\n        R.kv(**{'токенов в декодере': f'6 видов x {CFG.mld_tokens}^2 = '\n                                      f'{6 * CFG.mld_tokens ** 2} + 1 гео'})\n    del m, v, f; gc.collect(); torch.cuda.empty_cache()\n\n\n# Model checked before actual training by tensor shape and finite-input checks.\n"
  },
  {
    "source_cell": 38,
    "source": "from torch.cuda.amp import autocast, GradScaler\n\n\ndef per_class_f1(y, p):\n    return pd.Series([f1_score(y[:, i], p[:, i], zero_division=0) for i in range(11)],\n                     index=CFG.target_cols)\n\n\ndef exact_f1_threshold(y, p, plateau=0.995):\n    \"\"\"Точный argmax F1 по порогу за один проход сортировки.\n\n    Разрез допустим только там, где значение p МЕНЯЕТСЯ: иначе порог попадает между\n    двумя одинаковыми вероятностями и `p > th` выбрасывает оба объекта вместо одного.\n    При насыщенных сигмоидах и усреднении по фолдам совпадения массовые.\n    \"\"\"\n    y = np.asarray(y).astype(np.int32); p = np.asarray(p, np.float64)\n    P = int(y.sum())\n    if P == 0:\n        return 0.5, 0.0\n    o = np.argsort(-p, kind='stable'); ys, ps = y[o], p[o]\n    f1 = 2 * np.cumsum(ys) / (np.arange(1, len(y) + 1) + P)\n    fb = float(f1.max())\n    cut = np.empty(len(ps), bool); cut[-1] = True; cut[:-1] = ps[:-1] > ps[1:]\n    ok = np.where((f1 >= plateau * fb) & cut)[0]\n    if not len(ok):\n        ok = np.where(cut)[0][[int(np.argmax(f1[cut]))]]\n    i = int(ok[len(ok) // 2])\n    return float((ps[i] + ps[i + 1]) / 2 if i + 1 < len(ps) else ps[i] - 1e-6), fb\n\n\ndef quick_thresholds(y, prob):\n    return np.array([exact_f1_threshold(y[:, i], prob[:, i])[0] for i in range(11)])\n\n\ndef _fmt_eta(sec):\n    sec = int(max(sec, 0))\n    return f'{sec // 3600}ч {sec % 3600 // 60:02d}м' if sec >= 3600 else f'{sec // 60}м {sec % 60:02d}с'\n\n\ndef live_dashboard(hist, fold):\n    if not hist:\n        return\n    h = pd.DataFrame(hist)\n    fig, ax = plt.subplots(1, 3, figsize=(16, 4))\n    ax[0].plot(h['epoch'], h['val_metric'], '-o', ms=4, color=PAL['main'], label='raw')\n    if h['val_metric_ema'].notna().any():\n        ax[0].plot(h['epoch'], h['val_metric_ema'], '-s', ms=4, color=PAL['alt'], label='EMA')\n    ax[0].axhline(14.25, color=PAL['grey'], ls=':', lw=1, label='CNN в v5')\n    ax[0].set_xlabel('эпоха'); ax[0].legend(fontsize=8)\n    ax[0].set_title(f'{CFG.run_tag} fold {fold}: метрика')\n    ax[1].plot(h['epoch'], 10 * h['f1_quality'], '-o', ms=4, color=PAL['bad'], label='quality')\n    ax[1].plot(h['epoch'], 10 * h['f1_artefacts'], '-o', ms=4, color=PAL['warn'], label='artefacts')\n    ax[1].set_xlabel('эпоха'); ax[1].legend(fontsize=8); ax[1].set_title('Половины метрики')\n    f1c = [c for c in h.columns if c.startswith('f1_') and c not in ('f1_quality', 'f1_artefacts')]\n    last = h.iloc[-1][f1c].astype(float).sort_values()\n    ax[2].barh([c[3:] for c in last.index], last.values,\n               color=[PAL['bad'] if v < .4 else PAL['warn'] if v < .6 else PAL['good']\n                      for v in last.values])\n    ax[2].set_xlim(0, 1); ax[2].set_title(f'per-class F1, эпоха {int(h[\"epoch\"].iloc[-1])}')\n    plt.tight_layout(); plt.show()\n\n\ndef train_fold(fold):\n    if CFG.resume and has_artifact(rt(f'fold{fold}_preds')):\n        d = load_artifact(rt(f'fold{fold}_preds'))\n        R.ok(f'фолд {fold} взят из кэша (метрика {d[\"ens\"]:.3f}) — обучение пропускаем')\n        return (d['oof_ids'], d['oof']), (d['te_ids'], d['te'])\n\n    R.section(f'{CFG.run_tag}: фолд {fold} / {CFG.n_folds}',\n              f'{CFG.backbone.split(\".\")[0]} @ {CFG.tile} | эпох {CFG.epochs}')\n    seed_everything(CFG.seed + fold)\n    tr = train_df[train_df.fold != fold].reset_index(drop=True)\n    va = train_df[train_df.fold == fold].reset_index(drop=True)\n    ltr = DataLoader(MeshViewDataset(tr, IMG_TRAIN, feat_tr, train=True),\n                     batch_size=CFG.batch_size, shuffle=True, drop_last=True,\n                     num_workers=CFG.num_workers, pin_memory=True, persistent_workers=True)\n    lva = DataLoader(MeshViewDataset(va, IMG_TRAIN, feat_tr, train=False),\n                     batch_size=CFG.batch_size * 2, num_workers=CFG.num_workers, pin_memory=True)\n    R.kv(**{'объектов train / val': f'{len(tr)} / {len(va)}',\n            'шагов на эпоху': f'{len(ltr)} (оптимизаторских {len(ltr) // CFG.accum})'})\n\n    model = MultiViewNet(len(FEAT_COLS)).to(CFG.device)\n    ema = ModelEMA(model, CFG.ema_decay)\n    head = [p for n_, p in model.named_parameters() if not n_.startswith('backbone')]\n    head_ids = {id(p) for p in head}\n\n    if IS_VIT and CFG.layer_decay < 1.0:\n        blocks = getattr(model.backbone, 'blocks', [])\n        nb_ = len(blocks)\n        buckets = {}\n        for name, p in model.backbone.named_parameters():\n            m_ = re.search(r'blocks\\.(\\d+)\\.', name)\n            depth = (int(m_.group(1)) + 1) if m_ else (0 if any(\n                k in name for k in ('patch_embed', 'pos_embed', 'cls_token', 'reg_token')) else nb_)\n            buckets.setdefault(round(CFG.layer_decay ** (nb_ - depth), 5), []).append(p)\n        groups = [{'params': ps, 'lr': CFG.backbone_lr * s} for s, ps in buckets.items()]\n        R.kv(**{'LR бэкбона': f'{CFG.backbone_lr * min(buckets):.2e} … {CFG.backbone_lr:.2e} '\n                              f'(затухание {CFG.layer_decay}, блоков {nb_})'})\n    else:\n        groups = [{'params': [p for p in model.backbone.parameters()], 'lr': CFG.backbone_lr}]\n    groups.append({'params': head, 'lr': CFG.lr * CFG.head_lr_mult})\n    opt = torch.optim.AdamW(groups, weight_decay=CFG.weight_decay)\n\n    steps = CFG.epochs * max(len(ltr) // CFG.accum, 1)\n    warm = int(CFG.warmup_frac * steps)\n    sched = torch.optim.lr_scheduler.LambdaLR(\n        opt, lambda s: s / max(warm, 1) if s < warm\n        else 0.5 * (1 + math.cos(math.pi * (s - warm) / max(steps - warm, 1))))\n\n    pos = tr[CFG.target_cols].sum(0).values.astype(np.float32)\n    pw = np.clip((len(tr) - pos) / np.clip(pos, 1, None), 1.0, 20.0)\n    bce = nn.BCEWithLogitsLoss(pos_weight=torch.tensor(pw, device=CFG.device))\n    asl = AsymmetricLoss()\n    scaler = GradScaler(enabled=CFG.amp)\n\n    snaps, best, start_ep, hist, gstep = [], -1, 0, [], 0\n    ck = load_ckpt(fold)\n    if ck is not None:\n        model.load_state_dict(_to_fp32(ck['model']))\n        if ck.get('ema') is not None:\n            ema.load_state_dict(_to_fp32(ck['ema']))\n        try:\n            opt.load_state_dict(ck['opt']); sched.load_state_dict(ck['sched'])\n            scaler.load_state_dict(ck['scaler'])\n        except Exception as e:\n            R.warn(f'состояние оптимизатора не восстановлено: {e}')\n        snaps = [(s, _to_fp32(sd)) for s, sd in ck['snaps']]\n        best, start_ep, hist = ck['best'], ck['epoch'] + 1, ck.get('hist', [])\n        set_rng_state(ck['rng'])\n        R.ok(f'продолжаем с эпохи {start_ep + 1}/{CFG.epochs} (лучшая пока {best:.3f})')\n\n    yv = va[CFG.target_cols].values\n    pred, ep_times = None, []\n\n    for ep in range(start_ep, CFG.epochs):\n        t0 = time.time(); model.train()\n        tot, nb_i = 0.0, 0\n        opt.zero_grad(set_to_none=True)\n        pbar = tqdm(ltr, desc=f'{CFG.run_tag} f{fold} ep{ep + 1:>2}/{CFG.epochs}', leave=False)\n        for it, b in enumerate(pbar):\n            v = b['views'].to(CFG.device, non_blocking=True)\n            f = b['feats'].to(CFG.device, non_blocking=True)\n            y = b['y'].to(CFG.device, non_blocking=True)\n            y = y * (1 - CFG.label_smooth) + 0.5 * CFG.label_smooth\n            if CFG.mixup > 0 and random.random() < 0.5:\n                lam = float(np.random.beta(CFG.mixup, CFG.mixup))\n                pm = torch.randperm(v.size(0), device=v.device)\n                v = lam * v + (1 - lam) * v[pm]\n                f = lam * f + (1 - lam) * f[pm]\n                y = lam * y + (1 - lam) * y[pm]\n            with autocast(enabled=CFG.amp):\n                logits = model(v, f)\n                loss = (1 - CFG.asl_weight) * bce(logits, y) + CFG.asl_weight * asl(logits, y)\n            scaler.scale(loss / CFG.accum).backward()\n            if (it + 1) % CFG.accum == 0 or it + 1 == len(ltr):\n                scaler.unscale_(opt)\n                torch.nn.utils.clip_grad_norm_(model.parameters(), 5.0)\n                scaler.step(opt); scaler.update(); opt.zero_grad(set_to_none=True)\n                sched.step(); gstep += 1; ema.update(model, gstep)\n            tot += float(loss.detach()); nb_i += 1\n            if it % 20 == 0:\n                mem = torch.cuda.max_memory_allocated() / 1e9 if torch.cuda.is_available() else 0\n                pbar.set_postfix_str(f'L={tot / nb_i:.3f} lr={opt.param_groups[-1][\"lr\"]:.1e} '\n                                     f'{mem:.1f}G')\n\n        prob, _ = predict(model, lva, tta=1, desc=f'val ep{ep + 1}')\n        pred = (prob > quick_thresholds(yv, prob)).astype(int)\n        sc, fq, fa = competition_metric(yv, pred)\n        sc_ema = np.nan\n        if ep >= 1:\n            pe, _ = predict(ema.ema, lva, tta=1, desc=f'val-EMA ep{ep + 1}')\n            prd = (pe > quick_thresholds(yv, pe)).astype(int)\n            sc_ema, fqe, fae = competition_metric(yv, prd)\n            if sc_ema > sc:\n                prob, pred, sc, fq, fa = pe, prd, sc_ema, fqe, fae\n\n        dt = time.time() - t0; ep_times.append(dt)\n        left = (CFG.epochs - ep - 1) * np.median(ep_times)\n        src = 'EMA' if (not np.isnan(sc_ema) and sc_ema == sc) else 'raw'\n        print(f'  ep{ep + 1:>2}/{CFG.epochs} | метрика {sc:6.3f} ({src}) = quality {10 * fq:5.2f} '\n              f'+ artefacts {10 * fa:5.2f} | loss {tot / nb_i:.3f} | {dt / 60:.1f} мин | '\n              f'ост. {_fmt_eta(left)}' + (' <-- best' if sc > best else ''))\n        pcf = per_class_f1(yv, pred)\n        print('        ' + '  '.join(f'{c[:5]}={pcf[c]:.2f}' for c in CFG.target_cols))\n\n        row = {'run': CFG.run_tag, 'fold': fold, 'epoch': ep + 1, 'train_loss': tot / nb_i,\n               'val_metric': sc, 'val_metric_ema': sc_ema, 'f1_quality': fq,\n               'f1_artefacts': fa, 'sec': dt, 'time': time.strftime('%Y-%m-%d %H:%M:%S'),\n               **{f'f1_{c}': float(v_) for c, v_ in pcf.items()}}\n        log_epoch(row); hist.append(row)\n\n        if ep >= 1:\n            keep = ema.ema if src == 'EMA' else model\n            snaps.append((sc, {k: v_.detach().cpu().clone() for k, v_ in keep.state_dict().items()}))\n            snaps.sort(key=lambda x: -x[0]); del snaps[CFG.n_snapshots:]\n        best = max(best, sc)\n\n        if (ep + 1) % CFG.ckpt_every == 0 or ep == CFG.epochs - 1:\n            conv = _to_fp16 if CFG.ckpt_fp16 else (lambda x: x)\n            save_ckpt(fold, {'epoch': ep, 'best': best, 'rng': rng_state(), 'hist': hist,\n                             'model': conv(model.state_dict()), 'ema': conv(ema.state_dict()),\n                             'opt': opt.state_dict(), 'sched': sched.state_dict(),\n                             'scaler': scaler.state_dict(),\n                             'snaps': [(s, conv(sd)) for s, sd in snaps],\n                             'cfg': {'tile': CFG.tile, 'backbone': CFG.backbone,\n                                     'epochs': CFG.epochs, 'feat_ver': FEAT_VER}})\n\n    live_dashboard(hist, fold)\n    if not snaps:\n        snaps = [(best, {k: v_.detach().cpu().clone() for k, v_ in model.state_dict().items()})]\n\n    lte = DataLoader(MeshViewDataset(test_df.assign(**{c: 0 for c in CFG.target_cols}),\n                                     IMG_TEST, feat_te, train=False, labels=False),\n                     batch_size=CFG.batch_size * 2, num_workers=CFG.num_workers, pin_memory=True)\n    oa, ta = 0.0, 0.0\n    for k, (s_, st) in enumerate(snaps):\n        model.load_state_dict(st)\n        op, oof_ids = predict(model, lva, desc=f'OOF snap{k + 1}/{len(snaps)}')\n        tp, te_ids = predict(model, lte, desc=f'test snap{k + 1}/{len(snaps)}')\n        oa = oa + op; ta = ta + tp\n    oof_prob, te_prob = oa / len(snaps), ta / len(snaps)\n    ens = competition_metric(yv, (oof_prob > quick_thresholds(yv, oof_prob)).astype(int))[0]\n    print(f'  лучшая эпоха {best:.3f} -> снапшот-ансамбль + TTA {ens:.3f} ({ens - best:+.3f})')\n    R.bar(ens, label=f'фолд {fold}')\n\n    keep_snapshots(fold, snaps)\n    save_artifact(rt(f'fold{fold}_preds'),\n                  {'oof': oof_prob, 'oof_ids': list(oof_ids), 'te': te_prob,\n                   'te_ids': list(te_ids), 'best': best, 'ens': ens})\n    drop_ckpt(fold)\n    del model, ema, ltr, lva, snaps; gc.collect(); torch.cuda.empty_cache()\n    return (oof_ids, oof_prob), (te_ids, te_prob)\n"
  }
]


In [ ]:
%%writefile /content/meshqc_code/v5_224_source.py
                    
import os, gc, json, math, random, re, warnings, shutil
from pathlib import Path
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
from tqdm.auto import tqdm
import matplotlib.pyplot as plt

warnings.filterwarnings('ignore')

class CFG:
    seed = 42

                      
    dataset  = 'daniilantonov5/3d-mesh-quality-control'
    work     = LOCAL_DIR; work.mkdir(parents=True, exist_ok=True)
    cache    = work / 'cache';        cache.mkdir(parents=True, exist_ok=True)

                                 
                                                                                          
                                                                                
    view_mode = 'multiview'                                                                       
    tile      = 224                                                                        
    grid_rows, grid_cols = 2, 3

                   
                                                                                             
                                                                                         
                                                                                         
    backbone   = 'vit_small_patch14_reg4_dinov2.lvd142m'
                                                                                               
                                                                                           
    backbone_lr = 4e-5                                                                            
    layer_decay = 0.75                                                                   
    drop_rate  = 0.2
    epochs     = 16
    batch_size = 12
    mixup      = 0.4                                                                    

                                                  
    use_clip   = True                                                                       
    clip_model = ('ViT-B-32', 'laion2b_s34b_b79k')
    use_dino_frozen = True                                                                   
    dino_frozen_model = 'vit_base_patch14_reg4_dinov2.lvd142m'
    dino_pca   = 128
    lr         = 3e-4
    head_lr_mult = 10.0
    weight_decay = 0.05
    warmup_frac  = 0.1
    label_smooth = 0.01
    amp        = True
    num_workers = 4

                  
    n_folds = 5
    RUN_ALL_FOLDS = True                                                        
    folds_to_run  = [0]

                        
    tta = 2                                                                           
    n_snapshots = 3                                                                         
    run_tag = 'dinos224_geo2_v5'                                                              
                                                                                          
    ensemble_tags = ['dinos224_geo2_v5']                                                            
                                                                                                 
                                                                                                        
                                                                   
    asl_weight = 0.65                                                                      
                                                                                                
                                                                                                

                                           
    use_drive   = True                                                                                    
    drive_dir   = '/content/drive/MyDrive/sber_meshqc_v5'                                  
                                                                                           
                                                                     
    resume      = True                                                            
    ckpt_every  = 1                                                  
    ckpt_fp16   = True                                                             
    keep_fold_ckpt = False                                                             
    force_recompute = []                                                                      

                     
    artifact_cols = ['abstract','artifacts','intersection','lowpoly','noisy',
                     'open','partial','scale','set','simple']
    target_cols   = artifact_cols + ['quality']
    n_targets     = 11

    device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

                     
import kagglehub

root = Path(kagglehub.dataset_download(CFG.dataset))
print('dataset root:', root)

def _pick_dir(mode):
    """Папка с максимальным числом .npz, в пути которой встречается train/test."""
    best, best_n = None, -1
    for p in root.rglob('*'):
        if not p.is_dir():
            continue
        if mode not in str(p).lower():
            continue
        n = sum(1 for _ in p.glob('*.npz'))
        if n > best_n:
            best, best_n = p, n
    if best is None or best_n == 0:
        raise FileNotFoundError(f'не найдена папка с .npz для {mode}')
    return best

def _pick_csv(mode):
    cands = list(root.rglob('*.csv'))
    def score(p):
        n = p.name.lower(); s = 0
        if mode == 'train':
            s += 10 * ('train' in n) - 20 * ('submission' in n) - 20 * ('test' in n)
        else:
            s += 10 * ('test' in n) - 5 * ('submission' in n) - 20 * ('train' in n)
        return s - 0.001 * len(n)
    return sorted(cands, key=score, reverse=True)[0]

TRAIN_DIR, TEST_DIR = _pick_dir('train'), _pick_dir('test')
TRAIN_CSV, TEST_CSV = _pick_csv('train'), _pick_csv('test')
print('train dir :', TRAIN_DIR, len(list(TRAIN_DIR.glob('*.npz'))), 'npz /',
      len(list(TRAIN_DIR.glob('*.png'))), 'png')
print('test  dir :', TEST_DIR,  len(list(TEST_DIR.glob('*.npz'))),  'npz /',
      len(list(TEST_DIR.glob('*.png'))),  'png')
print('train csv :', TRAIN_CSV)
print('test  csv :', TEST_CSV)

train_df = pd.read_csv(TRAIN_CSV)
test_df  = pd.read_csv(TEST_CSV)[['item_id']].copy()
train_df['item_id'] = train_df['item_id'].astype(str)
test_df['item_id']  = test_df['item_id'].astype(str)

if 'quality' not in train_df.columns:
    train_df['quality'] = (train_df[CFG.artifact_cols].sum(1) == 0).astype(int)

print(train_df.shape, test_df.shape)
train_df.head()

                     
from sklearn.metrics import f1_score

def competition_metric(y_true, y_pred):
    """10*F1(quality) + 10*F1_weighted(10 дефектов) — точная формула организаторов."""
    f1_q = f1_score(y_true[:, 10], y_pred[:, 10], zero_division=0)
    f1_a = f1_score(y_true[:, :10], y_pred[:, :10], average='weighted', zero_division=0)
    return 10 * f1_q + 10 * f1_a, f1_q, f1_a

                                                                                          
Y = train_df[CFG.target_cols].values
for err in [0.02, 0.05, 0.10]:
    rng = np.random.default_rng(0)
    fake = Y.copy()
    flip = rng.random(fake[:, :10].shape) < err
    fake[:, :10] = np.abs(fake[:, :10] - flip)
    fake[:, 10] = (fake[:, :10].sum(1) == 0).astype(int)
    s, fq, fa = competition_metric(Y, fake)
    print(f'FPR/FNR по дефектам {err:.0%} -> метрика {s:5.2f} (quality F1={fq:.3f}, artefacts F1={fa:.3f})')
print('\n>>> Ошибка 5% по дефектам уже срезает quality F1 до ~0.7. '
      'Поэтому quality предсказывается ОТДЕЛЬНОЙ головой, а не только правилом.')

                     
from PIL import Image
from concurrent.futures import ProcessPoolExecutor

Image.MAX_IMAGE_PIXELS = None

def infer_grid(w, h):
    """(rows, cols) для 6 видов по соотношению сторон."""
    ar = w / h
    cands = {(2, 3): 3 / 2, (3, 2): 2 / 3, (1, 6): 6.0, (6, 1): 1 / 6}
    return min(cands, key=lambda k: abs(math.log(ar / cands[k])))

def split_views(img):
    """PIL.Image -> список из 6 PIL.Image в каноническом порядке чтения."""
    w, h = img.size
    r, c = infer_grid(w, h)
    tw, th = w // c, h // r
    return [img.crop((j * tw, i * th, (j + 1) * tw, (i + 1) * th)) for i in range(r) for j in range(c)]

def make_cache(args):
    src, dst, tile = args
    try:
        img = Image.open(src).convert('RGB')
        views = [v.resize((tile, tile), Image.BILINEAR) for v in split_views(img)]
        out = Image.new('RGB', (3 * tile, 2 * tile))
        for k, v in enumerate(views):
            out.paste(v, ((k % 3) * tile, (k // 3) * tile))
        out.save(dst, format='PNG', optimize=False, compress_level=1)
        return 1
    except Exception:
        return 0

def build_image_cache(ids, src_dir, split):
    out_dir = CFG.cache / f'img_{split}_{CFG.tile}'
    out_dir.mkdir(parents=True, exist_ok=True)
    jobs = [(str(Path(src_dir) / f'{i}.png'), str(out_dir / f'{i}.png'), CFG.tile)
            for i in ids if not (out_dir / f'{i}.png').exists()
            and (Path(src_dir) / f'{i}.png').exists()]
    if jobs:
        with ProcessPoolExecutor(max_workers=os.cpu_count()) as ex:
            ok = list(tqdm(ex.map(make_cache, jobs, chunksize=32), total=len(jobs),
                           desc=f'cache {split}'))
        print(f'{split}: закэшировано {sum(ok)}/{len(jobs)}')
    missing = [i for i in ids if not (out_dir / f'{i}.png').exists()]
    print(f'{split}: всего в кэше {len(ids) - len(missing)}/{len(ids)}, нет картинки у {len(missing)}')
    return out_dir, set(missing)

                                                                                       
_probe = [Image.open(Path(TRAIN_DIR) / f'{i}.png').size
          for i in train_df['item_id'].head(40) if (Path(TRAIN_DIR) / f'{i}.png').exists()]
if _probe:
    _sizes = pd.Series([f'{w}x{h}' for w, h in _probe]).value_counts()
    print('исходные размеры листа с 6 видами:'); print(_sizes.head(5).to_string())
    _w, _h = _probe[0]
    _r, _c = infer_grid(_w, _h)
    _native = min(_w // _c, _h // _r)
    print(f'-> раскладка {_r}x{_c}, нативный размер одного вида ~{_native} px')
    if CFG.tile > _native:
        print(f'!! CFG.tile = {CFG.tile} БОЛЬШЕ нативного {_native}: это апскейл, он удорожает '
              f'обучение, но новой информации не даёт. Снижаю tile до {_native}.')
        CFG.tile = int(_native)
    else:
        print(f'   CFG.tile = {CFG.tile} <= нативного — детали сохраняются')

IMG_TRAIN, MISS_TRAIN = build_image_cache(train_df['item_id'].tolist(), TRAIN_DIR, 'train')
IMG_TEST,  MISS_TEST  = build_image_cache(test_df['item_id'].tolist(),  TEST_DIR,  'test')

                     
import os, gc, math, numpy as np, pandas as pd
from pathlib import Path
from PIL import Image
from scipy import ndimage
from scipy.sparse import coo_matrix
from scipy.sparse.csgraph import connected_components
from scipy.spatial import ConvexHull, QhullError
from concurrent.futures import ProcessPoolExecutor
from concurrent.futures.process import BrokenProcessPool
import multiprocessing as mp

FACE_CAP  = 300_000
VERT_CAP  = 200_000
HULL_CAP  = 20_000
COMP_CAP  = 3_000_000
WELD_TOL  = 1e-6                              
FEAT_VER  = 'v4selfint'                                                                
                                                                                          
                                                                                              
N_WORKERS = min(4, os.cpu_count() or 2)
CHUNK     = 1024

def _pack21(q):
    """Три целых из [0, 2**21) -> один int64. Точно, без коллизий."""
    return (q[:, 0] << 42) | (q[:, 1] << 21) | q[:, 2]

def _hash3(q):
    q = q.astype(np.int64, copy=False)
    return (q[:, 0] * 73856093) ^ (q[:, 1] * 19349663) ^ (q[:, 2] * 83492791)

def _stats(x, pref):
    keys = ['mean', 'std', 'p10', 'p50', 'p90', 'max', 'cv']
    if len(x) == 0:
        return {f'{pref}_{k}': 0.0 for k in keys}
    x = np.asarray(x, dtype=np.float32)
    m, s = float(x.mean()), float(x.std())
    q10, q50, q90 = np.percentile(x, [10, 50, 90])
    return {f'{pref}_mean': m, f'{pref}_std': s, f'{pref}_p10': float(q10),
            f'{pref}_p50': float(q50), f'{pref}_p90': float(q90),
            f'{pref}_max': float(x.max()), f'{pref}_cv': float(s / (abs(m) + 1e-9))}

def _edge_table(Fi, nfw):
    """Рёбра -> (номер группы, число граней на ребро, отсортированные id граней)."""
    E = np.concatenate([Fi[:, [0, 1]], Fi[:, [1, 2]], Fi[:, [2, 0]]], 0)
    E.sort(axis=1)
    fid = np.tile(np.arange(nfw, dtype=np.int32), 3)
    order = np.lexsort((E[:, 1], E[:, 0]))
    Es, fids = E[order], fid[order]
    new = np.ones(len(Es), dtype=bool)
    new[1:] = (Es[1:] != Es[:-1]).any(1)
    grp = np.cumsum(new) - 1
    counts = np.bincount(grp)
    return E, Es, fids, grp, counts, new

def mesh_features(npz_path):
    f = {'geo_ok': 0.0, 'geo_subsampled': 0.0}
    try:
        f['npz_mb'] = os.path.getsize(npz_path) / 1e6
    except Exception:
        return f
    try:
        with np.load(npz_path, allow_pickle=False) as d:
            keys = list(d.keys())
            V = np.asarray(d['vertices' if 'vertices' in keys else keys[0]], dtype=np.float32)
            Fc = None
            for k in ('faces', 'triangles', 'f'):
                if k in keys:
                    Fc = np.asarray(d[k], dtype=np.int64); break
            if Fc is None and len(keys) > 1:
                Fc = np.asarray(d[keys[1]], dtype=np.int64)
    except Exception:
        return f
    if V.ndim != 2 or V.shape[1] != 3 or len(V) == 0:
        return f
    if Fc is None or Fc.ndim != 2 or Fc.shape[1] != 3 or len(Fc) == 0:
        Fc = np.zeros((0, 3), dtype=np.int64)
    else:
        Fc = Fc[(Fc >= 0).all(1) & (Fc < len(V)).all(1)]

    f['geo_ok'] = 1.0
    nv_raw, nf_raw = len(V), len(Fc)
    vmin, vmax = V.min(0), V.max(0)
    ext = (vmax - vmin).astype(np.float64)
    ext_s = np.sort(ext)[::-1]
    diag = float(np.linalg.norm(ext)) + 1e-12
    center = ((vmin + vmax) / 2).astype(np.float32)
    inv_s = np.float32(1.0 / diag)
    f.update({'n_verts_raw': float(nv_raw), 'n_faces': float(nf_raw),
              'log_verts': math.log1p(nv_raw), 'log_faces': math.log1p(nf_raw),
              'ext_max': float(ext_s[0]), 'ext_mid': float(ext_s[1]), 'ext_min': float(ext_s[2]),
              'ext_ratio_min': float(ext_s[2] / (ext_s[0] + 1e-12)),
              'ext_ratio_mid': float(ext_s[1] / (ext_s[0] + 1e-12)),
              'bbox_diag': diag,
              'bbox_fill': float(nv_raw / (ext.prod() + 1e-12)) if ext.prod() > 0 else 0.0})

                                                         
                                                                                           
                                                                                         
                                                                                                
    step = np.float32(diag * WELD_TOL)
    q = np.clip(np.rint((V - vmin) / step), 0, (1 << 21) - 1).astype(np.int64)
    code = _pack21(q)
    del q
    ucode, first, invmap = np.unique(code, return_index=True, return_inverse=True)
    del code, ucode
    nv = len(first)
    f['n_verts'] = float(nv)
    f['weld_ratio'] = float(1.0 - nv / max(nv_raw, 1))                                                         
    f['f_per_v'] = nf_raw / max(nv, 1)
    Vw_full = V[first]
    del first

    if nf_raw:
        Fw = invmap[Fc]
        ok = (Fw[:, 0] != Fw[:, 1]) & (Fw[:, 1] != Fw[:, 2]) & (Fw[:, 0] != Fw[:, 2])
        f['degen_after_weld'] = float(1.0 - ok.mean())
        Fw = Fw[ok]
        hf = _hash3(np.sort(Fw, axis=1))
        _, uidx = np.unique(hf, return_index=True)
        f['dup_face_ratio'] = float(1.0 - len(uidx) / max(len(Fw), 1))
        Fw = Fw[np.sort(uidx)]
        del hf, uidx, ok
    else:
        Fw = np.zeros((0, 3), dtype=np.int64)
        f['degen_after_weld'] = 0.0; f['dup_face_ratio'] = 0.0
    del invmap, Fc
    nf = len(Fw)

                                                 
    Vs = Vw_full[::max(1, int(np.ceil(nv / VERT_CAP)))]
    if nv > VERT_CAP:
        f['geo_subsampled'] = 1.0
    Vs = (Vs - center) * inv_s
    try:
        if len(Vs) < 3:
            raise ValueError('too few vertices')
        ev = np.clip(np.linalg.eigvalsh(np.cov(Vs.T.astype(np.float64)))[::-1], 1e-16, None)
        f.update({'pca_1': float(ev[0]), 'pca_2': float(ev[1]), 'pca_3': float(ev[2]),
                  'pca_flat': float(ev[2] / ev[0]), 'pca_lin': float(ev[1] / ev[0]),
                  'pca_aniso': float((ev[0] - ev[2]) / ev.sum())})
    except Exception:
        f.update({k: 0.0 for k in ['pca_1', 'pca_2', 'pca_3', 'pca_flat', 'pca_lin', 'pca_aniso']})

                                                                       
    f.update({'hull_ok': 0.0, 'solidity': 0.0, 'solidity_signed': 0.0, 'hull_area_ratio': 0.0,
              'hull_pts_frac': 0.0, 'sphericity': 0.0, 'hull_vol': 0.0, 'hull_area': 0.0})
    try:
        Vh = Vs[::max(1, int(np.ceil(len(Vs) / HULL_CAP)))].astype(np.float64)
        if len(Vh) >= 8:
            hull = ConvexHull(Vh, qhull_options='QJ')
            f['hull_ok'] = 1.0
            f['hull_vol'] = float(hull.volume)
            f['hull_pts_frac'] = len(hull.vertices) / len(Vh)
            f['hull_area'] = float(hull.area)
    except (QhullError, ValueError, MemoryError):
        pass

    if nf == 0:
        f['no_faces'] = 1.0
        f.update({'area_total': 0.0, 'volume': 0.0, 'abs_volume': 0.0, 'vol_ratio': 0.0,
                  'vol_over_area': 0.0, 'absvol_over_area': 0.0, 'vol_over_bbox': 0.0,
                  'absvol_over_bbox': 0.0})
        for pref in ('dihed', 'dihedf', 'farea', 'elen', 'aspect', 'valence', 'defect'):
            f.update(_stats([], pref))
        return f
    f['no_faces'] = 0.0

                                                               
    total_area, vol6, absvol6 = 0.0, 0.0, 0.0
    for s0 in range(0, nf, 500_000):
        Pc = ((Vw_full[Fw[s0:s0 + 500_000]] - center) * inv_s).astype(np.float64)
        cr_c = np.cross(Pc[:, 1] - Pc[:, 0], Pc[:, 2] - Pc[:, 0])
        total_area += 0.5 * float(np.linalg.norm(cr_c, axis=1).sum())
        contrib = (Pc[:, 0] * np.cross(Pc[:, 1], Pc[:, 2])).sum(1)
        vol6 += float(contrib.sum()); absvol6 += float(np.abs(contrib).sum())
        del Pc, cr_c, contrib
    total_area += 1e-12
    vol = abs(vol6 / 6.0)
    absvol = absvol6 / 6.0
                                                                                            
                                                                                            
    f.update({'area_total': total_area, 'volume': vol, 'abs_volume': absvol,
              'vol_ratio': vol / (absvol + 1e-12),
              'vol_over_area': vol / (total_area ** 1.5 + 1e-12),
              'absvol_over_area': absvol / (total_area ** 1.5 + 1e-12),
              'vol_over_bbox': vol / (float(np.prod(ext * inv_s)) + 1e-12),
              'absvol_over_bbox': absvol / (float(np.prod(ext * inv_s)) + 1e-12)})
    if f['hull_ok']:
        f['solidity'] = absvol / (f['hull_vol'] + 1e-12)
        f['solidity_signed'] = vol / (f['hull_vol'] + 1e-12)
        f['hull_area_ratio'] = total_area / (f.get('hull_area', 0.0) + 1e-12)
        f['sphericity'] = (math.pi ** (1 / 3)) * (6 * absvol) ** (2 / 3) / (total_area + 1e-12)

                                                             
    f.update({'n_comp': 1.0, 'n_comp_big': 1.0, 'comp_top1': 1.0, 'comp_top2': 0.0,
              'comp_entropy': 0.0, 'bbox_iou_max': 0.0, 'bbox_iou_mean': 0.0,
              'bbox_iou_frac_pos': 0.0, 'comp_sep_max': 0.0, 'comp_sep_mean': 0.0,
              'inside_frac_max': 0.0, 'comp_exact': 1.0})
    if nf <= COMP_CAP:
        try:
            Ef = np.concatenate([Fw[:, [0, 1]], Fw[:, [1, 2]], Fw[:, [2, 0]]], 0).astype(np.int32)
            A = coo_matrix((np.ones(len(Ef), dtype=np.int8), (Ef[:, 0], Ef[:, 1])), shape=(nv, nv))
            del Ef
            ncomp, lab = connected_components(A, directed=False)
            del A
            used = np.unique(Fw)
            sizes_all = np.bincount(lab[used], minlength=ncomp).astype(np.float64)
            sizes = np.sort(sizes_all[sizes_all > 0])[::-1]
            tot = sizes.sum()
            big_idx = np.where(sizes_all > 0.005 * tot)[0]
            f.update({'n_comp': float(len(sizes)), 'n_comp_big': float(len(big_idx)),
                      'comp_top1': float(sizes[0] / tot),
                      'comp_top2': float(sizes[1] / tot) if len(sizes) > 1 else 0.0,
                      'comp_entropy': float(-((sizes / tot) * np.log(sizes / tot + 1e-12)).sum())})
            if len(big_idx) > 1:
                top = big_idx[np.argsort(-sizes_all[big_idx])][:8]
                pts_l, boxes = [], []
                for c in top:
                    ic = used[lab[used] == c]
                    if len(ic) > 20_000:
                        ic = ic[::len(ic) // 20_000 + 1]
                    p = (Vw_full[ic] - center) * inv_s
                    pts_l.append(p); boxes.append((p.min(0), p.max(0)))
                ious, seps, insides = [], [], []
                for i in range(len(boxes)):
                    for j in range(len(boxes)):
                        if i == j:
                            continue
                        lo, hi = boxes[j]
                        insides.append(float(((pts_l[i] >= lo) & (pts_l[i] <= hi)).all(1).mean()))
                        if j <= i:
                            continue
                        lo2 = np.maximum(boxes[i][0], boxes[j][0])
                        hi2 = np.minimum(boxes[i][1], boxes[j][1])
                        it = float(np.prod(np.clip(hi2 - lo2, 0, None)))
                        vi = float(np.prod(np.clip(boxes[i][1] - boxes[i][0], 1e-6, None)))
                        vj = float(np.prod(np.clip(boxes[j][1] - boxes[j][0], 1e-6, None)))
                        ious.append(it / (vi + vj - it + 1e-12))
                        seps.append(float(np.linalg.norm(
                            (boxes[i][0] + boxes[i][1]) / 2 - (boxes[j][0] + boxes[j][1]) / 2)))
                f.update({'bbox_iou_max': float(max(ious)), 'bbox_iou_mean': float(np.mean(ious)),
                          'bbox_iou_frac_pos': float(np.mean(np.asarray(ious) > 1e-6)),
                          'comp_sep_max': float(max(seps)), 'comp_sep_mean': float(np.mean(seps)),
                          'inside_frac_max': float(max(insides))})
            del lab, sizes_all, sizes, used
        except Exception:
            f['comp_exact'] = 0.0
    else:
        f['comp_exact'] = 0.0

                                                            
    if nf > FACE_CAP:
        f['geo_subsampled'] = 1.0
        anchor = (Vw_full[Fw[:, 0]] - vmin) / (ext.astype(np.float32) + 1e-9)
        Fp, g = None, 1
        while g < 40:
            g += 1
            cid = np.minimum((anchor * g).astype(np.int32), g - 1)
            key = (cid[:, 0] * g + cid[:, 1]) * g + cid[:, 2]
            cnt = np.bincount(key, minlength=g ** 3)
            best = int(np.argmax(cnt))
            if cnt[best] <= FACE_CAP:
                Fp = Fw[key == best]; break
        if Fp is None or len(Fp) < 1000:
            Fp = Fw[:FACE_CAP]
        f['patch_grid'] = float(g)
        del anchor
    else:
        Fp = Fw
        f['patch_grid'] = 1.0
    f['patch_face_frac'] = len(Fp) / max(nf, 1)
    del Fw
    nfw = len(Fp)

    uvp, Fi = np.unique(Fp, return_inverse=True)
    Fi = Fi.reshape(nfw, 3).astype(np.int32)
    Vp = ((Vw_full[uvp] - center) * inv_s).astype(np.float32)
    nvp = len(uvp)
    del uvp, Fp, Vw_full, V

    P = Vp[Fi]
    cr = np.cross(P[:, 1] - P[:, 0], P[:, 2] - P[:, 0])
    area2 = np.linalg.norm(cr, axis=1)
    area = 0.5 * area2
    nrm = cr / (area2[:, None] + 1e-12)
    del cr
    f.update(_stats(area / (area.mean() + 1e-12), 'farea'))
    f['degenerate_ratio'] = float((area < 1e-10).mean())

    L = np.stack([np.linalg.norm(P[:, 1] - P[:, 0], axis=1),
                  np.linalg.norm(P[:, 2] - P[:, 1], axis=1),
                  np.linalg.norm(P[:, 0] - P[:, 2], axis=1)], 1)
    f.update(_stats(L.ravel() / (L.mean() + 1e-12), 'elen'))
    aspect = L.max(1) / (L.min(1) + 1e-12)
    f.update(_stats(np.log1p(aspect), 'aspect'))
    f['sliver_ratio'] = float((aspect > 20).mean())

                                                                                  
                                                                                      
                                                                                     
                                                                                          
                                                                                   
                                      
    L0, L1, L2 = L[:, 0], L[:, 1], L[:, 2]
    with np.errstate(divide='ignore', invalid='ignore'):
        cA = np.clip((L1 ** 2 + L2 ** 2 - L0 ** 2) / (2 * L1 * L2 + 1e-12), -1, 1)
        cB = np.clip((L0 ** 2 + L2 ** 2 - L1 ** 2) / (2 * L0 * L2 + 1e-12), -1, 1)
        cC = np.clip((L0 ** 2 + L1 ** 2 - L2 ** 2) / (2 * L0 * L1 + 1e-12), -1, 1)
    tri_ang = np.degrees(np.arccos(np.stack([cA, cB, cC], 1)))
    min_ang, max_ang = tri_ang.min(1), tri_ang.max(1)
    f.update(_stats(min_ang, 'minang')); f.update(_stats(max_ang, 'maxang'))
    f['sliver_angle_ratio'] = float((min_ang < 5).mean())
    tri_quality = np.clip((4 * math.sqrt(3) * area) / (L0 ** 2 + L1 ** 2 + L2 ** 2 + 1e-12), 0, 1)
    f.update(_stats(tri_quality, 'triq'))
    del L0, L1, L2, cA, cB, cC, tri_ang, min_ang, max_ang, tri_quality

    for qd in (1, 2):
        f[f'uniq_normals_q{qd}'] = len(np.unique(_hash3(np.rint(nrm * 10 ** qd)))) / nfw
                                                                               
    hn = _hash3(np.rint(nrm * 20))
    _, ninv = np.unique(hn, return_inverse=True)
    mass = np.bincount(ninv, weights=area)
    mass = np.sort(mass)[::-1]
    f['normal_top6_mass'] = float(mass[:6].sum() / (mass.sum() + 1e-12))
    f['normal_top1_mass'] = float(mass[0] / (mass.sum() + 1e-12))
    del hn, ninv, mass

                                                                                    
                                                                                 
                                                                                  
                                                                                   
                                                                                       
                                                                          
                                                                                          
                                                                                       
    f.update({'selfint_ok': 0.0, 'selfint_pair_ratio': 0.0, 'selfint_face_ratio': 0.0,
              'selfint_any': 0.0, 'selfint_capped': 0.0})
    SELFINT_FACE_CAP = 120_000
    SELFINT_PAIR_CAP = 400_000
    if 0 < nfw <= SELFINT_FACE_CAP:
        try:
            from scipy.spatial import cKDTree
            cen = Vp[Fi].mean(1)
            elen_med = float(np.median(L)) if len(L) else 1e-6
            r = max(elen_med * 2.5, 1e-6)
            pairs = cKDTree(cen).query_pairs(r=r, output_type='ndarray')
            if len(pairs):
                share = (Fi[pairs[:, 0], :, None] == Fi[pairs[:, 1], None, :]).any((1, 2))
                pairs = pairs[~share]
            if len(pairs) > SELFINT_PAIR_CAP:
                sel = np.random.default_rng(0).choice(len(pairs), SELFINT_PAIR_CAP, replace=False)
                pairs = pairs[sel]
                f['selfint_capped'] = 1.0
            f['selfint_ok'] = 1.0
            if len(pairs):
                ii, jj = pairs[:, 0], pairs[:, 1]
                PA, PB = Vp[Fi[ii]], Vp[Fi[jj]]
                nA, nB = nrm[ii], nrm[jj]

                def _edge_hits(a, b, p0, p1, p2, n):
                    d = np.einsum('ij,ij->i', n, p0)
                    ta = np.einsum('ij,ij->i', n, a) - d
                    tb = np.einsum('ij,ij->i', n, b) - d
                    denom = ta - tb
                    safe = np.abs(denom) > 1e-12
                    t = np.clip(np.divide(ta, denom, out=np.zeros_like(ta), where=safe), 0.0, 1.0)
                    pt = a + t[:, None] * (b - a)
                    e0 = np.cross(p1 - p0, pt - p0); e1 = np.cross(p2 - p1, pt - p1)
                    e2 = np.cross(p0 - p2, pt - p2)
                    s0 = np.einsum('ij,ij->i', e0, n); s1 = np.einsum('ij,ij->i', e1, n)
                    s2 = np.einsum('ij,ij->i', e2, n)
                    inside = (((s0 >= -1e-9) & (s1 >= -1e-9) & (s2 >= -1e-9)) |
                              ((s0 <= 1e-9) & (s1 <= 1e-9) & (s2 <= 1e-9)))
                    return (ta * tb <= 0) & safe & inside

                hit = np.zeros(len(pairs), dtype=bool)
                for e in range(3):
                    hit |= _edge_hits(PA[:, e], PA[:, (e + 1) % 3], PB[:, 0], PB[:, 1], PB[:, 2], nB)
                for e in range(3):
                    hit |= _edge_hits(PB[:, e], PB[:, (e + 1) % 3], PA[:, 0], PA[:, 1], PA[:, 2], nA)
                n_hit = int(hit.sum())
                f['selfint_pair_ratio'] = float(n_hit / max(nfw, 1))
                f['selfint_any'] = float(n_hit > 0)
                if n_hit:
                    f['selfint_face_ratio'] = float(
                        len(np.unique(np.concatenate([ii[hit], jj[hit]]))) / max(nfw, 1))
                del PA, PB, nA, nB, hit
            del cen, pairs
        except Exception:
            pass
    else:
        f['selfint_capped'] = 1.0

                                                                              
    E, Es, fids, grp, counts, _ = _edge_table(Fi, nfw)
    n_edges = len(counts)
    f.update({'n_edges': float(n_edges),
              'boundary_edge_ratio': float((counts == 1).mean()),
              'nonmanifold_edge_ratio': float((counts >= 3).mean()),
              'euler_norm': float((nvp - n_edges + nfw) / max(nfw, 1)),
              'euler_char': float(nvp - n_edges + nfw),
              'watertight': float((counts == 1).sum() == 0 and (counts >= 3).sum() == 0),
              'manifold_edge_ratio': float((counts == 2).mean())})

                                                                  
    bnd = np.where(counts == 1)[0]
    f['nonmanifold_vert_ratio'] = 0.0
    if len(bnd):
        st = np.searchsorted(grp, bnd)
        be = Es[st]
        try:
            Ab = coo_matrix((np.ones(len(be), dtype=np.int8), (be[:, 0], be[:, 1])), shape=(nvp, nvp))
            nb_comp, lb = connected_components(Ab, directed=False)
            used_b = np.unique(be)
            f['n_boundary_loops'] = float(len(np.unique(lb[used_b])))
            f['boundary_vert_frac'] = float(len(used_b) / nvp)
            del Ab, lb, used_b
        except Exception:
            f['n_boundary_loops'] = 0.0; f['boundary_vert_frac'] = 0.0
                                                                                   
                                                                               
                                                                                 
                                                                                
                                                                                 
                                                                                   
        deg_b = np.bincount(be.ravel(), minlength=nvp)
        f['nonmanifold_vert_ratio'] = float((deg_b > 2).sum() / max(nvp, 1))
        del be, st, deg_b
    else:
        f['n_boundary_loops'] = 0.0; f['boundary_vert_frac'] = 0.0

    two = np.where(counts == 2)[0]
    if len(two):
        st = np.searchsorted(grp, two)
        fa, fb = fids[st], fids[st + 1]
        cosang = np.clip((nrm[fa] * nrm[fb]).sum(1), -1, 1)
        ang = np.degrees(np.arccos(cosang))
        angf = np.degrees(np.arccos(np.abs(cosang)))
        f.update(_stats(ang, 'dihed')); f.update(_stats(angf, 'dihedf'))
        for t in (10, 30, 60, 90):
            f[f'dihed_gt{t}'] = float((ang > t).mean())
        for t in (5, 15, 30, 60):
            f[f'dihedf_gt{t}'] = float((angf > t).mean())
        f['flip_ratio'] = float((cosang < 0).mean())
        hist = np.bincount((angf / 5).astype(np.int32).clip(0, 17), minlength=18).astype(np.float64)
        p = hist / max(hist.sum(), 1)
        f['dihed_entropy'] = float(-(p[p > 0] * np.log(p[p > 0])).sum())
        f['smoothness'] = float(angf.mean() * math.sqrt(nfw) / 100.0)
                                                                           
        el = np.linalg.norm(Vp[Es[st][:, 0]] - Vp[Es[st][:, 1]], axis=1)
        f['dihedf_areaw'] = float((angf * el).sum() / (el.sum() + 1e-12))
        del fa, fb, cosang, ang, angf, el
    else:
        f.update(_stats([], 'dihed')); f.update(_stats([], 'dihedf'))
        for t in (10, 30, 60, 90):
            f[f'dihed_gt{t}'] = 0.0
        for t in (5, 15, 30, 60):
            f[f'dihedf_gt{t}'] = 0.0
        f.update({'flip_ratio': 0.0, 'dihed_entropy': 0.0, 'smoothness': 0.0, 'dihedf_areaw': 0.0})

                                                                       
    val = np.bincount(Fi.ravel(), minlength=nvp).astype(np.float32)
    f.update(_stats(val, 'valence'))
    f['valence_le3'] = float((val <= 3).mean())
    f['valence_ge8'] = float((val >= 8).mean())
    e01 = P[:, 1] - P[:, 0]; e12 = P[:, 2] - P[:, 1]; e20 = P[:, 0] - P[:, 2]
    def _ang(u, v):
        cu = (u * v).sum(1) / (np.linalg.norm(u, axis=1) * np.linalg.norm(v, axis=1) + 1e-12)
        return np.arccos(np.clip(cu, -1, 1))
    a0 = _ang(e01, -e20); a1 = _ang(e12, -e01); a2 = _ang(e20, -e12)
    defect = np.full(nvp, 2 * np.pi, dtype=np.float64)
    np.subtract.at(defect, Fi[:, 0], a0)
    np.subtract.at(defect, Fi[:, 1], a1)
    np.subtract.at(defect, Fi[:, 2], a2)
    interior = val > 0
    f.update(_stats(np.abs(defect[interior]), 'defect'))
    f['defect_sum'] = float(defect[interior].sum() / (2 * np.pi))                                
    f['defect_gt05'] = float((np.abs(defect[interior]) > 0.5).mean())
    del defect, val, a0, a1, a2, e01, e12, e20

    del E, Es, fids, grp, counts, two, nrm, P, Vp, Fi, L, aspect, area, area2
    return f

                     
def image_features(png_path):
    f = {'img_ok': 0.0}
    try:
        img = Image.open(png_path).convert('L')
    except Exception:
        return f
    a = np.asarray(img, dtype=np.float32) / 255.0
    t = CFG.tile
    f['img_ok'] = 1.0
    covs, comps, edens, stds, bws, bhs = [], [], [], [], [], []
    for k in range(6):
        v = a[(k // 3) * t:(k // 3 + 1) * t, (k % 3) * t:(k % 3 + 1) * t]
        bg = np.median(np.concatenate([v[0], v[-1], v[:, 0], v[:, -1]]))
        mask = np.abs(v - bg) > 0.06
        cov = float(mask.mean()); covs.append(cov)
        if mask.any():
            ys, xs = np.where(mask)
            bws.append((xs.max() - xs.min() + 1) / t)
            bhs.append((ys.max() - ys.min() + 1) / t)
            lab, ncc = ndimage.label(mask)
            if ncc:
                sz = np.bincount(lab.ravel())[1:]
                comps.append(float((sz > 0.01 * sz.sum()).sum()))
            else:
                comps.append(0.0)
        else:
            bws.append(0.0); bhs.append(0.0); comps.append(0.0)
        edens.append(float(np.abs(np.diff(v, axis=1)).mean() + np.abs(np.diff(v, axis=0)).mean()))
        stds.append(float(v.std()))
    def agg(vals, pref):
        v = np.asarray(vals, dtype=np.float64)
        return {f'{pref}_mean': v.mean(), f'{pref}_min': v.min(), f'{pref}_max': v.max(),
                f'{pref}_std': v.std(), f'{pref}_range': v.max() - v.min()}
    f.update(agg(covs, 'cov')); f.update(agg(comps, 'ncc')); f.update(agg(edens, 'edge'))
    f.update(agg(stds, 'pxstd')); f.update(agg(bws, 'bw')); f.update(agg(bhs, 'bh'))
    f['cov_empty_views'] = float((np.asarray(covs) < 0.01).sum())
    f['bbox_fill_view'] = float(np.mean(np.asarray(covs) /
                                        (np.asarray(bws) * np.asarray(bhs) + 1e-6)))
    return f

def _feat_one(args):
    iid, npz, png = args
    d = {'item_id': iid}
    try:
        d.update(mesh_features(npz))
    except (MemoryError, Exception):
        d['geo_ok'] = 0.0
    try:
        d.update(image_features(png))
    except Exception:
        d['img_ok'] = 0.0
    return d

def build_features(ids, npz_dir, img_dir, split):
    """Чанками, с чекпоинтом и резюмированием. Повторный запуск после падения
    продолжает с последнего сохранённого чанка."""
    final = CFG.cache / f'feats_{split}_{CFG.tile}_{FEAT_VER}.parquet'
    ckpt  = CFG.cache / f'feats_{split}_{CFG.tile}_{FEAT_VER}_partial.parquet'
    if final.exists():
        df = pd.read_parquet(final); print(f'{split}: фичи из кэша {df.shape}'); return df

    done = pd.read_parquet(ckpt) if ckpt.exists() else pd.DataFrame(columns=['item_id'])
    have = set(done['item_id'].astype(str)) if len(done) else set()
    todo = [i for i in ids if i not in have]
    print(f'{split}: уже готово {len(have)}, осталось {len(todo)}, воркеров {N_WORKERS}')

    rows = [done] if len(done) else []
    for s in range(0, len(todo), CHUNK):
        part = todo[s:s + CHUNK]
        jobs = [(i, str(Path(npz_dir) / f'{i}.npz'), str(Path(img_dir) / f'{i}.png')) for i in part]
                                                                                           
                                                                                               
                                                                                  
                                                                   
        try:
            ctx = mp.get_context('fork')
        except ValueError:
            ctx = None
        try:
            with ProcessPoolExecutor(max_workers=N_WORKERS, mp_context=ctx) as ex:
                got = list(tqdm(ex.map(_feat_one, jobs, chunksize=4), total=len(jobs),
                                desc=f'{split} {s + len(part)}/{len(todo)}', leave=False))
        except (BrokenProcessPool, OSError, MemoryError) as e:
            print(f'  пул упал ({type(e).__name__}), чанк считается последовательно')
            got = [_feat_one(j) for j in tqdm(jobs, desc='serial', leave=False)]
        rows.append(pd.DataFrame(got))
        _atomic(ckpt, lambda p: pd.concat(rows, ignore_index=True).to_parquet(p, index=False))
        gc.collect()

    df = pd.concat(rows, ignore_index=True).drop_duplicates('item_id')
    df = df.set_index('item_id').reindex([str(i) for i in ids]).reset_index()
    df = df.fillna(0.0)
    _atomic(final, lambda p: df.to_parquet(p, index=False))
    if ckpt.exists():
        ckpt.unlink()
    print(f'{split}: фичи посчитаны {df.shape}')
    return df

feat_tr = build_features(train_df['item_id'].tolist(), TRAIN_DIR, IMG_TRAIN, 'train')
feat_te = build_features(test_df['item_id'].tolist(),  TEST_DIR,  IMG_TEST,  'test')

                     
CLIP_PROMPTS = {
                                            
    'abs_text':     ['3D text lettering', 'a sign with written words', 'an extruded logo'],
    'abs_chart':    ['a bar chart', 'a pie chart', 'a graph plot', 'a diagram', 'a flowchart',
                     'a table with data', 'an infographic'],
    'abs_support':  ['3D printing support structure', 'scaffolding lattice'],
    'abs_voxel':    ['minecraft voxel blocks', 'blocky pixelated cubes'],
                                                     
    'obj_single':   ['a 3D model of a single object', 'a product render on white background'],
    'obj_semantic': ['a chair', 'a car', 'a building', 'a plant', 'a character figure',
                     'a tool', 'furniture'],
                                          
    'p_lowpoly':    ['a low-poly 3D model with visible flat facets'],
    'p_noisy':      ['a noisy 3D scan with rough surface', 'a point cloud scan'],
    'p_broken':     ['a broken mesh with holes and artifacts'],
    'p_hollow':     ['a hollow thin shell', 'a flat plane'],
    'p_multi':      ['several separate objects scattered apart', 'a collection of many objects'],
    'p_small':      ['a tiny object in the middle of a large empty frame'],
    'p_smooth':     ['a smooth clean 3D render of one object'],
}

def compute_clip_features(ids, img_dir, split):
    import open_clip
    name, pretrained = CFG.clip_model
    model, _, _ = open_clip.create_model_and_transforms(name, pretrained=pretrained)
    tokenizer = open_clip.get_tokenizer(name)
    model = model.to(CFG.device).eval()

    groups = list(CLIP_PROMPTS)
    flat, owner = [], []
    for g in groups:
        flat += CLIP_PROMPTS[g]; owner += [g] * len(CLIP_PROMPTS[g])
    with torch.no_grad():
        tf = model.encode_text(tokenizer(flat).to(CFG.device)).float()
        tf = tf / tf.norm(dim=-1, keepdim=True)
    owner = np.array(owner)

    size = model.visual.image_size
    size = size[0] if isinstance(size, (tuple, list)) else int(size)
    mean = torch.tensor([0.48145466, 0.4578275, 0.40821073], device=CFG.device).view(1, 3, 1, 1)
    std = torch.tensor([0.26862954, 0.26130258, 0.27577711], device=CFG.device).view(1, 3, 1, 1)

    rows, bs = [], 24
    for s0 in tqdm(range(0, len(ids), bs), desc=f'CLIP {split}'):
        chunk = ids[s0:s0 + bs]
        batch = []
        for iid in chunk:
            p = Path(img_dir) / f'{iid}.png'
            if p.exists():
                vs = [np.asarray(v.resize((size, size), Image.BILINEAR), dtype=np.uint8)
                      for v in split_views(Image.open(p).convert('RGB'))]
            else:
                vs = [np.full((size, size, 3), 255, np.uint8)] * 6
            batch.append(np.stack(vs))
        x = torch.from_numpy(np.stack(batch)).to(CFG.device)                              
        B = x.shape[0]
        x = x.permute(0, 1, 4, 2, 3).reshape(B * 6, 3, size, size).float() / 255.0
        x = (x - mean) / std
        with torch.no_grad():
            im = model.encode_image(x).float()
        im = im / im.norm(dim=-1, keepdim=True)
        sim = (im @ tf.T).view(B, 6, -1).cpu().numpy()

        gsim = np.stack([sim[:, :, owner == g].max(-1) for g in groups], -1)                   
        feat = {}
        for j, g in enumerate(groups):
            v = gsim[:, :, j]
            feat[f'clip_{g}_mean'] = v.mean(1)
            feat[f'clip_{g}_max'] = v.max(1)
            feat[f'clip_{g}_std'] = v.std(1)
        abs_max = np.stack([feat[f'clip_{g}_max'] for g in groups if g.startswith('abs_')], 1).max(1)
        obj_max = np.stack([feat[f'clip_{g}_max'] for g in groups if g.startswith('obj_')], 1).max(1)
        def_max = np.stack([feat[f'clip_{g}_max'] for g in groups
                            if g.startswith('p_') and g != 'p_smooth'], 1).max(1)
        feat['clip_abstract_margin'] = abs_max - obj_max
        feat['clip_abstract_prob'] = 1 / (1 + np.exp(-100 * (abs_max - obj_max)))
        feat['clip_defect_margin'] = def_max - feat['clip_p_smooth_max']
        rows.append(pd.DataFrame(feat, index=list(chunk)))
    del model
    gc.collect(); torch.cuda.empty_cache()
    return pd.concat(rows).rename_axis('item_id').reset_index()

if CFG.use_clip:
    clip_tr = cached(f'clip_train_{CFG.clip_model[0]}',
                     lambda: compute_clip_features(train_df['item_id'].tolist(), IMG_TRAIN, 'train'))
    clip_te = cached(f'clip_test_{CFG.clip_model[0]}',
                     lambda: compute_clip_features(test_df['item_id'].tolist(), IMG_TEST, 'test'))
    feat_tr = feat_tr.merge(clip_tr, on='item_id', how='left')
    feat_te = feat_te.merge(clip_te, on='item_id', how='left')
    print('добавлено CLIP-признаков:', clip_tr.shape[1] - 1)

    _m = train_df.merge(clip_tr, on='item_id', how='left').fillna(0)
    cc = pd.Series({c: np.corrcoef(_m[c], _m['abstract'])[0, 1]
                    for c in clip_tr.columns if c != 'item_id'}).sort_values(key=abs, ascending=False)
    print('\nкорреляция CLIP-признаков с abstract (топ-8):')
    print(cc.head(8).round(3).to_string())
    print('\nдля сравнения: лучший геометро-визуальный признак давал по abstract r = +0.51')
else:
    print('CLIP выключен (CFG.use_clip = False)')

                     
_ALL_COLS = [c for c in feat_tr.columns if c != 'item_id' and c in feat_te.columns]
feat_tr[_ALL_COLS] = feat_tr[_ALL_COLS].replace([np.inf, -np.inf], 0).astype(np.float32)
feat_te[_ALL_COLS] = feat_te[_ALL_COLS].replace([np.inf, -np.inf], 0).astype(np.float32)

def dedup_feature_columns():
    """Убираем константные и буквально совпадающие колонки: они только замедляют GBM
    и размывают важность фич между копиями. Сравнение по хешу содержимого колонки."""
    import hashlib
    keep, seen, const, dup = [], {}, [], []
    for c in _ALL_COLS:
        v = np.nan_to_num(feat_tr[c].values.astype(np.float64))
        if v.std() == 0:
            const.append(c); continue
        key = hashlib.md5(np.ascontiguousarray(np.round(v, 9)).tobytes()).hexdigest()
        if key in seen:
            dup.append((c, seen[key])); continue
        seen[key] = c; keep.append(c)
    return {'cols': keep, 'const': const, 'dup': dup}

_fc = cached(f'feat_cols_{CFG.tile}_{FEAT_VER}', dedup_feature_columns)
FEAT_COLS = [c for c in _fc['cols'] if c in feat_te.columns]
print(f'фич всего {len(_ALL_COLS)} -> после дедупликации {len(FEAT_COLS)}')
if _fc['const']:
    print('  константные (выброшены):', ', '.join(_fc['const'][:12]),
          '…' if len(_fc['const']) > 12 else '')
if _fc['dup']:
    print('  дубликаты (выброшены, в скобках — оставленный оригинал):',
          ', '.join(f'{a}({b})' for a, b in _fc['dup'][:12]), '…' if len(_fc['dup']) > 12 else '')
print('подвыборка применялась к', f"{feat_tr['geo_subsampled'].mean():.1%}", 'мешей train')

                     
from iterstrat.ml_stratifiers import MultilabelStratifiedKFold

mskf = MultilabelStratifiedKFold(n_splits=CFG.n_folds, shuffle=True, random_state=CFG.seed)
train_df['fold'] = -1
for k, (_, vi) in enumerate(mskf.split(train_df, train_df[CFG.target_cols].values)):
    train_df.loc[train_df.index[vi], 'fold'] = k
display(train_df.groupby('fold')[CFG.target_cols].mean().round(4))

                     
from torch.utils.data import Dataset, DataLoader

MEAN = np.array([0.485, 0.456, 0.406], dtype=np.float32)
STD  = np.array([0.229, 0.224, 0.225], dtype=np.float32)

def load_views(path, tile):
    """PNG-мозаика -> (6, tile, tile, 3) uint8."""
    a = np.asarray(Image.open(path).convert('RGB'), dtype=np.uint8)
    t = tile
    return np.stack([a[(k // 3) * t:(k // 3 + 1) * t, (k % 3) * t:(k % 3 + 1) * t] for k in range(6)])

class MeshViewDataset(Dataset):
    def __init__(self, df, img_dir, feats, train=True, labels=True):
        self.ids = df['item_id'].tolist()
        self.dir = Path(img_dir)
        self.train = train
        self.labels = df[CFG.target_cols].values.astype(np.float32) if labels else None
        fm = feats.set_index('item_id').reindex(self.ids)[FEAT_COLS].fillna(0.0)
        self.feats = fm.values.astype(np.float32)

    def __len__(self):
        return len(self.ids)

    def _aug(self, v):
                            
        rng = np.random
        if rng.rand() < 0.5:                                                   
            v = v[:, :, ::-1]
        if rng.rand() < 0.3:                                                   
            s = rng.randint(-8, 9, size=2)
            v = np.stack([np.roll(x, s, axis=(0, 1)) for x in v])
        if rng.rand() < 0.15:                                    
            k = rng.randint(6)
            v = v.copy(); v[k] = 255
        v = v.astype(np.float32)
        if rng.rand() < 0.3:                                         
            v = np.clip(v * rng.uniform(0.85, 1.15) + rng.uniform(-15, 15), 0, 255)
        return v

    def __getitem__(self, i):
        p = self.dir / f'{self.ids[i]}.png'
        if p.exists():
            v = load_views(p, CFG.tile)
        else:
            v = np.full((6, CFG.tile, CFG.tile, 3), 255, dtype=np.uint8)
        v = self._aug(v) if self.train else v.astype(np.float32)
        v = ((v / 255.0 - MEAN) / STD).astype(np.float32)
        v = torch.from_numpy(np.ascontiguousarray(v)).permute(0, 3, 1, 2)              
        out = {'views': v, 'feats': torch.from_numpy(self.feats[i]), 'item_id': self.ids[i]}
        if self.labels is not None:
            out['y'] = torch.from_numpy(self.labels[i])
        return out

def views_to_grid(v):
    """(B,6,3,t,t) -> (B,3,2t,3t) — мозаика для режима 'grid'."""
    B, N, C, H, W = v.shape
    v = v.view(B, 2, 3, C, H, W).permute(0, 3, 1, 4, 2, 5).reshape(B, C, 2 * H, 3 * W)
    return v

                     
import timm

class MultiViewNet(nn.Module):
    def __init__(self, n_feats, n_out=CFG.n_targets):
        super().__init__()
        self.mode = CFG.view_mode
        kw = dict(pretrained=True, num_classes=0, drop_rate=CFG.drop_rate)
        if IS_VIT:
                                                                                 
            kw['img_size'] = CFG.tile
        self.backbone = timm.create_model(CFG.backbone, **kw)
        d = self.backbone.num_features

                                                                            
        self.att_v = nn.Sequential(nn.Linear(d, 256), nn.Tanh())
        self.att_u = nn.Sequential(nn.Linear(d, 256), nn.Sigmoid())
        self.att_w = nn.Linear(256, 1)

        self.geo = nn.Sequential(
            nn.BatchNorm1d(n_feats), nn.Linear(n_feats, 256), nn.SiLU(), nn.Dropout(0.2),
            nn.Linear(256, 128), nn.SiLU())
        self.head = nn.Sequential(
            nn.Linear(d + 128, 512), nn.SiLU(), nn.Dropout(0.3), nn.Linear(512, n_out))

    def embed(self, views):
        if self.mode == 'grid':
            return self.backbone(views_to_grid(views)), None
        B, N = views.shape[:2]
        z = self.backbone(views.flatten(0, 1)).view(B, N, -1)
        a = self.att_w(self.att_v(z) * self.att_u(z))                     
        w = torch.softmax(a, dim=1)
        return (z * w).sum(1) + z.max(1).values, w.squeeze(-1)

    def forward(self, views, feats, return_att=False):
        z, w = self.embed(views)
        h = torch.cat([z, self.geo(feats)], 1)
        out = self.head(h)
        return (out, w) if return_att else out

class AsymmetricLoss(nn.Module):
    """Ridnik et al., ICCV 2021."""
    def __init__(self, gamma_neg=4.0, gamma_pos=0.0, clip=0.05, eps=1e-8):
        super().__init__()
        self.gn, self.gp, self.clip, self.eps = gamma_neg, gamma_pos, clip, eps

    def forward(self, logits, y):
        p = torch.sigmoid(logits)
        p_neg = (1 - p + self.clip).clamp(max=1.0)
        loss = y * torch.log(p.clamp(min=self.eps)) + (1 - y) * torch.log(p_neg.clamp(min=self.eps))
        pt = p * y + (1 - p) * (1 - y)
        gamma = self.gp * y + self.gn * (1 - y)
        return -(loss * (1 - pt).pow(gamma)).mean()

                     
from torch.cuda.amp import autocast, GradScaler

def per_class_f1(y, p):
    return pd.Series([f1_score(y[:, i], p[:, i], zero_division=0) for i in range(11)],
                     index=CFG.target_cols)

def quick_thresholds(y, prob):
    """Быстрые per-class пороги (для мониторинга по эпохам)."""
    th = np.full(11, 0.5)
    for i in range(11):
        best, bt = -1, 0.5
        for t in np.arange(0.05, 0.96, 0.02):
            s = f1_score(y[:, i], (prob[:, i] > t).astype(int), zero_division=0)
            if s > best:
                best, bt = s, t
        th[i] = bt
    return th

@torch.no_grad()
def predict(model, loader, tta=1):
    model.eval()
    probs, ids = [], []
    for b in tqdm(loader, desc='predict', leave=False):
        v = b['views'].to(CFG.device, non_blocking=True)
        f = b['feats'].to(CFG.device, non_blocking=True)
        acc = 0
        with autocast(enabled=CFG.amp):
            acc = torch.sigmoid(model(v, f)).float()
            if tta > 1:
                acc = acc + torch.sigmoid(model(torch.flip(v, dims=[-1]), f)).float()
        probs.append((acc / tta).cpu().numpy())
        ids.extend(b['item_id'])
    return np.vstack(probs), ids

def train_fold(fold):
                                                                                 
    if CFG.resume and has_artifact(rt(f'fold{fold}_preds')):
        d = load_artifact(rt(f'fold{fold}_preds'))
        print(f'[кэш] фолд {fold} готов (val {d["best"]:.3f}), обучение пропускаем')
        return (d['oof_ids'], d['oof']), (d['te_ids'], d['te'])

    seed_everything(CFG.seed + fold)
    tr = train_df[train_df.fold != fold].reset_index(drop=True)
    va = train_df[train_df.fold == fold].reset_index(drop=True)

    dtr = MeshViewDataset(tr, IMG_TRAIN, feat_tr, train=True)
    dva = MeshViewDataset(va, IMG_TRAIN, feat_tr, train=False)
    ltr = DataLoader(dtr, batch_size=CFG.batch_size, shuffle=True, drop_last=True,
                     num_workers=CFG.num_workers, pin_memory=True, persistent_workers=True)
    lva = DataLoader(dva, batch_size=CFG.batch_size * 2, shuffle=False,
                     num_workers=CFG.num_workers, pin_memory=True)

    model = MultiViewNet(len(FEAT_COLS)).to(CFG.device)
    head_mods = [model.head, model.geo, model.att_v, model.att_u, model.att_w]
    head_ids = {id(p) for m in head_mods for p in m.parameters()}

    def _block_index(name):
        """Номер блока трансформера в имени параметра (для послойного затухания LR)."""
        m = re.search(r'blocks\.(\d+)\.', name)
        if m:
            return int(m.group(1))
        return -1 if any(k in name for k in ('patch_embed', 'pos_embed', 'cls_token',
                                             'reg_token')) else None

    if IS_VIT:
        n_blocks = len(getattr(model.backbone, 'blocks', []))
        base_lr = CFG.backbone_lr
        buckets = {}
        for name, p in model.backbone.named_parameters():
            bi = _block_index(name)
            depth = 0 if bi == -1 else (bi + 1 if bi is not None else n_blocks)
            scale = CFG.layer_decay ** (n_blocks - depth)
            buckets.setdefault(round(scale, 4), []).append(p)
        groups = [{'params': ps, 'lr': base_lr * sc} for sc, ps in buckets.items()]
        groups.append({'params': [p for m in head_mods for p in m.parameters()],
                       'lr': CFG.lr * CFG.head_lr_mult})
        print(f'  ViT: {n_blocks} блоков, LR от {base_lr * min(buckets):.2e} до {base_lr:.2e}, '
              f'голова {CFG.lr * CFG.head_lr_mult:.2e}')
    else:
        groups = [{'params': [p for p in model.parameters() if id(p) not in head_ids], 'lr': CFG.lr},
                  {'params': [p for p in model.parameters() if id(p) in head_ids],
                   'lr': CFG.lr * CFG.head_lr_mult}]
    opt = torch.optim.AdamW(groups, weight_decay=CFG.weight_decay)

    steps = CFG.epochs * len(ltr)
    warm = int(CFG.warmup_frac * steps)
    sched = torch.optim.lr_scheduler.LambdaLR(
        opt, lambda s: s / max(warm, 1) if s < warm
        else 0.5 * (1 + math.cos(math.pi * (s - warm) / max(steps - warm, 1))))

    pos = tr[CFG.target_cols].sum(0).values.astype(np.float32)
    pw = np.clip((len(tr) - pos) / np.clip(pos, 1, None), 1.0, 20.0)
    bce = nn.BCEWithLogitsLoss(pos_weight=torch.tensor(pw, device=CFG.device))
    asl = AsymmetricLoss()
    scaler = GradScaler(enabled=CFG.amp)

                                                                                    
    snaps, best, start_ep = [], -1, 0
    ck = load_ckpt(fold)
    if ck is not None:
        model.load_state_dict(_to_fp32(ck['model']))
        try:
            opt.load_state_dict(ck['opt']); sched.load_state_dict(ck['sched'])
            scaler.load_state_dict(ck['scaler'])
        except Exception as e:
            print('  состояние оптимизатора не восстановлено, продолжаем с текущим:', e)
        snaps = [(sc_i, _to_fp32(sd)) for sc_i, sd in ck['snaps']]
        best, start_ep = ck['best'], ck['epoch'] + 1
        set_rng_state(ck['rng'])
        print(f'  продолжаем фолд {fold} с эпохи {start_ep + 1}/{CFG.epochs} '
              f'(лучшая метрика пока {best:.3f})')
        if start_ep >= CFG.epochs:
            print('  все эпохи пройдены, остаётся только инференс')

    yv = va[CFG.target_cols].values
    pred = None
    for ep in range(start_ep, CFG.epochs):
        model.train(); tot = 0.0
        pbar = tqdm(ltr, desc=f'fold{fold} ep{ep + 1}/{CFG.epochs}')
        for b in pbar:
            v = b['views'].to(CFG.device, non_blocking=True)
            f = b['feats'].to(CFG.device, non_blocking=True)
            y = b['y'].to(CFG.device, non_blocking=True)
            y = y * (1 - CFG.label_smooth) + 0.5 * CFG.label_smooth
                                                                                          
                                                                                    
                                                  
            if CFG.mixup > 0 and random.random() < 0.5:
                lam = float(np.random.beta(CFG.mixup, CFG.mixup))
                perm = torch.randperm(v.size(0), device=v.device)
                v = lam * v + (1 - lam) * v[perm]
                f = lam * f + (1 - lam) * f[perm]
                y = lam * y + (1 - lam) * y[perm]
            opt.zero_grad(set_to_none=True)
            with autocast(enabled=CFG.amp):
                logits = model(v, f)
                loss = (1 - CFG.asl_weight) * bce(logits, y) + CFG.asl_weight * asl(logits, y)
            scaler.scale(loss).backward()
            scaler.unscale_(opt)
            torch.nn.utils.clip_grad_norm_(model.parameters(), 5.0)
            scaler.step(opt); scaler.update(); sched.step()
            tot += loss.item(); pbar.set_postfix(loss=f'{tot / (pbar.n + 1):.4f}')

        prob, _ = predict(model, lva, tta=1)
        th = quick_thresholds(yv, prob)
        pred = (prob > th).astype(int)
        sc, fq, fa = competition_metric(yv, pred)
        print(f'  ep{ep + 1}: metric={sc:.3f} (quality {fq:.3f} | artefacts {fa:.3f})')
        if ep >= 2:                                        
            snaps.append((sc, {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}))
            snaps.sort(key=lambda x: -x[0])
            del snaps[CFG.n_snapshots:]
        if sc > best:
            best = sc
            print('   ^ best')

        log_epoch({'fold': fold, 'epoch': ep + 1, 'train_loss': tot / max(len(ltr), 1),
                   'val_metric': sc, 'f1_quality': fq, 'f1_artefacts': fa,
                   'lr': opt.param_groups[0]['lr'], 'time': time.strftime('%Y-%m-%d %H:%M:%S'),
                   **{f'f1_{c}': float(v) for c, v in per_class_f1(yv, pred).items()}})

        if (ep + 1) % CFG.ckpt_every == 0 or ep == CFG.epochs - 1:
            conv = _to_fp16 if CFG.ckpt_fp16 else (lambda x: x)
            save_ckpt(fold, {'epoch': ep, 'best': best, 'rng': rng_state(),
                             'model': conv(model.state_dict()),
                             'opt': opt.state_dict(), 'sched': sched.state_dict(),
                             'scaler': scaler.state_dict(),
                             'snaps': [(sc_i, conv(sd)) for sc_i, sd in snaps],
                             'cfg': {'tile': CFG.tile, 'backbone': CFG.backbone,
                                     'epochs': CFG.epochs, 'feat_ver': FEAT_VER}})
            print(f'   чекпоинт сохранён (эпоха {ep + 1})')

    if pred is None:                                                              
        prob, _ = predict(model, lva, tta=1)
        pred = (prob > quick_thresholds(yv, prob)).astype(int)
        best = max(best, competition_metric(yv, pred)[0])
    if not snaps:                                                             
        snaps = [(best, {k: v.detach().cpu().clone() for k, v in model.state_dict().items()})]
    print(f'fold {fold}: лучшая валидационная метрика {best:.3f}, '
          f'в ансамбле эпохи со скорами {[round(x[0], 2) for x in snaps]}')
    print(per_class_f1(yv, pred).round(3).to_string())

    _atomic(CFG.work / f'{CFG.run_tag}_cnn_fold{fold}.pth',
            lambda p: torch.save(_to_fp16(snaps[0][1]) if CFG.ckpt_fp16 else snaps[0][1], p))

    dte = MeshViewDataset(test_df.assign(**{c: 0 for c in CFG.target_cols}), IMG_TEST,
                          feat_te, train=False, labels=False)
    lte = DataLoader(dte, batch_size=CFG.batch_size * 2, shuffle=False,
                     num_workers=CFG.num_workers, pin_memory=True)

                                                                                                 
    oof_acc, te_acc = 0.0, 0.0
    for k, (sc_k, st) in enumerate(snaps):
        model.load_state_dict(st)
        op, oof_ids = predict(model, lva, tta=CFG.tta)
        tp, te_ids = predict(model, lte, tta=CFG.tta)
        oof_acc = oof_acc + op; te_acc = te_acc + tp
    oof_prob, te_prob = oof_acc / len(snaps), te_acc / len(snaps)
    sc_ens = competition_metric(yv, (oof_prob > quick_thresholds(yv, oof_prob)).astype(int))[0]
    print(f'fold {fold}: одна эпоха {best:.3f} -> snapshot-ансамбль {sc_ens:.3f}')

    keep_snapshots(fold, snaps)
    save_artifact(rt(f'fold{fold}_preds'), {'oof': oof_prob, 'oof_ids': list(oof_ids),
                                        'te': te_prob, 'te_ids': list(te_ids),
                                        'best': best, 'ens': sc_ens,
                                        'snap_scores': [x[0] for x in snaps]})
    drop_ckpt(fold)                                                                

    del model, ltr, lva, snaps; gc.collect(); torch.cuda.empty_cache()
    return (oof_ids, oof_prob), (te_ids, te_prob)


In [ ]:
%%writefile /content/meshqc_code/v5_224_stages.json
[
  {
    "source_cell": 6,
    "source": "import os, gc, json, math, random, re, warnings, shutil\nfrom pathlib import Path\nimport numpy as np\nimport pandas as pd\nimport torch\nimport torch.nn as nn\nimport torch.nn.functional as F\nfrom tqdm.auto import tqdm\nimport matplotlib.pyplot as plt\n\nwarnings.filterwarnings('ignore')\n\nclass CFG:\n    seed = 42\n\n    # ---- данные ----\n    dataset  = 'daniilantonov5/3d-mesh-quality-control'\n    work     = LOCAL_DIR; work.mkdir(parents=True, exist_ok=True)\n    cache    = work / 'cache';        cache.mkdir(parents=True, exist_ok=True)\n\n    # ---- вид входа для CNN ----\n    # 'grid'      : одна мозаика 2x3 из 6 рендеров -> 1 forward на объект (быстро, дефолт)\n    # 'multiview' : 6 отдельных видов + attention-pooling (точнее, но x6 дороже)\n    view_mode = 'multiview'   # для ViT только так: 6 квадратных видов вместо неквадратной мозаики\n    tile      = 224          # 224 = 14*16 патчей DINOv2; для свёрточных бэкбонов можно 288\n    grid_rows, grid_cols = 2, 3\n\n    # ---- CNN ----\n    # Self-supervised ViT. DINOv2 обучен на 142M изображений без меток; его признаки известны\n    # сильной линейной разделимостью по семантике формы — а abstract/simple/set это ровно\n    # семантические категории. Патч 14, поэтому tile должен быть кратен 14 (224 = 14*16).\n    backbone   = 'vit_small_patch14_reg4_dinov2.lvd142m'\n    # альтернативы: 'vit_base_patch14_reg4_dinov2.lvd142m' (втрое дороже, для второго прогона),\n    #               'convnext_tiny.fb_in22k_ft_in1k' (прежний, для диверсификации ансамбля)\n    backbone_lr = 4e-5        # ViT после SSL-претрейна требует малого LR, иначе признаки ломаются\n    layer_decay = 0.75        # послойное затухание LR: нижние блоки почти замораживаются\n    drop_rate  = 0.2\n    epochs     = 16\n    batch_size = 12\n    mixup      = 0.4         # 0 = выключить; мультилейбл-mixup по видам, фичам и меткам\n\n    # ---- дополнительные источники признаков ----\n    use_clip   = True         # zero-shot скоры CLIP -> признаки (прежде всего для abstract)\n    clip_model = ('ViT-B-32', 'laion2b_s34b_b79k')\n    use_dino_frozen = True    # замороженные эмбеддинги DINOv2 -> отдельный источник в бленде\n    dino_frozen_model = 'vit_base_patch14_reg4_dinov2.lvd142m'\n    dino_pca   = 128\n    lr         = 3e-4\n    head_lr_mult = 10.0\n    weight_decay = 0.05\n    warmup_frac  = 0.1\n    label_smooth = 0.01\n    amp        = True\n    num_workers = 4\n\n    # ---- CV ----\n    n_folds = 5\n    RUN_ALL_FOLDS = True         # <-- False для быстрой проверки на одном фолде\n    folds_to_run  = [0]\n\n    # ---- инференс ----\n    tta = 2                      # 1 = без TTA, 2 = + горизонтальный флип каждого вида\n    n_snapshots = 3              # усреднение предсказаний 3 лучших эпох (snapshot ensemble)\n    run_tag = 'dinos224_geo2_v5'  # версия 5: новые геометрические фичи меняют вход model.geo,\n                                 # поэтому веса прошлых прогонов несовместимы -> новый тег\n    ensemble_tags = ['dinos224_geo2_v5']  # прошлые прогоны (cnx288_v4, dinos224) считались в другой\n                                 # папке на диске; чтобы включить их в бленд §9.2 — скопируйте их\n                                 # *.joblib из старого artifacts/ в новый (см. use_drive/drive_dir ниже)\n                                 # и допишите их теги в этот список\n    asl_weight = 0.65            # вес AsymmetricLoss в блэнде с BCE (было зашито 0.5/0.5);\n                                 # выше -> сильнее давит на лёгкие негативы, что должно помогать\n                                 # редким/несбалансированным классам без отдельного oversampling\n\n    # ---- устойчивость к перезапускам ----\n    use_drive   = True    # хранить кэш и чекпоинты на Google Drive: /content стирается вместе с рантаймом\n    drive_dir   = '/content/drive/MyDrive/sber_meshqc_v5'  # НОВАЯ папка: прогон 4 остаётся\n                                 # нетронутым в 'sber_meshqc', этот прогон живёт отдельно и\n                                 # полностью резюмируется сам по себе\n    resume      = True    # продолжать с последнего чекпоинта, а не считать заново\n    ckpt_every  = 1       # сохранять состояние обучения раз в N эпох\n    ckpt_fp16   = True    # веса в чекпоинте в half: вчетверо меньше места на Drive\n    keep_fold_ckpt = False  # удалять промежуточный чекпоинт фолда после его завершения\n    force_recompute = []  # имена артефактов, которые пересчитать принудительно, напр. ['gbm']\n\n    # ---- метки ----\n    artifact_cols = ['abstract','artifacts','intersection','lowpoly','noisy',\n                     'open','partial','scale','set','simple']\n    target_cols   = artifact_cols + ['quality']\n    n_targets     = 11\n\n    device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')"
  },
  {
    "source_cell": 10,
    "source": "import kagglehub\n\nroot = Path(kagglehub.dataset_download(CFG.dataset))\nprint('dataset root:', root)\n\ndef _pick_dir(mode):\n    \"\"\"Папка с максимальным числом .npz, в пути которой встречается train/test.\"\"\"\n    best, best_n = None, -1\n    for p in root.rglob('*'):\n        if not p.is_dir():\n            continue\n        if mode not in str(p).lower():\n            continue\n        n = sum(1 for _ in p.glob('*.npz'))\n        if n > best_n:\n            best, best_n = p, n\n    if best is None or best_n == 0:\n        raise FileNotFoundError(f'не найдена папка с .npz для {mode}')\n    return best\n\ndef _pick_csv(mode):\n    cands = list(root.rglob('*.csv'))\n    def score(p):\n        n = p.name.lower(); s = 0\n        if mode == 'train':\n            s += 10 * ('train' in n) - 20 * ('submission' in n) - 20 * ('test' in n)\n        else:\n            s += 10 * ('test' in n) - 5 * ('submission' in n) - 20 * ('train' in n)\n        return s - 0.001 * len(n)\n    return sorted(cands, key=score, reverse=True)[0]\n\nTRAIN_DIR, TEST_DIR = _pick_dir('train'), _pick_dir('test')\nTRAIN_CSV, TEST_CSV = _pick_csv('train'), _pick_csv('test')\nprint('train dir :', TRAIN_DIR, len(list(TRAIN_DIR.glob('*.npz'))), 'npz /',\n      len(list(TRAIN_DIR.glob('*.png'))), 'png')\nprint('test  dir :', TEST_DIR,  len(list(TEST_DIR.glob('*.npz'))),  'npz /',\n      len(list(TEST_DIR.glob('*.png'))),  'png')\nprint('train csv :', TRAIN_CSV)\nprint('test  csv :', TEST_CSV)\n\ntrain_df = pd.read_csv(TRAIN_CSV)\ntest_df  = pd.read_csv(TEST_CSV)[['item_id']].copy()\ntrain_df['item_id'] = train_df['item_id'].astype(str)\ntest_df['item_id']  = test_df['item_id'].astype(str)\n\nif 'quality' not in train_df.columns:\n    train_df['quality'] = (train_df[CFG.artifact_cols].sum(1) == 0).astype(int)\n\nprint(train_df.shape, test_df.shape)\ntrain_df.head()"
  },
  {
    "source_cell": 15,
    "source": "from sklearn.metrics import f1_score\n\ndef competition_metric(y_true, y_pred):\n    \"\"\"10*F1(quality) + 10*F1_weighted(10 дефектов) — точная формула организаторов.\"\"\"\n    f1_q = f1_score(y_true[:, 10], y_pred[:, 10], zero_division=0)\n    f1_a = f1_score(y_true[:, :10], y_pred[:, :10], average='weighted', zero_division=0)\n    return 10 * f1_q + 10 * f1_a, f1_q, f1_a\n\n# сколько стоит \"OR-правило\" quality = нет дефектов, если по дефектам ошибаться независимо\nY = train_df[CFG.target_cols].values\nfor err in [0.02, 0.05, 0.10]:\n    rng = np.random.default_rng(0)\n    fake = Y.copy()\n    flip = rng.random(fake[:, :10].shape) < err\n    fake[:, :10] = np.abs(fake[:, :10] - flip)\n    fake[:, 10] = (fake[:, :10].sum(1) == 0).astype(int)\n    s, fq, fa = competition_metric(Y, fake)\n    print(f'FPR/FNR по дефектам {err:.0%} -> метрика {s:5.2f} (quality F1={fq:.3f}, artefacts F1={fa:.3f})')\nprint('\\n>>> Ошибка 5% по дефектам уже срезает quality F1 до ~0.7. '\n      'Поэтому quality предсказывается ОТДЕЛЬНОЙ головой, а не только правилом.')"
  },
  {
    "source_cell": 17,
    "source": "from PIL import Image\nfrom concurrent.futures import ProcessPoolExecutor\n\nImage.MAX_IMAGE_PIXELS = None\n\ndef infer_grid(w, h):\n    \"\"\"(rows, cols) для 6 видов по соотношению сторон.\"\"\"\n    ar = w / h\n    cands = {(2, 3): 3 / 2, (3, 2): 2 / 3, (1, 6): 6.0, (6, 1): 1 / 6}\n    return min(cands, key=lambda k: abs(math.log(ar / cands[k])))\n\ndef split_views(img):\n    \"\"\"PIL.Image -> список из 6 PIL.Image в каноническом порядке чтения.\"\"\"\n    w, h = img.size\n    r, c = infer_grid(w, h)\n    tw, th = w // c, h // r\n    return [img.crop((j * tw, i * th, (j + 1) * tw, (i + 1) * th)) for i in range(r) for j in range(c)]\n\ndef make_cache(args):\n    src, dst, tile = args\n    try:\n        img = Image.open(src).convert('RGB')\n        views = [v.resize((tile, tile), Image.BILINEAR) for v in split_views(img)]\n        out = Image.new('RGB', (3 * tile, 2 * tile))\n        for k, v in enumerate(views):\n            out.paste(v, ((k % 3) * tile, (k // 3) * tile))\n        out.save(dst, format='PNG', optimize=False, compress_level=1)\n        return 1\n    except Exception:\n        return 0\n\ndef build_image_cache(ids, src_dir, split):\n    out_dir = CFG.cache / f'img_{split}_{CFG.tile}'\n    out_dir.mkdir(parents=True, exist_ok=True)\n    jobs = [(str(Path(src_dir) / f'{i}.png'), str(out_dir / f'{i}.png'), CFG.tile)\n            for i in ids if not (out_dir / f'{i}.png').exists()\n            and (Path(src_dir) / f'{i}.png').exists()]\n    if jobs:\n        with ProcessPoolExecutor(max_workers=os.cpu_count()) as ex:\n            ok = list(tqdm(ex.map(make_cache, jobs, chunksize=32), total=len(jobs),\n                           desc=f'cache {split}'))\n        print(f'{split}: закэшировано {sum(ok)}/{len(jobs)}')\n    missing = [i for i in ids if not (out_dir / f'{i}.png').exists()]\n    print(f'{split}: всего в кэше {len(ids) - len(missing)}/{len(ids)}, нет картинки у {len(missing)}')\n    return out_dir, set(missing)\n\n# --- сколько пикселей на вид даёт исходный рендер: апскейл выше этого бессмысленно ---\n_probe = [Image.open(Path(TRAIN_DIR) / f'{i}.png').size\n          for i in train_df['item_id'].head(40) if (Path(TRAIN_DIR) / f'{i}.png').exists()]\nif _probe:\n    _sizes = pd.Series([f'{w}x{h}' for w, h in _probe]).value_counts()\n    print('исходные размеры листа с 6 видами:'); print(_sizes.head(5).to_string())\n    _w, _h = _probe[0]\n    _r, _c = infer_grid(_w, _h)\n    _native = min(_w // _c, _h // _r)\n    print(f'-> раскладка {_r}x{_c}, нативный размер одного вида ~{_native} px')\n    if CFG.tile > _native:\n        print(f'!! CFG.tile = {CFG.tile} БОЛЬШЕ нативного {_native}: это апскейл, он удорожает '\n              f'обучение, но новой информации не даёт. Снижаю tile до {_native}.')\n        CFG.tile = int(_native)\n    else:\n        print(f'   CFG.tile = {CFG.tile} <= нативного — детали сохраняются')\n\nIMG_TRAIN, MISS_TRAIN = build_image_cache(train_df['item_id'].tolist(), TRAIN_DIR, 'train')\nIMG_TEST,  MISS_TEST  = build_image_cache(test_df['item_id'].tolist(),  TEST_DIR,  'test')"
  },
  {
    "source_cell": 20,
    "source": "import os, gc, math, numpy as np, pandas as pd\nfrom pathlib import Path\nfrom PIL import Image\nfrom scipy import ndimage\nfrom scipy.sparse import coo_matrix\nfrom scipy.sparse.csgraph import connected_components\nfrom scipy.spatial import ConvexHull, QhullError\nfrom concurrent.futures import ProcessPoolExecutor\nfrom concurrent.futures.process import BrokenProcessPool\nimport multiprocessing as mp\n\nFACE_CAP  = 300_000\nVERT_CAP  = 200_000\nHULL_CAP  = 20_000\nCOMP_CAP  = 3_000_000\nWELD_TOL  = 1e-6         # доля диагонали bbox\nFEAT_VER  = 'v4selfint'  # версия набора фич: входит в имя кэша. Поднята с v3weld, т.к.\n                         # добавлены §4.5.1-4.5.3 (самопересечения/качество треугольников/\n                         # non-manifold вершины) — старый parquet без них не подхватится молча\nN_WORKERS = min(4, os.cpu_count() or 2)\nCHUNK     = 1024\n\ndef _pack21(q):\n    \"\"\"Три целых из [0, 2**21) -> один int64. Точно, без коллизий.\"\"\"\n    return (q[:, 0] << 42) | (q[:, 1] << 21) | q[:, 2]\n\ndef _hash3(q):\n    q = q.astype(np.int64, copy=False)\n    return (q[:, 0] * 73856093) ^ (q[:, 1] * 19349663) ^ (q[:, 2] * 83492791)\n\ndef _stats(x, pref):\n    keys = ['mean', 'std', 'p10', 'p50', 'p90', 'max', 'cv']\n    if len(x) == 0:\n        return {f'{pref}_{k}': 0.0 for k in keys}\n    x = np.asarray(x, dtype=np.float32)\n    m, s = float(x.mean()), float(x.std())\n    q10, q50, q90 = np.percentile(x, [10, 50, 90])\n    return {f'{pref}_mean': m, f'{pref}_std': s, f'{pref}_p10': float(q10),\n            f'{pref}_p50': float(q50), f'{pref}_p90': float(q90),\n            f'{pref}_max': float(x.max()), f'{pref}_cv': float(s / (abs(m) + 1e-9))}\n\ndef _edge_table(Fi, nfw):\n    \"\"\"Рёбра -> (номер группы, число граней на ребро, отсортированные id граней).\"\"\"\n    E = np.concatenate([Fi[:, [0, 1]], Fi[:, [1, 2]], Fi[:, [2, 0]]], 0)\n    E.sort(axis=1)\n    fid = np.tile(np.arange(nfw, dtype=np.int32), 3)\n    order = np.lexsort((E[:, 1], E[:, 0]))\n    Es, fids = E[order], fid[order]\n    new = np.ones(len(Es), dtype=bool)\n    new[1:] = (Es[1:] != Es[:-1]).any(1)\n    grp = np.cumsum(new) - 1\n    counts = np.bincount(grp)\n    return E, Es, fids, grp, counts, new\n\ndef mesh_features(npz_path):\n    f = {'geo_ok': 0.0, 'geo_subsampled': 0.0}\n    try:\n        f['npz_mb'] = os.path.getsize(npz_path) / 1e6\n    except Exception:\n        return f\n    try:\n        with np.load(npz_path, allow_pickle=False) as d:\n            keys = list(d.keys())\n            V = np.asarray(d['vertices' if 'vertices' in keys else keys[0]], dtype=np.float32)\n            Fc = None\n            for k in ('faces', 'triangles', 'f'):\n                if k in keys:\n                    Fc = np.asarray(d[k], dtype=np.int64); break\n            if Fc is None and len(keys) > 1:\n                Fc = np.asarray(d[keys[1]], dtype=np.int64)\n    except Exception:\n        return f\n    if V.ndim != 2 or V.shape[1] != 3 or len(V) == 0:\n        return f\n    if Fc is None or Fc.ndim != 2 or Fc.shape[1] != 3 or len(Fc) == 0:\n        Fc = np.zeros((0, 3), dtype=np.int64)\n    else:\n        Fc = Fc[(Fc >= 0).all(1) & (Fc < len(V)).all(1)]\n\n    f['geo_ok'] = 1.0\n    nv_raw, nf_raw = len(V), len(Fc)\n    vmin, vmax = V.min(0), V.max(0)\n    ext = (vmax - vmin).astype(np.float64)\n    ext_s = np.sort(ext)[::-1]\n    diag = float(np.linalg.norm(ext)) + 1e-12\n    center = ((vmin + vmax) / 2).astype(np.float32)\n    inv_s = np.float32(1.0 / diag)\n    f.update({'n_verts_raw': float(nv_raw), 'n_faces': float(nf_raw),\n              'log_verts': math.log1p(nv_raw), 'log_faces': math.log1p(nf_raw),\n              'ext_max': float(ext_s[0]), 'ext_mid': float(ext_s[1]), 'ext_min': float(ext_s[2]),\n              'ext_ratio_min': float(ext_s[2] / (ext_s[0] + 1e-12)),\n              'ext_ratio_mid': float(ext_s[1] / (ext_s[0] + 1e-12)),\n              'bbox_diag': diag,\n              'bbox_fill': float(nv_raw / (ext.prod() + 1e-12)) if ext.prod() > 0 else 0.0})\n\n    # ================== СВАРКА ВЕРШИН ==================\n    # Экспортёры (glTF/OBJ с раздельными нормалями и UV) дублируют вершины на каждую грань.\n    # Без сварки НИ ОДНО ребро не имеет двух граней: двугранные углы пусты, boundary=1.0,\n    # компонент столько же, сколько треугольников. Вся топология обязана считаться после сварки.\n    step = np.float32(diag * WELD_TOL)\n    q = np.clip(np.rint((V - vmin) / step), 0, (1 << 21) - 1).astype(np.int64)\n    code = _pack21(q)\n    del q\n    ucode, first, invmap = np.unique(code, return_index=True, return_inverse=True)\n    del code, ucode\n    nv = len(first)\n    f['n_verts'] = float(nv)\n    f['weld_ratio'] = float(1.0 - nv / max(nv_raw, 1))          # 0 = уже сварен, ~0.83 = по 3 вершины на грань\n    f['f_per_v'] = nf_raw / max(nv, 1)\n    Vw_full = V[first]\n    del first\n\n    if nf_raw:\n        Fw = invmap[Fc]\n        ok = (Fw[:, 0] != Fw[:, 1]) & (Fw[:, 1] != Fw[:, 2]) & (Fw[:, 0] != Fw[:, 2])\n        f['degen_after_weld'] = float(1.0 - ok.mean())\n        Fw = Fw[ok]\n        hf = _hash3(np.sort(Fw, axis=1))\n        _, uidx = np.unique(hf, return_index=True)\n        f['dup_face_ratio'] = float(1.0 - len(uidx) / max(len(Fw), 1))\n        Fw = Fw[np.sort(uidx)]\n        del hf, uidx, ok\n    else:\n        Fw = np.zeros((0, 3), dtype=np.int64)\n        f['degen_after_weld'] = 0.0; f['dup_face_ratio'] = 0.0\n    del invmap, Fc\n    nf = len(Fw)\n\n    # ---- вершинные статистики (подвыборка) ----\n    Vs = Vw_full[::max(1, int(np.ceil(nv / VERT_CAP)))]\n    if nv > VERT_CAP:\n        f['geo_subsampled'] = 1.0\n    Vs = (Vs - center) * inv_s\n    try:\n        if len(Vs) < 3:\n            raise ValueError('too few vertices')\n        ev = np.clip(np.linalg.eigvalsh(np.cov(Vs.T.astype(np.float64)))[::-1], 1e-16, None)\n        f.update({'pca_1': float(ev[0]), 'pca_2': float(ev[1]), 'pca_3': float(ev[2]),\n                  'pca_flat': float(ev[2] / ev[0]), 'pca_lin': float(ev[1] / ev[0]),\n                  'pca_aniso': float((ev[0] - ev[2]) / ev.sum())})\n    except Exception:\n        f.update({k: 0.0 for k in ['pca_1', 'pca_2', 'pca_3', 'pca_flat', 'pca_lin', 'pca_aniso']})\n\n    # ---- выпуклая оболочка: solidity/сферичность -> simple, open ----\n    f.update({'hull_ok': 0.0, 'solidity': 0.0, 'solidity_signed': 0.0, 'hull_area_ratio': 0.0,\n              'hull_pts_frac': 0.0, 'sphericity': 0.0, 'hull_vol': 0.0, 'hull_area': 0.0})\n    try:\n        Vh = Vs[::max(1, int(np.ceil(len(Vs) / HULL_CAP)))].astype(np.float64)\n        if len(Vh) >= 8:\n            hull = ConvexHull(Vh, qhull_options='QJ')\n            f['hull_ok'] = 1.0\n            f['hull_vol'] = float(hull.volume)\n            f['hull_pts_frac'] = len(hull.vertices) / len(Vh)\n            f['hull_area'] = float(hull.area)\n    except (QhullError, ValueError, MemoryError):\n        pass\n\n    if nf == 0:\n        f['no_faces'] = 1.0\n        f.update({'area_total': 0.0, 'volume': 0.0, 'abs_volume': 0.0, 'vol_ratio': 0.0,\n                  'vol_over_area': 0.0, 'absvol_over_area': 0.0, 'vol_over_bbox': 0.0,\n                  'absvol_over_bbox': 0.0})\n        for pref in ('dihed', 'dihedf', 'farea', 'elen', 'aspect', 'valence', 'defect'):\n            f.update(_stats([], pref))\n        return f\n    f['no_faces'] = 0.0\n\n    # ---- точные площадь/объём по полному сваренному мешу ----\n    total_area, vol6, absvol6 = 0.0, 0.0, 0.0\n    for s0 in range(0, nf, 500_000):\n        Pc = ((Vw_full[Fw[s0:s0 + 500_000]] - center) * inv_s).astype(np.float64)\n        cr_c = np.cross(Pc[:, 1] - Pc[:, 0], Pc[:, 2] - Pc[:, 0])\n        total_area += 0.5 * float(np.linalg.norm(cr_c, axis=1).sum())\n        contrib = (Pc[:, 0] * np.cross(Pc[:, 1], Pc[:, 2])).sum(1)\n        vol6 += float(contrib.sum()); absvol6 += float(np.abs(contrib).sum())\n        del Pc, cr_c, contrib\n    total_area += 1e-12\n    vol = abs(vol6 / 6.0)\n    absvol = absvol6 / 6.0\n    # vol_ratio ~1 у корректно ориентированного меша и ~0 при несогласованной намотке граней\n    # (сама по себе сильная улика для artifacts/noisy). Объём для solidity берём устойчивый.\n    f.update({'area_total': total_area, 'volume': vol, 'abs_volume': absvol,\n              'vol_ratio': vol / (absvol + 1e-12),\n              'vol_over_area': vol / (total_area ** 1.5 + 1e-12),\n              'absvol_over_area': absvol / (total_area ** 1.5 + 1e-12),\n              'vol_over_bbox': vol / (float(np.prod(ext * inv_s)) + 1e-12),\n              'absvol_over_bbox': absvol / (float(np.prod(ext * inv_s)) + 1e-12)})\n    if f['hull_ok']:\n        f['solidity'] = absvol / (f['hull_vol'] + 1e-12)\n        f['solidity_signed'] = vol / (f['hull_vol'] + 1e-12)\n        f['hull_area_ratio'] = total_area / (f.get('hull_area', 0.0) + 1e-12)\n        f['sphericity'] = (math.pi ** (1 / 3)) * (6 * absvol) ** (2 / 3) / (total_area + 1e-12)\n\n    # ---- компоненты связности на полном СВАРЕННОМ меше ----\n    f.update({'n_comp': 1.0, 'n_comp_big': 1.0, 'comp_top1': 1.0, 'comp_top2': 0.0,\n              'comp_entropy': 0.0, 'bbox_iou_max': 0.0, 'bbox_iou_mean': 0.0,\n              'bbox_iou_frac_pos': 0.0, 'comp_sep_max': 0.0, 'comp_sep_mean': 0.0,\n              'inside_frac_max': 0.0, 'comp_exact': 1.0})\n    if nf <= COMP_CAP:\n        try:\n            Ef = np.concatenate([Fw[:, [0, 1]], Fw[:, [1, 2]], Fw[:, [2, 0]]], 0).astype(np.int32)\n            A = coo_matrix((np.ones(len(Ef), dtype=np.int8), (Ef[:, 0], Ef[:, 1])), shape=(nv, nv))\n            del Ef\n            ncomp, lab = connected_components(A, directed=False)\n            del A\n            used = np.unique(Fw)\n            sizes_all = np.bincount(lab[used], minlength=ncomp).astype(np.float64)\n            sizes = np.sort(sizes_all[sizes_all > 0])[::-1]\n            tot = sizes.sum()\n            big_idx = np.where(sizes_all > 0.005 * tot)[0]\n            f.update({'n_comp': float(len(sizes)), 'n_comp_big': float(len(big_idx)),\n                      'comp_top1': float(sizes[0] / tot),\n                      'comp_top2': float(sizes[1] / tot) if len(sizes) > 1 else 0.0,\n                      'comp_entropy': float(-((sizes / tot) * np.log(sizes / tot + 1e-12)).sum())})\n            if len(big_idx) > 1:\n                top = big_idx[np.argsort(-sizes_all[big_idx])][:8]\n                pts_l, boxes = [], []\n                for c in top:\n                    ic = used[lab[used] == c]\n                    if len(ic) > 20_000:\n                        ic = ic[::len(ic) // 20_000 + 1]\n                    p = (Vw_full[ic] - center) * inv_s\n                    pts_l.append(p); boxes.append((p.min(0), p.max(0)))\n                ious, seps, insides = [], [], []\n                for i in range(len(boxes)):\n                    for j in range(len(boxes)):\n                        if i == j:\n                            continue\n                        lo, hi = boxes[j]\n                        insides.append(float(((pts_l[i] >= lo) & (pts_l[i] <= hi)).all(1).mean()))\n                        if j <= i:\n                            continue\n                        lo2 = np.maximum(boxes[i][0], boxes[j][0])\n                        hi2 = np.minimum(boxes[i][1], boxes[j][1])\n                        it = float(np.prod(np.clip(hi2 - lo2, 0, None)))\n                        vi = float(np.prod(np.clip(boxes[i][1] - boxes[i][0], 1e-6, None)))\n                        vj = float(np.prod(np.clip(boxes[j][1] - boxes[j][0], 1e-6, None)))\n                        ious.append(it / (vi + vj - it + 1e-12))\n                        seps.append(float(np.linalg.norm(\n                            (boxes[i][0] + boxes[i][1]) / 2 - (boxes[j][0] + boxes[j][1]) / 2)))\n                f.update({'bbox_iou_max': float(max(ious)), 'bbox_iou_mean': float(np.mean(ious)),\n                          'bbox_iou_frac_pos': float(np.mean(np.asarray(ious) > 1e-6)),\n                          'comp_sep_max': float(max(seps)), 'comp_sep_mean': float(np.mean(seps)),\n                          'inside_frac_max': float(max(insides))})\n            del lab, sizes_all, sizes, used\n        except Exception:\n            f['comp_exact'] = 0.0\n    else:\n        f['comp_exact'] = 0.0\n\n    # ---- пространственный патч для рёберной топологии ----\n    if nf > FACE_CAP:\n        f['geo_subsampled'] = 1.0\n        anchor = (Vw_full[Fw[:, 0]] - vmin) / (ext.astype(np.float32) + 1e-9)\n        Fp, g = None, 1\n        while g < 40:\n            g += 1\n            cid = np.minimum((anchor * g).astype(np.int32), g - 1)\n            key = (cid[:, 0] * g + cid[:, 1]) * g + cid[:, 2]\n            cnt = np.bincount(key, minlength=g ** 3)\n            best = int(np.argmax(cnt))\n            if cnt[best] <= FACE_CAP:\n                Fp = Fw[key == best]; break\n        if Fp is None or len(Fp) < 1000:\n            Fp = Fw[:FACE_CAP]\n        f['patch_grid'] = float(g)\n        del anchor\n    else:\n        Fp = Fw\n        f['patch_grid'] = 1.0\n    f['patch_face_frac'] = len(Fp) / max(nf, 1)\n    del Fw\n    nfw = len(Fp)\n\n    uvp, Fi = np.unique(Fp, return_inverse=True)\n    Fi = Fi.reshape(nfw, 3).astype(np.int32)\n    Vp = ((Vw_full[uvp] - center) * inv_s).astype(np.float32)\n    nvp = len(uvp)\n    del uvp, Fp, Vw_full, V\n\n    P = Vp[Fi]\n    cr = np.cross(P[:, 1] - P[:, 0], P[:, 2] - P[:, 0])\n    area2 = np.linalg.norm(cr, axis=1)\n    area = 0.5 * area2\n    nrm = cr / (area2[:, None] + 1e-12)\n    del cr\n    f.update(_stats(area / (area.mean() + 1e-12), 'farea'))\n    f['degenerate_ratio'] = float((area < 1e-10).mean())\n\n    L = np.stack([np.linalg.norm(P[:, 1] - P[:, 0], axis=1),\n                  np.linalg.norm(P[:, 2] - P[:, 1], axis=1),\n                  np.linalg.norm(P[:, 0] - P[:, 2], axis=1)], 1)\n    f.update(_stats(L.ravel() / (L.mean() + 1e-12), 'elen'))\n    aspect = L.max(1) / (L.min(1) + 1e-12)\n    f.update(_stats(np.log1p(aspect), 'aspect'))\n    f['sliver_ratio'] = float((aspect > 20).mean())\n\n    # ---- §4.5.2 качество треугольников: худший угол + нормализованная форма ----\n    # min-угол — классический индикатор \"плохого\" элемента (aspect ratio его не всегда\n    # ловит: длинный тупоугольный треугольник может иметь умеренный aspect, но острый\n    # min-угол). tri_quality = 4*sqrt(3)*area/(l0^2+l1^2+l2^2) -> 1.0 для равностороннего,\n    # ->0 для вырожденного; оба считаются векторно из уже посчитанных L и area, без\n    # дополнительного прохода по мешу.\n    L0, L1, L2 = L[:, 0], L[:, 1], L[:, 2]\n    with np.errstate(divide='ignore', invalid='ignore'):\n        cA = np.clip((L1 ** 2 + L2 ** 2 - L0 ** 2) / (2 * L1 * L2 + 1e-12), -1, 1)\n        cB = np.clip((L0 ** 2 + L2 ** 2 - L1 ** 2) / (2 * L0 * L2 + 1e-12), -1, 1)\n        cC = np.clip((L0 ** 2 + L1 ** 2 - L2 ** 2) / (2 * L0 * L1 + 1e-12), -1, 1)\n    tri_ang = np.degrees(np.arccos(np.stack([cA, cB, cC], 1)))\n    min_ang, max_ang = tri_ang.min(1), tri_ang.max(1)\n    f.update(_stats(min_ang, 'minang')); f.update(_stats(max_ang, 'maxang'))\n    f['sliver_angle_ratio'] = float((min_ang < 5).mean())\n    tri_quality = np.clip((4 * math.sqrt(3) * area) / (L0 ** 2 + L1 ** 2 + L2 ** 2 + 1e-12), 0, 1)\n    f.update(_stats(tri_quality, 'triq'))\n    del L0, L1, L2, cA, cB, cC, tri_ang, min_ang, max_ang, tri_quality\n\n    for qd in (1, 2):\n        f[f'uniq_normals_q{qd}'] = len(np.unique(_hash3(np.rint(nrm * 10 ** qd)))) / nfw\n    # концентрация нормалей: доля площади в 6 крупнейших кластерах -> примитивы\n    hn = _hash3(np.rint(nrm * 20))\n    _, ninv = np.unique(hn, return_inverse=True)\n    mass = np.bincount(ninv, weights=area)\n    mass = np.sort(mass)[::-1]\n    f['normal_top6_mass'] = float(mass[:6].sum() / (mass.sum() + 1e-12))\n    f['normal_top1_mass'] = float(mass[0] / (mass.sum() + 1e-12))\n    del hn, ninv, mass\n\n    # ---- §4.5.1 самопересечения: близкие по центроиду грани без общей вершины ----\n    # intersection сейчас ловится только CNN с рендеров; хуже того, его confusion\n    # портит quality (10 из 16 баллов), даже когда сам класс почти ничего не весит\n    # (см. §14, \"потолок по классам\": intersection+scale = 0.18 балла при идеальном\n    # предсказании) — поэтому цель этих фич не \"поднять F1(intersection)\" сама по себе,\n    # а снизить количество ложных срабатываний, которые размывают quality.\n    # cKDTree.query_pairs даёт кандидатов почти даром; точный тест — 6 рёберно-плоскостных\n    # проверок на кандидата (Möller-style), полностью векторизован по всем парам разом.\n    f.update({'selfint_ok': 0.0, 'selfint_pair_ratio': 0.0, 'selfint_face_ratio': 0.0,\n              'selfint_any': 0.0, 'selfint_capped': 0.0})\n    SELFINT_FACE_CAP = 120_000\n    SELFINT_PAIR_CAP = 400_000\n    if 0 < nfw <= SELFINT_FACE_CAP:\n        try:\n            from scipy.spatial import cKDTree\n            cen = Vp[Fi].mean(1)\n            elen_med = float(np.median(L)) if len(L) else 1e-6\n            r = max(elen_med * 2.5, 1e-6)\n            pairs = cKDTree(cen).query_pairs(r=r, output_type='ndarray')\n            if len(pairs):\n                share = (Fi[pairs[:, 0], :, None] == Fi[pairs[:, 1], None, :]).any((1, 2))\n                pairs = pairs[~share]\n            if len(pairs) > SELFINT_PAIR_CAP:\n                sel = np.random.default_rng(0).choice(len(pairs), SELFINT_PAIR_CAP, replace=False)\n                pairs = pairs[sel]\n                f['selfint_capped'] = 1.0\n            f['selfint_ok'] = 1.0\n            if len(pairs):\n                ii, jj = pairs[:, 0], pairs[:, 1]\n                PA, PB = Vp[Fi[ii]], Vp[Fi[jj]]\n                nA, nB = nrm[ii], nrm[jj]\n\n                def _edge_hits(a, b, p0, p1, p2, n):\n                    d = np.einsum('ij,ij->i', n, p0)\n                    ta = np.einsum('ij,ij->i', n, a) - d\n                    tb = np.einsum('ij,ij->i', n, b) - d\n                    denom = ta - tb\n                    safe = np.abs(denom) > 1e-12\n                    t = np.clip(np.divide(ta, denom, out=np.zeros_like(ta), where=safe), 0.0, 1.0)\n                    pt = a + t[:, None] * (b - a)\n                    e0 = np.cross(p1 - p0, pt - p0); e1 = np.cross(p2 - p1, pt - p1)\n                    e2 = np.cross(p0 - p2, pt - p2)\n                    s0 = np.einsum('ij,ij->i', e0, n); s1 = np.einsum('ij,ij->i', e1, n)\n                    s2 = np.einsum('ij,ij->i', e2, n)\n                    inside = (((s0 >= -1e-9) & (s1 >= -1e-9) & (s2 >= -1e-9)) |\n                              ((s0 <= 1e-9) & (s1 <= 1e-9) & (s2 <= 1e-9)))\n                    return (ta * tb <= 0) & safe & inside\n\n                hit = np.zeros(len(pairs), dtype=bool)\n                for e in range(3):\n                    hit |= _edge_hits(PA[:, e], PA[:, (e + 1) % 3], PB[:, 0], PB[:, 1], PB[:, 2], nB)\n                for e in range(3):\n                    hit |= _edge_hits(PB[:, e], PB[:, (e + 1) % 3], PA[:, 0], PA[:, 1], PA[:, 2], nA)\n                n_hit = int(hit.sum())\n                f['selfint_pair_ratio'] = float(n_hit / max(nfw, 1))\n                f['selfint_any'] = float(n_hit > 0)\n                if n_hit:\n                    f['selfint_face_ratio'] = float(\n                        len(np.unique(np.concatenate([ii[hit], jj[hit]]))) / max(nfw, 1))\n                del PA, PB, nA, nB, hit\n            del cen, pairs\n        except Exception:\n            pass\n    else:\n        f['selfint_capped'] = 1.0\n\n    # ---- рёбра/углы ПОСЛЕ сварки + диагностика \"как было бы БЕЗ сварки\" ----\n    E, Es, fids, grp, counts, _ = _edge_table(Fi, nfw)\n    n_edges = len(counts)\n    f.update({'n_edges': float(n_edges),\n              'boundary_edge_ratio': float((counts == 1).mean()),\n              'nonmanifold_edge_ratio': float((counts >= 3).mean()),\n              'euler_norm': float((nvp - n_edges + nfw) / max(nfw, 1)),\n              'euler_char': float(nvp - n_edges + nfw),\n              'watertight': float((counts == 1).sum() == 0 and (counts >= 3).sum() == 0),\n              'manifold_edge_ratio': float((counts == 2).mean())})\n\n    # число граничных петель -> сколько «дыр» в поверхности (open)\n    bnd = np.where(counts == 1)[0]\n    f['nonmanifold_vert_ratio'] = 0.0\n    if len(bnd):\n        st = np.searchsorted(grp, bnd)\n        be = Es[st]\n        try:\n            Ab = coo_matrix((np.ones(len(be), dtype=np.int8), (be[:, 0], be[:, 1])), shape=(nvp, nvp))\n            nb_comp, lb = connected_components(Ab, directed=False)\n            used_b = np.unique(be)\n            f['n_boundary_loops'] = float(len(np.unique(lb[used_b])))\n            f['boundary_vert_frac'] = float(len(used_b) / nvp)\n            del Ab, lb, used_b\n        except Exception:\n            f['n_boundary_loops'] = 0.0; f['boundary_vert_frac'] = 0.0\n        # ---- §4.5.3 non-manifold вершины: степень >2 в графе граничных рёбер ----\n        # у нормальной граничной петли каждая вершина имеет ровно 2 инцидентных\n        # граничных ребра; степень >2 -> вершина, где встречаются несколько \"дыр\"\n        # или веток границы (типичная примета artifacts/open после плохого шва).\n        # Это дополняет уже имеющиеся nonmanifold_edge_ratio/watertight на уровне\n        # вершин, а не рёбер, и считается одним bincount без дополнительного графа.\n        deg_b = np.bincount(be.ravel(), minlength=nvp)\n        f['nonmanifold_vert_ratio'] = float((deg_b > 2).sum() / max(nvp, 1))\n        del be, st, deg_b\n    else:\n        f['n_boundary_loops'] = 0.0; f['boundary_vert_frac'] = 0.0\n\n    two = np.where(counts == 2)[0]\n    if len(two):\n        st = np.searchsorted(grp, two)\n        fa, fb = fids[st], fids[st + 1]\n        cosang = np.clip((nrm[fa] * nrm[fb]).sum(1), -1, 1)\n        ang = np.degrees(np.arccos(cosang))\n        angf = np.degrees(np.arccos(np.abs(cosang)))\n        f.update(_stats(ang, 'dihed')); f.update(_stats(angf, 'dihedf'))\n        for t in (10, 30, 60, 90):\n            f[f'dihed_gt{t}'] = float((ang > t).mean())\n        for t in (5, 15, 30, 60):\n            f[f'dihedf_gt{t}'] = float((angf > t).mean())\n        f['flip_ratio'] = float((cosang < 0).mean())\n        hist = np.bincount((angf / 5).astype(np.int32).clip(0, 17), minlength=18).astype(np.float64)\n        p = hist / max(hist.sum(), 1)\n        f['dihed_entropy'] = float(-(p[p > 0] * np.log(p[p > 0])).sum())\n        f['smoothness'] = float(angf.mean() * math.sqrt(nfw) / 100.0)\n        # взвешенный по длине ребра угол: устойчивее к мелким треугольникам\n        el = np.linalg.norm(Vp[Es[st][:, 0]] - Vp[Es[st][:, 1]], axis=1)\n        f['dihedf_areaw'] = float((angf * el).sum() / (el.sum() + 1e-12))\n        del fa, fb, cosang, ang, angf, el\n    else:\n        f.update(_stats([], 'dihed')); f.update(_stats([], 'dihedf'))\n        for t in (10, 30, 60, 90):\n            f[f'dihed_gt{t}'] = 0.0\n        for t in (5, 15, 30, 60):\n            f[f'dihedf_gt{t}'] = 0.0\n        f.update({'flip_ratio': 0.0, 'dihed_entropy': 0.0, 'smoothness': 0.0, 'dihedf_areaw': 0.0})\n\n    # ---- валентность вершин и угловой дефект (гауссова кривизна) ----\n    val = np.bincount(Fi.ravel(), minlength=nvp).astype(np.float32)\n    f.update(_stats(val, 'valence'))\n    f['valence_le3'] = float((val <= 3).mean())\n    f['valence_ge8'] = float((val >= 8).mean())\n    e01 = P[:, 1] - P[:, 0]; e12 = P[:, 2] - P[:, 1]; e20 = P[:, 0] - P[:, 2]\n    def _ang(u, v):\n        cu = (u * v).sum(1) / (np.linalg.norm(u, axis=1) * np.linalg.norm(v, axis=1) + 1e-12)\n        return np.arccos(np.clip(cu, -1, 1))\n    a0 = _ang(e01, -e20); a1 = _ang(e12, -e01); a2 = _ang(e20, -e12)\n    defect = np.full(nvp, 2 * np.pi, dtype=np.float64)\n    np.subtract.at(defect, Fi[:, 0], a0)\n    np.subtract.at(defect, Fi[:, 1], a1)\n    np.subtract.at(defect, Fi[:, 2], a2)\n    interior = val > 0\n    f.update(_stats(np.abs(defect[interior]), 'defect'))\n    f['defect_sum'] = float(defect[interior].sum() / (2 * np.pi))     # ~ эйлерова характеристика\n    f['defect_gt05'] = float((np.abs(defect[interior]) > 0.5).mean())\n    del defect, val, a0, a1, a2, e01, e12, e20\n\n    del E, Es, fids, grp, counts, two, nrm, P, Vp, Fi, L, aspect, area, area2\n    return f"
  },
  {
    "source_cell": 22,
    "source": "def image_features(png_path):\n    f = {'img_ok': 0.0}\n    try:\n        img = Image.open(png_path).convert('L')\n    except Exception:\n        return f\n    a = np.asarray(img, dtype=np.float32) / 255.0\n    t = CFG.tile\n    f['img_ok'] = 1.0\n    covs, comps, edens, stds, bws, bhs = [], [], [], [], [], []\n    for k in range(6):\n        v = a[(k // 3) * t:(k // 3 + 1) * t, (k % 3) * t:(k % 3 + 1) * t]\n        bg = np.median(np.concatenate([v[0], v[-1], v[:, 0], v[:, -1]]))\n        mask = np.abs(v - bg) > 0.06\n        cov = float(mask.mean()); covs.append(cov)\n        if mask.any():\n            ys, xs = np.where(mask)\n            bws.append((xs.max() - xs.min() + 1) / t)\n            bhs.append((ys.max() - ys.min() + 1) / t)\n            lab, ncc = ndimage.label(mask)\n            if ncc:\n                sz = np.bincount(lab.ravel())[1:]\n                comps.append(float((sz > 0.01 * sz.sum()).sum()))\n            else:\n                comps.append(0.0)\n        else:\n            bws.append(0.0); bhs.append(0.0); comps.append(0.0)\n        edens.append(float(np.abs(np.diff(v, axis=1)).mean() + np.abs(np.diff(v, axis=0)).mean()))\n        stds.append(float(v.std()))\n    def agg(vals, pref):\n        v = np.asarray(vals, dtype=np.float64)\n        return {f'{pref}_mean': v.mean(), f'{pref}_min': v.min(), f'{pref}_max': v.max(),\n                f'{pref}_std': v.std(), f'{pref}_range': v.max() - v.min()}\n    f.update(agg(covs, 'cov')); f.update(agg(comps, 'ncc')); f.update(agg(edens, 'edge'))\n    f.update(agg(stds, 'pxstd')); f.update(agg(bws, 'bw')); f.update(agg(bhs, 'bh'))\n    f['cov_empty_views'] = float((np.asarray(covs) < 0.01).sum())\n    f['bbox_fill_view'] = float(np.mean(np.asarray(covs) /\n                                        (np.asarray(bws) * np.asarray(bhs) + 1e-6)))\n    return f\n\ndef _feat_one(args):\n    iid, npz, png = args\n    d = {'item_id': iid}\n    try:\n        d.update(mesh_features(npz))\n    except (MemoryError, Exception):\n        d['geo_ok'] = 0.0\n    try:\n        d.update(image_features(png))\n    except Exception:\n        d['img_ok'] = 0.0\n    return d\n\ndef build_features(ids, npz_dir, img_dir, split):\n    \"\"\"Чанками, с чекпоинтом и резюмированием. Повторный запуск после падения\n    продолжает с последнего сохранённого чанка.\"\"\"\n    final = CFG.cache / f'feats_{split}_{CFG.tile}_{FEAT_VER}.parquet'\n    ckpt  = CFG.cache / f'feats_{split}_{CFG.tile}_{FEAT_VER}_partial.parquet'\n    if final.exists():\n        df = pd.read_parquet(final); print(f'{split}: фичи из кэша {df.shape}'); return df\n\n    done = pd.read_parquet(ckpt) if ckpt.exists() else pd.DataFrame(columns=['item_id'])\n    have = set(done['item_id'].astype(str)) if len(done) else set()\n    todo = [i for i in ids if i not in have]\n    print(f'{split}: уже готово {len(have)}, осталось {len(todo)}, воркеров {N_WORKERS}')\n\n    rows = [done] if len(done) else []\n    for s in range(0, len(todo), CHUNK):\n        part = todo[s:s + CHUNK]\n        jobs = [(i, str(Path(npz_dir) / f'{i}.npz'), str(Path(img_dir) / f'{i}.png')) for i in part]\n        # ВАЖНО: max_tasks_per_child переключает старт-метод на 'spawn', а при spawn воркер\n        # заново импортирует __main__ — в ноутбуке там пусто, функция не находится и пул падает\n        # (в прошлом прогоне так упал КАЖДЫЙ чанк, всё считалось последовательно).\n        # Явно требуем 'fork': воркер наследует определения ячейки.\n        try:\n            ctx = mp.get_context('fork')\n        except ValueError:\n            ctx = None\n        try:\n            with ProcessPoolExecutor(max_workers=N_WORKERS, mp_context=ctx) as ex:\n                got = list(tqdm(ex.map(_feat_one, jobs, chunksize=4), total=len(jobs),\n                                desc=f'{split} {s + len(part)}/{len(todo)}', leave=False))\n        except (BrokenProcessPool, OSError, MemoryError) as e:\n            print(f'  пул упал ({type(e).__name__}), чанк считается последовательно')\n            got = [_feat_one(j) for j in tqdm(jobs, desc='serial', leave=False)]\n        rows.append(pd.DataFrame(got))\n        _atomic(ckpt, lambda p: pd.concat(rows, ignore_index=True).to_parquet(p, index=False))\n        gc.collect()\n\n    df = pd.concat(rows, ignore_index=True).drop_duplicates('item_id')\n    df = df.set_index('item_id').reindex([str(i) for i in ids]).reset_index()\n    df = df.fillna(0.0)\n    _atomic(final, lambda p: df.to_parquet(p, index=False))\n    if ckpt.exists():\n        ckpt.unlink()\n    print(f'{split}: фичи посчитаны {df.shape}')\n    return df\n\nfeat_tr = build_features(train_df['item_id'].tolist(), TRAIN_DIR, IMG_TRAIN, 'train')\nfeat_te = build_features(test_df['item_id'].tolist(),  TEST_DIR,  IMG_TEST,  'test')"
  },
  {
    "source_cell": 24,
    "source": "CLIP_PROMPTS = {\n    # --- то, что размечено как abstract ---\n    'abs_text':     ['3D text lettering', 'a sign with written words', 'an extruded logo'],\n    'abs_chart':    ['a bar chart', 'a pie chart', 'a graph plot', 'a diagram', 'a flowchart',\n                     'a table with data', 'an infographic'],\n    'abs_support':  ['3D printing support structure', 'scaffolding lattice'],\n    'abs_voxel':    ['minecraft voxel blocks', 'blocky pixelated cubes'],\n    # --- предметные объекты: контраст к abstract ---\n    'obj_single':   ['a 3D model of a single object', 'a product render on white background'],\n    'obj_semantic': ['a chair', 'a car', 'a building', 'a plant', 'a character figure',\n                     'a tool', 'furniture'],\n    # --- прокси для остальных классов ---\n    'p_lowpoly':    ['a low-poly 3D model with visible flat facets'],\n    'p_noisy':      ['a noisy 3D scan with rough surface', 'a point cloud scan'],\n    'p_broken':     ['a broken mesh with holes and artifacts'],\n    'p_hollow':     ['a hollow thin shell', 'a flat plane'],\n    'p_multi':      ['several separate objects scattered apart', 'a collection of many objects'],\n    'p_small':      ['a tiny object in the middle of a large empty frame'],\n    'p_smooth':     ['a smooth clean 3D render of one object'],\n}\n\ndef compute_clip_features(ids, img_dir, split):\n    import open_clip\n    name, pretrained = CFG.clip_model\n    model, _, _ = open_clip.create_model_and_transforms(name, pretrained=pretrained)\n    tokenizer = open_clip.get_tokenizer(name)\n    model = model.to(CFG.device).eval()\n\n    groups = list(CLIP_PROMPTS)\n    flat, owner = [], []\n    for g in groups:\n        flat += CLIP_PROMPTS[g]; owner += [g] * len(CLIP_PROMPTS[g])\n    with torch.no_grad():\n        tf = model.encode_text(tokenizer(flat).to(CFG.device)).float()\n        tf = tf / tf.norm(dim=-1, keepdim=True)\n    owner = np.array(owner)\n\n    size = model.visual.image_size\n    size = size[0] if isinstance(size, (tuple, list)) else int(size)\n    mean = torch.tensor([0.48145466, 0.4578275, 0.40821073], device=CFG.device).view(1, 3, 1, 1)\n    std = torch.tensor([0.26862954, 0.26130258, 0.27577711], device=CFG.device).view(1, 3, 1, 1)\n\n    rows, bs = [], 24\n    for s0 in tqdm(range(0, len(ids), bs), desc=f'CLIP {split}'):\n        chunk = ids[s0:s0 + bs]\n        batch = []\n        for iid in chunk:\n            p = Path(img_dir) / f'{iid}.png'\n            if p.exists():\n                vs = [np.asarray(v.resize((size, size), Image.BILINEAR), dtype=np.uint8)\n                      for v in split_views(Image.open(p).convert('RGB'))]\n            else:\n                vs = [np.full((size, size, 3), 255, np.uint8)] * 6\n            batch.append(np.stack(vs))\n        x = torch.from_numpy(np.stack(batch)).to(CFG.device)           # (B,6,size,size,3)\n        B = x.shape[0]\n        x = x.permute(0, 1, 4, 2, 3).reshape(B * 6, 3, size, size).float() / 255.0\n        x = (x - mean) / std\n        with torch.no_grad():\n            im = model.encode_image(x).float()\n        im = im / im.norm(dim=-1, keepdim=True)\n        sim = (im @ tf.T).view(B, 6, -1).cpu().numpy()\n\n        gsim = np.stack([sim[:, :, owner == g].max(-1) for g in groups], -1)   # (B,6,n_groups)\n        feat = {}\n        for j, g in enumerate(groups):\n            v = gsim[:, :, j]\n            feat[f'clip_{g}_mean'] = v.mean(1)\n            feat[f'clip_{g}_max'] = v.max(1)\n            feat[f'clip_{g}_std'] = v.std(1)\n        abs_max = np.stack([feat[f'clip_{g}_max'] for g in groups if g.startswith('abs_')], 1).max(1)\n        obj_max = np.stack([feat[f'clip_{g}_max'] for g in groups if g.startswith('obj_')], 1).max(1)\n        def_max = np.stack([feat[f'clip_{g}_max'] for g in groups\n                            if g.startswith('p_') and g != 'p_smooth'], 1).max(1)\n        feat['clip_abstract_margin'] = abs_max - obj_max\n        feat['clip_abstract_prob'] = 1 / (1 + np.exp(-100 * (abs_max - obj_max)))\n        feat['clip_defect_margin'] = def_max - feat['clip_p_smooth_max']\n        rows.append(pd.DataFrame(feat, index=list(chunk)))\n    del model\n    gc.collect(); torch.cuda.empty_cache()\n    return pd.concat(rows).rename_axis('item_id').reset_index()\n\nif CFG.use_clip:\n    clip_tr = cached(f'clip_train_{CFG.clip_model[0]}',\n                     lambda: compute_clip_features(train_df['item_id'].tolist(), IMG_TRAIN, 'train'))\n    clip_te = cached(f'clip_test_{CFG.clip_model[0]}',\n                     lambda: compute_clip_features(test_df['item_id'].tolist(), IMG_TEST, 'test'))\n    feat_tr = feat_tr.merge(clip_tr, on='item_id', how='left')\n    feat_te = feat_te.merge(clip_te, on='item_id', how='left')\n    print('добавлено CLIP-признаков:', clip_tr.shape[1] - 1)\n\n    _m = train_df.merge(clip_tr, on='item_id', how='left').fillna(0)\n    cc = pd.Series({c: np.corrcoef(_m[c], _m['abstract'])[0, 1]\n                    for c in clip_tr.columns if c != 'item_id'}).sort_values(key=abs, ascending=False)\n    print('\\nкорреляция CLIP-признаков с abstract (топ-8):')\n    print(cc.head(8).round(3).to_string())\n    print('\\nдля сравнения: лучший геометро-визуальный признак давал по abstract r = +0.51')\nelse:\n    print('CLIP выключен (CFG.use_clip = False)')"
  },
  {
    "source_cell": 25,
    "source": "_ALL_COLS = [c for c in feat_tr.columns if c != 'item_id' and c in feat_te.columns]\nfeat_tr[_ALL_COLS] = feat_tr[_ALL_COLS].replace([np.inf, -np.inf], 0).astype(np.float32)\nfeat_te[_ALL_COLS] = feat_te[_ALL_COLS].replace([np.inf, -np.inf], 0).astype(np.float32)\n\ndef dedup_feature_columns():\n    \"\"\"Убираем константные и буквально совпадающие колонки: они только замедляют GBM\n    и размывают важность фич между копиями. Сравнение по хешу содержимого колонки.\"\"\"\n    import hashlib\n    keep, seen, const, dup = [], {}, [], []\n    for c in _ALL_COLS:\n        v = np.nan_to_num(feat_tr[c].values.astype(np.float64))\n        if v.std() == 0:\n            const.append(c); continue\n        key = hashlib.md5(np.ascontiguousarray(np.round(v, 9)).tobytes()).hexdigest()\n        if key in seen:\n            dup.append((c, seen[key])); continue\n        seen[key] = c; keep.append(c)\n    return {'cols': keep, 'const': const, 'dup': dup}\n\n_fc = cached(f'feat_cols_{CFG.tile}_{FEAT_VER}', dedup_feature_columns)\nFEAT_COLS = [c for c in _fc['cols'] if c in feat_te.columns]\nprint(f'фич всего {len(_ALL_COLS)} -> после дедупликации {len(FEAT_COLS)}')\nif _fc['const']:\n    print('  константные (выброшены):', ', '.join(_fc['const'][:12]),\n          '…' if len(_fc['const']) > 12 else '')\nif _fc['dup']:\n    print('  дубликаты (выброшены, в скобках — оставленный оригинал):',\n          ', '.join(f'{a}({b})' for a, b in _fc['dup'][:12]), '…' if len(_fc['dup']) > 12 else '')\nprint('подвыборка применялась к', f\"{feat_tr['geo_subsampled'].mean():.1%}\", 'мешей train')"
  },
  {
    "source_cell": 30,
    "source": "from iterstrat.ml_stratifiers import MultilabelStratifiedKFold\n\nmskf = MultilabelStratifiedKFold(n_splits=CFG.n_folds, shuffle=True, random_state=CFG.seed)\ntrain_df['fold'] = -1\nfor k, (_, vi) in enumerate(mskf.split(train_df, train_df[CFG.target_cols].values)):\n    train_df.loc[train_df.index[vi], 'fold'] = k\ndisplay(train_df.groupby('fold')[CFG.target_cols].mean().round(4))"
  },
  {
    "source_cell": 32,
    "source": "from torch.utils.data import Dataset, DataLoader\n\nMEAN = np.array([0.485, 0.456, 0.406], dtype=np.float32)\nSTD  = np.array([0.229, 0.224, 0.225], dtype=np.float32)\n\ndef load_views(path, tile):\n    \"\"\"PNG-мозаика -> (6, tile, tile, 3) uint8.\"\"\"\n    a = np.asarray(Image.open(path).convert('RGB'), dtype=np.uint8)\n    t = tile\n    return np.stack([a[(k // 3) * t:(k // 3 + 1) * t, (k % 3) * t:(k % 3 + 1) * t] for k in range(6)])\n\nclass MeshViewDataset(Dataset):\n    def __init__(self, df, img_dir, feats, train=True, labels=True):\n        self.ids = df['item_id'].tolist()\n        self.dir = Path(img_dir)\n        self.train = train\n        self.labels = df[CFG.target_cols].values.astype(np.float32) if labels else None\n        fm = feats.set_index('item_id').reindex(self.ids)[FEAT_COLS].fillna(0.0)\n        self.feats = fm.values.astype(np.float32)\n\n    def __len__(self):\n        return len(self.ids)\n\n    def _aug(self, v):\n        # v: (6,t,t,3) uint8\n        rng = np.random\n        if rng.rand() < 0.5:                       # флип каждого вида отдельно\n            v = v[:, :, ::-1]\n        if rng.rand() < 0.3:                       # сдвиг всех видов синхронно\n            s = rng.randint(-8, 9, size=2)\n            v = np.stack([np.roll(x, s, axis=(0, 1)) for x in v])\n        if rng.rand() < 0.15:                      # view dropout\n            k = rng.randint(6)\n            v = v.copy(); v[k] = 255\n        v = v.astype(np.float32)\n        if rng.rand() < 0.3:                       # яркость/контраст\n            v = np.clip(v * rng.uniform(0.85, 1.15) + rng.uniform(-15, 15), 0, 255)\n        return v\n\n    def __getitem__(self, i):\n        p = self.dir / f'{self.ids[i]}.png'\n        if p.exists():\n            v = load_views(p, CFG.tile)\n        else:\n            v = np.full((6, CFG.tile, CFG.tile, 3), 255, dtype=np.uint8)\n        v = self._aug(v) if self.train else v.astype(np.float32)\n        v = ((v / 255.0 - MEAN) / STD).astype(np.float32)\n        v = torch.from_numpy(np.ascontiguousarray(v)).permute(0, 3, 1, 2)   # (6,3,t,t)\n        out = {'views': v, 'feats': torch.from_numpy(self.feats[i]), 'item_id': self.ids[i]}\n        if self.labels is not None:\n            out['y'] = torch.from_numpy(self.labels[i])\n        return out\n\ndef views_to_grid(v):\n    \"\"\"(B,6,3,t,t) -> (B,3,2t,3t) — мозаика для режима 'grid'.\"\"\"\n    B, N, C, H, W = v.shape\n    v = v.view(B, 2, 3, C, H, W).permute(0, 3, 1, 4, 2, 5).reshape(B, C, 2 * H, 3 * W)\n    return v"
  },
  {
    "source_cell": 34,
    "source": "import timm\n\nclass MultiViewNet(nn.Module):\n    def __init__(self, n_feats, n_out=CFG.n_targets):\n        super().__init__()\n        self.mode = CFG.view_mode\n        kw = dict(pretrained=True, num_classes=0, drop_rate=CFG.drop_rate)\n        if IS_VIT:\n            # позиционные эмбеддинги интерполируются timm-ом под наш размер входа\n            kw['img_size'] = CFG.tile\n        self.backbone = timm.create_model(CFG.backbone, **kw)\n        d = self.backbone.num_features\n\n        # gated attention pooling по видам (используется только в multiview)\n        self.att_v = nn.Sequential(nn.Linear(d, 256), nn.Tanh())\n        self.att_u = nn.Sequential(nn.Linear(d, 256), nn.Sigmoid())\n        self.att_w = nn.Linear(256, 1)\n\n        self.geo = nn.Sequential(\n            nn.BatchNorm1d(n_feats), nn.Linear(n_feats, 256), nn.SiLU(), nn.Dropout(0.2),\n            nn.Linear(256, 128), nn.SiLU())\n        self.head = nn.Sequential(\n            nn.Linear(d + 128, 512), nn.SiLU(), nn.Dropout(0.3), nn.Linear(512, n_out))\n\n    def embed(self, views):\n        if self.mode == 'grid':\n            return self.backbone(views_to_grid(views)), None\n        B, N = views.shape[:2]\n        z = self.backbone(views.flatten(0, 1)).view(B, N, -1)\n        a = self.att_w(self.att_v(z) * self.att_u(z))            # (B,N,1)\n        w = torch.softmax(a, dim=1)\n        return (z * w).sum(1) + z.max(1).values, w.squeeze(-1)\n\n    def forward(self, views, feats, return_att=False):\n        z, w = self.embed(views)\n        h = torch.cat([z, self.geo(feats)], 1)\n        out = self.head(h)\n        return (out, w) if return_att else out\n\nclass AsymmetricLoss(nn.Module):\n    \"\"\"Ridnik et al., ICCV 2021.\"\"\"\n    def __init__(self, gamma_neg=4.0, gamma_pos=0.0, clip=0.05, eps=1e-8):\n        super().__init__()\n        self.gn, self.gp, self.clip, self.eps = gamma_neg, gamma_pos, clip, eps\n\n    def forward(self, logits, y):\n        p = torch.sigmoid(logits)\n        p_neg = (1 - p + self.clip).clamp(max=1.0)\n        loss = y * torch.log(p.clamp(min=self.eps)) + (1 - y) * torch.log(p_neg.clamp(min=self.eps))\n        pt = p * y + (1 - p) * (1 - y)\n        gamma = self.gp * y + self.gn * (1 - y)\n        return -(loss * (1 - pt).pow(gamma)).mean()"
  },
  {
    "source_cell": 36,
    "source": "from torch.cuda.amp import autocast, GradScaler\n\ndef per_class_f1(y, p):\n    return pd.Series([f1_score(y[:, i], p[:, i], zero_division=0) for i in range(11)],\n                     index=CFG.target_cols)\n\ndef quick_thresholds(y, prob):\n    \"\"\"Быстрые per-class пороги (для мониторинга по эпохам).\"\"\"\n    th = np.full(11, 0.5)\n    for i in range(11):\n        best, bt = -1, 0.5\n        for t in np.arange(0.05, 0.96, 0.02):\n            s = f1_score(y[:, i], (prob[:, i] > t).astype(int), zero_division=0)\n            if s > best:\n                best, bt = s, t\n        th[i] = bt\n    return th\n\n@torch.no_grad()\ndef predict(model, loader, tta=1):\n    model.eval()\n    probs, ids = [], []\n    for b in tqdm(loader, desc='predict', leave=False):\n        v = b['views'].to(CFG.device, non_blocking=True)\n        f = b['feats'].to(CFG.device, non_blocking=True)\n        acc = 0\n        with autocast(enabled=CFG.amp):\n            acc = torch.sigmoid(model(v, f)).float()\n            if tta > 1:\n                acc = acc + torch.sigmoid(model(torch.flip(v, dims=[-1]), f)).float()\n        probs.append((acc / tta).cpu().numpy())\n        ids.extend(b['item_id'])\n    return np.vstack(probs), ids\n\ndef train_fold(fold):\n    # 1) фолд уже посчитан целиком -> отдаём сохранённые предсказания, не обучаем\n    if CFG.resume and has_artifact(rt(f'fold{fold}_preds')):\n        d = load_artifact(rt(f'fold{fold}_preds'))\n        print(f'[кэш] фолд {fold} готов (val {d[\"best\"]:.3f}), обучение пропускаем')\n        return (d['oof_ids'], d['oof']), (d['te_ids'], d['te'])\n\n    seed_everything(CFG.seed + fold)\n    tr = train_df[train_df.fold != fold].reset_index(drop=True)\n    va = train_df[train_df.fold == fold].reset_index(drop=True)\n\n    dtr = MeshViewDataset(tr, IMG_TRAIN, feat_tr, train=True)\n    dva = MeshViewDataset(va, IMG_TRAIN, feat_tr, train=False)\n    ltr = DataLoader(dtr, batch_size=CFG.batch_size, shuffle=True, drop_last=True,\n                     num_workers=CFG.num_workers, pin_memory=True, persistent_workers=True)\n    lva = DataLoader(dva, batch_size=CFG.batch_size * 2, shuffle=False,\n                     num_workers=CFG.num_workers, pin_memory=True)\n\n    model = MultiViewNet(len(FEAT_COLS)).to(CFG.device)\n    head_mods = [model.head, model.geo, model.att_v, model.att_u, model.att_w]\n    head_ids = {id(p) for m in head_mods for p in m.parameters()}\n\n    def _block_index(name):\n        \"\"\"Номер блока трансформера в имени параметра (для послойного затухания LR).\"\"\"\n        m = re.search(r'blocks\\.(\\d+)\\.', name)\n        if m:\n            return int(m.group(1))\n        return -1 if any(k in name for k in ('patch_embed', 'pos_embed', 'cls_token',\n                                             'reg_token')) else None\n\n    if IS_VIT:\n        n_blocks = len(getattr(model.backbone, 'blocks', []))\n        base_lr = CFG.backbone_lr\n        buckets = {}\n        for name, p in model.backbone.named_parameters():\n            bi = _block_index(name)\n            depth = 0 if bi == -1 else (bi + 1 if bi is not None else n_blocks)\n            scale = CFG.layer_decay ** (n_blocks - depth)\n            buckets.setdefault(round(scale, 4), []).append(p)\n        groups = [{'params': ps, 'lr': base_lr * sc} for sc, ps in buckets.items()]\n        groups.append({'params': [p for m in head_mods for p in m.parameters()],\n                       'lr': CFG.lr * CFG.head_lr_mult})\n        print(f'  ViT: {n_blocks} блоков, LR от {base_lr * min(buckets):.2e} до {base_lr:.2e}, '\n              f'голова {CFG.lr * CFG.head_lr_mult:.2e}')\n    else:\n        groups = [{'params': [p for p in model.parameters() if id(p) not in head_ids], 'lr': CFG.lr},\n                  {'params': [p for p in model.parameters() if id(p) in head_ids],\n                   'lr': CFG.lr * CFG.head_lr_mult}]\n    opt = torch.optim.AdamW(groups, weight_decay=CFG.weight_decay)\n\n    steps = CFG.epochs * len(ltr)\n    warm = int(CFG.warmup_frac * steps)\n    sched = torch.optim.lr_scheduler.LambdaLR(\n        opt, lambda s: s / max(warm, 1) if s < warm\n        else 0.5 * (1 + math.cos(math.pi * (s - warm) / max(steps - warm, 1))))\n\n    pos = tr[CFG.target_cols].sum(0).values.astype(np.float32)\n    pw = np.clip((len(tr) - pos) / np.clip(pos, 1, None), 1.0, 20.0)\n    bce = nn.BCEWithLogitsLoss(pos_weight=torch.tensor(pw, device=CFG.device))\n    asl = AsymmetricLoss()\n    scaler = GradScaler(enabled=CFG.amp)\n\n    # 2) незавершённый чекпоинт -> продолжаем с той же эпохи и того же состояния ГСЧ\n    snaps, best, start_ep = [], -1, 0\n    ck = load_ckpt(fold)\n    if ck is not None:\n        model.load_state_dict(_to_fp32(ck['model']))\n        try:\n            opt.load_state_dict(ck['opt']); sched.load_state_dict(ck['sched'])\n            scaler.load_state_dict(ck['scaler'])\n        except Exception as e:\n            print('  состояние оптимизатора не восстановлено, продолжаем с текущим:', e)\n        snaps = [(sc_i, _to_fp32(sd)) for sc_i, sd in ck['snaps']]\n        best, start_ep = ck['best'], ck['epoch'] + 1\n        set_rng_state(ck['rng'])\n        print(f'  продолжаем фолд {fold} с эпохи {start_ep + 1}/{CFG.epochs} '\n              f'(лучшая метрика пока {best:.3f})')\n        if start_ep >= CFG.epochs:\n            print('  все эпохи пройдены, остаётся только инференс')\n\n    yv = va[CFG.target_cols].values\n    pred = None\n    for ep in range(start_ep, CFG.epochs):\n        model.train(); tot = 0.0\n        pbar = tqdm(ltr, desc=f'fold{fold} ep{ep + 1}/{CFG.epochs}')\n        for b in pbar:\n            v = b['views'].to(CFG.device, non_blocking=True)\n            f = b['feats'].to(CFG.device, non_blocking=True)\n            y = b['y'].to(CFG.device, non_blocking=True)\n            y = y * (1 - CFG.label_smooth) + 0.5 * CFG.label_smooth\n            # mixup для мультилейбла: смешиваем виды, гео-фичи и метки одним и тем же lam.\n            # Метки остаются вещественными — BCE/ASL это допускают, а сеть перестаёт\n            # заучивать редкие комбинации целиком.\n            if CFG.mixup > 0 and random.random() < 0.5:\n                lam = float(np.random.beta(CFG.mixup, CFG.mixup))\n                perm = torch.randperm(v.size(0), device=v.device)\n                v = lam * v + (1 - lam) * v[perm]\n                f = lam * f + (1 - lam) * f[perm]\n                y = lam * y + (1 - lam) * y[perm]\n            opt.zero_grad(set_to_none=True)\n            with autocast(enabled=CFG.amp):\n                logits = model(v, f)\n                loss = (1 - CFG.asl_weight) * bce(logits, y) + CFG.asl_weight * asl(logits, y)\n            scaler.scale(loss).backward()\n            scaler.unscale_(opt)\n            torch.nn.utils.clip_grad_norm_(model.parameters(), 5.0)\n            scaler.step(opt); scaler.update(); sched.step()\n            tot += loss.item(); pbar.set_postfix(loss=f'{tot / (pbar.n + 1):.4f}')\n\n        prob, _ = predict(model, lva, tta=1)\n        th = quick_thresholds(yv, prob)\n        pred = (prob > th).astype(int)\n        sc, fq, fa = competition_metric(yv, pred)\n        print(f'  ep{ep + 1}: metric={sc:.3f} (quality {fq:.3f} | artefacts {fa:.3f})')\n        if ep >= 2:      # первые эпохи в ансамбль не берём\n            snaps.append((sc, {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}))\n            snaps.sort(key=lambda x: -x[0])\n            del snaps[CFG.n_snapshots:]\n        if sc > best:\n            best = sc\n            print('   ^ best')\n\n        log_epoch({'fold': fold, 'epoch': ep + 1, 'train_loss': tot / max(len(ltr), 1),\n                   'val_metric': sc, 'f1_quality': fq, 'f1_artefacts': fa,\n                   'lr': opt.param_groups[0]['lr'], 'time': time.strftime('%Y-%m-%d %H:%M:%S'),\n                   **{f'f1_{c}': float(v) for c, v in per_class_f1(yv, pred).items()}})\n\n        if (ep + 1) % CFG.ckpt_every == 0 or ep == CFG.epochs - 1:\n            conv = _to_fp16 if CFG.ckpt_fp16 else (lambda x: x)\n            save_ckpt(fold, {'epoch': ep, 'best': best, 'rng': rng_state(),\n                             'model': conv(model.state_dict()),\n                             'opt': opt.state_dict(), 'sched': sched.state_dict(),\n                             'scaler': scaler.state_dict(),\n                             'snaps': [(sc_i, conv(sd)) for sc_i, sd in snaps],\n                             'cfg': {'tile': CFG.tile, 'backbone': CFG.backbone,\n                                     'epochs': CFG.epochs, 'feat_ver': FEAT_VER}})\n            print(f'   чекпоинт сохранён (эпоха {ep + 1})')\n\n    if pred is None:          # обучение было полностью восстановлено из чекпоинта\n        prob, _ = predict(model, lva, tta=1)\n        pred = (prob > quick_thresholds(yv, prob)).astype(int)\n        best = max(best, competition_metric(yv, pred)[0])\n    if not snaps:             # чекпоинт с ранней эпохи: снапшотов ещё не было\n        snaps = [(best, {k: v.detach().cpu().clone() for k, v in model.state_dict().items()})]\n    print(f'fold {fold}: лучшая валидационная метрика {best:.3f}, '\n          f'в ансамбле эпохи со скорами {[round(x[0], 2) for x in snaps]}')\n    print(per_class_f1(yv, pred).round(3).to_string())\n\n    _atomic(CFG.work / f'{CFG.run_tag}_cnn_fold{fold}.pth',\n            lambda p: torch.save(_to_fp16(snaps[0][1]) if CFG.ckpt_fp16 else snaps[0][1], p))\n\n    dte = MeshViewDataset(test_df.assign(**{c: 0 for c in CFG.target_cols}), IMG_TEST,\n                          feat_te, train=False, labels=False)\n    lte = DataLoader(dte, batch_size=CFG.batch_size * 2, shuffle=False,\n                     num_workers=CFG.num_workers, pin_memory=True)\n\n    # snapshot-ensemble: усредняем вероятности лучших эпох (обучения не требует, только инференс)\n    oof_acc, te_acc = 0.0, 0.0\n    for k, (sc_k, st) in enumerate(snaps):\n        model.load_state_dict(st)\n        op, oof_ids = predict(model, lva, tta=CFG.tta)\n        tp, te_ids = predict(model, lte, tta=CFG.tta)\n        oof_acc = oof_acc + op; te_acc = te_acc + tp\n    oof_prob, te_prob = oof_acc / len(snaps), te_acc / len(snaps)\n    sc_ens = competition_metric(yv, (oof_prob > quick_thresholds(yv, oof_prob)).astype(int))[0]\n    print(f'fold {fold}: одна эпоха {best:.3f} -> snapshot-ансамбль {sc_ens:.3f}')\n\n    keep_snapshots(fold, snaps)\n    save_artifact(rt(f'fold{fold}_preds'), {'oof': oof_prob, 'oof_ids': list(oof_ids),\n                                        'te': te_prob, 'te_ids': list(te_ids),\n                                        'best': best, 'ens': sc_ens,\n                                        'snap_scores': [x[0] for x in snaps]})\n    drop_ckpt(fold)          # фолд закрыт, промежуточное состояние больше не нужно\n\n    del model, ltr, lva, snaps; gc.collect(); torch.cuda.empty_cache()\n    return (oof_ids, oof_prob), (te_ids, te_prob)"
  }
]


In [ ]:
%%writefile /content/meshqc_code/dino448_s2_source.py
                    
import os, gc, json, math, random, re, warnings, shutil
from pathlib import Path
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
from tqdm.auto import tqdm
import matplotlib.pyplot as plt

warnings.filterwarnings('ignore')
                                                                                  
                                                                              
import logging
logging.getLogger('py.warnings').setLevel(logging.ERROR)

class CFG:
    seed = 42

                      
    dataset  = 'daniilantonov5/3d-mesh-quality-control'
    work     = LOCAL_DIR; work.mkdir(parents=True, exist_ok=True)
    cache    = work / 'cache';        cache.mkdir(parents=True, exist_ok=True)

                                 
                                                                                          
                                                                                
    view_mode = 'multiview'                           
    tile      = 448                                                      
                                                                                                    
    grid_rows, grid_cols = 2, 3

                   
                                                                                             
                                                                                         
                                                                                         
    backbone   = 'vit_small_patch14_reg4_dinov2.lvd142m'                                                              
                                                                                               
                                                                                           
    backbone_lr = 4e-5                                                                            
    layer_decay = 0.75                                                                   
    drop_rate  = 0.2
    epochs     = 16                    
    batch_size = 6                                              
    mixup      = 0.4                                                                    

                                                  
    use_clip   = True                                                                       
    clip_model = ('ViT-B-32', 'laion2b_s34b_b79k')
    use_dino_frozen = True                                                                   
    dino_frozen_model = 'vit_base_patch14_reg4_dinov2.lvd142m'
    dino_pca   = 128
    lr         = 3e-4
    head_lr_mult = 10.0
    weight_decay = 0.05
    warmup_frac  = 0.1
    label_smooth = 0.01
    amp        = True
    num_workers = 4

                  
    n_folds = 5
    RUN_ALL_FOLDS = True                                                        
    folds_to_run  = [0]

                        
    tta = 2                                                                           
    n_snapshots = 3                                                                         
    run_tag = 'dino448_s2'                                               
                                                                                             
    ensemble_tags = ['dino448_v2', 'dino448_s2', 'cnx288_v4', 'cnx288_v5', 'dino448', 'dinos224']                                       
                                                                                                 
                                                                                             
    rebuild_specs = {'dino448': ('vit_small_patch14_reg4_dinov2.lvd142m', 448, 'multiview')}
    ensemble_select = True                                                                          
    level2_seeds = 3                                                                        
    ema_decay = 0.999                                                                                      
    model_seed = 1                                                                                       
    level2_all_candidates = True                                                                         
    ensemble_weights = True                                                                            
    postproc_select = True                                                                       
    level2_member_inputs = True                                                                 
                                                                                                  
    rule_select_by_nested = True                                                             
    use_geo_branch = True                                                                       
                                                                                    

                                           
    use_drive   = True                                                                                    
    drive_dir   = '/content/drive/MyDrive/sber_meshqc'
    resume      = True                                                            
    ckpt_every  = 1                                                  
    ckpt_fp16   = True                                                             
    keep_fold_ckpt = False                                                             
    force_recompute = []                                                                      

                     
    artifact_cols = ['abstract','artifacts','intersection','lowpoly','noisy',
                     'open','partial','scale','set','simple']
    target_cols   = artifact_cols + ['quality']
    n_targets     = 11

    device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

                     
import kagglehub

root = Path(kagglehub.dataset_download(CFG.dataset))
print('dataset root:', root)

def _pick_dir(mode):
    """Папка с максимальным числом .npz, в пути которой встречается train/test."""
    best, best_n = None, -1
    for p in root.rglob('*'):
        if not p.is_dir():
            continue
        if mode not in str(p).lower():
            continue
        n = sum(1 for _ in p.glob('*.npz'))
        if n > best_n:
            best, best_n = p, n
    if best is None or best_n == 0:
        raise FileNotFoundError(f'не найдена папка с .npz для {mode}')
    return best

def _pick_csv(mode):
    cands = list(root.rglob('*.csv'))
    def score(p):
        n = p.name.lower(); s = 0
        if mode == 'train':
            s += 10 * ('train' in n) - 20 * ('submission' in n) - 20 * ('test' in n)
        else:
            s += 10 * ('test' in n) - 5 * ('submission' in n) - 20 * ('train' in n)
        return s - 0.001 * len(n)
    return sorted(cands, key=score, reverse=True)[0]

TRAIN_DIR, TEST_DIR = _pick_dir('train'), _pick_dir('test')
TRAIN_CSV, TEST_CSV = _pick_csv('train'), _pick_csv('test')
print('train dir :', TRAIN_DIR, len(list(TRAIN_DIR.glob('*.npz'))), 'npz /',
      len(list(TRAIN_DIR.glob('*.png'))), 'png')
print('test  dir :', TEST_DIR,  len(list(TEST_DIR.glob('*.npz'))),  'npz /',
      len(list(TEST_DIR.glob('*.png'))),  'png')
print('train csv :', TRAIN_CSV)
print('test  csv :', TEST_CSV)

train_df = pd.read_csv(TRAIN_CSV)
test_df  = pd.read_csv(TEST_CSV)[['item_id']].copy()
train_df['item_id'] = train_df['item_id'].astype(str)
test_df['item_id']  = test_df['item_id'].astype(str)

if 'quality' not in train_df.columns:
    train_df['quality'] = (train_df[CFG.artifact_cols].sum(1) == 0).astype(int)

print(train_df.shape, test_df.shape)
train_df.head()

                     
from sklearn.metrics import f1_score

def competition_metric(y_true, y_pred):
    """10*F1(quality) + 10*F1_weighted(10 дефектов) — точная формула организаторов."""
    f1_q = f1_score(y_true[:, 10], y_pred[:, 10], zero_division=0)
    f1_a = f1_score(y_true[:, :10], y_pred[:, :10], average='weighted', zero_division=0)
    return 10 * f1_q + 10 * f1_a, f1_q, f1_a

                                                                                          
Y = train_df[CFG.target_cols].values
for err in [0.02, 0.05, 0.10]:
    rng = np.random.default_rng(0)
    fake = Y.copy()
    flip = rng.random(fake[:, :10].shape) < err
    fake[:, :10] = np.abs(fake[:, :10] - flip)
    fake[:, 10] = (fake[:, :10].sum(1) == 0).astype(int)
    s, fq, fa = competition_metric(Y, fake)
    print(f'FPR/FNR по дефектам {err:.0%} -> метрика {s:5.2f} (quality F1={fq:.3f}, artefacts F1={fa:.3f})')
print('\n>>> Ошибка 5% по дефектам уже срезает quality F1 до ~0.7. '
      'Поэтому quality предсказывается ОТДЕЛЬНОЙ головой, а не только правилом.')

                     
from PIL import Image
from concurrent.futures import ProcessPoolExecutor

Image.MAX_IMAGE_PIXELS = None

def infer_grid(w, h):
    """(rows, cols) для 6 видов по соотношению сторон."""
    ar = w / h
    cands = {(2, 3): 3 / 2, (3, 2): 2 / 3, (1, 6): 6.0, (6, 1): 1 / 6}
    return min(cands, key=lambda k: abs(math.log(ar / cands[k])))

def split_views(img):
    """PIL.Image -> список из 6 PIL.Image в каноническом порядке чтения."""
    w, h = img.size
    r, c = infer_grid(w, h)
    tw, th = w // c, h // r
    return [img.crop((j * tw, i * th, (j + 1) * tw, (i + 1) * th)) for i in range(r) for j in range(c)]

def make_cache(args):
    src, dst, tile = args
    try:
        img = Image.open(src).convert('RGB')
        views = [v.resize((tile, tile), Image.BILINEAR) for v in split_views(img)]
        out = Image.new('RGB', (3 * tile, 2 * tile))
        for k, v in enumerate(views):
            out.paste(v, ((k % 3) * tile, (k // 3) * tile))
        out.save(dst, format='PNG', optimize=False, compress_level=1)
        return 1
    except Exception:
        return 0

def build_image_cache(ids, src_dir, split):
    out_dir = CFG.cache / f'img_{split}_{CFG.tile}'
    out_dir.mkdir(parents=True, exist_ok=True)
    jobs = [(str(Path(src_dir) / f'{i}.png'), str(out_dir / f'{i}.png'), CFG.tile)
            for i in ids if not (out_dir / f'{i}.png').exists()
            and (Path(src_dir) / f'{i}.png').exists()]
    if jobs:
        with ProcessPoolExecutor(max_workers=os.cpu_count()) as ex:
            ok = list(tqdm(ex.map(make_cache, jobs, chunksize=32), total=len(jobs),
                           desc=f'cache {split}'))
        print(f'{split}: закэшировано {sum(ok)}/{len(jobs)}')
    missing = [i for i in ids if not (out_dir / f'{i}.png').exists()]
    print(f'{split}: всего в кэше {len(ids) - len(missing)}/{len(ids)}, нет картинки у {len(missing)}')
    return out_dir, set(missing)

                                                                                       
_probe = [Image.open(Path(TRAIN_DIR) / f'{i}.png').size
          for i in train_df['item_id'].head(40) if (Path(TRAIN_DIR) / f'{i}.png').exists()]
if _probe:
    _sizes = pd.Series([f'{w}x{h}' for w, h in _probe]).value_counts()
    print('исходные размеры листа с 6 видами:'); print(_sizes.head(5).to_string())
    _w, _h = _probe[0]
    _r, _c = infer_grid(_w, _h)
    _native = min(_w // _c, _h // _r)
    print(f'-> раскладка {_r}x{_c}, нативный размер одного вида ~{_native} px')
    if CFG.tile > _native:
        print(f'!! CFG.tile = {CFG.tile} БОЛЬШЕ нативного {_native}: это апскейл, он удорожает '
              f'обучение, но новой информации не даёт. Снижаю tile до {_native}.')
        CFG.tile = int(_native)
    else:
        print(f'   CFG.tile = {CFG.tile} <= нативного — детали сохраняются')

IMG_TRAIN, MISS_TRAIN = build_image_cache(train_df['item_id'].tolist(), TRAIN_DIR, 'train')
IMG_TEST,  MISS_TEST  = build_image_cache(test_df['item_id'].tolist(),  TEST_DIR,  'test')

                     
import os, gc, math, numpy as np, pandas as pd
from pathlib import Path
from PIL import Image
from scipy import ndimage
from scipy.sparse import coo_matrix
from scipy.sparse.csgraph import connected_components
from scipy.spatial import ConvexHull, QhullError
from concurrent.futures import ProcessPoolExecutor
from concurrent.futures.process import BrokenProcessPool
import multiprocessing as mp

FACE_CAP  = 300_000
VERT_CAP  = 200_000
HULL_CAP  = 20_000
COMP_CAP  = 3_000_000
WELD_TOL  = 1e-6                              
FEAT_VER  = 'v3weld'                                                                 
                                                                               
N_WORKERS = min(4, os.cpu_count() or 2)
CHUNK     = 1024

def _pack21(q):
    """Три целых из [0, 2**21) -> один int64. Точно, без коллизий."""
    return (q[:, 0] << 42) | (q[:, 1] << 21) | q[:, 2]

def _hash3(q):
    q = q.astype(np.int64, copy=False)
    return (q[:, 0] * 73856093) ^ (q[:, 1] * 19349663) ^ (q[:, 2] * 83492791)

def _stats(x, pref):
    keys = ['mean', 'std', 'p10', 'p50', 'p90', 'max', 'cv']
    if len(x) == 0:
        return {f'{pref}_{k}': 0.0 for k in keys}
    x = np.asarray(x, dtype=np.float32)
    m, s = float(x.mean()), float(x.std())
    q10, q50, q90 = np.percentile(x, [10, 50, 90])
    return {f'{pref}_mean': m, f'{pref}_std': s, f'{pref}_p10': float(q10),
            f'{pref}_p50': float(q50), f'{pref}_p90': float(q90),
            f'{pref}_max': float(x.max()), f'{pref}_cv': float(s / (abs(m) + 1e-9))}

def _edge_table(Fi, nfw):
    """Рёбра -> (номер группы, число граней на ребро, отсортированные id граней)."""
    E = np.concatenate([Fi[:, [0, 1]], Fi[:, [1, 2]], Fi[:, [2, 0]]], 0)
    E.sort(axis=1)
    fid = np.tile(np.arange(nfw, dtype=np.int32), 3)
    order = np.lexsort((E[:, 1], E[:, 0]))
    Es, fids = E[order], fid[order]
    new = np.ones(len(Es), dtype=bool)
    new[1:] = (Es[1:] != Es[:-1]).any(1)
    grp = np.cumsum(new) - 1
    counts = np.bincount(grp)
    return E, Es, fids, grp, counts, new

def mesh_features(npz_path):
    f = {'geo_ok': 0.0, 'geo_subsampled': 0.0}
    try:
        f['npz_mb'] = os.path.getsize(npz_path) / 1e6
    except Exception:
        return f
    try:
        with np.load(npz_path, allow_pickle=False) as d:
            keys = list(d.keys())
            V = np.asarray(d['vertices' if 'vertices' in keys else keys[0]], dtype=np.float32)
            Fc = None
            for k in ('faces', 'triangles', 'f'):
                if k in keys:
                    Fc = np.asarray(d[k], dtype=np.int64); break
            if Fc is None and len(keys) > 1:
                Fc = np.asarray(d[keys[1]], dtype=np.int64)
    except Exception:
        return f
    if V.ndim != 2 or V.shape[1] != 3 or len(V) == 0:
        return f
    if Fc is None or Fc.ndim != 2 or Fc.shape[1] != 3 or len(Fc) == 0:
        Fc = np.zeros((0, 3), dtype=np.int64)
    else:
        Fc = Fc[(Fc >= 0).all(1) & (Fc < len(V)).all(1)]

    f['geo_ok'] = 1.0
    nv_raw, nf_raw = len(V), len(Fc)
    vmin, vmax = V.min(0), V.max(0)
    ext = (vmax - vmin).astype(np.float64)
    ext_s = np.sort(ext)[::-1]
    diag = float(np.linalg.norm(ext)) + 1e-12
    center = ((vmin + vmax) / 2).astype(np.float32)
    inv_s = np.float32(1.0 / diag)
    f.update({'n_verts_raw': float(nv_raw), 'n_faces': float(nf_raw),
              'log_verts': math.log1p(nv_raw), 'log_faces': math.log1p(nf_raw),
              'ext_max': float(ext_s[0]), 'ext_mid': float(ext_s[1]), 'ext_min': float(ext_s[2]),
              'ext_ratio_min': float(ext_s[2] / (ext_s[0] + 1e-12)),
              'ext_ratio_mid': float(ext_s[1] / (ext_s[0] + 1e-12)),
              'bbox_diag': diag,
              'bbox_fill': float(nv_raw / (ext.prod() + 1e-12)) if ext.prod() > 0 else 0.0})

                                                         
                                                                                           
                                                                                         
                                                                                                
    step = np.float32(diag * WELD_TOL)
    q = np.clip(np.rint((V - vmin) / step), 0, (1 << 21) - 1).astype(np.int64)
    code = _pack21(q)
    del q
    ucode, first, invmap = np.unique(code, return_index=True, return_inverse=True)
    del code, ucode
    nv = len(first)
    f['n_verts'] = float(nv)
    f['weld_ratio'] = float(1.0 - nv / max(nv_raw, 1))                                                         
    f['f_per_v'] = nf_raw / max(nv, 1)
    Vw_full = V[first]
    del first

    if nf_raw:
        Fw = invmap[Fc]
        ok = (Fw[:, 0] != Fw[:, 1]) & (Fw[:, 1] != Fw[:, 2]) & (Fw[:, 0] != Fw[:, 2])
        f['degen_after_weld'] = float(1.0 - ok.mean())
        Fw = Fw[ok]
        hf = _hash3(np.sort(Fw, axis=1))
        _, uidx = np.unique(hf, return_index=True)
        f['dup_face_ratio'] = float(1.0 - len(uidx) / max(len(Fw), 1))
        Fw = Fw[np.sort(uidx)]
        del hf, uidx, ok
    else:
        Fw = np.zeros((0, 3), dtype=np.int64)
        f['degen_after_weld'] = 0.0; f['dup_face_ratio'] = 0.0
    del invmap, Fc
    nf = len(Fw)

                                                 
    Vs = Vw_full[::max(1, int(np.ceil(nv / VERT_CAP)))]
    if nv > VERT_CAP:
        f['geo_subsampled'] = 1.0
    Vs = (Vs - center) * inv_s
    try:
        if len(Vs) < 3:
            raise ValueError('too few vertices')
        ev = np.clip(np.linalg.eigvalsh(np.cov(Vs.T.astype(np.float64)))[::-1], 1e-16, None)
        f.update({'pca_1': float(ev[0]), 'pca_2': float(ev[1]), 'pca_3': float(ev[2]),
                  'pca_flat': float(ev[2] / ev[0]), 'pca_lin': float(ev[1] / ev[0]),
                  'pca_aniso': float((ev[0] - ev[2]) / ev.sum())})
    except Exception:
        f.update({k: 0.0 for k in ['pca_1', 'pca_2', 'pca_3', 'pca_flat', 'pca_lin', 'pca_aniso']})

                                                                       
    f.update({'hull_ok': 0.0, 'solidity': 0.0, 'solidity_signed': 0.0, 'hull_area_ratio': 0.0,
              'hull_pts_frac': 0.0, 'sphericity': 0.0, 'hull_vol': 0.0, 'hull_area': 0.0})
    try:
        Vh = Vs[::max(1, int(np.ceil(len(Vs) / HULL_CAP)))].astype(np.float64)
        if len(Vh) >= 8:
            hull = ConvexHull(Vh, qhull_options='QJ')
            f['hull_ok'] = 1.0
            f['hull_vol'] = float(hull.volume)
            f['hull_pts_frac'] = len(hull.vertices) / len(Vh)
            f['hull_area'] = float(hull.area)
    except (QhullError, ValueError, MemoryError):
        pass

    if nf == 0:
        f['no_faces'] = 1.0
        f.update({'area_total': 0.0, 'volume': 0.0, 'abs_volume': 0.0, 'vol_ratio': 0.0,
                  'vol_over_area': 0.0, 'absvol_over_area': 0.0, 'vol_over_bbox': 0.0,
                  'absvol_over_bbox': 0.0})
        for pref in ('dihed', 'dihedf', 'farea', 'elen', 'aspect', 'valence', 'defect'):
            f.update(_stats([], pref))
        return f
    f['no_faces'] = 0.0

                                                               
    total_area, vol6, absvol6 = 0.0, 0.0, 0.0
    for s0 in range(0, nf, 500_000):
        Pc = ((Vw_full[Fw[s0:s0 + 500_000]] - center) * inv_s).astype(np.float64)
        cr_c = np.cross(Pc[:, 1] - Pc[:, 0], Pc[:, 2] - Pc[:, 0])
        total_area += 0.5 * float(np.linalg.norm(cr_c, axis=1).sum())
        contrib = (Pc[:, 0] * np.cross(Pc[:, 1], Pc[:, 2])).sum(1)
        vol6 += float(contrib.sum()); absvol6 += float(np.abs(contrib).sum())
        del Pc, cr_c, contrib
    total_area += 1e-12
    vol = abs(vol6 / 6.0)
    absvol = absvol6 / 6.0
                                                                                            
                                                                                            
    f.update({'area_total': total_area, 'volume': vol, 'abs_volume': absvol,
              'vol_ratio': vol / (absvol + 1e-12),
              'vol_over_area': vol / (total_area ** 1.5 + 1e-12),
              'absvol_over_area': absvol / (total_area ** 1.5 + 1e-12),
              'vol_over_bbox': vol / (float(np.prod(ext * inv_s)) + 1e-12),
              'absvol_over_bbox': absvol / (float(np.prod(ext * inv_s)) + 1e-12)})
    if f['hull_ok']:
        f['solidity'] = absvol / (f['hull_vol'] + 1e-12)
        f['solidity_signed'] = vol / (f['hull_vol'] + 1e-12)
        f['hull_area_ratio'] = total_area / (f.get('hull_area', 0.0) + 1e-12)
        f['sphericity'] = (math.pi ** (1 / 3)) * (6 * absvol) ** (2 / 3) / (total_area + 1e-12)

                                                             
    f.update({'n_comp': 1.0, 'n_comp_big': 1.0, 'comp_top1': 1.0, 'comp_top2': 0.0,
              'comp_entropy': 0.0, 'bbox_iou_max': 0.0, 'bbox_iou_mean': 0.0,
              'bbox_iou_frac_pos': 0.0, 'comp_sep_max': 0.0, 'comp_sep_mean': 0.0,
              'inside_frac_max': 0.0, 'comp_exact': 1.0})
    if nf <= COMP_CAP:
        try:
            Ef = np.concatenate([Fw[:, [0, 1]], Fw[:, [1, 2]], Fw[:, [2, 0]]], 0).astype(np.int32)
            A = coo_matrix((np.ones(len(Ef), dtype=np.int8), (Ef[:, 0], Ef[:, 1])), shape=(nv, nv))
            del Ef
            ncomp, lab = connected_components(A, directed=False)
            del A
            used = np.unique(Fw)
            sizes_all = np.bincount(lab[used], minlength=ncomp).astype(np.float64)
            sizes = np.sort(sizes_all[sizes_all > 0])[::-1]
            tot = sizes.sum()
            big_idx = np.where(sizes_all > 0.005 * tot)[0]
            f.update({'n_comp': float(len(sizes)), 'n_comp_big': float(len(big_idx)),
                      'comp_top1': float(sizes[0] / tot),
                      'comp_top2': float(sizes[1] / tot) if len(sizes) > 1 else 0.0,
                      'comp_entropy': float(-((sizes / tot) * np.log(sizes / tot + 1e-12)).sum())})
            if len(big_idx) > 1:
                top = big_idx[np.argsort(-sizes_all[big_idx])][:8]
                pts_l, boxes = [], []
                for c in top:
                    ic = used[lab[used] == c]
                    if len(ic) > 20_000:
                        ic = ic[::len(ic) // 20_000 + 1]
                    p = (Vw_full[ic] - center) * inv_s
                    pts_l.append(p); boxes.append((p.min(0), p.max(0)))
                ious, seps, insides = [], [], []
                for i in range(len(boxes)):
                    for j in range(len(boxes)):
                        if i == j:
                            continue
                        lo, hi = boxes[j]
                        insides.append(float(((pts_l[i] >= lo) & (pts_l[i] <= hi)).all(1).mean()))
                        if j <= i:
                            continue
                        lo2 = np.maximum(boxes[i][0], boxes[j][0])
                        hi2 = np.minimum(boxes[i][1], boxes[j][1])
                        it = float(np.prod(np.clip(hi2 - lo2, 0, None)))
                        vi = float(np.prod(np.clip(boxes[i][1] - boxes[i][0], 1e-6, None)))
                        vj = float(np.prod(np.clip(boxes[j][1] - boxes[j][0], 1e-6, None)))
                        ious.append(it / (vi + vj - it + 1e-12))
                        seps.append(float(np.linalg.norm(
                            (boxes[i][0] + boxes[i][1]) / 2 - (boxes[j][0] + boxes[j][1]) / 2)))
                f.update({'bbox_iou_max': float(max(ious)), 'bbox_iou_mean': float(np.mean(ious)),
                          'bbox_iou_frac_pos': float(np.mean(np.asarray(ious) > 1e-6)),
                          'comp_sep_max': float(max(seps)), 'comp_sep_mean': float(np.mean(seps)),
                          'inside_frac_max': float(max(insides))})
            del lab, sizes_all, sizes, used
        except Exception:
            f['comp_exact'] = 0.0
    else:
        f['comp_exact'] = 0.0

                                                            
    if nf > FACE_CAP:
        f['geo_subsampled'] = 1.0
        anchor = (Vw_full[Fw[:, 0]] - vmin) / (ext.astype(np.float32) + 1e-9)
        Fp, g = None, 1
        while g < 40:
            g += 1
            cid = np.minimum((anchor * g).astype(np.int32), g - 1)
            key = (cid[:, 0] * g + cid[:, 1]) * g + cid[:, 2]
            cnt = np.bincount(key, minlength=g ** 3)
            best = int(np.argmax(cnt))
            if cnt[best] <= FACE_CAP:
                Fp = Fw[key == best]; break
        if Fp is None or len(Fp) < 1000:
            Fp = Fw[:FACE_CAP]
        f['patch_grid'] = float(g)
        del anchor
    else:
        Fp = Fw
        f['patch_grid'] = 1.0
    f['patch_face_frac'] = len(Fp) / max(nf, 1)
    del Fw
    nfw = len(Fp)

    uvp, Fi = np.unique(Fp, return_inverse=True)
    Fi = Fi.reshape(nfw, 3).astype(np.int32)
    Vp = ((Vw_full[uvp] - center) * inv_s).astype(np.float32)
    nvp = len(uvp)
    del uvp, Fp, Vw_full, V

    P = Vp[Fi]
    cr = np.cross(P[:, 1] - P[:, 0], P[:, 2] - P[:, 0])
    area2 = np.linalg.norm(cr, axis=1)
    area = 0.5 * area2
    nrm = cr / (area2[:, None] + 1e-12)
    del cr
    f.update(_stats(area / (area.mean() + 1e-12), 'farea'))
    f['degenerate_ratio'] = float((area < 1e-10).mean())

    L = np.stack([np.linalg.norm(P[:, 1] - P[:, 0], axis=1),
                  np.linalg.norm(P[:, 2] - P[:, 1], axis=1),
                  np.linalg.norm(P[:, 0] - P[:, 2], axis=1)], 1)
    f.update(_stats(L.ravel() / (L.mean() + 1e-12), 'elen'))
    aspect = L.max(1) / (L.min(1) + 1e-12)
    f.update(_stats(np.log1p(aspect), 'aspect'))
    f['sliver_ratio'] = float((aspect > 20).mean())

    for qd in (1, 2):
        f[f'uniq_normals_q{qd}'] = len(np.unique(_hash3(np.rint(nrm * 10 ** qd)))) / nfw
                                                                               
    hn = _hash3(np.rint(nrm * 20))
    _, ninv = np.unique(hn, return_inverse=True)
    mass = np.bincount(ninv, weights=area)
    mass = np.sort(mass)[::-1]
    f['normal_top6_mass'] = float(mass[:6].sum() / (mass.sum() + 1e-12))
    f['normal_top1_mass'] = float(mass[0] / (mass.sum() + 1e-12))
    del hn, ninv, mass

                                                                              
    E, Es, fids, grp, counts, _ = _edge_table(Fi, nfw)
    n_edges = len(counts)
    f.update({'n_edges': float(n_edges),
              'boundary_edge_ratio': float((counts == 1).mean()),
              'nonmanifold_edge_ratio': float((counts >= 3).mean()),
              'euler_norm': float((nvp - n_edges + nfw) / max(nfw, 1)),
              'euler_char': float(nvp - n_edges + nfw),
              'watertight': float((counts == 1).sum() == 0 and (counts >= 3).sum() == 0),
              'manifold_edge_ratio': float((counts == 2).mean())})

                                                                  
    bnd = np.where(counts == 1)[0]
    if len(bnd):
        st = np.searchsorted(grp, bnd)
        be = Es[st]
        try:
            Ab = coo_matrix((np.ones(len(be), dtype=np.int8), (be[:, 0], be[:, 1])), shape=(nvp, nvp))
            nb_comp, lb = connected_components(Ab, directed=False)
            used_b = np.unique(be)
            f['n_boundary_loops'] = float(len(np.unique(lb[used_b])))
            f['boundary_vert_frac'] = float(len(used_b) / nvp)
            del Ab, lb, used_b
        except Exception:
            f['n_boundary_loops'] = 0.0; f['boundary_vert_frac'] = 0.0
        del be, st
    else:
        f['n_boundary_loops'] = 0.0; f['boundary_vert_frac'] = 0.0

    two = np.where(counts == 2)[0]
    if len(two):
        st = np.searchsorted(grp, two)
        fa, fb = fids[st], fids[st + 1]
        cosang = np.clip((nrm[fa] * nrm[fb]).sum(1), -1, 1)
        ang = np.degrees(np.arccos(cosang))
        angf = np.degrees(np.arccos(np.abs(cosang)))
        f.update(_stats(ang, 'dihed')); f.update(_stats(angf, 'dihedf'))
        for t in (10, 30, 60, 90):
            f[f'dihed_gt{t}'] = float((ang > t).mean())
        for t in (5, 15, 30, 60):
            f[f'dihedf_gt{t}'] = float((angf > t).mean())
        f['flip_ratio'] = float((cosang < 0).mean())
        hist = np.bincount((angf / 5).astype(np.int32).clip(0, 17), minlength=18).astype(np.float64)
        p = hist / max(hist.sum(), 1)
        f['dihed_entropy'] = float(-(p[p > 0] * np.log(p[p > 0])).sum())
        f['smoothness'] = float(angf.mean() * math.sqrt(nfw) / 100.0)
                                                                           
        el = np.linalg.norm(Vp[Es[st][:, 0]] - Vp[Es[st][:, 1]], axis=1)
        f['dihedf_areaw'] = float((angf * el).sum() / (el.sum() + 1e-12))
        del fa, fb, cosang, ang, angf, el
    else:
        f.update(_stats([], 'dihed')); f.update(_stats([], 'dihedf'))
        for t in (10, 30, 60, 90):
            f[f'dihed_gt{t}'] = 0.0
        for t in (5, 15, 30, 60):
            f[f'dihedf_gt{t}'] = 0.0
        f.update({'flip_ratio': 0.0, 'dihed_entropy': 0.0, 'smoothness': 0.0, 'dihedf_areaw': 0.0})

                                                                       
    val = np.bincount(Fi.ravel(), minlength=nvp).astype(np.float32)
    f.update(_stats(val, 'valence'))
    f['valence_le3'] = float((val <= 3).mean())
    f['valence_ge8'] = float((val >= 8).mean())
    e01 = P[:, 1] - P[:, 0]; e12 = P[:, 2] - P[:, 1]; e20 = P[:, 0] - P[:, 2]
    def _ang(u, v):
        cu = (u * v).sum(1) / (np.linalg.norm(u, axis=1) * np.linalg.norm(v, axis=1) + 1e-12)
        return np.arccos(np.clip(cu, -1, 1))
    a0 = _ang(e01, -e20); a1 = _ang(e12, -e01); a2 = _ang(e20, -e12)
    defect = np.full(nvp, 2 * np.pi, dtype=np.float64)
    np.subtract.at(defect, Fi[:, 0], a0)
    np.subtract.at(defect, Fi[:, 1], a1)
    np.subtract.at(defect, Fi[:, 2], a2)
    interior = val > 0
    f.update(_stats(np.abs(defect[interior]), 'defect'))
    f['defect_sum'] = float(defect[interior].sum() / (2 * np.pi))                                
    f['defect_gt05'] = float((np.abs(defect[interior]) > 0.5).mean())
    del defect, val, a0, a1, a2, e01, e12, e20

    del E, Es, fids, grp, counts, two, nrm, P, Vp, Fi, L, aspect, area, area2
    return f

                     
def image_features(png_path):
    f = {'img_ok': 0.0}
    try:
        img = Image.open(png_path).convert('L')
    except Exception:
        return f
    a = np.asarray(img, dtype=np.float32) / 255.0
    t = CFG.tile
    f['img_ok'] = 1.0
    covs, comps, edens, stds, bws, bhs = [], [], [], [], [], []
    for k in range(6):
        v = a[(k // 3) * t:(k // 3 + 1) * t, (k % 3) * t:(k % 3 + 1) * t]
        bg = np.median(np.concatenate([v[0], v[-1], v[:, 0], v[:, -1]]))
        mask = np.abs(v - bg) > 0.06
        cov = float(mask.mean()); covs.append(cov)
        if mask.any():
            ys, xs = np.where(mask)
            bws.append((xs.max() - xs.min() + 1) / t)
            bhs.append((ys.max() - ys.min() + 1) / t)
            lab, ncc = ndimage.label(mask)
            if ncc:
                sz = np.bincount(lab.ravel())[1:]
                comps.append(float((sz > 0.01 * sz.sum()).sum()))
            else:
                comps.append(0.0)
        else:
            bws.append(0.0); bhs.append(0.0); comps.append(0.0)
        edens.append(float(np.abs(np.diff(v, axis=1)).mean() + np.abs(np.diff(v, axis=0)).mean()))
        stds.append(float(v.std()))
    def agg(vals, pref):
        v = np.asarray(vals, dtype=np.float64)
        return {f'{pref}_mean': v.mean(), f'{pref}_min': v.min(), f'{pref}_max': v.max(),
                f'{pref}_std': v.std(), f'{pref}_range': v.max() - v.min()}
    f.update(agg(covs, 'cov')); f.update(agg(comps, 'ncc')); f.update(agg(edens, 'edge'))
    f.update(agg(stds, 'pxstd')); f.update(agg(bws, 'bw')); f.update(agg(bhs, 'bh'))
    f['cov_empty_views'] = float((np.asarray(covs) < 0.01).sum())
    f['bbox_fill_view'] = float(np.mean(np.asarray(covs) /
                                        (np.asarray(bws) * np.asarray(bhs) + 1e-6)))
    return f

def _feat_one(args):
    iid, npz, png = args
    d = {'item_id': iid}
    try:
        d.update(mesh_features(npz))
    except (MemoryError, Exception):
        d['geo_ok'] = 0.0
    try:
        d.update(image_features(png))
    except Exception:
        d['img_ok'] = 0.0
    return d

def build_features(ids, npz_dir, img_dir, split):
    """Чанками, с чекпоинтом и резюмированием. Повторный запуск после падения
    продолжает с последнего сохранённого чанка."""
    final = CFG.cache / f'feats_{split}_{CFG.tile}_{FEAT_VER}.parquet'
    ckpt  = CFG.cache / f'feats_{split}_{CFG.tile}_{FEAT_VER}_partial.parquet'
    if final.exists():
        df = pd.read_parquet(final); print(f'{split}: фичи из кэша {df.shape}'); return df

    done = pd.read_parquet(ckpt) if ckpt.exists() else pd.DataFrame(columns=['item_id'])
    have = set(done['item_id'].astype(str)) if len(done) else set()
    todo = [i for i in ids if i not in have]
    print(f'{split}: уже готово {len(have)}, осталось {len(todo)}, воркеров {N_WORKERS}')

    rows = [done] if len(done) else []
    for s in range(0, len(todo), CHUNK):
        part = todo[s:s + CHUNK]
        jobs = [(i, str(Path(npz_dir) / f'{i}.npz'), str(Path(img_dir) / f'{i}.png')) for i in part]
                                                                                           
                                                                                               
                                                                                  
                                                                   
        try:
            ctx = mp.get_context('fork')
        except ValueError:
            ctx = None
        try:
            with ProcessPoolExecutor(max_workers=N_WORKERS, mp_context=ctx) as ex:
                got = list(tqdm(ex.map(_feat_one, jobs, chunksize=4), total=len(jobs),
                                desc=f'{split} {s + len(part)}/{len(todo)}', leave=False))
        except (BrokenProcessPool, OSError, MemoryError) as e:
            print(f'  пул упал ({type(e).__name__}), чанк считается последовательно')
            got = [_feat_one(j) for j in tqdm(jobs, desc='serial', leave=False)]
        rows.append(pd.DataFrame(got))
        _atomic(ckpt, lambda p: pd.concat(rows, ignore_index=True).to_parquet(p, index=False))
        gc.collect()

    df = pd.concat(rows, ignore_index=True).drop_duplicates('item_id')
    df = df.set_index('item_id').reindex([str(i) for i in ids]).reset_index()
    df = df.fillna(0.0)
    _atomic(final, lambda p: df.to_parquet(p, index=False))
    if ckpt.exists():
        ckpt.unlink()
    print(f'{split}: фичи посчитаны {df.shape}')
    return df

feat_tr = build_features(train_df['item_id'].tolist(), TRAIN_DIR, IMG_TRAIN, 'train')
feat_te = build_features(test_df['item_id'].tolist(),  TEST_DIR,  IMG_TEST,  'test')

                     
CLIP_PROMPTS = {
                                            
    'abs_text':     ['3D text lettering', 'a sign with written words', 'an extruded logo'],
    'abs_chart':    ['a bar chart', 'a pie chart', 'a graph plot', 'a diagram', 'a flowchart',
                     'a table with data', 'an infographic'],
    'abs_support':  ['3D printing support structure', 'scaffolding lattice'],
    'abs_voxel':    ['minecraft voxel blocks', 'blocky pixelated cubes'],
                                                     
    'obj_single':   ['a 3D model of a single object', 'a product render on white background'],
    'obj_semantic': ['a chair', 'a car', 'a building', 'a plant', 'a character figure',
                     'a tool', 'furniture'],
                                          
    'p_lowpoly':    ['a low-poly 3D model with visible flat facets'],
    'p_noisy':      ['a noisy 3D scan with rough surface', 'a point cloud scan'],
    'p_broken':     ['a broken mesh with holes and artifacts'],
    'p_hollow':     ['a hollow thin shell', 'a flat plane'],
    'p_multi':      ['several separate objects scattered apart', 'a collection of many objects'],
    'p_small':      ['a tiny object in the middle of a large empty frame'],
    'p_smooth':     ['a smooth clean 3D render of one object'],
}

def compute_clip_features(ids, img_dir, split):
    import open_clip
    name, pretrained = CFG.clip_model
    model, _, _ = open_clip.create_model_and_transforms(name, pretrained=pretrained)
    tokenizer = open_clip.get_tokenizer(name)
    model = model.to(CFG.device).eval()

    groups = list(CLIP_PROMPTS)
    flat, owner = [], []
    for g in groups:
        flat += CLIP_PROMPTS[g]; owner += [g] * len(CLIP_PROMPTS[g])
    with torch.no_grad():
        tf = model.encode_text(tokenizer(flat).to(CFG.device)).float()
        tf = tf / tf.norm(dim=-1, keepdim=True)
    owner = np.array(owner)

    size = model.visual.image_size
    size = size[0] if isinstance(size, (tuple, list)) else int(size)
    mean = torch.tensor([0.48145466, 0.4578275, 0.40821073], device=CFG.device).view(1, 3, 1, 1)
    std = torch.tensor([0.26862954, 0.26130258, 0.27577711], device=CFG.device).view(1, 3, 1, 1)

    rows, bs = [], 24
    for s0 in tqdm(range(0, len(ids), bs), desc=f'CLIP {split}'):
        chunk = ids[s0:s0 + bs]
        batch = []
        for iid in chunk:
            p = Path(img_dir) / f'{iid}.png'
            if p.exists():
                vs = [np.asarray(v.resize((size, size), Image.BILINEAR), dtype=np.uint8)
                      for v in split_views(Image.open(p).convert('RGB'))]
            else:
                vs = [np.full((size, size, 3), 255, np.uint8)] * 6
            batch.append(np.stack(vs))
        x = torch.from_numpy(np.stack(batch)).to(CFG.device)                              
        B = x.shape[0]
        x = x.permute(0, 1, 4, 2, 3).reshape(B * 6, 3, size, size).float() / 255.0
        x = (x - mean) / std
        with torch.no_grad():
            im = model.encode_image(x).float()
        im = im / im.norm(dim=-1, keepdim=True)
        sim = (im @ tf.T).view(B, 6, -1).cpu().numpy()

        gsim = np.stack([sim[:, :, owner == g].max(-1) for g in groups], -1)                   
        feat = {}
        for j, g in enumerate(groups):
            v = gsim[:, :, j]
            feat[f'clip_{g}_mean'] = v.mean(1)
            feat[f'clip_{g}_max'] = v.max(1)
            feat[f'clip_{g}_std'] = v.std(1)
        abs_max = np.stack([feat[f'clip_{g}_max'] for g in groups if g.startswith('abs_')], 1).max(1)
        obj_max = np.stack([feat[f'clip_{g}_max'] for g in groups if g.startswith('obj_')], 1).max(1)
        def_max = np.stack([feat[f'clip_{g}_max'] for g in groups
                            if g.startswith('p_') and g != 'p_smooth'], 1).max(1)
        feat['clip_abstract_margin'] = abs_max - obj_max
        feat['clip_abstract_prob'] = 1 / (1 + np.exp(-100 * (abs_max - obj_max)))
        feat['clip_defect_margin'] = def_max - feat['clip_p_smooth_max']
        rows.append(pd.DataFrame(feat, index=list(chunk)))
    del model
    gc.collect(); torch.cuda.empty_cache()
    return pd.concat(rows).rename_axis('item_id').reset_index()

if CFG.use_clip:
    clip_tr = cached(f'clip_train_{CFG.clip_model[0]}',
                     lambda: compute_clip_features(train_df['item_id'].tolist(), IMG_TRAIN, 'train'))
    clip_te = cached(f'clip_test_{CFG.clip_model[0]}',
                     lambda: compute_clip_features(test_df['item_id'].tolist(), IMG_TEST, 'test'))
    feat_tr = feat_tr.merge(clip_tr, on='item_id', how='left')
    feat_te = feat_te.merge(clip_te, on='item_id', how='left')
    print('добавлено CLIP-признаков:', clip_tr.shape[1] - 1)

    _m = train_df.merge(clip_tr, on='item_id', how='left').fillna(0)
    cc = pd.Series({c: np.corrcoef(_m[c], _m['abstract'])[0, 1]
                    for c in clip_tr.columns if c != 'item_id'}).sort_values(key=abs, ascending=False)
    print('\nкорреляция CLIP-признаков с abstract (топ-8):')
    print(cc.head(8).round(3).to_string())
    print('\nдля сравнения: лучший геометро-визуальный признак давал по abstract r = +0.51')
else:
    print('CLIP выключен (CFG.use_clip = False)')

                     
_ALL_COLS = [c for c in feat_tr.columns if c != 'item_id' and c in feat_te.columns]
                                                                                        
                                                                                             
                                                                                         
FEAT_CLIP = 1e9                                                      

def sanitize_feature_frame(df, cols):
    x = df[cols].to_numpy(dtype=np.float64, copy=True)
    x[~np.isfinite(x)] = 0.0
    np.clip(x, -FEAT_CLIP, FEAT_CLIP, out=x)
    df[cols] = x.astype(np.float32)
    return df

_raw = feat_tr[_ALL_COLS].to_numpy(dtype=np.float64)
print(f'признаки train: нечисловых значений {int((~np.isfinite(_raw)).sum())}, '
      f'по модулю больше {FEAT_CLIP:.0e}: {int((np.abs(np.where(np.isfinite(_raw), _raw, 0)) > FEAT_CLIP).sum())} '
      f'— обнуляются и обрезаются')
sanitize_feature_frame(feat_tr, _ALL_COLS)
sanitize_feature_frame(feat_te, _ALL_COLS)

def dedup_feature_columns():
    """Убираем константные и буквально совпадающие колонки: они только замедляют GBM
    и размывают важность фич между копиями. Сравнение по хешу содержимого колонки."""
    import hashlib
    keep, seen, const, dup = [], {}, [], []
    for c in _ALL_COLS:
        v = np.nan_to_num(feat_tr[c].values.astype(np.float64))
        if v.std() == 0:
            const.append(c); continue
        key = hashlib.md5(np.ascontiguousarray(np.round(v, 9)).tobytes()).hexdigest()
        if key in seen:
            dup.append((c, seen[key])); continue
        seen[key] = c; keep.append(c)
    return {'cols': keep, 'const': const, 'dup': dup}

_fc = cached(f'feat_cols_{CFG.tile}_{FEAT_VER}', dedup_feature_columns)
FEAT_COLS = [c for c in _fc['cols'] if c in feat_te.columns]
print(f'фич всего {len(_ALL_COLS)} -> после дедупликации {len(FEAT_COLS)}')
if _fc['const']:
    print('  константные (выброшены):', ', '.join(_fc['const'][:12]),
          '…' if len(_fc['const']) > 12 else '')
if _fc['dup']:
    print('  дубликаты (выброшены, в скобках — оставленный оригинал):',
          ', '.join(f'{a}({b})' for a, b in _fc['dup'][:12]), '…' if len(_fc['dup']) > 12 else '')
print('подвыборка применялась к', f"{feat_tr['geo_subsampled'].mean():.1%}", 'мешей train')

                                                                                           
                                                                                        
FEAT_SIG = f'{FEAT_VER}_t{CFG.tile}_f{len(FEAT_COLS)}'
L2_SIG = f'{FEAT_SIG}_s{max(1, getattr(CFG, "level2_seeds", 1))}'                          
print('подпись набора признаков:', FEAT_SIG, '| level-2:', L2_SIG)

                     
from iterstrat.ml_stratifiers import MultilabelStratifiedKFold

mskf = MultilabelStratifiedKFold(n_splits=CFG.n_folds, shuffle=True, random_state=CFG.seed)
train_df['fold'] = -1
for k, (_, vi) in enumerate(mskf.split(train_df, train_df[CFG.target_cols].values)):
    train_df.loc[train_df.index[vi], 'fold'] = k
display(train_df.groupby('fold')[CFG.target_cols].mean().round(4))

                     
from torch.utils.data import Dataset, DataLoader

MEAN = np.array([0.485, 0.456, 0.406], dtype=np.float32)
STD  = np.array([0.229, 0.224, 0.225], dtype=np.float32)

def load_views(path, tile):
    """PNG-мозаика -> (6, tile, tile, 3) uint8."""
    a = np.asarray(Image.open(path).convert('RGB'), dtype=np.uint8)
    t = tile
    return np.stack([a[(k // 3) * t:(k // 3 + 1) * t, (k % 3) * t:(k % 3 + 1) * t] for k in range(6)])

class MeshViewDataset(Dataset):
    def __init__(self, df, img_dir, feats, train=True, labels=True):
        self.ids = df['item_id'].tolist()
        self.dir = Path(img_dir)
        self.train = train
        self.labels = df[CFG.target_cols].values.astype(np.float32) if labels else None
        x = feats.set_index('item_id').reindex(self.ids)[FEAT_COLS].to_numpy(dtype=np.float64, copy=True)
        x[~np.isfinite(x)] = 0.0                                                                  
        self.feats = np.clip(x, -FEAT_CLIP, FEAT_CLIP).astype(np.float32)

    def __len__(self):
        return len(self.ids)

    def _aug(self, v):
                            
        rng = np.random
        if rng.rand() < 0.5:                                                   
            v = v[:, :, ::-1]
        if rng.rand() < 0.3:                                                   
            s = rng.randint(-8, 9, size=2)
            v = np.stack([np.roll(x, s, axis=(0, 1)) for x in v])
        if rng.rand() < 0.15:                                    
            k = rng.randint(6)
            v = v.copy(); v[k] = 255
        v = v.astype(np.float32)
        if rng.rand() < 0.3:                                         
            v = np.clip(v * rng.uniform(0.85, 1.15) + rng.uniform(-15, 15), 0, 255)
        return v

    def __getitem__(self, i):
        p = self.dir / f'{self.ids[i]}.png'
        if p.exists():
            v = load_views(p, CFG.tile)
        else:
            v = np.full((6, CFG.tile, CFG.tile, 3), 255, dtype=np.uint8)
        v = self._aug(v) if self.train else v.astype(np.float32)
        v = ((v / 255.0 - MEAN) / STD).astype(np.float32)
        v = torch.from_numpy(np.ascontiguousarray(v)).permute(0, 3, 1, 2)              
        out = {'views': v, 'feats': torch.from_numpy(self.feats[i]), 'item_id': self.ids[i]}
        if self.labels is not None:
            out['y'] = torch.from_numpy(self.labels[i])
        return out

def views_to_grid(v):
    """(B,6,3,t,t) -> (B,3,2t,3t) — мозаика для режима 'grid'."""
    B, N, C, H, W = v.shape
    v = v.view(B, 2, 3, C, H, W).permute(0, 3, 1, 4, 2, 5).reshape(B, C, 2 * H, 3 * W)
    return v

                     
import timm

class MultiViewNet(nn.Module):
    def __init__(self, n_feats, n_out=CFG.n_targets):
        super().__init__()
        self.mode = CFG.view_mode
        kw = dict(pretrained=True, num_classes=0, drop_rate=CFG.drop_rate)
        if IS_VIT:
                                                                                 
            kw['img_size'] = CFG.tile
        self.backbone = timm.create_model(CFG.backbone, **kw)
        d = self.backbone.num_features

                                                                            
        self.att_v = nn.Sequential(nn.Linear(d, 256), nn.Tanh())
        self.att_u = nn.Sequential(nn.Linear(d, 256), nn.Sigmoid())
        self.att_w = nn.Linear(256, 1)

                                                                                    
                                                                              
        self.geo = nn.Sequential(
            nn.BatchNorm1d(n_feats), nn.Linear(n_feats, 256), nn.SiLU(), nn.Dropout(0.2),
            nn.Linear(256, 128), nn.SiLU()) if getattr(CFG, 'use_geo_branch', True) else None
        self.head = nn.Sequential(
            nn.Linear(d + (128 if self.geo is not None else 0), 512), nn.SiLU(), nn.Dropout(0.3),
            nn.Linear(512, n_out))

    def embed(self, views):
        if self.mode == 'grid':
            return self.backbone(views_to_grid(views)), None
        B, N = views.shape[:2]
        z = self.backbone(views.flatten(0, 1)).view(B, N, -1)
        a = self.att_w(self.att_v(z) * self.att_u(z))                     
        w = torch.softmax(a, dim=1)
        return (z * w).sum(1) + z.max(1).values, w.squeeze(-1)

    def forward(self, views, feats, return_att=False):
        z, w = self.embed(views)
        h = torch.cat([z, self.geo(feats)], 1) if self.geo is not None else z
        out = self.head(h)
        return (out, w) if return_att else out

class AsymmetricLoss(nn.Module):
    """Ridnik et al., ICCV 2021."""
    def __init__(self, gamma_neg=4.0, gamma_pos=0.0, clip=0.05, eps=1e-8):
        super().__init__()
        self.gn, self.gp, self.clip, self.eps = gamma_neg, gamma_pos, clip, eps

    def forward(self, logits, y):
        p = torch.sigmoid(logits)
        p_neg = (1 - p + self.clip).clamp(max=1.0)
        loss = y * torch.log(p.clamp(min=self.eps)) + (1 - y) * torch.log(p_neg.clamp(min=self.eps))
        pt = p * y + (1 - p) * (1 - y)
        gamma = self.gp * y + self.gn * (1 - y)
        return -(loss * (1 - pt).pow(gamma)).mean()

                     
from torch.cuda.amp import autocast, GradScaler

def per_class_f1(y, p):
    return pd.Series([f1_score(y[:, i], p[:, i], zero_division=0) for i in range(11)],
                     index=CFG.target_cols)

def quick_thresholds(y, prob):
    """Быстрые per-class пороги (для мониторинга по эпохам)."""
    th = np.full(11, 0.5)
    for i in range(11):
        best, bt = -1, 0.5
        for t in np.arange(0.05, 0.96, 0.02):
            s = f1_score(y[:, i], (prob[:, i] > t).astype(int), zero_division=0)
            if s > best:
                best, bt = s, t
        th[i] = bt
    return th

@torch.no_grad()
def predict(model, loader, tta=1):
    model.eval()
    probs, ids = [], []
    for b in tqdm(loader, desc='predict', leave=False):
        v = b['views'].to(CFG.device, non_blocking=True)
        f = b['feats'].to(CFG.device, non_blocking=True)
        acc = 0
        with autocast(enabled=CFG.amp):
            acc = torch.sigmoid(model(v, f)).float()
            if tta > 1:
                acc = acc + torch.sigmoid(model(torch.flip(v, dims=[-1]), f)).float()
        probs.append((acc / tta).cpu().numpy())
        ids.extend(b['item_id'])
    return np.vstack(probs), ids

class ModelEMA:
    """Экспоненциальное среднее весов (включая буферы BatchNorm). Валидация, снапшоты и сохранённые
    веса берутся по нему: в dino448_v2 пик был на ~11-й эпохе, а к 16-й метрика падала на 0.3."""
    def __init__(self, model, decay):
        import copy
        self.decay, self.updates = float(decay), 0
        self.module = copy.deepcopy(model).eval()
        for p in self.module.parameters():
            p.requires_grad_(False)

    @torch.no_grad()
    def update(self, model):
        self.updates += 1
        d = min(self.decay, (1 + self.updates) / (10 + self.updates))                           
        msd = model.state_dict()
        for k, v in self.module.state_dict().items():
            if v.dtype.is_floating_point:
                v.mul_(d).add_(msd[k].detach(), alpha=1 - d)
            else:
                v.copy_(msd[k])

def train_fold(fold):
                                                                                 
    if CFG.resume and has_artifact(rt(f'fold{fold}_preds')):
        d = load_artifact(rt(f'fold{fold}_preds'))
        diff = _cfg_mismatch(d.get('cfg'))
        if diff:
            print(f'[!] сохранённые предсказания фолда {fold} — от другой конфигурации '
                  f'({"; ".join(diff)}), пересчитываю')
        else:
            print(f'[кэш] фолд {fold} готов (val {d["best"]:.3f}), обучение пропускаем')
            return (d['oof_ids'], d['oof']), (d['te_ids'], d['te'])

                                                                                                
    seed_everything(CFG.seed + fold + 1000 * int(getattr(CFG, 'model_seed', 0)))
    tr = train_df[train_df.fold != fold].reset_index(drop=True)
    va = train_df[train_df.fold == fold].reset_index(drop=True)

    dtr = MeshViewDataset(tr, IMG_TRAIN, feat_tr, train=True)
    dva = MeshViewDataset(va, IMG_TRAIN, feat_tr, train=False)
    ltr = DataLoader(dtr, batch_size=CFG.batch_size, shuffle=True, drop_last=True,
                     num_workers=CFG.num_workers, pin_memory=True, persistent_workers=True)
    lva = DataLoader(dva, batch_size=CFG.batch_size * 2, shuffle=False,
                     num_workers=CFG.num_workers, pin_memory=True)

    model = MultiViewNet(len(FEAT_COLS)).to(CFG.device)
    ema = ModelEMA(model, CFG.ema_decay) if getattr(CFG, 'ema_decay', 0) > 0 else None
    eval_model = lambda: ema.module if ema is not None else model
    head_mods = [m for m in (model.head, model.geo, model.att_v, model.att_u, model.att_w) if m is not None]
    head_ids = {id(p) for m in head_mods for p in m.parameters()}

    def _block_index(name):
        """Номер блока трансформера в имени параметра (для послойного затухания LR)."""
        m = re.search(r'blocks\.(\d+)\.', name)
        if m:
            return int(m.group(1))
        return -1 if any(k in name for k in ('patch_embed', 'pos_embed', 'cls_token',
                                             'reg_token')) else None

    if IS_VIT:
        n_blocks = len(getattr(model.backbone, 'blocks', []))
        base_lr = CFG.backbone_lr
        buckets = {}
        for name, p in model.backbone.named_parameters():
            bi = _block_index(name)
            depth = 0 if bi == -1 else (bi + 1 if bi is not None else n_blocks)
            scale = CFG.layer_decay ** (n_blocks - depth)
            buckets.setdefault(round(scale, 4), []).append(p)
        groups = [{'params': ps, 'lr': base_lr * sc} for sc, ps in buckets.items()]
        groups.append({'params': [p for m in head_mods for p in m.parameters()],
                       'lr': CFG.lr * CFG.head_lr_mult})
        print(f'  ViT: {n_blocks} блоков, LR от {base_lr * min(buckets):.2e} до {base_lr:.2e}, '
              f'голова {CFG.lr * CFG.head_lr_mult:.2e}')
    else:
        groups = [{'params': [p for p in model.parameters() if id(p) not in head_ids], 'lr': CFG.lr},
                  {'params': [p for p in model.parameters() if id(p) in head_ids],
                   'lr': CFG.lr * CFG.head_lr_mult}]
    opt = torch.optim.AdamW(groups, weight_decay=CFG.weight_decay)

    steps = CFG.epochs * len(ltr)
    warm = int(CFG.warmup_frac * steps)
    sched = torch.optim.lr_scheduler.LambdaLR(
        opt, lambda s: s / max(warm, 1) if s < warm
        else 0.5 * (1 + math.cos(math.pi * (s - warm) / max(steps - warm, 1))))

    pos = tr[CFG.target_cols].sum(0).values.astype(np.float32)
    pw = np.clip((len(tr) - pos) / np.clip(pos, 1, None), 1.0, 20.0)
    bce = nn.BCEWithLogitsLoss(pos_weight=torch.tensor(pw, device=CFG.device))
    asl = AsymmetricLoss()
    scaler = GradScaler(enabled=CFG.amp)

                                                                                    
    snaps, best, start_ep = [], -1, 0
    ck = load_ckpt(fold)
    if ck is not None:
        model.load_state_dict(_to_fp32(ck['model']))
        if ema is not None:
            ema.module.load_state_dict(_to_fp32(ck['ema']) if ck.get('ema') is not None else model.state_dict())
            ema.updates = int(ck.get('ema_updates', 0))
        try:
            opt.load_state_dict(ck['opt']); sched.load_state_dict(ck['sched'])
            scaler.load_state_dict(ck['scaler'])
        except Exception as e:
            print('  состояние оптимизатора не восстановлено, продолжаем с текущим:', e)
        snaps = [(sc_i, _to_fp32(sd)) for sc_i, sd in ck['snaps']]
        best, start_ep = ck['best'], ck['epoch'] + 1
        set_rng_state(ck['rng'])
        print(f'  продолжаем фолд {fold} с эпохи {start_ep + 1}/{CFG.epochs} '
              f'(лучшая метрика пока {best:.3f})')
        if start_ep >= CFG.epochs:
            print('  все эпохи пройдены, остаётся только инференс')

    yv = va[CFG.target_cols].values
    pred = None
    for ep in range(start_ep, CFG.epochs):
        model.train(); tot = 0.0
        pbar = tqdm(ltr, desc=f'fold{fold} ep{ep + 1}/{CFG.epochs}')
        for b in pbar:
            v = b['views'].to(CFG.device, non_blocking=True)
            f = b['feats'].to(CFG.device, non_blocking=True)
            y = b['y'].to(CFG.device, non_blocking=True)
            y = y * (1 - CFG.label_smooth) + 0.5 * CFG.label_smooth
                                                                                          
                                                                                    
                                                  
            if CFG.mixup > 0 and random.random() < 0.5:
                lam = float(np.random.beta(CFG.mixup, CFG.mixup))
                perm = torch.randperm(v.size(0), device=v.device)
                v = lam * v + (1 - lam) * v[perm]
                f = lam * f + (1 - lam) * f[perm]
                y = lam * y + (1 - lam) * y[perm]
            opt.zero_grad(set_to_none=True)
            with autocast(enabled=CFG.amp):
                logits = model(v, f)
                loss = 0.5 * bce(logits, y) + 0.5 * asl(logits, y)
            scaler.scale(loss).backward()
            scaler.unscale_(opt)
            torch.nn.utils.clip_grad_norm_(model.parameters(), 5.0)
            scaler.step(opt); scaler.update(); sched.step()
            if ema is not None:
                ema.update(model)
            tot += loss.item(); pbar.set_postfix(loss=f'{tot / (pbar.n + 1):.4f}')

        prob, _ = predict(eval_model(), lva, tta=1)
        th = quick_thresholds(yv, prob)
        pred = (prob > th).astype(int)
        sc, fq, fa = competition_metric(yv, pred)
        print(f'  ep{ep + 1}: metric={sc:.3f} (quality {fq:.3f} | artefacts {fa:.3f})')
        if ep >= 2:                                        
            snaps.append((sc, {k: v.detach().cpu().clone() for k, v in eval_model().state_dict().items()}))
            snaps.sort(key=lambda x: -x[0])
            del snaps[CFG.n_snapshots:]
        if sc > best:
            best = sc
            print('   ^ best')

        log_epoch({'fold': fold, 'epoch': ep + 1, 'train_loss': tot / max(len(ltr), 1),
                   'val_metric': sc, 'f1_quality': fq, 'f1_artefacts': fa,
                   'lr': opt.param_groups[0]['lr'], 'time': time.strftime('%Y-%m-%d %H:%M:%S'),
                   **{f'f1_{c}': float(v) for c, v in per_class_f1(yv, pred).items()}})

        if (ep + 1) % CFG.ckpt_every == 0 or ep == CFG.epochs - 1:
            conv = _to_fp16 if CFG.ckpt_fp16 else (lambda x: x)
            save_ckpt(fold, {'epoch': ep, 'best': best, 'rng': rng_state(),
                             'model': conv(model.state_dict()),
                             'opt': opt.state_dict(), 'sched': sched.state_dict(),
                             'scaler': scaler.state_dict(),
                             'snaps': [(sc_i, conv(sd)) for sc_i, sd in snaps],
                             'ema': conv(ema.module.state_dict()) if ema is not None else None,
                             'ema_updates': ema.updates if ema is not None else 0,
                             'cfg': run_cfg()})
            print(f'   чекпоинт сохранён (эпоха {ep + 1})')

    if pred is None:                                                              
        prob, _ = predict(eval_model(), lva, tta=1)
        pred = (prob > quick_thresholds(yv, prob)).astype(int)
        best = max(best, competition_metric(yv, pred)[0])
    if not snaps:                                                             
        snaps = [(best, {k: v.detach().cpu().clone() for k, v in eval_model().state_dict().items()})]
    print(f'fold {fold}: лучшая валидационная метрика {best:.3f}, '
          f'в ансамбле эпохи со скорами {[round(x[0], 2) for x in snaps]}')
    print(per_class_f1(yv, pred).round(3).to_string())

                                                                                       
    _atomic(ART.parent / f'{CFG.run_tag}_cnn_fold{fold}.pth',
            lambda p: torch.save(_to_fp16(snaps[0][1]) if CFG.ckpt_fp16 else snaps[0][1], p))

    dte = MeshViewDataset(test_df.assign(**{c: 0 for c in CFG.target_cols}), IMG_TEST,
                          feat_te, train=False, labels=False)
    lte = DataLoader(dte, batch_size=CFG.batch_size * 2, shuffle=False,
                     num_workers=CFG.num_workers, pin_memory=True)

                                                                                                 
    oof_acc, te_acc = 0.0, 0.0
    for k, (sc_k, st) in enumerate(snaps):
        model.load_state_dict(st)
        op, oof_ids = predict(model, lva, tta=CFG.tta)
        tp, te_ids = predict(model, lte, tta=CFG.tta)
        oof_acc = oof_acc + op; te_acc = te_acc + tp
    oof_prob, te_prob = oof_acc / len(snaps), te_acc / len(snaps)
    sc_ens = competition_metric(yv, (oof_prob > quick_thresholds(yv, oof_prob)).astype(int))[0]
    print(f'fold {fold}: одна эпоха {best:.3f} -> snapshot-ансамбль {sc_ens:.3f}')

                                                                                   
                                                                         
    if sc_ens < 1.0 or not np.isfinite(oof_prob).all():
        print('  !! ВЫРОЖДЕННЫЙ РЕЗУЛЬТАТ ФОЛДА — диагностика:')
        print(f'     NaN/inf в OOF: {int((~np.isfinite(oof_prob)).sum())} из {oof_prob.size}')
        print(f'     диапазон вероятностей: [{np.nanmin(oof_prob):.4f}, {np.nanmax(oof_prob):.4f}], '
              f'разброс по объектам {np.nanstd(oof_prob, 0).mean():.4f}')
        print(f'     доля предсказанных единиц по классам: '
              f'{np.round((oof_prob > 0.5).mean(0), 3).tolist()}')
        print(f'     позитивов в валидации по классам: {yv.sum(0).tolist()}')
        print('     если разброс ~0 — сеть схлопнулась (проверьте lr и mixup);')
        print('     если NaN — расходимость под AMP (снизьте lr или поставьте CFG.amp = False);')
        print('     если разброс нормальный, а метрика 0 — проверьте, что метки фолда не пустые.')

    keep_snapshots(fold, snaps)
    save_artifact(rt(f'fold{fold}_preds'), {'oof': oof_prob, 'oof_ids': list(oof_ids),
                                            'te': te_prob, 'te_ids': list(te_ids),
                                            'best': best, 'ens': sc_ens, 'cfg': run_cfg(),
                                            'snap_scores': [x[0] for x in snaps]})
    drop_ckpt(fold)                                                                

    del model, ema, ltr, lva, snaps; gc.collect(); torch.cuda.empty_cache()
    return (oof_ids, oof_prob), (te_ids, te_prob)


In [ ]:
%%writefile /content/meshqc_code/dino448_s2_stages.json
[
  {
    "source_cell": 5,
    "source": "import os, gc, json, math, random, re, warnings, shutil\nfrom pathlib import Path\nimport numpy as np\nimport pandas as pd\nimport torch\nimport torch.nn as nn\nimport torch.nn.functional as F\nfrom tqdm.auto import tqdm\nimport matplotlib.pyplot as plt\n\nwarnings.filterwarnings('ignore')\n# Colab печатает 'AssertionError: can only test a child process' при сборке мусора\n# DataLoader-а с воркерами. Это безвредный шум завершения, не ошибка обучения.\nimport logging\nlogging.getLogger('py.warnings').setLevel(logging.ERROR)\n\nclass CFG:\n    seed = 42\n\n    # ---- данные ----\n    dataset  = 'daniilantonov5/3d-mesh-quality-control'\n    work     = LOCAL_DIR; work.mkdir(parents=True, exist_ok=True)\n    cache    = work / 'cache';        cache.mkdir(parents=True, exist_ok=True)\n\n    # ---- вид входа для CNN ----\n    # 'grid'      : одна мозаика 2x3 из 6 рендеров -> 1 forward на объект (быстро, дефолт)\n    # 'multiview' : 6 отдельных видов + attention-pooling (точнее, но x6 дороже)\n    view_mode = 'multiview'  # ViT: 6 квадратных видов\n    tile      = 448  # должен совпадать с dino448_v2, иначе §9 переобучит\n                             # 448 = 14*32 ближе к нативным ~512 px рендера и к 518 претрейна DINOv2\n    grid_rows, grid_cols = 2, 3\n\n    # ---- CNN ----\n    # Self-supervised ViT. DINOv2 обучен на 142M изображений без меток; его признаки известны\n    # сильной линейной разделимостью по семантике формы — а abstract/simple/set это ровно\n    # семантические категории. Патч 14, поэтому tile должен быть кратен 14 (224 = 14*16).\n    backbone   = 'vit_small_patch14_reg4_dinov2.lvd142m'  # как в прогоне dino448_v2: его предсказания берутся из кэша\n    # альтернативы: 'vit_base_patch14_reg4_dinov2.lvd142m' (втрое дороже, для второго прогона),\n    #               'convnext_tiny.fb_in22k_ft_in1k' (прежний, для диверсификации ансамбля)\n    backbone_lr = 4e-5        # ViT после SSL-претрейна требует малого LR, иначе признаки ломаются\n    layer_decay = 0.75        # послойное затухание LR: нижние блоки почти замораживаются\n    drop_rate  = 0.2\n    epochs     = 16  # как в dino448_v2\n    batch_size = 6  # для 448 multiview; при нехватке памяти — 4\n    mixup      = 0.4         # 0 = выключить; мультилейбл-mixup по видам, фичам и меткам\n\n    # ---- дополнительные источники признаков ----\n    use_clip   = True         # zero-shot скоры CLIP -> признаки (прежде всего для abstract)\n    clip_model = ('ViT-B-32', 'laion2b_s34b_b79k')\n    use_dino_frozen = True    # замороженные эмбеддинги DINOv2 -> отдельный источник в бленде\n    dino_frozen_model = 'vit_base_patch14_reg4_dinov2.lvd142m'\n    dino_pca   = 128\n    lr         = 3e-4\n    head_lr_mult = 10.0\n    weight_decay = 0.05\n    warmup_frac  = 0.1\n    label_smooth = 0.01\n    amp        = True\n    num_workers = 4\n\n    # ---- CV ----\n    n_folds = 5\n    RUN_ALL_FOLDS = True         # <-- False для быстрой проверки на одном фолде\n    folds_to_run  = [0]\n\n    # ---- инференс ----\n    tta = 2                      # 1 = без TTA, 2 = + горизонтальный флип каждого вида\n    n_snapshots = 3              # усреднение предсказаний 3 лучших эпох (snapshot ensemble)\n    run_tag = 'dino448_s2'  # текущий прогон — уже обучен, §9 возьмёт кэш\n                                 # несколько обученных конфигураций рядом и ансамблировать их\n    ensemble_tags = ['dino448_v2', 'dino448_s2', 'cnx288_v4', 'cnx288_v5', 'dino448', 'dinos224']  # кандидаты; состав и веса — в §9.2.3\n    # Члены ансамбля, предсказания которых при необходимости восстанавливаются из весов (§9.2.1):\n    # тег -> (бэкбон, tile, режим видов) — ровно та конфигурация, на которой прогон обучался.\n    rebuild_specs = {'dino448': ('vit_small_patch14_reg4_dinov2.lvd142m', 448, 'multiview')}\n    ensemble_select = True       # отбор членов из кандидатов по вложенной оценке (жадно, по одному)\n    level2_seeds = 3             # стэкинг и powerset усредняются по стольким сидам LightGBM\n    ema_decay = 0.999            # EMA весов при обучении: оценка и снапшоты по сглаженным весам (0 = выкл)\n    model_seed = 1               # сид модели для повторного прогона той же конфигурации; фолды НЕ меняет\n    level2_all_candidates = True # в стэкинг и powerset — все исправные кандидаты, а не только отобранные\n    ensemble_weights = True      # веса членов ансамбля подбираются по вложенной оценке (иначе поровну)\n    postproc_select = True       # согласование quality и дефектов выбирается по вложенной оценке\n    level2_member_inputs = True  # члены ансамбля подаются в стэкинг и powerset ещё и раздельно:\n                                 # level-2 обучается по фолдам и сам взвешивает прогоны по классам\n    rule_select_by_nested = True # сложность решающего правила выбирается по вложенной оценке\n    use_geo_branch = True        # False = сеть только по рендерам: её ошибки меньше коррелируют\n                                 # с GBM и с сетями, которым подаются те же признаки\n\n    # ---- устойчивость к перезапускам ----\n    use_drive   = True    # хранить кэш и чекпоинты на Google Drive: /content стирается вместе с рантаймом\n    drive_dir   = '/content/drive/MyDrive/sber_meshqc'\n    resume      = True    # продолжать с последнего чекпоинта, а не считать заново\n    ckpt_every  = 1       # сохранять состояние обучения раз в N эпох\n    ckpt_fp16   = True    # веса в чекпоинте в half: вчетверо меньше места на Drive\n    keep_fold_ckpt = False  # удалять промежуточный чекпоинт фолда после его завершения\n    force_recompute = []  # имена артефактов, которые пересчитать принудительно, напр. ['gbm']\n\n    # ---- метки ----\n    artifact_cols = ['abstract','artifacts','intersection','lowpoly','noisy',\n                     'open','partial','scale','set','simple']\n    target_cols   = artifact_cols + ['quality']\n    n_targets     = 11\n\n    device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')"
  },
  {
    "source_cell": 11,
    "source": "import kagglehub\n\nroot = Path(kagglehub.dataset_download(CFG.dataset))\nprint('dataset root:', root)\n\ndef _pick_dir(mode):\n    \"\"\"Папка с максимальным числом .npz, в пути которой встречается train/test.\"\"\"\n    best, best_n = None, -1\n    for p in root.rglob('*'):\n        if not p.is_dir():\n            continue\n        if mode not in str(p).lower():\n            continue\n        n = sum(1 for _ in p.glob('*.npz'))\n        if n > best_n:\n            best, best_n = p, n\n    if best is None or best_n == 0:\n        raise FileNotFoundError(f'не найдена папка с .npz для {mode}')\n    return best\n\ndef _pick_csv(mode):\n    cands = list(root.rglob('*.csv'))\n    def score(p):\n        n = p.name.lower(); s = 0\n        if mode == 'train':\n            s += 10 * ('train' in n) - 20 * ('submission' in n) - 20 * ('test' in n)\n        else:\n            s += 10 * ('test' in n) - 5 * ('submission' in n) - 20 * ('train' in n)\n        return s - 0.001 * len(n)\n    return sorted(cands, key=score, reverse=True)[0]\n\nTRAIN_DIR, TEST_DIR = _pick_dir('train'), _pick_dir('test')\nTRAIN_CSV, TEST_CSV = _pick_csv('train'), _pick_csv('test')\nprint('train dir :', TRAIN_DIR, len(list(TRAIN_DIR.glob('*.npz'))), 'npz /',\n      len(list(TRAIN_DIR.glob('*.png'))), 'png')\nprint('test  dir :', TEST_DIR,  len(list(TEST_DIR.glob('*.npz'))),  'npz /',\n      len(list(TEST_DIR.glob('*.png'))),  'png')\nprint('train csv :', TRAIN_CSV)\nprint('test  csv :', TEST_CSV)\n\ntrain_df = pd.read_csv(TRAIN_CSV)\ntest_df  = pd.read_csv(TEST_CSV)[['item_id']].copy()\ntrain_df['item_id'] = train_df['item_id'].astype(str)\ntest_df['item_id']  = test_df['item_id'].astype(str)\n\nif 'quality' not in train_df.columns:\n    train_df['quality'] = (train_df[CFG.artifact_cols].sum(1) == 0).astype(int)\n\nprint(train_df.shape, test_df.shape)\ntrain_df.head()"
  },
  {
    "source_cell": 16,
    "source": "from sklearn.metrics import f1_score\n\ndef competition_metric(y_true, y_pred):\n    \"\"\"10*F1(quality) + 10*F1_weighted(10 дефектов) — точная формула организаторов.\"\"\"\n    f1_q = f1_score(y_true[:, 10], y_pred[:, 10], zero_division=0)\n    f1_a = f1_score(y_true[:, :10], y_pred[:, :10], average='weighted', zero_division=0)\n    return 10 * f1_q + 10 * f1_a, f1_q, f1_a\n\n# сколько стоит \"OR-правило\" quality = нет дефектов, если по дефектам ошибаться независимо\nY = train_df[CFG.target_cols].values\nfor err in [0.02, 0.05, 0.10]:\n    rng = np.random.default_rng(0)\n    fake = Y.copy()\n    flip = rng.random(fake[:, :10].shape) < err\n    fake[:, :10] = np.abs(fake[:, :10] - flip)\n    fake[:, 10] = (fake[:, :10].sum(1) == 0).astype(int)\n    s, fq, fa = competition_metric(Y, fake)\n    print(f'FPR/FNR по дефектам {err:.0%} -> метрика {s:5.2f} (quality F1={fq:.3f}, artefacts F1={fa:.3f})')\nprint('\\n>>> Ошибка 5% по дефектам уже срезает quality F1 до ~0.7. '\n      'Поэтому quality предсказывается ОТДЕЛЬНОЙ головой, а не только правилом.')"
  },
  {
    "source_cell": 18,
    "source": "from PIL import Image\nfrom concurrent.futures import ProcessPoolExecutor\n\nImage.MAX_IMAGE_PIXELS = None\n\ndef infer_grid(w, h):\n    \"\"\"(rows, cols) для 6 видов по соотношению сторон.\"\"\"\n    ar = w / h\n    cands = {(2, 3): 3 / 2, (3, 2): 2 / 3, (1, 6): 6.0, (6, 1): 1 / 6}\n    return min(cands, key=lambda k: abs(math.log(ar / cands[k])))\n\ndef split_views(img):\n    \"\"\"PIL.Image -> список из 6 PIL.Image в каноническом порядке чтения.\"\"\"\n    w, h = img.size\n    r, c = infer_grid(w, h)\n    tw, th = w // c, h // r\n    return [img.crop((j * tw, i * th, (j + 1) * tw, (i + 1) * th)) for i in range(r) for j in range(c)]\n\ndef make_cache(args):\n    src, dst, tile = args\n    try:\n        img = Image.open(src).convert('RGB')\n        views = [v.resize((tile, tile), Image.BILINEAR) for v in split_views(img)]\n        out = Image.new('RGB', (3 * tile, 2 * tile))\n        for k, v in enumerate(views):\n            out.paste(v, ((k % 3) * tile, (k // 3) * tile))\n        out.save(dst, format='PNG', optimize=False, compress_level=1)\n        return 1\n    except Exception:\n        return 0\n\ndef build_image_cache(ids, src_dir, split):\n    out_dir = CFG.cache / f'img_{split}_{CFG.tile}'\n    out_dir.mkdir(parents=True, exist_ok=True)\n    jobs = [(str(Path(src_dir) / f'{i}.png'), str(out_dir / f'{i}.png'), CFG.tile)\n            for i in ids if not (out_dir / f'{i}.png').exists()\n            and (Path(src_dir) / f'{i}.png').exists()]\n    if jobs:\n        with ProcessPoolExecutor(max_workers=os.cpu_count()) as ex:\n            ok = list(tqdm(ex.map(make_cache, jobs, chunksize=32), total=len(jobs),\n                           desc=f'cache {split}'))\n        print(f'{split}: закэшировано {sum(ok)}/{len(jobs)}')\n    missing = [i for i in ids if not (out_dir / f'{i}.png').exists()]\n    print(f'{split}: всего в кэше {len(ids) - len(missing)}/{len(ids)}, нет картинки у {len(missing)}')\n    return out_dir, set(missing)\n\n# --- сколько пикселей на вид даёт исходный рендер: апскейл выше этого бессмысленно ---\n_probe = [Image.open(Path(TRAIN_DIR) / f'{i}.png').size\n          for i in train_df['item_id'].head(40) if (Path(TRAIN_DIR) / f'{i}.png').exists()]\nif _probe:\n    _sizes = pd.Series([f'{w}x{h}' for w, h in _probe]).value_counts()\n    print('исходные размеры листа с 6 видами:'); print(_sizes.head(5).to_string())\n    _w, _h = _probe[0]\n    _r, _c = infer_grid(_w, _h)\n    _native = min(_w // _c, _h // _r)\n    print(f'-> раскладка {_r}x{_c}, нативный размер одного вида ~{_native} px')\n    if CFG.tile > _native:\n        print(f'!! CFG.tile = {CFG.tile} БОЛЬШЕ нативного {_native}: это апскейл, он удорожает '\n              f'обучение, но новой информации не даёт. Снижаю tile до {_native}.')\n        CFG.tile = int(_native)\n    else:\n        print(f'   CFG.tile = {CFG.tile} <= нативного — детали сохраняются')\n\nIMG_TRAIN, MISS_TRAIN = build_image_cache(train_df['item_id'].tolist(), TRAIN_DIR, 'train')\nIMG_TEST,  MISS_TEST  = build_image_cache(test_df['item_id'].tolist(),  TEST_DIR,  'test')"
  },
  {
    "source_cell": 21,
    "source": "import os, gc, math, numpy as np, pandas as pd\nfrom pathlib import Path\nfrom PIL import Image\nfrom scipy import ndimage\nfrom scipy.sparse import coo_matrix\nfrom scipy.sparse.csgraph import connected_components\nfrom scipy.spatial import ConvexHull, QhullError\nfrom concurrent.futures import ProcessPoolExecutor\nfrom concurrent.futures.process import BrokenProcessPool\nimport multiprocessing as mp\n\nFACE_CAP  = 300_000\nVERT_CAP  = 200_000\nHULL_CAP  = 20_000\nCOMP_CAP  = 3_000_000\nWELD_TOL  = 1e-6         # доля диагонали bbox\nFEAT_VER  = 'v3weld'     # версия набора фич: входит в имя кэша, чтобы старый parquet\n                         # (посчитанный БЕЗ сварки вершин) не подхватился молча\nN_WORKERS = min(4, os.cpu_count() or 2)\nCHUNK     = 1024\n\ndef _pack21(q):\n    \"\"\"Три целых из [0, 2**21) -> один int64. Точно, без коллизий.\"\"\"\n    return (q[:, 0] << 42) | (q[:, 1] << 21) | q[:, 2]\n\ndef _hash3(q):\n    q = q.astype(np.int64, copy=False)\n    return (q[:, 0] * 73856093) ^ (q[:, 1] * 19349663) ^ (q[:, 2] * 83492791)\n\ndef _stats(x, pref):\n    keys = ['mean', 'std', 'p10', 'p50', 'p90', 'max', 'cv']\n    if len(x) == 0:\n        return {f'{pref}_{k}': 0.0 for k in keys}\n    x = np.asarray(x, dtype=np.float32)\n    m, s = float(x.mean()), float(x.std())\n    q10, q50, q90 = np.percentile(x, [10, 50, 90])\n    return {f'{pref}_mean': m, f'{pref}_std': s, f'{pref}_p10': float(q10),\n            f'{pref}_p50': float(q50), f'{pref}_p90': float(q90),\n            f'{pref}_max': float(x.max()), f'{pref}_cv': float(s / (abs(m) + 1e-9))}\n\ndef _edge_table(Fi, nfw):\n    \"\"\"Рёбра -> (номер группы, число граней на ребро, отсортированные id граней).\"\"\"\n    E = np.concatenate([Fi[:, [0, 1]], Fi[:, [1, 2]], Fi[:, [2, 0]]], 0)\n    E.sort(axis=1)\n    fid = np.tile(np.arange(nfw, dtype=np.int32), 3)\n    order = np.lexsort((E[:, 1], E[:, 0]))\n    Es, fids = E[order], fid[order]\n    new = np.ones(len(Es), dtype=bool)\n    new[1:] = (Es[1:] != Es[:-1]).any(1)\n    grp = np.cumsum(new) - 1\n    counts = np.bincount(grp)\n    return E, Es, fids, grp, counts, new\n\ndef mesh_features(npz_path):\n    f = {'geo_ok': 0.0, 'geo_subsampled': 0.0}\n    try:\n        f['npz_mb'] = os.path.getsize(npz_path) / 1e6\n    except Exception:\n        return f\n    try:\n        with np.load(npz_path, allow_pickle=False) as d:\n            keys = list(d.keys())\n            V = np.asarray(d['vertices' if 'vertices' in keys else keys[0]], dtype=np.float32)\n            Fc = None\n            for k in ('faces', 'triangles', 'f'):\n                if k in keys:\n                    Fc = np.asarray(d[k], dtype=np.int64); break\n            if Fc is None and len(keys) > 1:\n                Fc = np.asarray(d[keys[1]], dtype=np.int64)\n    except Exception:\n        return f\n    if V.ndim != 2 or V.shape[1] != 3 or len(V) == 0:\n        return f\n    if Fc is None or Fc.ndim != 2 or Fc.shape[1] != 3 or len(Fc) == 0:\n        Fc = np.zeros((0, 3), dtype=np.int64)\n    else:\n        Fc = Fc[(Fc >= 0).all(1) & (Fc < len(V)).all(1)]\n\n    f['geo_ok'] = 1.0\n    nv_raw, nf_raw = len(V), len(Fc)\n    vmin, vmax = V.min(0), V.max(0)\n    ext = (vmax - vmin).astype(np.float64)\n    ext_s = np.sort(ext)[::-1]\n    diag = float(np.linalg.norm(ext)) + 1e-12\n    center = ((vmin + vmax) / 2).astype(np.float32)\n    inv_s = np.float32(1.0 / diag)\n    f.update({'n_verts_raw': float(nv_raw), 'n_faces': float(nf_raw),\n              'log_verts': math.log1p(nv_raw), 'log_faces': math.log1p(nf_raw),\n              'ext_max': float(ext_s[0]), 'ext_mid': float(ext_s[1]), 'ext_min': float(ext_s[2]),\n              'ext_ratio_min': float(ext_s[2] / (ext_s[0] + 1e-12)),\n              'ext_ratio_mid': float(ext_s[1] / (ext_s[0] + 1e-12)),\n              'bbox_diag': diag,\n              'bbox_fill': float(nv_raw / (ext.prod() + 1e-12)) if ext.prod() > 0 else 0.0})\n\n    # ================== СВАРКА ВЕРШИН ==================\n    # Экспортёры (glTF/OBJ с раздельными нормалями и UV) дублируют вершины на каждую грань.\n    # Без сварки НИ ОДНО ребро не имеет двух граней: двугранные углы пусты, boundary=1.0,\n    # компонент столько же, сколько треугольников. Вся топология обязана считаться после сварки.\n    step = np.float32(diag * WELD_TOL)\n    q = np.clip(np.rint((V - vmin) / step), 0, (1 << 21) - 1).astype(np.int64)\n    code = _pack21(q)\n    del q\n    ucode, first, invmap = np.unique(code, return_index=True, return_inverse=True)\n    del code, ucode\n    nv = len(first)\n    f['n_verts'] = float(nv)\n    f['weld_ratio'] = float(1.0 - nv / max(nv_raw, 1))          # 0 = уже сварен, ~0.83 = по 3 вершины на грань\n    f['f_per_v'] = nf_raw / max(nv, 1)\n    Vw_full = V[first]\n    del first\n\n    if nf_raw:\n        Fw = invmap[Fc]\n        ok = (Fw[:, 0] != Fw[:, 1]) & (Fw[:, 1] != Fw[:, 2]) & (Fw[:, 0] != Fw[:, 2])\n        f['degen_after_weld'] = float(1.0 - ok.mean())\n        Fw = Fw[ok]\n        hf = _hash3(np.sort(Fw, axis=1))\n        _, uidx = np.unique(hf, return_index=True)\n        f['dup_face_ratio'] = float(1.0 - len(uidx) / max(len(Fw), 1))\n        Fw = Fw[np.sort(uidx)]\n        del hf, uidx, ok\n    else:\n        Fw = np.zeros((0, 3), dtype=np.int64)\n        f['degen_after_weld'] = 0.0; f['dup_face_ratio'] = 0.0\n    del invmap, Fc\n    nf = len(Fw)\n\n    # ---- вершинные статистики (подвыборка) ----\n    Vs = Vw_full[::max(1, int(np.ceil(nv / VERT_CAP)))]\n    if nv > VERT_CAP:\n        f['geo_subsampled'] = 1.0\n    Vs = (Vs - center) * inv_s\n    try:\n        if len(Vs) < 3:\n            raise ValueError('too few vertices')\n        ev = np.clip(np.linalg.eigvalsh(np.cov(Vs.T.astype(np.float64)))[::-1], 1e-16, None)\n        f.update({'pca_1': float(ev[0]), 'pca_2': float(ev[1]), 'pca_3': float(ev[2]),\n                  'pca_flat': float(ev[2] / ev[0]), 'pca_lin': float(ev[1] / ev[0]),\n                  'pca_aniso': float((ev[0] - ev[2]) / ev.sum())})\n    except Exception:\n        f.update({k: 0.0 for k in ['pca_1', 'pca_2', 'pca_3', 'pca_flat', 'pca_lin', 'pca_aniso']})\n\n    # ---- выпуклая оболочка: solidity/сферичность -> simple, open ----\n    f.update({'hull_ok': 0.0, 'solidity': 0.0, 'solidity_signed': 0.0, 'hull_area_ratio': 0.0,\n              'hull_pts_frac': 0.0, 'sphericity': 0.0, 'hull_vol': 0.0, 'hull_area': 0.0})\n    try:\n        Vh = Vs[::max(1, int(np.ceil(len(Vs) / HULL_CAP)))].astype(np.float64)\n        if len(Vh) >= 8:\n            hull = ConvexHull(Vh, qhull_options='QJ')\n            f['hull_ok'] = 1.0\n            f['hull_vol'] = float(hull.volume)\n            f['hull_pts_frac'] = len(hull.vertices) / len(Vh)\n            f['hull_area'] = float(hull.area)\n    except (QhullError, ValueError, MemoryError):\n        pass\n\n    if nf == 0:\n        f['no_faces'] = 1.0\n        f.update({'area_total': 0.0, 'volume': 0.0, 'abs_volume': 0.0, 'vol_ratio': 0.0,\n                  'vol_over_area': 0.0, 'absvol_over_area': 0.0, 'vol_over_bbox': 0.0,\n                  'absvol_over_bbox': 0.0})\n        for pref in ('dihed', 'dihedf', 'farea', 'elen', 'aspect', 'valence', 'defect'):\n            f.update(_stats([], pref))\n        return f\n    f['no_faces'] = 0.0\n\n    # ---- точные площадь/объём по полному сваренному мешу ----\n    total_area, vol6, absvol6 = 0.0, 0.0, 0.0\n    for s0 in range(0, nf, 500_000):\n        Pc = ((Vw_full[Fw[s0:s0 + 500_000]] - center) * inv_s).astype(np.float64)\n        cr_c = np.cross(Pc[:, 1] - Pc[:, 0], Pc[:, 2] - Pc[:, 0])\n        total_area += 0.5 * float(np.linalg.norm(cr_c, axis=1).sum())\n        contrib = (Pc[:, 0] * np.cross(Pc[:, 1], Pc[:, 2])).sum(1)\n        vol6 += float(contrib.sum()); absvol6 += float(np.abs(contrib).sum())\n        del Pc, cr_c, contrib\n    total_area += 1e-12\n    vol = abs(vol6 / 6.0)\n    absvol = absvol6 / 6.0\n    # vol_ratio ~1 у корректно ориентированного меша и ~0 при несогласованной намотке граней\n    # (сама по себе сильная улика для artifacts/noisy). Объём для solidity берём устойчивый.\n    f.update({'area_total': total_area, 'volume': vol, 'abs_volume': absvol,\n              'vol_ratio': vol / (absvol + 1e-12),\n              'vol_over_area': vol / (total_area ** 1.5 + 1e-12),\n              'absvol_over_area': absvol / (total_area ** 1.5 + 1e-12),\n              'vol_over_bbox': vol / (float(np.prod(ext * inv_s)) + 1e-12),\n              'absvol_over_bbox': absvol / (float(np.prod(ext * inv_s)) + 1e-12)})\n    if f['hull_ok']:\n        f['solidity'] = absvol / (f['hull_vol'] + 1e-12)\n        f['solidity_signed'] = vol / (f['hull_vol'] + 1e-12)\n        f['hull_area_ratio'] = total_area / (f.get('hull_area', 0.0) + 1e-12)\n        f['sphericity'] = (math.pi ** (1 / 3)) * (6 * absvol) ** (2 / 3) / (total_area + 1e-12)\n\n    # ---- компоненты связности на полном СВАРЕННОМ меше ----\n    f.update({'n_comp': 1.0, 'n_comp_big': 1.0, 'comp_top1': 1.0, 'comp_top2': 0.0,\n              'comp_entropy': 0.0, 'bbox_iou_max': 0.0, 'bbox_iou_mean': 0.0,\n              'bbox_iou_frac_pos': 0.0, 'comp_sep_max': 0.0, 'comp_sep_mean': 0.0,\n              'inside_frac_max': 0.0, 'comp_exact': 1.0})\n    if nf <= COMP_CAP:\n        try:\n            Ef = np.concatenate([Fw[:, [0, 1]], Fw[:, [1, 2]], Fw[:, [2, 0]]], 0).astype(np.int32)\n            A = coo_matrix((np.ones(len(Ef), dtype=np.int8), (Ef[:, 0], Ef[:, 1])), shape=(nv, nv))\n            del Ef\n            ncomp, lab = connected_components(A, directed=False)\n            del A\n            used = np.unique(Fw)\n            sizes_all = np.bincount(lab[used], minlength=ncomp).astype(np.float64)\n            sizes = np.sort(sizes_all[sizes_all > 0])[::-1]\n            tot = sizes.sum()\n            big_idx = np.where(sizes_all > 0.005 * tot)[0]\n            f.update({'n_comp': float(len(sizes)), 'n_comp_big': float(len(big_idx)),\n                      'comp_top1': float(sizes[0] / tot),\n                      'comp_top2': float(sizes[1] / tot) if len(sizes) > 1 else 0.0,\n                      'comp_entropy': float(-((sizes / tot) * np.log(sizes / tot + 1e-12)).sum())})\n            if len(big_idx) > 1:\n                top = big_idx[np.argsort(-sizes_all[big_idx])][:8]\n                pts_l, boxes = [], []\n                for c in top:\n                    ic = used[lab[used] == c]\n                    if len(ic) > 20_000:\n                        ic = ic[::len(ic) // 20_000 + 1]\n                    p = (Vw_full[ic] - center) * inv_s\n                    pts_l.append(p); boxes.append((p.min(0), p.max(0)))\n                ious, seps, insides = [], [], []\n                for i in range(len(boxes)):\n                    for j in range(len(boxes)):\n                        if i == j:\n                            continue\n                        lo, hi = boxes[j]\n                        insides.append(float(((pts_l[i] >= lo) & (pts_l[i] <= hi)).all(1).mean()))\n                        if j <= i:\n                            continue\n                        lo2 = np.maximum(boxes[i][0], boxes[j][0])\n                        hi2 = np.minimum(boxes[i][1], boxes[j][1])\n                        it = float(np.prod(np.clip(hi2 - lo2, 0, None)))\n                        vi = float(np.prod(np.clip(boxes[i][1] - boxes[i][0], 1e-6, None)))\n                        vj = float(np.prod(np.clip(boxes[j][1] - boxes[j][0], 1e-6, None)))\n                        ious.append(it / (vi + vj - it + 1e-12))\n                        seps.append(float(np.linalg.norm(\n                            (boxes[i][0] + boxes[i][1]) / 2 - (boxes[j][0] + boxes[j][1]) / 2)))\n                f.update({'bbox_iou_max': float(max(ious)), 'bbox_iou_mean': float(np.mean(ious)),\n                          'bbox_iou_frac_pos': float(np.mean(np.asarray(ious) > 1e-6)),\n                          'comp_sep_max': float(max(seps)), 'comp_sep_mean': float(np.mean(seps)),\n                          'inside_frac_max': float(max(insides))})\n            del lab, sizes_all, sizes, used\n        except Exception:\n            f['comp_exact'] = 0.0\n    else:\n        f['comp_exact'] = 0.0\n\n    # ---- пространственный патч для рёберной топологии ----\n    if nf > FACE_CAP:\n        f['geo_subsampled'] = 1.0\n        anchor = (Vw_full[Fw[:, 0]] - vmin) / (ext.astype(np.float32) + 1e-9)\n        Fp, g = None, 1\n        while g < 40:\n            g += 1\n            cid = np.minimum((anchor * g).astype(np.int32), g - 1)\n            key = (cid[:, 0] * g + cid[:, 1]) * g + cid[:, 2]\n            cnt = np.bincount(key, minlength=g ** 3)\n            best = int(np.argmax(cnt))\n            if cnt[best] <= FACE_CAP:\n                Fp = Fw[key == best]; break\n        if Fp is None or len(Fp) < 1000:\n            Fp = Fw[:FACE_CAP]\n        f['patch_grid'] = float(g)\n        del anchor\n    else:\n        Fp = Fw\n        f['patch_grid'] = 1.0\n    f['patch_face_frac'] = len(Fp) / max(nf, 1)\n    del Fw\n    nfw = len(Fp)\n\n    uvp, Fi = np.unique(Fp, return_inverse=True)\n    Fi = Fi.reshape(nfw, 3).astype(np.int32)\n    Vp = ((Vw_full[uvp] - center) * inv_s).astype(np.float32)\n    nvp = len(uvp)\n    del uvp, Fp, Vw_full, V\n\n    P = Vp[Fi]\n    cr = np.cross(P[:, 1] - P[:, 0], P[:, 2] - P[:, 0])\n    area2 = np.linalg.norm(cr, axis=1)\n    area = 0.5 * area2\n    nrm = cr / (area2[:, None] + 1e-12)\n    del cr\n    f.update(_stats(area / (area.mean() + 1e-12), 'farea'))\n    f['degenerate_ratio'] = float((area < 1e-10).mean())\n\n    L = np.stack([np.linalg.norm(P[:, 1] - P[:, 0], axis=1),\n                  np.linalg.norm(P[:, 2] - P[:, 1], axis=1),\n                  np.linalg.norm(P[:, 0] - P[:, 2], axis=1)], 1)\n    f.update(_stats(L.ravel() / (L.mean() + 1e-12), 'elen'))\n    aspect = L.max(1) / (L.min(1) + 1e-12)\n    f.update(_stats(np.log1p(aspect), 'aspect'))\n    f['sliver_ratio'] = float((aspect > 20).mean())\n\n    for qd in (1, 2):\n        f[f'uniq_normals_q{qd}'] = len(np.unique(_hash3(np.rint(nrm * 10 ** qd)))) / nfw\n    # концентрация нормалей: доля площади в 6 крупнейших кластерах -> примитивы\n    hn = _hash3(np.rint(nrm * 20))\n    _, ninv = np.unique(hn, return_inverse=True)\n    mass = np.bincount(ninv, weights=area)\n    mass = np.sort(mass)[::-1]\n    f['normal_top6_mass'] = float(mass[:6].sum() / (mass.sum() + 1e-12))\n    f['normal_top1_mass'] = float(mass[0] / (mass.sum() + 1e-12))\n    del hn, ninv, mass\n\n    # ---- рёбра/углы ПОСЛЕ сварки + диагностика \"как было бы БЕЗ сварки\" ----\n    E, Es, fids, grp, counts, _ = _edge_table(Fi, nfw)\n    n_edges = len(counts)\n    f.update({'n_edges': float(n_edges),\n              'boundary_edge_ratio': float((counts == 1).mean()),\n              'nonmanifold_edge_ratio': float((counts >= 3).mean()),\n              'euler_norm': float((nvp - n_edges + nfw) / max(nfw, 1)),\n              'euler_char': float(nvp - n_edges + nfw),\n              'watertight': float((counts == 1).sum() == 0 and (counts >= 3).sum() == 0),\n              'manifold_edge_ratio': float((counts == 2).mean())})\n\n    # число граничных петель -> сколько «дыр» в поверхности (open)\n    bnd = np.where(counts == 1)[0]\n    if len(bnd):\n        st = np.searchsorted(grp, bnd)\n        be = Es[st]\n        try:\n            Ab = coo_matrix((np.ones(len(be), dtype=np.int8), (be[:, 0], be[:, 1])), shape=(nvp, nvp))\n            nb_comp, lb = connected_components(Ab, directed=False)\n            used_b = np.unique(be)\n            f['n_boundary_loops'] = float(len(np.unique(lb[used_b])))\n            f['boundary_vert_frac'] = float(len(used_b) / nvp)\n            del Ab, lb, used_b\n        except Exception:\n            f['n_boundary_loops'] = 0.0; f['boundary_vert_frac'] = 0.0\n        del be, st\n    else:\n        f['n_boundary_loops'] = 0.0; f['boundary_vert_frac'] = 0.0\n\n    two = np.where(counts == 2)[0]\n    if len(two):\n        st = np.searchsorted(grp, two)\n        fa, fb = fids[st], fids[st + 1]\n        cosang = np.clip((nrm[fa] * nrm[fb]).sum(1), -1, 1)\n        ang = np.degrees(np.arccos(cosang))\n        angf = np.degrees(np.arccos(np.abs(cosang)))\n        f.update(_stats(ang, 'dihed')); f.update(_stats(angf, 'dihedf'))\n        for t in (10, 30, 60, 90):\n            f[f'dihed_gt{t}'] = float((ang > t).mean())\n        for t in (5, 15, 30, 60):\n            f[f'dihedf_gt{t}'] = float((angf > t).mean())\n        f['flip_ratio'] = float((cosang < 0).mean())\n        hist = np.bincount((angf / 5).astype(np.int32).clip(0, 17), minlength=18).astype(np.float64)\n        p = hist / max(hist.sum(), 1)\n        f['dihed_entropy'] = float(-(p[p > 0] * np.log(p[p > 0])).sum())\n        f['smoothness'] = float(angf.mean() * math.sqrt(nfw) / 100.0)\n        # взвешенный по длине ребра угол: устойчивее к мелким треугольникам\n        el = np.linalg.norm(Vp[Es[st][:, 0]] - Vp[Es[st][:, 1]], axis=1)\n        f['dihedf_areaw'] = float((angf * el).sum() / (el.sum() + 1e-12))\n        del fa, fb, cosang, ang, angf, el\n    else:\n        f.update(_stats([], 'dihed')); f.update(_stats([], 'dihedf'))\n        for t in (10, 30, 60, 90):\n            f[f'dihed_gt{t}'] = 0.0\n        for t in (5, 15, 30, 60):\n            f[f'dihedf_gt{t}'] = 0.0\n        f.update({'flip_ratio': 0.0, 'dihed_entropy': 0.0, 'smoothness': 0.0, 'dihedf_areaw': 0.0})\n\n    # ---- валентность вершин и угловой дефект (гауссова кривизна) ----\n    val = np.bincount(Fi.ravel(), minlength=nvp).astype(np.float32)\n    f.update(_stats(val, 'valence'))\n    f['valence_le3'] = float((val <= 3).mean())\n    f['valence_ge8'] = float((val >= 8).mean())\n    e01 = P[:, 1] - P[:, 0]; e12 = P[:, 2] - P[:, 1]; e20 = P[:, 0] - P[:, 2]\n    def _ang(u, v):\n        cu = (u * v).sum(1) / (np.linalg.norm(u, axis=1) * np.linalg.norm(v, axis=1) + 1e-12)\n        return np.arccos(np.clip(cu, -1, 1))\n    a0 = _ang(e01, -e20); a1 = _ang(e12, -e01); a2 = _ang(e20, -e12)\n    defect = np.full(nvp, 2 * np.pi, dtype=np.float64)\n    np.subtract.at(defect, Fi[:, 0], a0)\n    np.subtract.at(defect, Fi[:, 1], a1)\n    np.subtract.at(defect, Fi[:, 2], a2)\n    interior = val > 0\n    f.update(_stats(np.abs(defect[interior]), 'defect'))\n    f['defect_sum'] = float(defect[interior].sum() / (2 * np.pi))     # ~ эйлерова характеристика\n    f['defect_gt05'] = float((np.abs(defect[interior]) > 0.5).mean())\n    del defect, val, a0, a1, a2, e01, e12, e20\n\n    del E, Es, fids, grp, counts, two, nrm, P, Vp, Fi, L, aspect, area, area2\n    return f"
  },
  {
    "source_cell": 23,
    "source": "def image_features(png_path):\n    f = {'img_ok': 0.0}\n    try:\n        img = Image.open(png_path).convert('L')\n    except Exception:\n        return f\n    a = np.asarray(img, dtype=np.float32) / 255.0\n    t = CFG.tile\n    f['img_ok'] = 1.0\n    covs, comps, edens, stds, bws, bhs = [], [], [], [], [], []\n    for k in range(6):\n        v = a[(k // 3) * t:(k // 3 + 1) * t, (k % 3) * t:(k % 3 + 1) * t]\n        bg = np.median(np.concatenate([v[0], v[-1], v[:, 0], v[:, -1]]))\n        mask = np.abs(v - bg) > 0.06\n        cov = float(mask.mean()); covs.append(cov)\n        if mask.any():\n            ys, xs = np.where(mask)\n            bws.append((xs.max() - xs.min() + 1) / t)\n            bhs.append((ys.max() - ys.min() + 1) / t)\n            lab, ncc = ndimage.label(mask)\n            if ncc:\n                sz = np.bincount(lab.ravel())[1:]\n                comps.append(float((sz > 0.01 * sz.sum()).sum()))\n            else:\n                comps.append(0.0)\n        else:\n            bws.append(0.0); bhs.append(0.0); comps.append(0.0)\n        edens.append(float(np.abs(np.diff(v, axis=1)).mean() + np.abs(np.diff(v, axis=0)).mean()))\n        stds.append(float(v.std()))\n    def agg(vals, pref):\n        v = np.asarray(vals, dtype=np.float64)\n        return {f'{pref}_mean': v.mean(), f'{pref}_min': v.min(), f'{pref}_max': v.max(),\n                f'{pref}_std': v.std(), f'{pref}_range': v.max() - v.min()}\n    f.update(agg(covs, 'cov')); f.update(agg(comps, 'ncc')); f.update(agg(edens, 'edge'))\n    f.update(agg(stds, 'pxstd')); f.update(agg(bws, 'bw')); f.update(agg(bhs, 'bh'))\n    f['cov_empty_views'] = float((np.asarray(covs) < 0.01).sum())\n    f['bbox_fill_view'] = float(np.mean(np.asarray(covs) /\n                                        (np.asarray(bws) * np.asarray(bhs) + 1e-6)))\n    return f\n\ndef _feat_one(args):\n    iid, npz, png = args\n    d = {'item_id': iid}\n    try:\n        d.update(mesh_features(npz))\n    except (MemoryError, Exception):\n        d['geo_ok'] = 0.0\n    try:\n        d.update(image_features(png))\n    except Exception:\n        d['img_ok'] = 0.0\n    return d\n\ndef build_features(ids, npz_dir, img_dir, split):\n    \"\"\"Чанками, с чекпоинтом и резюмированием. Повторный запуск после падения\n    продолжает с последнего сохранённого чанка.\"\"\"\n    final = CFG.cache / f'feats_{split}_{CFG.tile}_{FEAT_VER}.parquet'\n    ckpt  = CFG.cache / f'feats_{split}_{CFG.tile}_{FEAT_VER}_partial.parquet'\n    if final.exists():\n        df = pd.read_parquet(final); print(f'{split}: фичи из кэша {df.shape}'); return df\n\n    done = pd.read_parquet(ckpt) if ckpt.exists() else pd.DataFrame(columns=['item_id'])\n    have = set(done['item_id'].astype(str)) if len(done) else set()\n    todo = [i for i in ids if i not in have]\n    print(f'{split}: уже готово {len(have)}, осталось {len(todo)}, воркеров {N_WORKERS}')\n\n    rows = [done] if len(done) else []\n    for s in range(0, len(todo), CHUNK):\n        part = todo[s:s + CHUNK]\n        jobs = [(i, str(Path(npz_dir) / f'{i}.npz'), str(Path(img_dir) / f'{i}.png')) for i in part]\n        # ВАЖНО: max_tasks_per_child переключает старт-метод на 'spawn', а при spawn воркер\n        # заново импортирует __main__ — в ноутбуке там пусто, функция не находится и пул падает\n        # (в прошлом прогоне так упал КАЖДЫЙ чанк, всё считалось последовательно).\n        # Явно требуем 'fork': воркер наследует определения ячейки.\n        try:\n            ctx = mp.get_context('fork')\n        except ValueError:\n            ctx = None\n        try:\n            with ProcessPoolExecutor(max_workers=N_WORKERS, mp_context=ctx) as ex:\n                got = list(tqdm(ex.map(_feat_one, jobs, chunksize=4), total=len(jobs),\n                                desc=f'{split} {s + len(part)}/{len(todo)}', leave=False))\n        except (BrokenProcessPool, OSError, MemoryError) as e:\n            print(f'  пул упал ({type(e).__name__}), чанк считается последовательно')\n            got = [_feat_one(j) for j in tqdm(jobs, desc='serial', leave=False)]\n        rows.append(pd.DataFrame(got))\n        _atomic(ckpt, lambda p: pd.concat(rows, ignore_index=True).to_parquet(p, index=False))\n        gc.collect()\n\n    df = pd.concat(rows, ignore_index=True).drop_duplicates('item_id')\n    df = df.set_index('item_id').reindex([str(i) for i in ids]).reset_index()\n    df = df.fillna(0.0)\n    _atomic(final, lambda p: df.to_parquet(p, index=False))\n    if ckpt.exists():\n        ckpt.unlink()\n    print(f'{split}: фичи посчитаны {df.shape}')\n    return df\n\nfeat_tr = build_features(train_df['item_id'].tolist(), TRAIN_DIR, IMG_TRAIN, 'train')\nfeat_te = build_features(test_df['item_id'].tolist(),  TEST_DIR,  IMG_TEST,  'test')"
  },
  {
    "source_cell": 25,
    "source": "CLIP_PROMPTS = {\n    # --- то, что размечено как abstract ---\n    'abs_text':     ['3D text lettering', 'a sign with written words', 'an extruded logo'],\n    'abs_chart':    ['a bar chart', 'a pie chart', 'a graph plot', 'a diagram', 'a flowchart',\n                     'a table with data', 'an infographic'],\n    'abs_support':  ['3D printing support structure', 'scaffolding lattice'],\n    'abs_voxel':    ['minecraft voxel blocks', 'blocky pixelated cubes'],\n    # --- предметные объекты: контраст к abstract ---\n    'obj_single':   ['a 3D model of a single object', 'a product render on white background'],\n    'obj_semantic': ['a chair', 'a car', 'a building', 'a plant', 'a character figure',\n                     'a tool', 'furniture'],\n    # --- прокси для остальных классов ---\n    'p_lowpoly':    ['a low-poly 3D model with visible flat facets'],\n    'p_noisy':      ['a noisy 3D scan with rough surface', 'a point cloud scan'],\n    'p_broken':     ['a broken mesh with holes and artifacts'],\n    'p_hollow':     ['a hollow thin shell', 'a flat plane'],\n    'p_multi':      ['several separate objects scattered apart', 'a collection of many objects'],\n    'p_small':      ['a tiny object in the middle of a large empty frame'],\n    'p_smooth':     ['a smooth clean 3D render of one object'],\n}\n\ndef compute_clip_features(ids, img_dir, split):\n    import open_clip\n    name, pretrained = CFG.clip_model\n    model, _, _ = open_clip.create_model_and_transforms(name, pretrained=pretrained)\n    tokenizer = open_clip.get_tokenizer(name)\n    model = model.to(CFG.device).eval()\n\n    groups = list(CLIP_PROMPTS)\n    flat, owner = [], []\n    for g in groups:\n        flat += CLIP_PROMPTS[g]; owner += [g] * len(CLIP_PROMPTS[g])\n    with torch.no_grad():\n        tf = model.encode_text(tokenizer(flat).to(CFG.device)).float()\n        tf = tf / tf.norm(dim=-1, keepdim=True)\n    owner = np.array(owner)\n\n    size = model.visual.image_size\n    size = size[0] if isinstance(size, (tuple, list)) else int(size)\n    mean = torch.tensor([0.48145466, 0.4578275, 0.40821073], device=CFG.device).view(1, 3, 1, 1)\n    std = torch.tensor([0.26862954, 0.26130258, 0.27577711], device=CFG.device).view(1, 3, 1, 1)\n\n    rows, bs = [], 24\n    for s0 in tqdm(range(0, len(ids), bs), desc=f'CLIP {split}'):\n        chunk = ids[s0:s0 + bs]\n        batch = []\n        for iid in chunk:\n            p = Path(img_dir) / f'{iid}.png'\n            if p.exists():\n                vs = [np.asarray(v.resize((size, size), Image.BILINEAR), dtype=np.uint8)\n                      for v in split_views(Image.open(p).convert('RGB'))]\n            else:\n                vs = [np.full((size, size, 3), 255, np.uint8)] * 6\n            batch.append(np.stack(vs))\n        x = torch.from_numpy(np.stack(batch)).to(CFG.device)           # (B,6,size,size,3)\n        B = x.shape[0]\n        x = x.permute(0, 1, 4, 2, 3).reshape(B * 6, 3, size, size).float() / 255.0\n        x = (x - mean) / std\n        with torch.no_grad():\n            im = model.encode_image(x).float()\n        im = im / im.norm(dim=-1, keepdim=True)\n        sim = (im @ tf.T).view(B, 6, -1).cpu().numpy()\n\n        gsim = np.stack([sim[:, :, owner == g].max(-1) for g in groups], -1)   # (B,6,n_groups)\n        feat = {}\n        for j, g in enumerate(groups):\n            v = gsim[:, :, j]\n            feat[f'clip_{g}_mean'] = v.mean(1)\n            feat[f'clip_{g}_max'] = v.max(1)\n            feat[f'clip_{g}_std'] = v.std(1)\n        abs_max = np.stack([feat[f'clip_{g}_max'] for g in groups if g.startswith('abs_')], 1).max(1)\n        obj_max = np.stack([feat[f'clip_{g}_max'] for g in groups if g.startswith('obj_')], 1).max(1)\n        def_max = np.stack([feat[f'clip_{g}_max'] for g in groups\n                            if g.startswith('p_') and g != 'p_smooth'], 1).max(1)\n        feat['clip_abstract_margin'] = abs_max - obj_max\n        feat['clip_abstract_prob'] = 1 / (1 + np.exp(-100 * (abs_max - obj_max)))\n        feat['clip_defect_margin'] = def_max - feat['clip_p_smooth_max']\n        rows.append(pd.DataFrame(feat, index=list(chunk)))\n    del model\n    gc.collect(); torch.cuda.empty_cache()\n    return pd.concat(rows).rename_axis('item_id').reset_index()\n\nif CFG.use_clip:\n    clip_tr = cached(f'clip_train_{CFG.clip_model[0]}',\n                     lambda: compute_clip_features(train_df['item_id'].tolist(), IMG_TRAIN, 'train'))\n    clip_te = cached(f'clip_test_{CFG.clip_model[0]}',\n                     lambda: compute_clip_features(test_df['item_id'].tolist(), IMG_TEST, 'test'))\n    feat_tr = feat_tr.merge(clip_tr, on='item_id', how='left')\n    feat_te = feat_te.merge(clip_te, on='item_id', how='left')\n    print('добавлено CLIP-признаков:', clip_tr.shape[1] - 1)\n\n    _m = train_df.merge(clip_tr, on='item_id', how='left').fillna(0)\n    cc = pd.Series({c: np.corrcoef(_m[c], _m['abstract'])[0, 1]\n                    for c in clip_tr.columns if c != 'item_id'}).sort_values(key=abs, ascending=False)\n    print('\\nкорреляция CLIP-признаков с abstract (топ-8):')\n    print(cc.head(8).round(3).to_string())\n    print('\\nдля сравнения: лучший геометро-визуальный признак давал по abstract r = +0.51')\nelse:\n    print('CLIP выключен (CFG.use_clip = False)')"
  },
  {
    "source_cell": 26,
    "source": "_ALL_COLS = [c for c in feat_tr.columns if c != 'item_id' and c in feat_te.columns]\n# Экстремальные значения (напр. bbox_fill у почти плоских мешей) при переводе во float32\n# превращались в inf, а их квадраты переполняли статистики BatchNorm. Отсюда NaN у нескольких\n# объектов в предсказаниях всех сетей и испорченные running_mean/var гео-ветки у dino448.\nFEAT_CLIP = 1e9                  # квадрат 1e18 помещается во float32\n\ndef sanitize_feature_frame(df, cols):\n    x = df[cols].to_numpy(dtype=np.float64, copy=True)\n    x[~np.isfinite(x)] = 0.0\n    np.clip(x, -FEAT_CLIP, FEAT_CLIP, out=x)\n    df[cols] = x.astype(np.float32)\n    return df\n\n_raw = feat_tr[_ALL_COLS].to_numpy(dtype=np.float64)\nprint(f'признаки train: нечисловых значений {int((~np.isfinite(_raw)).sum())}, '\n      f'по модулю больше {FEAT_CLIP:.0e}: {int((np.abs(np.where(np.isfinite(_raw), _raw, 0)) > FEAT_CLIP).sum())} '\n      f'— обнуляются и обрезаются')\nsanitize_feature_frame(feat_tr, _ALL_COLS)\nsanitize_feature_frame(feat_te, _ALL_COLS)\n\ndef dedup_feature_columns():\n    \"\"\"Убираем константные и буквально совпадающие колонки: они только замедляют GBM\n    и размывают важность фич между копиями. Сравнение по хешу содержимого колонки.\"\"\"\n    import hashlib\n    keep, seen, const, dup = [], {}, [], []\n    for c in _ALL_COLS:\n        v = np.nan_to_num(feat_tr[c].values.astype(np.float64))\n        if v.std() == 0:\n            const.append(c); continue\n        key = hashlib.md5(np.ascontiguousarray(np.round(v, 9)).tobytes()).hexdigest()\n        if key in seen:\n            dup.append((c, seen[key])); continue\n        seen[key] = c; keep.append(c)\n    return {'cols': keep, 'const': const, 'dup': dup}\n\n_fc = cached(f'feat_cols_{CFG.tile}_{FEAT_VER}', dedup_feature_columns)\nFEAT_COLS = [c for c in _fc['cols'] if c in feat_te.columns]\nprint(f'фич всего {len(_ALL_COLS)} -> после дедупликации {len(FEAT_COLS)}')\nif _fc['const']:\n    print('  константные (выброшены):', ', '.join(_fc['const'][:12]),\n          '…' if len(_fc['const']) > 12 else '')\nif _fc['dup']:\n    print('  дубликаты (выброшены, в скобках — оставленный оригинал):',\n          ', '.join(f'{a}({b})' for a, b in _fc['dup'][:12]), '…' if len(_fc['dup']) > 12 else '')\nprint('подвыборка применялась к', f\"{feat_tr['geo_subsampled'].mean():.1%}\", 'мешей train')\n\n# Подпись набора признаков входит в имена артефактов 2-го уровня. Без неё GBM, обученный на\n# другом наборе (например, без CLIP), молча подгружался из кэша и ломал ячейку важности.\nFEAT_SIG = f'{FEAT_VER}_t{CFG.tile}_f{len(FEAT_COLS)}'\nL2_SIG = f'{FEAT_SIG}_s{max(1, getattr(CFG, \"level2_seeds\", 1))}'   # + число сидов level-2\nprint('подпись набора признаков:', FEAT_SIG, '| level-2:', L2_SIG)"
  },
  {
    "source_cell": 31,
    "source": "from iterstrat.ml_stratifiers import MultilabelStratifiedKFold\n\nmskf = MultilabelStratifiedKFold(n_splits=CFG.n_folds, shuffle=True, random_state=CFG.seed)\ntrain_df['fold'] = -1\nfor k, (_, vi) in enumerate(mskf.split(train_df, train_df[CFG.target_cols].values)):\n    train_df.loc[train_df.index[vi], 'fold'] = k\ndisplay(train_df.groupby('fold')[CFG.target_cols].mean().round(4))"
  },
  {
    "source_cell": 33,
    "source": "from torch.utils.data import Dataset, DataLoader\n\nMEAN = np.array([0.485, 0.456, 0.406], dtype=np.float32)\nSTD  = np.array([0.229, 0.224, 0.225], dtype=np.float32)\n\ndef load_views(path, tile):\n    \"\"\"PNG-мозаика -> (6, tile, tile, 3) uint8.\"\"\"\n    a = np.asarray(Image.open(path).convert('RGB'), dtype=np.uint8)\n    t = tile\n    return np.stack([a[(k // 3) * t:(k // 3 + 1) * t, (k % 3) * t:(k % 3 + 1) * t] for k in range(6)])\n\nclass MeshViewDataset(Dataset):\n    def __init__(self, df, img_dir, feats, train=True, labels=True):\n        self.ids = df['item_id'].tolist()\n        self.dir = Path(img_dir)\n        self.train = train\n        self.labels = df[CFG.target_cols].values.astype(np.float32) if labels else None\n        x = feats.set_index('item_id').reindex(self.ids)[FEAT_COLS].to_numpy(dtype=np.float64, copy=True)\n        x[~np.isfinite(x)] = 0.0                  # как в таблице признаков: NaN/inf -> 0, обрезка\n        self.feats = np.clip(x, -FEAT_CLIP, FEAT_CLIP).astype(np.float32)\n\n    def __len__(self):\n        return len(self.ids)\n\n    def _aug(self, v):\n        # v: (6,t,t,3) uint8\n        rng = np.random\n        if rng.rand() < 0.5:                       # флип каждого вида отдельно\n            v = v[:, :, ::-1]\n        if rng.rand() < 0.3:                       # сдвиг всех видов синхронно\n            s = rng.randint(-8, 9, size=2)\n            v = np.stack([np.roll(x, s, axis=(0, 1)) for x in v])\n        if rng.rand() < 0.15:                      # view dropout\n            k = rng.randint(6)\n            v = v.copy(); v[k] = 255\n        v = v.astype(np.float32)\n        if rng.rand() < 0.3:                       # яркость/контраст\n            v = np.clip(v * rng.uniform(0.85, 1.15) + rng.uniform(-15, 15), 0, 255)\n        return v\n\n    def __getitem__(self, i):\n        p = self.dir / f'{self.ids[i]}.png'\n        if p.exists():\n            v = load_views(p, CFG.tile)\n        else:\n            v = np.full((6, CFG.tile, CFG.tile, 3), 255, dtype=np.uint8)\n        v = self._aug(v) if self.train else v.astype(np.float32)\n        v = ((v / 255.0 - MEAN) / STD).astype(np.float32)\n        v = torch.from_numpy(np.ascontiguousarray(v)).permute(0, 3, 1, 2)   # (6,3,t,t)\n        out = {'views': v, 'feats': torch.from_numpy(self.feats[i]), 'item_id': self.ids[i]}\n        if self.labels is not None:\n            out['y'] = torch.from_numpy(self.labels[i])\n        return out\n\ndef views_to_grid(v):\n    \"\"\"(B,6,3,t,t) -> (B,3,2t,3t) — мозаика для режима 'grid'.\"\"\"\n    B, N, C, H, W = v.shape\n    v = v.view(B, 2, 3, C, H, W).permute(0, 3, 1, 4, 2, 5).reshape(B, C, 2 * H, 3 * W)\n    return v"
  },
  {
    "source_cell": 35,
    "source": "import timm\n\nclass MultiViewNet(nn.Module):\n    def __init__(self, n_feats, n_out=CFG.n_targets):\n        super().__init__()\n        self.mode = CFG.view_mode\n        kw = dict(pretrained=True, num_classes=0, drop_rate=CFG.drop_rate)\n        if IS_VIT:\n            # позиционные эмбеддинги интерполируются timm-ом под наш размер входа\n            kw['img_size'] = CFG.tile\n        self.backbone = timm.create_model(CFG.backbone, **kw)\n        d = self.backbone.num_features\n\n        # gated attention pooling по видам (используется только в multiview)\n        self.att_v = nn.Sequential(nn.Linear(d, 256), nn.Tanh())\n        self.att_u = nn.Sequential(nn.Linear(d, 256), nn.Sigmoid())\n        self.att_w = nn.Linear(256, 1)\n\n        # гео-ветка опциональна: без неё сеть смотрит только на рендеры, и её ошибки\n        # меньше коррелируют с GBM и с сетями, которым подаются те же признаки\n        self.geo = nn.Sequential(\n            nn.BatchNorm1d(n_feats), nn.Linear(n_feats, 256), nn.SiLU(), nn.Dropout(0.2),\n            nn.Linear(256, 128), nn.SiLU()) if getattr(CFG, 'use_geo_branch', True) else None\n        self.head = nn.Sequential(\n            nn.Linear(d + (128 if self.geo is not None else 0), 512), nn.SiLU(), nn.Dropout(0.3),\n            nn.Linear(512, n_out))\n\n    def embed(self, views):\n        if self.mode == 'grid':\n            return self.backbone(views_to_grid(views)), None\n        B, N = views.shape[:2]\n        z = self.backbone(views.flatten(0, 1)).view(B, N, -1)\n        a = self.att_w(self.att_v(z) * self.att_u(z))            # (B,N,1)\n        w = torch.softmax(a, dim=1)\n        return (z * w).sum(1) + z.max(1).values, w.squeeze(-1)\n\n    def forward(self, views, feats, return_att=False):\n        z, w = self.embed(views)\n        h = torch.cat([z, self.geo(feats)], 1) if self.geo is not None else z\n        out = self.head(h)\n        return (out, w) if return_att else out\n\nclass AsymmetricLoss(nn.Module):\n    \"\"\"Ridnik et al., ICCV 2021.\"\"\"\n    def __init__(self, gamma_neg=4.0, gamma_pos=0.0, clip=0.05, eps=1e-8):\n        super().__init__()\n        self.gn, self.gp, self.clip, self.eps = gamma_neg, gamma_pos, clip, eps\n\n    def forward(self, logits, y):\n        p = torch.sigmoid(logits)\n        p_neg = (1 - p + self.clip).clamp(max=1.0)\n        loss = y * torch.log(p.clamp(min=self.eps)) + (1 - y) * torch.log(p_neg.clamp(min=self.eps))\n        pt = p * y + (1 - p) * (1 - y)\n        gamma = self.gp * y + self.gn * (1 - y)\n        return -(loss * (1 - pt).pow(gamma)).mean()"
  },
  {
    "source_cell": 37,
    "source": "from torch.cuda.amp import autocast, GradScaler\n\ndef per_class_f1(y, p):\n    return pd.Series([f1_score(y[:, i], p[:, i], zero_division=0) for i in range(11)],\n                     index=CFG.target_cols)\n\ndef quick_thresholds(y, prob):\n    \"\"\"Быстрые per-class пороги (для мониторинга по эпохам).\"\"\"\n    th = np.full(11, 0.5)\n    for i in range(11):\n        best, bt = -1, 0.5\n        for t in np.arange(0.05, 0.96, 0.02):\n            s = f1_score(y[:, i], (prob[:, i] > t).astype(int), zero_division=0)\n            if s > best:\n                best, bt = s, t\n        th[i] = bt\n    return th\n\n@torch.no_grad()\ndef predict(model, loader, tta=1):\n    model.eval()\n    probs, ids = [], []\n    for b in tqdm(loader, desc='predict', leave=False):\n        v = b['views'].to(CFG.device, non_blocking=True)\n        f = b['feats'].to(CFG.device, non_blocking=True)\n        acc = 0\n        with autocast(enabled=CFG.amp):\n            acc = torch.sigmoid(model(v, f)).float()\n            if tta > 1:\n                acc = acc + torch.sigmoid(model(torch.flip(v, dims=[-1]), f)).float()\n        probs.append((acc / tta).cpu().numpy())\n        ids.extend(b['item_id'])\n    return np.vstack(probs), ids\n\nclass ModelEMA:\n    \"\"\"Экспоненциальное среднее весов (включая буферы BatchNorm). Валидация, снапшоты и сохранённые\n    веса берутся по нему: в dino448_v2 пик был на ~11-й эпохе, а к 16-й метрика падала на 0.3.\"\"\"\n    def __init__(self, model, decay):\n        import copy\n        self.decay, self.updates = float(decay), 0\n        self.module = copy.deepcopy(model).eval()\n        for p in self.module.parameters():\n            p.requires_grad_(False)\n\n    @torch.no_grad()\n    def update(self, model):\n        self.updates += 1\n        d = min(self.decay, (1 + self.updates) / (10 + self.updates))   # мягкий разгон в начале\n        msd = model.state_dict()\n        for k, v in self.module.state_dict().items():\n            if v.dtype.is_floating_point:\n                v.mul_(d).add_(msd[k].detach(), alpha=1 - d)\n            else:\n                v.copy_(msd[k])\n\ndef train_fold(fold):\n    # 1) фолд уже посчитан целиком -> отдаём сохранённые предсказания, не обучаем\n    if CFG.resume and has_artifact(rt(f'fold{fold}_preds')):\n        d = load_artifact(rt(f'fold{fold}_preds'))\n        diff = _cfg_mismatch(d.get('cfg'))\n        if diff:\n            print(f'[!] сохранённые предсказания фолда {fold} — от другой конфигурации '\n                  f'({\"; \".join(diff)}), пересчитываю')\n        else:\n            print(f'[кэш] фолд {fold} готов (val {d[\"best\"]:.3f}), обучение пропускаем')\n            return (d['oof_ids'], d['oof']), (d['te_ids'], d['te'])\n\n    # model_seed меняет только случайность обучения; фолды задаются CFG.seed и остаются прежними\n    seed_everything(CFG.seed + fold + 1000 * int(getattr(CFG, 'model_seed', 0)))\n    tr = train_df[train_df.fold != fold].reset_index(drop=True)\n    va = train_df[train_df.fold == fold].reset_index(drop=True)\n\n    dtr = MeshViewDataset(tr, IMG_TRAIN, feat_tr, train=True)\n    dva = MeshViewDataset(va, IMG_TRAIN, feat_tr, train=False)\n    ltr = DataLoader(dtr, batch_size=CFG.batch_size, shuffle=True, drop_last=True,\n                     num_workers=CFG.num_workers, pin_memory=True, persistent_workers=True)\n    lva = DataLoader(dva, batch_size=CFG.batch_size * 2, shuffle=False,\n                     num_workers=CFG.num_workers, pin_memory=True)\n\n    model = MultiViewNet(len(FEAT_COLS)).to(CFG.device)\n    ema = ModelEMA(model, CFG.ema_decay) if getattr(CFG, 'ema_decay', 0) > 0 else None\n    eval_model = lambda: ema.module if ema is not None else model\n    head_mods = [m for m in (model.head, model.geo, model.att_v, model.att_u, model.att_w) if m is not None]\n    head_ids = {id(p) for m in head_mods for p in m.parameters()}\n\n    def _block_index(name):\n        \"\"\"Номер блока трансформера в имени параметра (для послойного затухания LR).\"\"\"\n        m = re.search(r'blocks\\.(\\d+)\\.', name)\n        if m:\n            return int(m.group(1))\n        return -1 if any(k in name for k in ('patch_embed', 'pos_embed', 'cls_token',\n                                             'reg_token')) else None\n\n    if IS_VIT:\n        n_blocks = len(getattr(model.backbone, 'blocks', []))\n        base_lr = CFG.backbone_lr\n        buckets = {}\n        for name, p in model.backbone.named_parameters():\n            bi = _block_index(name)\n            depth = 0 if bi == -1 else (bi + 1 if bi is not None else n_blocks)\n            scale = CFG.layer_decay ** (n_blocks - depth)\n            buckets.setdefault(round(scale, 4), []).append(p)\n        groups = [{'params': ps, 'lr': base_lr * sc} for sc, ps in buckets.items()]\n        groups.append({'params': [p for m in head_mods for p in m.parameters()],\n                       'lr': CFG.lr * CFG.head_lr_mult})\n        print(f'  ViT: {n_blocks} блоков, LR от {base_lr * min(buckets):.2e} до {base_lr:.2e}, '\n              f'голова {CFG.lr * CFG.head_lr_mult:.2e}')\n    else:\n        groups = [{'params': [p for p in model.parameters() if id(p) not in head_ids], 'lr': CFG.lr},\n                  {'params': [p for p in model.parameters() if id(p) in head_ids],\n                   'lr': CFG.lr * CFG.head_lr_mult}]\n    opt = torch.optim.AdamW(groups, weight_decay=CFG.weight_decay)\n\n    steps = CFG.epochs * len(ltr)\n    warm = int(CFG.warmup_frac * steps)\n    sched = torch.optim.lr_scheduler.LambdaLR(\n        opt, lambda s: s / max(warm, 1) if s < warm\n        else 0.5 * (1 + math.cos(math.pi * (s - warm) / max(steps - warm, 1))))\n\n    pos = tr[CFG.target_cols].sum(0).values.astype(np.float32)\n    pw = np.clip((len(tr) - pos) / np.clip(pos, 1, None), 1.0, 20.0)\n    bce = nn.BCEWithLogitsLoss(pos_weight=torch.tensor(pw, device=CFG.device))\n    asl = AsymmetricLoss()\n    scaler = GradScaler(enabled=CFG.amp)\n\n    # 2) незавершённый чекпоинт -> продолжаем с той же эпохи и того же состояния ГСЧ\n    snaps, best, start_ep = [], -1, 0\n    ck = load_ckpt(fold)\n    if ck is not None:\n        model.load_state_dict(_to_fp32(ck['model']))\n        if ema is not None:\n            ema.module.load_state_dict(_to_fp32(ck['ema']) if ck.get('ema') is not None else model.state_dict())\n            ema.updates = int(ck.get('ema_updates', 0))\n        try:\n            opt.load_state_dict(ck['opt']); sched.load_state_dict(ck['sched'])\n            scaler.load_state_dict(ck['scaler'])\n        except Exception as e:\n            print('  состояние оптимизатора не восстановлено, продолжаем с текущим:', e)\n        snaps = [(sc_i, _to_fp32(sd)) for sc_i, sd in ck['snaps']]\n        best, start_ep = ck['best'], ck['epoch'] + 1\n        set_rng_state(ck['rng'])\n        print(f'  продолжаем фолд {fold} с эпохи {start_ep + 1}/{CFG.epochs} '\n              f'(лучшая метрика пока {best:.3f})')\n        if start_ep >= CFG.epochs:\n            print('  все эпохи пройдены, остаётся только инференс')\n\n    yv = va[CFG.target_cols].values\n    pred = None\n    for ep in range(start_ep, CFG.epochs):\n        model.train(); tot = 0.0\n        pbar = tqdm(ltr, desc=f'fold{fold} ep{ep + 1}/{CFG.epochs}')\n        for b in pbar:\n            v = b['views'].to(CFG.device, non_blocking=True)\n            f = b['feats'].to(CFG.device, non_blocking=True)\n            y = b['y'].to(CFG.device, non_blocking=True)\n            y = y * (1 - CFG.label_smooth) + 0.5 * CFG.label_smooth\n            # mixup для мультилейбла: смешиваем виды, гео-фичи и метки одним и тем же lam.\n            # Метки остаются вещественными — BCE/ASL это допускают, а сеть перестаёт\n            # заучивать редкие комбинации целиком.\n            if CFG.mixup > 0 and random.random() < 0.5:\n                lam = float(np.random.beta(CFG.mixup, CFG.mixup))\n                perm = torch.randperm(v.size(0), device=v.device)\n                v = lam * v + (1 - lam) * v[perm]\n                f = lam * f + (1 - lam) * f[perm]\n                y = lam * y + (1 - lam) * y[perm]\n            opt.zero_grad(set_to_none=True)\n            with autocast(enabled=CFG.amp):\n                logits = model(v, f)\n                loss = 0.5 * bce(logits, y) + 0.5 * asl(logits, y)\n            scaler.scale(loss).backward()\n            scaler.unscale_(opt)\n            torch.nn.utils.clip_grad_norm_(model.parameters(), 5.0)\n            scaler.step(opt); scaler.update(); sched.step()\n            if ema is not None:\n                ema.update(model)\n            tot += loss.item(); pbar.set_postfix(loss=f'{tot / (pbar.n + 1):.4f}')\n\n        prob, _ = predict(eval_model(), lva, tta=1)\n        th = quick_thresholds(yv, prob)\n        pred = (prob > th).astype(int)\n        sc, fq, fa = competition_metric(yv, pred)\n        print(f'  ep{ep + 1}: metric={sc:.3f} (quality {fq:.3f} | artefacts {fa:.3f})')\n        if ep >= 2:      # первые эпохи в ансамбль не берём\n            snaps.append((sc, {k: v.detach().cpu().clone() for k, v in eval_model().state_dict().items()}))\n            snaps.sort(key=lambda x: -x[0])\n            del snaps[CFG.n_snapshots:]\n        if sc > best:\n            best = sc\n            print('   ^ best')\n\n        log_epoch({'fold': fold, 'epoch': ep + 1, 'train_loss': tot / max(len(ltr), 1),\n                   'val_metric': sc, 'f1_quality': fq, 'f1_artefacts': fa,\n                   'lr': opt.param_groups[0]['lr'], 'time': time.strftime('%Y-%m-%d %H:%M:%S'),\n                   **{f'f1_{c}': float(v) for c, v in per_class_f1(yv, pred).items()}})\n\n        if (ep + 1) % CFG.ckpt_every == 0 or ep == CFG.epochs - 1:\n            conv = _to_fp16 if CFG.ckpt_fp16 else (lambda x: x)\n            save_ckpt(fold, {'epoch': ep, 'best': best, 'rng': rng_state(),\n                             'model': conv(model.state_dict()),\n                             'opt': opt.state_dict(), 'sched': sched.state_dict(),\n                             'scaler': scaler.state_dict(),\n                             'snaps': [(sc_i, conv(sd)) for sc_i, sd in snaps],\n                             'ema': conv(ema.module.state_dict()) if ema is not None else None,\n                             'ema_updates': ema.updates if ema is not None else 0,\n                             'cfg': run_cfg()})\n            print(f'   чекпоинт сохранён (эпоха {ep + 1})')\n\n    if pred is None:          # обучение было полностью восстановлено из чекпоинта\n        prob, _ = predict(eval_model(), lva, tta=1)\n        pred = (prob > quick_thresholds(yv, prob)).astype(int)\n        best = max(best, competition_metric(yv, pred)[0])\n    if not snaps:             # чекпоинт с ранней эпохи: снапшотов ещё не было\n        snaps = [(best, {k: v.detach().cpu().clone() for k, v in eval_model().state_dict().items()})]\n    print(f'fold {fold}: лучшая валидационная метрика {best:.3f}, '\n          f'в ансамбле эпохи со скорами {[round(x[0], 2) for x in snaps]}')\n    print(per_class_f1(yv, pred).round(3).to_string())\n\n    # рядом с артефактами: CFG.work зависит от перезапуска §1, а ART фиксируется в §1.1\n    _atomic(ART.parent / f'{CFG.run_tag}_cnn_fold{fold}.pth',\n            lambda p: torch.save(_to_fp16(snaps[0][1]) if CFG.ckpt_fp16 else snaps[0][1], p))\n\n    dte = MeshViewDataset(test_df.assign(**{c: 0 for c in CFG.target_cols}), IMG_TEST,\n                          feat_te, train=False, labels=False)\n    lte = DataLoader(dte, batch_size=CFG.batch_size * 2, shuffle=False,\n                     num_workers=CFG.num_workers, pin_memory=True)\n\n    # snapshot-ensemble: усредняем вероятности лучших эпох (обучения не требует, только инференс)\n    oof_acc, te_acc = 0.0, 0.0\n    for k, (sc_k, st) in enumerate(snaps):\n        model.load_state_dict(st)\n        op, oof_ids = predict(model, lva, tta=CFG.tta)\n        tp, te_ids = predict(model, lte, tta=CFG.tta)\n        oof_acc = oof_acc + op; te_acc = te_acc + tp\n    oof_prob, te_prob = oof_acc / len(snaps), te_acc / len(snaps)\n    sc_ens = competition_metric(yv, (oof_prob > quick_thresholds(yv, oof_prob)).astype(int))[0]\n    print(f'fold {fold}: одна эпоха {best:.3f} -> snapshot-ансамбль {sc_ens:.3f}')\n\n    # Метрика около нуля означает вырожденные предсказания, а не «плохо обучилось».\n    # Печатаем причину сразу, иначе она всплывёт только на этапе сабмита.\n    if sc_ens < 1.0 or not np.isfinite(oof_prob).all():\n        print('  !! ВЫРОЖДЕННЫЙ РЕЗУЛЬТАТ ФОЛДА — диагностика:')\n        print(f'     NaN/inf в OOF: {int((~np.isfinite(oof_prob)).sum())} из {oof_prob.size}')\n        print(f'     диапазон вероятностей: [{np.nanmin(oof_prob):.4f}, {np.nanmax(oof_prob):.4f}], '\n              f'разброс по объектам {np.nanstd(oof_prob, 0).mean():.4f}')\n        print(f'     доля предсказанных единиц по классам: '\n              f'{np.round((oof_prob > 0.5).mean(0), 3).tolist()}')\n        print(f'     позитивов в валидации по классам: {yv.sum(0).tolist()}')\n        print('     если разброс ~0 — сеть схлопнулась (проверьте lr и mixup);')\n        print('     если NaN — расходимость под AMP (снизьте lr или поставьте CFG.amp = False);')\n        print('     если разброс нормальный, а метрика 0 — проверьте, что метки фолда не пустые.')\n\n    keep_snapshots(fold, snaps)\n    save_artifact(rt(f'fold{fold}_preds'), {'oof': oof_prob, 'oof_ids': list(oof_ids),\n                                            'te': te_prob, 'te_ids': list(te_ids),\n                                            'best': best, 'ens': sc_ens, 'cfg': run_cfg(),\n                                            'snap_scores': [x[0] for x in snaps]})\n    drop_ckpt(fold)          # фолд закрыт, промежуточное состояние больше не нужно\n\n    del model, ema, ltr, lva, snaps; gc.collect(); torch.cuda.empty_cache()\n    return (oof_ids, oof_prob), (te_ids, te_prob)"
  }
]


In [ ]:
%%writefile /content/meshqc_code/submitted_reference.csv
item_id,abstract,artifacts,intersection,lowpoly,noisy,open,partial,scale,set,simple,quality
002f4a73-0a18-40a8-b2bf-5f5e63743850,1,0,0,0,0,0,0,0,0,0,0
0054f4fe-ae32-48b1-bbc4-8e978cd20038,0,0,0,0,0,0,0,0,0,1,0
0148ab6e-5b6a-44fd-8710-9df11206676e,0,0,0,0,1,0,0,0,0,0,0
02fea41f-ad32-4277-a3d3-cb4390f931c7,1,0,0,1,0,0,0,0,0,0,0
03bcb9bd-df4a-4354-9113-e7111f6b373d,0,0,0,0,0,0,0,0,0,1,0
040ca85b-44ba-4ce9-8c84-3dcc48fe3c07,0,0,0,0,0,0,1,1,0,0,0
04513ba4-49e9-4d64-b754-232fceeecf7c,0,0,0,0,0,0,0,0,1,0,0
05dc7916-715f-4764-8f9f-197dcbfd822a,0,0,0,0,1,0,0,0,0,0,0
066758a2-cc21-4cc3-9d28-945dea44a9a2,0,0,0,0,0,0,0,0,0,1,0
066b1e4a-7520-4ed3-8b9a-54e087fd4a42,0,0,0,0,1,0,0,0,0,0,0
06abeb06-598d-4576-9281-ea4d9bfb8c25,0,0,0,0,0,1,0,0,0,1,0
0734fb9b-2584-4ec1-a76d-4df09016eace,1,0,0,0,0,0,0,0,0,0,0
07d04b21-7c58-464f-ac28-c113909c9dea,0,0,0,0,0,1,0,0,0,1,0
08154acc-097c-4a4f-b3b0-2df0ae2bf472,0,0,0,0,0,0,0,0,0,0,1
08408b74-c280-4123-b89a-14c56101f370,0,0,0,0,0,1,0,0,0,1,0
08bfb5c9-abdb-4e3d-932b-a57960f6a4a1,0,0,0,0,0,0,0,0,1,0,0
094d2414-43ec-43a5-b39a-977a0bca09cd,1,0,0,0,0,0,1,0,0,0,0
09571847-0ac5-4640-a691-a417f99dc769,0,0,0,0,0,0,0,0,0,1,0
09b63d29-00cc-454f-ad07-da7672fadc7b,1,0,0,0,0,0,0,0,0,0,0
09e3c426-885c-47e1-a793-cf669eda36ea,0,1,0,0,1,0,0,0,0,0,0
0a5c4d53-3460-46e5-b08d-f379d4b72d04,0,0,0,0,0,0,0,0,0,0,1
0a66d785-c878-4774-823d-90d9a2286732,0,0,0,0,0,0,0,0,0,0,1
0af836df-ea3e-4414-bf22-6088076ac613,0,0,0,0,0,0,0,0,1,0,0
0b1fe806-7d12-4a33-8860-4daa9cf27b79,0,1,0,0,1,0,0,0,0,0,0
0b671dd7-ee49-464b-920b-5a73857b471c,0,0,0,0,0,0,0,0,0,0,1
0b7d54e9-a282-4f28-90d9-77f7403cd238,1,0,0,0,0,0,0,0,1,0,0
0cd69e3c-e1d6-443c-908e-afb242e8beb5,0,0,0,0,0,0,0,0,1,0,0
0d8053f9-d6b6-4581-96fb-76614ea5859c,0,0,1,0,0,0,0,0,0,1,0
0e1c2f46-edb0-4fc5-b1c9-5368c1e22538,0,1,0,1,0,0,0,0,0,0,0
0e261ea2-596f-48b8-b3bc-ec6a8ff4e0d9,1,0,0,0,0,0,0,0,0,0,0
0f33b5b6-9214-403f-9896-2a0d8d0c2fb4,0,0,1,0,0,0,1,1,0,0,0
0f91941c-49a3-4907-9c4c-76d4f70fd84c,0,0,0,0,0,1,0,0,0,1,0
0fee2c9d-eae3-4568-ab6f-f5098bb8f3cb,1,0,0,0,0,0,0,0,0,0,0
101877ad-ef53-453a-8691-7bc6c9bb158f,0,0,0,0,0,0,0,0,1,0,0
104b450b-695b-4535-8064-4f32f27d8958,1,0,0,0,0,0,0,0,0,1,0
106dacb6-032a-4957-8928-bd455173654b,1,0,0,0,0,0,0,0,0,0,0
11f261fc-26d2-4151-9a40-b646f14d5433,1,0,0,0,0,0,0,0,1,0,0
1201b23f-2376-48ce-ac88-9a10cad5f0ad,0,1,0,0,1,0,0,0,0,0,0
121bd464-e019-4e10-be1f-54cb680f9300,0,0,0,0,0,0,0,0,0,0,1
12d30781-eb1c-4bec-8d43-3f54f63cec08,0,0,0,1,0,0,0,0,1,0,0
12df8cd0-7e7e-4643-9936-4a07811d9824,0,1,0,0,1,0,0,0,0,0,0
12fdbacb-070f-4ff9-8bca-2a306ae016d8,0,0,0,0,0,0,0,0,0,1,0
136c7d9c-2dcd-496a-87d6-eb271c165401,0,0,0,0,1,0,0,0,0,0,0
13b2e1c9-f251-4989-ae5e-e33914a93742,0,0,0,1,0,0,0,0,0,1,0
1558ff1e-8483-4f04-92c9-a10e468fd30d,1,0,0,0,0,0,0,0,1,0,0
155a4adf-e090-4dfd-8837-371980883b65,0,0,0,0,0,0,0,0,1,1,0
1586f53d-ec58-4884-ae88-e3ee74f501d2,0,0,0,0,0,0,0,0,0,0,1
15b2164b-721a-41e0-a749-bb1a7bd8c8d1,0,1,0,0,1,0,0,0,0,0,0
15ccd49d-7224-4c00-b921-aac9c3ea482f,0,0,1,0,0,0,0,0,0,0,0
1613adc0-38f0-4939-9ea7-e780fa8dddda,0,0,0,0,0,0,0,0,1,1,0
1691c2b6-c835-43f6-8ce6-b96d3aa65b0a,0,0,0,0,1,0,0,0,0,0,0
1694166c-7024-4c4b-ba9a-ffaebe73bc82,0,1,0,0,1,0,0,0,0,0,0
16998b08-5e5d-4923-8a72-fdd9327e7049,0,0,0,0,0,0,0,0,0,0,1
16ac0cc1-f755-4e31-b30e-d291ac770ef6,0,0,0,1,0,0,0,0,0,0,0
176c457d-7319-4506-95fb-0e1adc4088e8,1,0,0,0,0,0,0,0,0,0,0
17798f30-c6ae-4a1e-a196-462c953f3a26,0,0,0,0,1,0,0,0,0,0,0
177c9840-a374-4269-8c37-dc8d03e7e2c2,0,0,0,1,0,0,0,0,0,0,0
17ae97ea-8164-4946-bfd3-b5ce0c993697,0,0,0,0,1,0,0,0,0,0,0
17c60800-9f58-4a67-be74-e11159750530,0,0,0,0,1,0,0,0,0,0,0
181a1ff0-0998-452d-b99e-6c143c9b29be,0,0,0,0,0,0,0,0,0,1,0
1978bda2-d5d9-40c4-beca-49afb86a672d,0,1,0,0,0,0,0,0,1,0,0
1a16e5fb-6bc9-4f4d-9332-d1860b9b38e3,0,0,0,1,0,0,0,0,0,0,0
1a18ec73-1158-41ac-a54a-3ccfc5014d9a,1,0,0,0,0,0,0,0,0,0,0
1a37810f-4b4c-4e15-b2a3-a29450dbf0e2,0,0,0,0,0,0,0,0,0,0,1
1a3fc946-7e33-45ec-9d05-685a35a33743,0,0,0,0,1,0,0,0,0,0,0
1a990cf2-6b29-4ffe-807f-57b892c688d5,0,0,0,0,0,0,1,0,0,0,0
1adeb6c1-7d28-4efe-9fbd-8ee9993ecae8,0,0,0,0,0,0,0,0,1,0,0
1aea8246-e5e7-4f20-bcf7-0c06945aabec,0,0,0,0,1,0,0,0,0,0,0
1afeb814-1c5a-4f66-8d0f-fd81b64a9acd,1,0,0,0,0,0,0,0,0,0,0
1c3305a5-9bb6-45d8-a488-a3465a4b64c1,0,0,0,0,0,0,0,0,1,0,0
1cb0741a-857b-4ed9-9f42-7f2e5b2a5a63,0,1,0,0,1,0,0,0,0,0,0
1d031257-61f8-46f1-a108-d8ec8ff9a41b,0,0,0,0,0,0,0,0,0,0,1
1d0bb4f7-2a7f-4c3e-9e17-11967b6833d2,0,0,0,0,1,0,0,0,0,0,0
1d15cab1-446d-434c-98f7-b38d580ff85c,0,0,0,0,1,0,0,0,0,0,0
1d4ece8c-0ed0-49ac-b0db-6effa50bc214,0,0,0,1,0,0,1,0,0,0,0
1e716a94-5685-4453-9fb8-a60ec290389d,0,0,0,0,0,0,0,0,0,1,0
1ed0e20d-2724-448e-8967-5b7c568b45a2,0,0,0,0,0,1,0,0,0,1,0
1f1d0b91-64fb-4035-87ee-bcdc1af5ed72,0,0,0,0,0,0,0,0,1,0,0
20798d1a-0366-4925-9b42-67814dbbb4d0,0,0,0,0,1,0,0,0,0,0,0
21224dc3-7f05-4ee1-8985-0eb36d2ae223,1,0,0,0,0,0,0,0,0,1,0
212ca901-de99-49fc-b746-267364dbd4ac,1,0,0,0,0,0,0,0,0,1,0
216e6849-6bfe-459e-978d-6c764b53070a,0,0,0,1,0,0,0,0,0,0,0
219b561d-aa86-42a2-9b68-e1d25d430ea8,0,1,0,1,0,0,0,0,0,0,0
229044a6-939c-43ea-8d04-679e46af4812,0,0,0,0,0,0,0,0,0,0,1
229807e9-2d80-40d2-af49-862ac6b20fc1,0,0,0,0,1,0,0,0,0,0,0
22a5fe91-f1d9-4463-bd52-97ac3a0f17a2,0,1,0,0,1,0,0,0,0,0,0
23045453-e380-4f2c-8b15-4d2b68abe370,0,0,0,0,1,0,0,0,0,0,0
2348f4f9-45dc-497f-975e-509431a21d9d,0,0,0,1,0,0,0,0,0,0,0
259c8565-f407-41c9-b8b6-765e05b8a122,0,0,0,0,0,0,1,0,0,0,0
26572cb4-c30a-4850-bd03-513828e351a5,0,0,0,0,0,0,1,0,0,0,0
276007f8-e906-4ff3-8389-060218bcbdef,0,0,0,0,1,0,0,0,0,0,0
27f3dfa0-0212-41ba-a5a4-7c1f903f98fc,0,0,0,0,0,0,0,0,0,1,0
2824ace3-190c-43e2-adc0-647b93c0ba64,0,0,0,0,0,0,0,0,0,0,1
285c3198-e790-4811-b4ed-d6a870a521a4,0,1,0,0,1,0,0,0,1,0,0
28a7a1ea-ae09-46c9-a544-bf9736143d7e,0,0,0,0,1,0,0,0,0,0,0
295ac08f-9c03-4a81-8553-dad76b9d54ba,1,0,0,0,0,0,0,0,0,1,0
29acae3f-486f-4433-9940-197e9093a63a,0,0,0,1,0,0,0,0,0,0,0
29cbc7c4-dbf7-49c1-a75d-fbd3970d1fde,1,0,1,0,0,0,0,0,0,0,0
2a5317e1-df0a-46c9-9f26-6cb0131076aa,0,0,0,0,0,0,0,0,0,1,0
2a5bdc1c-d0f9-4440-9d66-a6c47cb1df0b,0,0,0,0,1,0,0,0,0,0,0
2a6b9527-46a1-462f-9864-9f99d2931679,1,0,0,0,0,0,0,0,0,1,0
2ad6c387-d7e1-4d92-8392-421faf396282,0,0,0,0,0,0,0,0,0,0,1
2aef58c1-7852-4705-8109-a130a99140bf,0,0,0,0,0,1,0,0,0,1,0
2b20459c-b061-4822-9295-85c7ff1f4188,1,0,0,0,0,0,0,0,0,1,0
2b52b5d1-e315-4ec2-87cb-8313cffbdd85,0,0,0,0,0,0,0,0,0,1,0
2b7eb110-2935-4464-bdfd-c16c0523ea07,0,0,1,0,0,0,1,0,0,0,0
2c507ff6-1f03-443e-a508-cce9b7f81d18,0,0,0,0,0,0,0,0,1,0,0
2c559756-83ad-4a0a-9b09-601d49b19890,0,1,0,0,1,1,0,0,0,0,0
2ce50229-64d2-477f-81b9-dad120e92e68,0,1,0,0,1,0,0,0,0,0,0
2d0bf27e-3c78-4fa8-abb7-a590feedcaf2,0,0,0,1,0,0,0,0,0,0,1
2ddf1c50-6929-4231-8444-6a27f9f6aebc,0,0,0,1,0,0,0,0,0,1,0
2e728cfb-8f64-4601-a448-171d72bac60f,0,0,0,1,0,0,0,0,0,0,0
2ffa35ee-e147-4624-91e8-21f4c6d53485,0,1,0,0,1,0,0,0,0,0,0
301dbc86-343c-4095-b952-2f976a579141,0,0,0,0,0,0,0,0,0,1,0
30678bd7-6196-4599-ad35-f50040d9a722,0,0,0,0,0,0,0,0,0,0,1
30780d3c-8542-4648-9815-1a44ccdf2a3a,1,0,0,0,0,0,0,0,0,0,0
30b9fb9b-dd1f-4d2b-918f-6894dbac74ba,0,0,0,0,1,0,0,0,0,0,0
30bc2f18-cb9e-4156-b08a-5a2ec266b55c,1,0,0,0,0,0,0,0,0,0,0
30c6160e-eb71-4bc4-97b0-d705a2d47179,0,0,0,1,0,0,0,0,0,0,1
31a97633-3264-4d75-bdea-b808ce3b2d37,0,0,0,0,1,0,0,0,0,0,0
31d71a50-7d39-483f-bb9f-4db3ddb41ce0,0,0,0,0,1,0,0,0,0,0,0
325126b8-7483-45e3-b744-59b591023de6,0,0,0,0,0,0,1,0,0,0,0
3282cc6f-98bd-418a-a0c5-02b7eaad81b8,0,0,0,0,0,0,0,0,0,0,1
3348948b-8b3a-4e1a-be78-5b7d71cf7a75,0,0,0,1,0,0,0,0,0,0,0
33731fc8-a791-4ec8-84a4-639a6d0952e6,0,0,0,1,0,0,0,0,0,0,0
33e8ec03-eb80-46bd-a883-78c3bcc4ab55,1,0,0,0,0,0,0,0,0,1,0
3403225f-54b9-4054-9dd8-d3a439c2620b,0,0,0,0,1,0,0,0,0,0,0
34209482-9cbc-4d0e-bc4b-eac1564c0210,0,0,0,1,0,0,0,0,0,0,0
343e3480-b637-4668-8c61-35c05eb6a4a2,0,0,0,1,0,0,0,0,0,0,0
353c1bc3-f83a-4cb1-8226-d6b95f8514a2,0,0,0,0,1,0,0,0,0,0,0
355a4a81-86bd-4f3f-be5e-2cc13d742d34,0,0,0,0,0,0,0,0,0,0,1
359b1be5-88dc-4264-b7ec-6da9256eb0f9,0,0,0,0,0,1,0,0,0,1,0
35a86d3b-c509-4772-b0ce-47b0adb1764d,0,0,0,0,1,0,0,0,0,0,0
3710e084-139b-4bf4-b6c9-46e2072e1ba2,0,0,0,0,0,0,0,0,0,1,1
37142c36-8ec1-4ebd-a84a-3a00894f55a9,0,0,0,0,1,0,0,0,0,0,0
374c0f15-a359-4e7b-ad71-09a7b31bd897,0,0,0,0,1,0,0,0,0,0,0
378421b3-bc4c-499a-b701-59587bfed0df,0,0,0,0,1,0,0,0,0,0,0
37bc2802-e83b-48ee-89fc-6d9ca88f1147,0,0,0,0,0,0,0,0,0,1,0
3889b93f-47fb-483f-b98c-74ad5b85e7e5,0,0,0,0,0,0,0,0,1,0,0
391bc7f5-a27e-4a33-8bbd-ae2af86859a7,0,0,0,1,0,0,0,0,0,0,0
3936569e-d35c-4954-851a-532a4091c210,0,0,0,0,1,0,0,0,0,0,0
3ad163ed-6278-44d2-8441-ae46727af562,0,0,0,0,1,0,0,0,0,0,0
3b610a19-bca2-4346-b691-c3aa17604001,0,0,0,0,0,0,1,0,1,0,0
3c7e5267-30a3-4e0f-9b01-85ad451814b9,0,0,0,0,0,0,0,0,0,0,1
3ca1764f-f155-48a6-ab35-92cf7c823ca8,0,0,0,0,1,0,0,0,0,0,0
3cf28e62-6aff-4b58-955d-dfd6b3ab93fa,1,0,0,0,0,0,0,0,0,0,0
3d00aeda-93f6-43f0-9325-023297ccdccf,0,0,1,0,0,1,1,1,1,0,0
3d49672e-b750-4fc5-9fc9-661eec28fe04,0,0,0,0,0,0,0,0,0,0,1
3daa394b-a841-455e-aac9-a31a17318a94,0,0,0,0,0,0,0,0,1,0,0
3e30d729-1358-406e-9c47-a833fa80cc3c,0,0,0,0,0,0,0,0,1,0,0
3ebb35b7-ead9-4946-b6c6-e42315dbbfee,1,0,0,0,0,0,0,0,0,0,0
3f6b79b5-93e8-40da-9225-f0f47754d843,0,0,0,1,0,0,0,0,0,0,0
3f6bb1ef-d323-42a0-8a5a-611513c79c1c,1,0,0,0,0,0,0,0,0,0,0
409bbfe9-a18f-4b26-8cd5-b5f880a1980a,0,0,0,1,0,0,0,0,1,0,0
40a01d37-9b82-4295-8f9e-18ab2d32711e,0,0,0,0,0,1,0,0,0,0,0
40c88735-942c-4620-81bc-cfb7b7d5ea3e,1,0,0,0,0,0,0,0,0,0,0
412cd886-8b20-4ac6-bb87-e56606440bc8,0,0,0,1,0,0,0,0,0,0,0
4174b78a-31ec-4266-89b8-93ef6da2ada2,0,0,0,0,0,0,0,0,1,0,0
41dc28e5-e5e3-4c46-84c0-4ebc98d59d91,0,0,0,0,0,0,0,0,1,0,1
42067e4c-ca77-4703-b355-77db401af4a7,0,0,0,0,0,0,0,0,0,1,0
42ffd2f8-92d8-42a4-ac76-abc5085fb8dd,1,0,0,0,0,1,0,0,1,0,0
4327833d-60af-4177-a56c-ac6033a8ef18,0,1,0,0,1,0,0,0,0,0,0
434cb8ca-fc05-4966-8f01-e0e966c5c542,0,1,0,0,1,0,0,0,0,0,0
435fe428-e892-431b-9c55-71b0ba8570be,0,0,0,0,1,0,0,0,0,0,0
43dc82d0-2a9a-44a0-bb59-67906b4080f2,0,0,0,0,0,0,0,0,0,1,0
4501d6f4-c313-465b-a357-9c03ab2832c0,0,0,0,0,0,0,0,0,0,1,0
465025dd-6bd7-4244-b3fa-699a1bf78429,0,0,0,0,1,0,0,0,0,0,0
46c80079-b249-4005-984e-6da6f17b55b8,0,0,0,0,0,0,0,0,0,1,0
46f6c5df-4317-4cb6-80fc-9da1672a701b,0,0,0,0,0,0,0,0,0,0,1
47a3aac1-cd70-4860-a09d-addf18f326c8,0,0,0,0,0,0,1,1,0,0,0
47a79305-cb90-4c76-922d-3785f7e2fe82,0,0,0,0,0,0,0,0,1,0,0
47b667bc-27c9-4859-b86d-e8279ef5dcd8,0,0,0,0,0,0,0,0,1,0,0
480c5922-ea2b-4232-9329-ead7064a461f,1,0,0,0,0,0,0,0,0,1,0
481b085f-ffdb-4c15-a27a-bd3005131619,0,0,0,1,0,0,0,0,0,0,0
481e2593-247f-4ef3-9d89-7b17a53f2091,0,0,0,0,0,0,0,0,0,1,0
4830f60f-9ae6-40c4-89e4-85a00d68a66e,0,0,0,0,0,1,0,0,0,1,0
483c799d-a08c-4946-a29a-2d6f2c3027fa,1,0,0,0,0,0,0,0,0,0,0
486eb241-99ec-4c9c-a878-96c95c43450d,0,0,0,0,0,0,0,0,0,1,0
487e0f0f-0a2a-4378-9006-46ad746da8c8,0,0,0,1,0,0,0,0,1,0,0
48af511a-83e6-4454-9b52-8c912b80e629,0,0,0,0,0,0,0,0,0,1,0
496f82ea-8c7c-49de-b977-6c83b3972d93,1,0,0,0,0,0,0,0,0,0,0
4982164f-6e2d-4365-a455-a16546e1ceaa,0,1,0,0,1,0,0,0,0,0,0
4a57aaf6-9ec2-483e-9a09-50bccb4d9ad5,0,0,0,0,1,0,0,0,0,0,0
4a823cfd-ca14-4973-b9e9-2b3ba5831f2f,0,0,0,0,0,0,0,0,0,0,1
4b9f898e-3be9-4878-9e69-9c06a251e407,0,1,0,0,1,1,0,0,0,0,0
4c72f631-0d8b-48e0-822f-0753c51424ea,0,0,0,0,1,0,0,0,0,0,0
4c82a537-270a-49f7-a779-02d48bdb95ea,0,0,0,0,1,1,0,0,0,0,0
4ca18641-e48b-4fe3-82ca-a7c452c37c4d,0,0,0,0,0,0,0,0,0,0,1
4ddf5cf5-b404-4a2e-9059-9c0c6e743bf0,0,0,0,0,0,1,0,0,1,0,0
4e32b895-2130-40b8-b613-35d601bb7832,0,0,0,0,0,0,0,0,0,1,0
4ed0af52-7257-4ae0-ae51-249bc13ec1e1,0,0,0,1,0,0,0,0,0,0,0
4f3eb8db-188b-45b2-b4bb-1c1b4eceaeda,0,1,0,0,0,0,0,0,1,0,0
4f56333a-4d6c-4451-882b-f6c6d92e9083,0,1,0,0,1,0,0,0,0,0,0
500c47ad-2352-470a-8299-669ad6ec92e6,1,0,0,0,0,0,0,0,0,0,0
50e3fb8a-d536-4bba-a53f-c2e96258e321,0,0,0,0,0,0,0,0,0,1,0
511f30ef-159d-4e30-9f22-5d94437234bd,0,0,0,0,1,0,0,0,0,0,0
516e4c91-b69d-4f40-a087-525f5edd6c5d,0,1,0,1,0,0,0,0,0,0,0
51888fea-3b94-4bbd-880d-6ed43c755510,0,0,0,0,1,0,0,0,0,0,0
519a92b5-aedd-493a-9380-1bc5403a88f6,1,0,0,0,0,0,0,0,0,0,0
5206de20-b07c-4f13-9d18-1c0169d4996e,0,0,0,0,0,0,0,0,0,0,1
52634bdd-329b-4cd7-91ec-9a878aeee464,0,0,0,0,0,0,0,0,0,1,0
5363972e-6add-461c-bd02-4b4fd2c0a19b,0,0,0,0,0,0,0,0,1,0,0
54234390-bef3-4fc5-a633-cf385f832b49,0,0,0,0,0,0,0,0,0,0,1
542d8852-8242-47dc-b0be-fa2e2564b953,0,0,0,0,0,0,1,1,0,0,0
544dd643-5ab1-4f9a-9a2c-a6efc6a498a1,0,0,0,0,0,0,0,1,0,0,0
54798bac-4e38-49fa-8deb-f0b908278f88,0,0,0,0,0,0,0,0,1,0,0
5483c749-5695-498c-b748-b599a459f6cd,0,0,0,0,0,0,0,0,0,1,0
54f4cff7-629c-426c-a034-924f74e3acb5,0,0,0,0,0,0,0,0,0,0,1
555ee972-1fed-4aac-b58d-895b0d639b86,1,0,0,0,0,0,0,0,0,0,0
563f3603-1902-49ab-92f7-57aff5b9a044,0,1,0,0,1,0,0,0,0,0,0
5656410c-b107-495b-815d-f6727a1e15b0,0,0,1,0,0,0,1,1,0,0,0
569dce40-1d04-4f73-bcf1-c5b3a98478a8,0,0,0,0,0,0,0,0,0,0,1
58211943-6f96-4411-bfef-45fe49428654,0,0,0,0,0,0,0,0,0,1,0
5834c5ec-dde9-4533-b878-8569d0863acc,0,0,0,0,1,0,0,0,0,0,0
58a8c541-bda8-479a-9833-200b91339d46,0,0,0,0,1,0,0,0,0,0,0
58ed7cf9-2ca9-4f8e-8bf5-f5069311f6ad,0,1,0,0,1,0,0,0,0,0,0
59147e01-bb5e-49de-b0e2-4ed9c31b488e,0,0,0,0,0,0,0,0,0,0,1
593675ee-b727-4f90-8676-dc5dee46a025,0,0,0,0,1,0,0,0,0,0,0
5976f17d-996c-49ce-8f10-dcf07c53fb32,0,1,0,0,1,0,0,0,0,0,0
5a5f0616-0989-4ee3-89b5-ac073b0efb9b,0,0,0,0,0,0,0,0,0,0,1
5bd65f68-0973-4aee-8129-e2c1f611e73f,0,0,0,0,0,0,0,0,0,0,1
5c30d6f6-2170-40d1-902f-e99a99838436,0,0,0,0,0,0,0,0,0,0,1
5c533377-728d-4410-ae82-5bef1656351f,0,1,0,1,1,0,0,0,0,0,0
5c734b3b-eef0-45ff-9b57-31ac8f5edaad,0,0,0,0,0,0,0,0,1,0,0
5cb2228b-4635-4b7f-ac8e-064970fb3eff,0,0,0,0,0,0,0,0,0,0,1
5cea88e7-1dd8-4897-bfe6-b1a67c24fbaf,0,0,0,0,1,0,0,0,0,0,0
5d0eaa8f-48fe-4274-8717-533edfc377a8,0,0,0,0,0,0,0,0,0,0,1
5d6e1013-1e3b-44ab-aec9-ba005d08d22f,0,0,0,0,0,0,0,0,0,1,0
5dedb4fc-a692-403d-8b5a-c4d1813aeaac,1,0,0,0,0,0,0,0,0,0,0
5e3d275e-63ea-4fdd-99db-2e5f91badf98,0,0,0,1,0,0,0,0,0,0,0
5f18b90d-302e-4967-a45c-22b7fdb4802f,0,1,0,0,1,0,0,0,0,0,0
5fa50f0b-9a73-4be0-aae3-e78e63a139fe,0,0,0,0,0,0,0,0,0,0,1
5fcea791-fc62-4bd1-8769-9d1d49a6ae5c,0,1,0,1,1,0,0,0,0,0,0
5fd0672b-0180-4c7c-9670-1b6b409c1800,0,0,0,0,0,0,0,0,0,0,1
5fd82693-6567-4308-b72e-75dac49e796e,1,0,0,0,0,0,0,0,0,0,0
600a5351-d885-4684-a804-50a3f3ad56dc,0,0,0,0,1,0,0,0,0,0,0
60e62925-6f37-41f4-b122-bbdcad845e6c,1,0,0,0,0,0,0,0,0,0,0
60f38f4d-216d-40dc-b878-7096f2fa9998,0,0,0,0,1,0,0,0,0,0,0
61298d2c-8f9b-4461-b2dc-852fe5f728c6,0,0,0,1,0,0,0,0,0,1,0
624c912e-4b37-4662-adef-8c621061ae6d,0,0,0,1,0,0,0,0,0,0,0
6306d5a2-8521-4b9f-9625-6bb6ac1e4174,0,0,1,0,0,0,0,1,0,1,0
63562365-97d2-407c-8cad-3e5cfb522744,0,0,0,0,0,0,0,0,0,0,1
63ab2dcc-d4f0-42cc-9a39-0cf018f6ff60,0,0,0,1,0,0,0,0,0,1,0
640c1582-0ca0-49ea-8d0d-b93164956be5,0,0,0,0,0,0,0,0,0,1,0
641f66bc-5a16-49da-b9e9-96d2b2d3f6e0,0,0,0,0,1,1,0,0,0,0,0
648a6c2e-ca07-48b0-93de-818b8d74ca01,0,0,0,0,0,0,0,0,1,0,0
64a5cb65-7330-4fad-b61a-578fbd6fbd41,0,0,0,0,0,0,0,0,0,0,1
64c583be-6f5a-4be7-a974-87ddcadf5443,0,0,0,0,0,0,0,0,0,1,0
65a9b483-1562-4b20-bf9e-26adf1145694,0,0,0,1,0,0,1,0,0,0,0
666081b6-9d80-4639-bbe8-6477b0f580ad,0,0,0,0,0,1,0,0,0,1,0
6669709f-34b0-4115-bb23-6f09c30f5132,0,0,0,0,0,1,0,0,0,1,0
668c0258-c681-4b1d-905b-6f824c70eeea,0,0,0,0,1,0,0,0,0,0,0
671891b9-0809-4587-ac23-6420a95916d3,1,0,0,0,0,0,0,0,1,0,0
6761b6d5-ffa4-4ef0-a16e-699efbfd1e22,0,0,0,0,1,0,0,0,0,0,0
67b295e8-06d5-4ad3-8452-f91fe136283f,0,0,0,1,0,0,0,0,0,0,0
6852ebfc-0635-42a8-8827-ab13ecd8f92e,0,0,0,0,1,0,0,0,0,0,0
6a2d9181-3aeb-4ac4-8e89-1fc40f42a62f,0,0,0,0,0,0,0,0,0,1,0
6a56063d-9ef2-42de-8e90-895fe19c5219,0,0,0,0,1,0,0,0,0,0,0
6a9c6cad-2949-4075-a797-b3ff41afd2e1,0,0,0,0,0,0,0,0,0,0,1
6b060ce6-6078-4c04-a642-f8a93da346a7,0,1,0,0,1,0,0,0,0,0,0
6b4cf548-d269-403e-838d-f35fd54e2b6a,0,0,0,0,1,0,0,0,0,0,0
6bb9550e-73dc-4b63-8435-2b605ecf2086,0,0,0,0,1,0,0,0,0,0,0
6c4df972-b2c8-40ec-82f0-fadebea4ba03,0,0,0,1,0,0,0,0,0,0,0
6d63bb12-01ac-45d7-95cd-fe342d96b5a1,0,0,0,0,1,0,0,0,0,0,0
6d97c4b9-d62c-489b-b416-254d9c4d5a26,0,0,0,0,0,0,0,0,0,0,1
6dfc2d71-7258-49b7-b9e9-d96e77760b42,1,0,0,0,0,0,0,0,0,0,0
6e189561-b70d-419a-a59f-4415244b40e1,0,0,0,0,0,0,0,0,0,0,1
6e2a6b19-bbf9-450b-8326-dc5acf72d8a4,0,0,0,0,0,0,0,0,0,1,0
6e3d0393-f562-4864-aab2-f4c239950d22,1,0,0,0,0,0,0,0,0,1,0
6e7fbe92-2ca0-47b8-8d48-2085f8652fff,0,0,0,0,0,0,0,0,0,0,1
6eadb65b-2fd6-4362-8cf7-27fb178c1ac7,0,0,0,0,0,0,0,0,0,1,0
6ef33593-2af7-4471-9618-f38cb4e6e683,0,0,0,0,0,0,1,0,0,0,0
6f3abeb8-6216-4997-a973-087955633858,1,0,0,0,0,0,0,0,0,0,0
6f8c6205-0e5b-42a0-bf72-973379f106d4,0,0,0,0,0,1,0,0,0,1,0
6fa9ccb0-107a-47ba-9575-47d6b178b900,0,0,0,0,0,0,0,0,0,0,1
6fae49bb-6652-4688-86a0-fadfe894a6dc,0,0,0,0,1,0,0,0,0,0,0
6fed6d18-82c5-4771-b501-789351a03af6,1,0,0,0,0,0,0,0,0,1,0
701b6742-53bc-419f-a964-5e0d74c7818a,0,0,0,0,1,0,0,0,0,0,0
702a87bc-1274-4988-8af2-1631f89694eb,0,0,0,0,0,0,0,0,0,0,1
706850fa-04a9-4e55-aaca-bf850a4aef99,0,0,0,0,0,0,0,0,0,1,0
70bfe788-e617-4c6e-841b-b3468d80a5a5,0,0,0,0,0,0,0,0,0,1,0
712d1ff5-9c57-4e66-84f1-ff0db8680607,0,0,0,0,0,0,0,0,0,1,0
71a69eb8-1906-4ff1-a4c7-cada67d8e999,0,0,0,0,1,0,0,0,0,0,0
71d3c560-5aea-4333-bccb-66cc383b4f28,0,0,0,0,0,0,0,0,0,0,1
71de0aaf-b731-44bc-9901-db4def4f3460,0,0,0,0,0,0,0,0,0,0,1
722cb749-41b5-41c5-91f6-6e20eb845597,0,1,0,0,0,0,0,0,0,0,0
72901925-0837-47db-b94a-f754a3025a00,0,1,0,0,1,0,0,0,0,0,0
7329a9c6-4565-416a-bd5e-a839c36e2adf,0,0,0,0,0,1,0,0,0,1,0
738e6b44-c3a7-4b41-bd5e-3f3ec1865d8e,0,0,0,0,1,0,0,0,0,0,0
73c2ff77-35bb-479b-83ca-ce662c6d0f60,0,0,0,1,0,0,0,0,1,0,0
7439523f-5734-4cc3-a18c-e98b3427a01b,0,0,0,0,0,0,1,0,0,0,0
74681c06-56d2-4763-8a5f-2ae386417367,0,0,0,0,0,0,0,0,0,0,1
75d57418-1afd-41b2-9579-c8ba908fe641,0,0,0,0,0,0,0,0,0,0,1
75db0388-2031-486d-a63f-414b2e58c006,0,0,0,1,0,0,0,0,0,0,0
76252d70-0b34-4b73-abad-36062e3b04c9,0,0,0,0,0,0,0,0,0,1,0
76345c2c-3580-42e4-b206-18a45e7f36c3,0,0,0,1,0,0,0,0,0,0,0
7639b412-4806-4479-9db4-0ca9025d71e3,0,0,1,0,0,0,0,1,0,0,0
7655779b-efdb-44e2-8a32-ede65775a6b7,0,0,0,1,0,0,0,0,0,0,0
778ef636-5b62-46de-b520-f7193e112967,0,1,0,1,0,0,0,0,0,0,0
77a7d094-41e1-40a4-8a2a-11b05a492821,0,0,0,0,0,1,0,0,0,1,0
782114ae-cb53-4b3b-8625-6618f7d36a80,1,0,0,0,0,0,0,0,1,0,0
7821804a-973f-47cd-a99f-86dc5724563e,0,0,0,0,0,0,0,0,0,1,1
78228afd-d15f-4a2e-ad5f-aa9a87b911ad,0,0,0,0,1,0,0,0,0,0,0
783c3e94-0777-4134-a753-079a6e3bb79c,0,0,0,0,0,0,0,0,0,1,0
78eedb84-0501-4004-8316-101eae778e1f,0,1,0,0,1,0,0,0,0,0,0
790b9549-307e-4a19-b379-6d6915057f1b,0,0,0,0,0,0,0,0,0,0,1
79d67b55-dfdd-42ad-adae-79cd0b31df0a,1,0,0,0,0,0,0,0,0,0,0
7a4b732a-01c7-426a-b5a2-d8ba66831108,1,0,0,0,0,0,0,0,0,1,0
7ba7d6f2-1cb3-4c1d-b9d2-23bc49ca9a08,0,0,0,0,1,0,0,0,0,0,0
7ba82145-d0d8-4790-9d91-f8619694d896,0,0,0,0,0,0,0,0,0,0,1
7bf5b393-c794-476b-a388-ecbcf37b9960,0,0,0,0,0,0,0,0,1,0,0
7c3379f3-4c7b-4347-83f2-c1741ba4a39e,0,0,0,0,0,0,0,0,0,1,1
7ccf5fac-a218-4a4e-a058-8a3efc297921,0,0,0,0,0,0,0,0,0,0,1
7ced4454-9956-4a95-bf4b-1ce210e02137,0,0,0,0,0,0,0,0,0,0,1
7d571af4-27c1-478f-9d93-54cae6d127e0,1,0,0,0,0,0,0,0,1,0,0
7e75e4c1-c31f-420b-9dc0-27909bf28b8f,0,0,0,0,1,0,0,0,0,0,0
7ea3e906-5db2-4c8e-8c24-781d1d53f629,0,0,0,0,0,0,0,0,0,1,0
7efd39b6-efd3-4160-bb7e-9da28d1c541c,1,0,0,0,0,0,0,0,1,0,0
7f04320f-cbda-499f-b623-5a591645aa2f,1,0,0,0,0,0,0,0,0,0,0
7f84a225-2cc1-4700-9a3e-9215bb9d04d9,0,0,0,0,0,0,0,0,0,0,1
7fa2beee-2093-4414-b82d-6534ea9b45fe,0,0,0,0,0,0,0,0,0,0,1
8012c7c8-efa2-4b86-85dc-8ca48491e259,0,0,0,0,0,0,0,0,0,1,0
80742eeb-a455-4a83-a867-8e603b2acbe3,1,0,0,0,0,0,0,0,0,0,0
8127b703-1d36-4105-ab74-1f5487017839,0,0,0,0,1,0,0,0,0,0,0
81c1db17-9c29-4e26-927e-aef57498fa85,0,0,0,0,0,0,0,0,1,0,0
81edefff-2b2a-49b0-9299-e3f02024eee6,0,0,0,0,0,0,0,0,0,0,1
8213e9e9-3c57-4da0-8f88-7623708dbabe,0,0,0,1,0,0,0,0,0,0,0
83807adf-4684-433e-850b-489de95cdf09,0,0,0,0,1,0,0,0,0,0,0
83b4ddc7-ce6c-425c-9870-c191f3a0285e,0,0,0,0,0,0,0,0,0,1,0
846aefa9-41ad-4709-9ea3-789e1fccd513,0,0,0,0,1,0,0,0,0,0,0
848b878c-caa3-4816-96d2-22f514992f80,0,0,0,0,0,0,0,0,0,0,1
84afbd2c-f89c-47b3-b7c2-3cd18c64cd2d,0,0,0,0,0,0,0,0,0,1,0
84c3626e-a241-499a-94cf-0e2e6d69958d,0,0,0,0,0,0,0,0,0,0,1
8613e16a-537b-4ea1-8a2e-cc48f79f3c13,0,0,0,0,0,0,0,0,0,1,0
86bcabd5-be14-463b-8982-0415b0887d1e,0,0,0,0,1,0,0,0,0,0,0
8733e4c7-9af2-4551-826a-94e2c131596c,0,0,0,0,0,0,0,0,0,1,1
877a3288-ea9f-4490-ad32-e36b0e4bfc0a,0,1,0,1,0,0,0,0,0,0,0
877f71ba-93c1-4e68-ac14-67797ebbf448,1,0,0,0,0,0,0,0,0,1,0
87f87bf1-7502-457b-8b2c-dac37ac987d4,0,0,0,0,1,0,0,0,0,0,0
883685b9-80e2-46e8-aa0e-b940c7216651,0,0,0,0,0,0,0,0,0,1,0
88f22a77-7e42-4000-af9b-929e2936754e,0,0,0,0,0,0,0,0,0,1,0
89212b4a-d9c7-47ec-a8da-51b3ea5a8e73,0,0,0,0,1,0,0,0,0,0,0
8966b86f-5944-4135-84d4-6b4b9d24f4e9,0,0,0,1,0,0,0,0,0,0,1
89c67f6a-0cca-4295-a37d-03437db1f236,0,0,0,0,1,0,0,0,0,0,0
8a04c0b3-dbdc-481f-8e18-28311d65903b,0,0,0,1,0,0,0,0,0,0,0
8a0fc6bf-de49-4901-bc91-dd7f1d50a270,1,0,0,0,0,0,0,0,0,0,0
8a20a443-508e-47d6-a740-3a51711fcf3c,0,0,0,0,0,0,0,0,0,1,1
8a432b88-57ef-4fae-842b-01386863e9b5,0,0,0,0,1,0,0,0,0,1,0
8b47a5c5-4114-47df-9440-6c856797bb07,0,0,0,0,1,0,0,0,0,0,0
8b7860f5-4091-4044-bc08-c84908c3e6a4,0,0,0,0,0,0,0,0,0,0,1
8b8208a7-ef49-4a47-af9a-ad8bb3553e25,0,0,0,0,0,0,0,0,0,0,1
8b9d8083-641d-4b6d-b6a4-d855ff2b75eb,0,0,0,0,0,0,0,0,0,0,1
8be8cc21-b58a-41aa-bb25-9d54a8f0db35,0,0,0,0,1,0,0,0,0,0,0
8c26d2c2-04b7-4b6d-aad4-a610a2f4a44e,0,0,0,0,1,0,0,0,0,0,0
8cf48d1a-7079-437e-ba35-2275abc76874,0,0,0,0,1,0,0,0,0,0,0
8dde8430-8ebb-44fb-88e4-1d049b49e15c,0,0,0,1,0,0,0,0,0,0,0
8e1854d3-d742-41a5-bba3-8f6ca5d03ad7,1,0,0,0,0,0,0,0,0,0,0
8e19f97c-3f48-4d75-bccd-d478fcc068c0,0,0,0,0,0,0,1,0,0,0,0
8e51325f-730e-4279-845a-b3e7e18ab3d6,0,0,0,0,1,0,0,0,0,0,0
8e906234-94ab-4182-8734-84cdd571a4bf,0,0,0,0,1,0,0,0,0,0,0
8f85b222-ddca-4320-aa96-749c0fdfedfa,0,0,0,0,1,0,0,0,0,0,0
8fd4bc00-9c2b-452b-b5f7-72000fee4a8c,0,0,0,0,0,1,0,0,0,1,0
8fe85106-7499-4e7e-bedd-2ea5391f8232,0,0,0,0,0,1,0,0,0,1,0
901f9cc2-78cc-468b-b6a6-4beddb3594f1,0,0,0,0,0,0,0,0,0,1,0
905afb1a-0ec2-4e0a-a3e6-9f9383b1ba96,0,0,0,0,0,0,0,0,0,1,0
90a2edee-3343-43f7-ab5f-1cb5bf720d43,1,0,0,0,0,0,0,0,0,0,0
90bb8256-1cf9-46a4-b38e-dfbd43170038,0,0,0,0,0,1,0,0,0,1,0
91138cc8-a559-436b-b670-a79c52f85a15,1,0,0,0,0,0,0,0,0,0,0
914fa8cf-93b5-4d7c-9786-4509ac330a40,1,0,0,0,0,0,0,0,0,0,0
91809d46-9581-424b-ae03-fb6a2fb29166,0,0,0,1,0,0,0,0,0,0,0
9181aad9-759b-4de9-b7dd-9d31a3bd8d11,0,0,0,1,0,0,0,0,0,0,0
91a5f53c-af1a-4523-87c8-0636f3db04c1,0,1,0,1,0,0,0,0,0,0,0
91e2906e-9752-423b-bd4b-02293b8e44a1,0,0,0,0,1,0,0,0,0,0,0
922f0a2f-2da4-4e62-8143-0b2d9578fed9,0,0,0,0,0,0,0,0,0,1,0
9248a223-b5c8-4868-8a67-f9ab63953a03,0,0,0,0,0,0,0,0,0,0,1
93614f3a-33e5-4743-a46f-807bdbd6b729,0,0,0,0,1,0,0,0,0,0,0
93866643-687a-4eb7-a216-67ff99a53f18,0,0,0,1,0,0,0,0,0,0,0
93891cc6-f742-4809-9129-7e0f2441d0c7,0,0,0,0,0,0,0,0,1,0,0
940b1434-1cc1-4355-a512-e38bb705b6cd,1,0,0,0,0,0,0,0,0,0,0
94364732-0c65-4214-99c0-ff7cb9f0c43f,1,0,0,0,0,0,0,0,1,0,0
944deb5f-d2ca-472f-a6e4-3fcedb2ccfb9,0,0,0,0,0,0,0,0,0,0,1
9466ce71-2a52-4c19-8af2-ea06a9d8403c,0,0,0,0,0,0,0,0,0,0,1
946c3b34-f4a9-471b-944b-56c1763f52a2,0,0,0,0,0,0,0,0,0,1,0
94825551-0a0f-48f2-abce-ef84ecde79f9,0,1,0,0,1,0,0,0,1,0,0
9497e720-3d6a-4030-bec5-6330b43b4d48,0,0,0,0,0,0,0,0,0,0,1
9598b9e7-f10b-4c6a-92de-6fde0e948126,0,0,0,0,0,0,0,0,0,1,0
959c8de1-f444-4197-a3f2-e8b9588bfbff,1,0,0,0,0,0,0,0,0,0,0
95ac56cb-365b-4394-ae8e-4f54d094d2e2,0,0,0,0,0,0,0,0,0,1,0
966824fa-68ab-4065-b87f-bb6ce2038b2f,0,0,0,0,0,0,0,0,0,1,0
96b4d7d1-64ca-47b2-b1e6-c6492ec27f79,0,0,0,0,0,0,0,0,0,1,0
970e1dde-d686-44df-817a-d78658480d6b,0,0,0,0,1,0,0,0,0,0,0
9764d7c1-1ac6-4d5e-89a0-75fab7e6c45d,1,0,0,0,0,0,0,0,0,1,0
9795d61b-23ff-4179-9d7f-3b37fc0dc29c,1,0,0,1,0,0,0,0,0,0,0
980f1b8b-4c1b-46af-a3f7-895e141087f2,0,0,0,0,0,0,0,0,0,0,1
983d3211-3a82-4912-b758-46283f879b57,0,0,0,0,0,0,0,0,0,1,0
98820763-37bb-49fb-983f-6dddef6eda46,0,0,0,0,0,0,0,0,0,1,0
9890b6ed-fffc-4fc7-bed7-326ac7b4d8df,0,0,0,0,0,0,0,0,0,1,0
98f2095d-70b2-4da4-a891-f64302861868,1,0,0,0,0,0,0,0,0,0,0
98f87cf0-cec4-4443-949d-b923fa056ea2,0,0,0,0,0,0,0,0,0,1,0
99064d6a-5551-4abc-b552-584491668414,0,0,0,0,1,0,0,0,0,0,1
99df60aa-07c6-49a8-9d0f-67181fb225a5,0,0,0,0,0,0,0,0,0,0,1
9b1826d3-7dd8-43d2-9a0c-c37ec30b13ae,0,0,0,0,1,0,0,0,0,0,0
9b206d63-69a8-4947-9e9d-5969fe56ad21,0,0,0,0,0,0,0,0,0,1,0
9baa866c-d5b7-44bd-94a6-acc44e9f053d,0,0,0,0,1,0,0,0,0,0,0
9bac0925-105b-4e26-bb8f-23e2374dbf2f,0,0,0,0,0,0,0,0,0,1,0
9c054771-73fe-4f2c-ae52-fbcb6f8ad5e2,0,0,0,0,0,0,0,0,1,0,0
9cab1e08-ddf2-4808-b556-e2c348fa3744,0,0,1,0,0,0,1,0,0,0,0
9d94c569-8c63-45db-b1fb-3da101d80f2d,0,0,0,0,0,0,0,0,0,0,1
9dae31fa-decd-445f-b553-8347e3f8e44b,0,0,0,0,1,0,0,0,0,0,0
9dc00bc1-78a4-4b4a-9665-6f9e0b589b91,0,0,0,0,0,0,0,0,0,1,0
9dd4f37f-878e-484e-9484-ffcc928e546b,1,0,0,0,0,0,0,0,0,1,0
9e38e995-afb4-4090-bbd7-f6cff2ddec5f,0,0,0,0,0,0,0,0,0,1,0
9e7f6a05-a362-4aee-a678-d2f53c2db661,0,0,0,0,0,0,0,0,0,1,0
9eac7262-3be4-4ed2-9a77-8ad2a334b47d,0,0,0,0,0,0,0,0,1,0,0
9fab91b0-5110-4166-9893-c8c0e3946795,0,0,0,0,0,0,0,0,0,0,1
9fd111c8-bc03-44df-9726-dea428f8d752,0,0,0,0,0,0,0,0,0,1,0
a1236d5d-e5f5-4143-a3c5-66849c5efe9e,0,0,0,1,0,0,0,0,0,0,0
a1bd6cfe-d3aa-4748-8fae-5263647e1a3d,0,0,0,0,0,0,0,0,0,1,0
a216ba67-64c3-4cfc-af03-7bfed7145914,0,0,0,0,0,0,0,0,0,1,0
a2269ee2-e90d-4d14-ac18-9f58f582bab1,1,0,0,0,0,0,0,0,1,0,0
a30d0d97-906c-426d-9cbc-3c205ba2ae70,0,0,0,0,1,0,0,0,0,0,0
a317b6fe-e7df-4fc0-a457-ca805116dab2,0,0,0,1,0,0,0,0,0,0,0
a3328b83-4311-4ecb-bf35-bb4f831cda6e,0,0,1,0,0,0,0,1,0,0,0
a381f497-7b59-4c3a-91b1-a65a10fb6166,0,0,0,0,0,0,0,0,0,0,1
a3c4c414-f35b-4aae-ad87-57dbdc0e27c0,0,0,0,0,0,0,0,0,0,1,0
a4ad3b5e-b108-4ff2-aedf-ac2547796833,0,0,0,0,1,0,0,0,0,0,0
a4bcb651-1266-46f7-95e1-b39746f86866,0,0,0,0,0,0,0,0,1,0,0
a55597fb-dff7-489f-a337-7fba88447257,0,0,0,0,1,0,0,0,0,0,0
a606e3c7-44c3-428f-b488-7ebaaf4cf9ca,0,0,0,1,0,0,0,0,0,0,0
a6281fdd-556c-4236-ad74-b699b5dffe37,1,0,0,0,0,0,0,0,0,1,0
a635ddfb-fe33-4837-a948-a379aa508731,1,0,0,0,0,0,0,0,0,0,0
a650ade0-0165-47a5-9866-3216038795cb,0,0,0,0,1,0,0,0,0,0,0
a6ec6118-d0d8-40e7-8bbf-e9a745db1c4e,0,0,0,0,1,0,0,0,0,0,0
a7cdf116-87ac-44f2-a7d9-b407a3b044e5,0,0,0,1,0,0,0,0,0,0,1
a8ab7416-b7f2-49ee-a5d5-9af944d1dfd4,0,0,0,0,0,0,0,0,1,0,0
a9167e42-a392-4f2b-b08e-8471fcfabd7a,0,0,1,0,0,0,0,0,0,0,0
a9425817-09a0-4f52-855a-b02f44093667,0,0,0,0,1,0,0,0,0,0,0
a9e751be-68d7-41b5-b498-a97b0b3c757a,0,1,0,0,1,0,0,0,0,0,0
aa7e18f3-07f9-430e-b489-407e520bdf45,1,0,0,0,0,0,0,0,0,1,0
aa879823-3150-490f-814f-9c3d98ce2093,0,0,0,0,0,0,0,0,0,1,0
ab66e4c2-6e9b-456f-8e5a-d5799dc3b850,0,0,1,0,0,0,0,0,0,1,0
ab898f78-d2c2-4e58-b941-3a265b4ed758,0,0,0,0,0,0,0,0,0,0,1
aba1355b-ba32-4f06-a52e-4d05b823075e,0,0,0,0,0,0,0,0,0,1,0
abe82a25-078b-43ab-9203-8a77f242b33c,0,0,0,1,0,0,0,0,0,0,0
ac6dfcc8-f7ad-4de4-9498-07861ca1b871,0,0,0,0,0,0,0,0,0,0,1
ad894c75-7267-4db0-8387-2328d9220f2e,0,0,0,0,1,0,0,0,0,0,0
adfb88d0-0d14-400d-9b48-4a6bd610f53e,0,0,0,0,1,0,0,0,0,0,0
aec2b08b-4932-430f-b2e5-be97d8579266,0,0,0,0,1,0,0,0,0,0,0
aeff7bbe-4c56-4119-b418-51ee0d616ba6,0,0,0,0,0,0,0,0,0,0,1
af0d2a59-777c-4f1d-9acf-e0f863acdfe3,0,0,0,0,0,0,1,0,0,0,0
af6574b9-5a31-4238-8784-8e44b3d4bd86,0,0,0,1,0,0,0,0,0,0,0
afa9897f-de8a-4eaf-b7a7-f54856628f74,0,0,0,0,1,0,0,0,0,1,0
b0908fb8-eded-4100-aca8-1e2b7dbb10e2,0,0,0,0,1,0,0,0,0,0,1
b111914f-65e9-4c13-8ed6-690bf266db18,0,0,0,0,0,0,0,0,0,1,0
b185d0fb-5689-44df-b86b-4d91bc113e89,0,0,0,0,0,0,0,0,0,0,1
b22ed2ed-338b-44ca-8f2a-65001daff04d,0,0,0,0,0,0,0,0,0,1,0
b29378a4-3bac-49c0-923b-aeef7f35077a,0,0,0,0,0,0,0,0,0,0,1
b2fd67f5-4d61-44d0-bba1-82782c65e7ea,0,0,0,0,0,0,0,0,1,0,0
b300622c-bcd9-45c8-a6bd-005694af7fc5,0,0,0,1,0,0,0,0,1,0,0
b33ad53e-49f1-4ca3-827f-e529ffe0214e,0,0,0,0,1,0,0,0,0,0,0
b403f77f-b240-4c05-ba4c-f6f429171e41,0,0,0,1,0,0,0,0,0,0,0
b4eaa0de-6f1b-40b6-908e-72bf3d80387c,0,0,0,0,1,0,0,0,0,0,0
b5c29dbe-dbd9-422e-bfa4-1e0dbb9a6d85,0,0,0,0,1,1,0,0,0,0,0
b65266cf-f2c8-4559-8649-6d92b0fe7667,0,1,0,0,1,0,0,0,0,0,0
b6793d5c-62d5-4a77-a2ad-69b46e1e4896,0,0,0,0,0,0,0,0,1,0,0
b6b9ec08-3749-4c92-a199-96c2e97e7906,1,0,0,0,0,0,0,0,0,0,0
b6dfea59-e072-416f-8322-006c3dbbd1d6,0,0,0,0,0,1,0,0,0,1,0
b7c3c308-e42a-454f-af28-d48063abe05d,0,1,0,0,1,0,0,0,0,0,0
b7dc5d5b-d7ca-462f-a471-a6a5ecab4243,0,0,0,0,0,0,0,0,0,1,0
b83dc701-e64c-42fd-9743-32119491bd9e,1,0,0,0,0,0,0,0,0,0,0
b8994252-bfad-4dd9-8167-1b6d4005611e,0,0,0,0,0,0,0,0,0,0,1
b8b5ef71-c107-4b20-9d3b-7b561787c7e5,0,1,0,0,1,0,0,0,0,0,0
b8df7cfe-e501-4c1f-bbef-9e17a5cb863b,0,0,0,0,0,0,1,0,0,0,0
b95f2b15-f8d3-4550-9b77-e474e914ebe2,0,0,0,0,0,0,0,0,0,0,1
b9bdfa29-35e2-4224-92b3-bccb0d8658bd,0,0,1,0,0,0,0,0,0,0,0
ba40f328-371c-4f08-9729-0e11336d15db,0,0,0,1,0,0,1,0,0,0,0
bacdb56a-7a1e-47cf-84cd-397513db75ca,0,0,0,0,0,0,0,0,0,0,1
bb1d9e2e-4589-43f1-b2e3-ec7943e332d1,0,0,0,0,0,0,0,0,0,0,1
bb3a0d3e-afbd-466b-8edf-fa7979170ea2,0,0,0,0,0,0,0,0,0,0,1
bb9c1e8c-725a-4cda-87af-b318e4bc3227,0,0,0,0,1,0,0,0,0,0,1
bcf6a714-ea59-4662-b746-e638a1859855,0,0,0,0,1,0,0,0,0,0,0
bcfe32fb-93ff-4493-bd5e-0a782b6900b8,0,0,0,0,1,0,0,0,0,0,0
bd0d341e-4f6e-4014-a594-48ec1754aed0,1,0,0,0,0,0,0,0,0,0,0
bd907421-fc83-4830-a19c-03b5df18e619,0,0,0,0,0,0,0,0,1,0,0
bd9f12bf-8340-4a49-a084-2ec673225f58,0,0,0,0,1,0,0,0,0,0,0
bddd689e-6e91-45fc-b778-ec22e35455dd,0,0,0,0,0,0,0,0,0,0,1
be1d9729-5648-4117-8935-6601441887de,1,0,0,0,0,0,0,0,0,0,0
be440cc0-a31c-4a6c-ba82-8fc07c4f6566,0,0,0,0,1,0,0,0,0,0,0
bece999a-f9b7-4da4-ba87-2a5c2d7dbeb1,0,0,0,0,1,0,0,0,0,0,0
bf09e5ad-fd99-4229-a188-81065efdce98,0,0,0,1,0,0,0,0,0,0,0
bf4bdd86-6351-4bce-aebd-2151614ff2d6,0,0,0,0,1,0,0,0,0,0,0
bf700779-68a3-4205-8c04-37a7ecce004a,0,0,0,0,0,0,0,0,0,1,0
bf7901a5-ade6-4ec8-a64e-01e727ee0115,0,0,0,0,0,1,0,0,0,1,0
bf8956ab-5a39-4e07-a8be-ba7bceaa3317,0,0,0,0,0,1,0,0,0,1,0
c0229641-6bf8-4ed6-befb-51b70d9e1564,0,0,1,0,0,0,1,1,0,0,0
c02b0c37-74f3-48bb-9552-7644ee2a5fc5,1,0,0,0,0,0,0,0,1,0,0
c02fafec-1ee9-485b-acd0-a21f1c4644ce,0,0,0,1,0,0,0,0,0,0,0
c0429e1b-5f7f-4eb5-a5c4-f4fe4ccbc53c,0,0,0,0,1,0,0,0,0,0,0
c1044976-4439-424c-b61d-96b450d50ca3,0,0,0,0,0,0,0,0,0,0,1
c1438c63-697e-4633-80d0-dfb8cacfb224,0,1,0,0,1,0,0,0,0,0,0
c20fe2c6-2de0-4301-a4c5-141e03763350,0,0,0,0,1,0,0,0,0,0,0
c217912c-be67-4f99-a799-9b8ec8972d9d,0,0,0,0,0,0,0,0,0,0,1
c258928b-b9b6-4c79-904d-fb381af680b4,0,0,0,0,1,0,0,0,0,0,0
c29a33fa-fe5c-427e-b32a-e3cdf9609468,0,0,0,1,0,0,0,0,0,0,0
c301ae81-5ad8-4e75-bb7a-1fec61cf75ef,0,0,0,0,1,0,0,0,0,0,0
c3389b23-4ac5-4dc4-b380-c122b4888ef3,0,0,0,0,0,0,0,0,0,1,0
c41255c9-a83a-4d5c-a942-fd6dfd0cf4b2,0,0,0,0,1,0,0,0,0,0,0
c46aa42e-29c1-428f-ad26-0c399a2863b7,1,0,0,0,0,0,0,0,0,1,0
c4a4c5fe-89c3-4478-ab15-a0f64af9d516,0,0,0,0,0,0,0,0,0,1,0
c4f5e899-041a-4bb3-a101-887f7b0be402,1,0,0,0,0,0,0,0,0,0,0
c50700b4-71d1-4243-8d72-7bc68606c1b5,0,0,0,1,0,0,0,0,0,0,0
c52c0731-cafd-438e-9407-e5f8b6efb180,0,0,0,1,0,0,0,0,0,0,0
c532ba74-211d-4e85-88ab-c3d91c8e6036,0,0,0,0,0,0,0,0,0,1,0
c56a333f-d957-43c4-92fa-a117ff3dffd1,0,0,0,0,0,0,0,0,1,0,0
c6ace7d9-3158-4c05-8f00-d950534476da,0,0,0,0,0,0,0,0,0,1,0
c6f42f9b-2d8b-4863-bf67-e63939ee1ee8,0,0,0,0,0,0,0,0,1,1,0
c7137b3c-fe5c-487e-b1bc-db5a0496d0b1,0,0,0,0,0,0,0,0,0,0,1
c71ea9b0-092a-4377-8a92-9a67dbb87655,0,0,0,0,0,0,0,0,0,0,1
c78c10a8-7f6e-4b93-8a9c-11be807b8f64,0,0,1,0,0,0,0,0,0,0,0
c8b86aab-be56-437a-b5e2-77647fad3459,0,0,0,0,1,0,0,0,0,0,1
c8fb9945-ca45-41cf-84b3-10ecbf1611e6,0,1,1,1,0,0,0,0,0,0,0
c90049d9-3405-4b3f-822f-bdf45de9dd1c,0,1,0,0,1,0,0,0,0,0,0
c93181a7-86e6-43df-9310-b728ba67b7ac,0,0,0,0,1,0,0,0,0,0,0
c9352f71-e9b3-41dc-b7fd-bb0f6a8989d5,0,1,0,0,1,0,0,0,0,0,0
c9836d24-7bf7-4ff8-a747-935a8f5519ce,1,0,0,0,0,0,0,0,0,0,0
c98fb238-a576-4096-b7af-fa95ae4a73fa,0,0,0,0,1,0,0,0,0,0,0
ca8c49cc-6332-4097-a862-61d7c61da1a0,1,0,0,0,0,0,0,0,0,1,0
cb2d9346-aa11-4413-a33a-644aa805f7bb,0,0,0,0,0,0,0,0,0,1,0
cb629dab-4a88-41a1-98d9-e0a498fd1be3,0,0,0,0,0,1,0,0,0,0,0
cbc57bd9-fe22-4995-9c26-e0973ba7989a,0,0,0,1,0,0,0,0,0,0,0
cc0dfea7-387b-4ea7-8a13-6b3a32ab9f94,1,0,0,0,0,0,0,0,0,0,0
cc28c944-a65a-42db-b265-1093087a1774,0,0,0,0,0,1,0,0,0,1,0
cc2f4b31-c4c5-4d85-a398-0dd8260225ea,1,0,0,0,0,0,0,0,0,0,0
cc5579d2-58cc-4f5e-a23b-334e4a7cf012,0,0,0,0,0,0,0,0,0,1,0
ccefb2b4-0885-4407-a9cb-8754614e6659,0,0,0,0,1,1,0,0,0,0,0
ccf76460-3baf-4587-a1d8-c14ef13812d4,0,0,0,0,0,0,0,0,0,1,0
ccfcb275-92df-47ca-8c4a-17084b3d3d8d,0,0,0,0,1,0,0,0,0,1,0
cd182d02-8321-4d7d-912b-8819cc4cc50f,0,0,0,0,1,0,0,0,0,0,0
ce915f41-82e2-4b8d-9d55-0b2e6cb26aa8,0,0,0,0,0,0,0,0,0,0,1
cfa7a998-9153-46e7-9562-5bbd2c2a21eb,0,0,0,0,0,0,0,0,0,0,1
cfb9cbb5-c969-45d2-ab83-8b9474b1c183,1,0,0,0,0,0,0,0,0,0,0
cfbe4e8d-13e9-4eea-9003-f7f617877f00,1,0,0,0,0,0,0,0,0,1,0
d01a59c9-bfc3-4711-a13e-bd5e7a5e07cb,0,0,0,0,0,0,0,1,1,0,0
d06fb24a-876a-4f89-ba2f-deb7aabcb2df,0,0,0,0,0,0,0,0,0,0,1
d0fbf20f-0594-4514-91ee-d4ef788f3d5c,0,0,0,0,1,1,0,0,0,1,0
d18b26da-77b2-46c4-9c4b-ba8b08496637,0,1,0,1,0,0,0,0,0,0,0
d1ef867d-0478-45e7-a3f6-64dead8fd043,0,0,1,0,0,0,0,0,0,0,0
d21f4a81-186a-4cc2-be9c-afd23c2e277e,1,0,0,0,0,0,0,0,0,0,0
d245b8c8-43fb-4946-82b8-a5564f5078c4,0,0,0,0,0,0,0,0,0,1,0
d3330b95-0a3c-4d36-96c9-79b216c388ff,0,0,0,0,0,0,0,0,1,0,0
d377e518-e0e8-4a88-bf09-84282495f9f6,0,0,0,0,0,0,0,0,0,1,0
d3b71476-c6ef-4706-810b-203c9f60a346,0,0,0,0,0,0,0,0,1,0,0
d3f1bfa8-15e7-4f5c-98f6-c40f0294b34e,0,0,0,0,0,0,0,0,0,0,1
d403d40b-16e3-49b0-b263-45286174b91c,1,0,0,0,0,0,0,0,0,0,0
d4164d6e-52dc-4609-9b0b-c0805cecc0ee,0,0,0,0,1,0,0,0,0,0,0
d43573e8-8fd4-42c5-95f1-0cca830e8f47,0,0,0,0,0,0,0,0,1,0,1
d4a62b93-1e7a-4d35-a97c-ff818f674ad7,0,0,1,0,0,1,0,0,0,0,0
d4b926fe-bfe3-4043-b9aa-80d9c1e1f393,0,0,0,1,0,0,0,0,0,0,0
d4ccbb21-3518-45d6-97e8-fa2a87b7a7c3,0,1,0,0,1,0,0,0,0,0,0
d4e3e141-f330-4706-90a9-a8762a184a20,0,0,0,0,0,0,0,0,0,1,0
d50fdbbb-154e-46c8-afed-7112ed38fa26,0,0,0,0,0,0,0,0,0,0,1
d5954059-e89d-48fa-a22f-b048b141ad8b,0,0,0,0,0,1,0,0,0,1,0
d5bb6263-7d7a-4c34-b91e-7b9f332a9cfe,0,0,0,0,0,0,0,0,0,1,0
d5e91487-4d39-44e2-af84-1ae5dfe4f9bd,0,0,0,0,0,0,0,0,0,0,1
d5f60aa4-5c1f-4dea-8cec-b696f92b2783,0,0,0,0,0,0,0,0,0,0,1
d61bd232-c5d8-4f35-ab0f-ed64f891a0e1,1,0,0,0,0,0,0,0,0,1,0
d62f67ea-ad77-450e-a2f8-dedc20e68454,0,0,0,0,1,0,0,0,0,0,0
d6b05aef-05c2-4233-9db5-008e1e6d482f,0,0,0,0,0,0,0,0,0,0,1
d6f08971-c18f-48a1-b503-316860b9bc66,0,0,0,0,0,0,0,0,1,0,0
d7452242-3525-4615-a216-d7a928ccef90,0,0,0,0,1,1,0,0,0,1,0
d7e4d404-82ef-4442-8480-f93945f2bfc9,0,0,0,0,1,0,0,0,0,0,0
d80f168c-90de-481d-86c7-d5dff5e6e642,0,0,0,0,0,0,0,0,0,0,1
d82124b3-93e1-46f1-bc44-9aea957e458f,0,0,0,0,0,0,0,0,0,1,0
d87f8cf2-ace6-417b-83f4-2f81de49d0bb,0,0,0,0,1,0,0,0,0,0,0
d8bf50b7-5987-438c-b42d-f07f739afbaf,0,0,0,0,1,0,0,0,0,0,0
d8e06e05-5c02-4cad-b6b2-eecc6426a6d9,0,0,0,0,1,0,0,0,0,0,0
d8e2b221-7341-4455-843a-b588a9f2e9b6,0,0,0,0,1,0,0,0,0,0,0
da2b52df-4bb3-4542-9085-0ff881a6be0c,0,0,0,1,0,0,0,0,0,0,1
da3ee997-5928-4d83-b36d-9fc56b9578f6,0,0,0,0,0,0,0,0,0,0,1
dc33c83d-90ad-4c0d-b022-5e0839ec3b14,0,0,0,0,0,0,0,0,0,1,0
dc43236c-4069-4d8e-9656-c5021c481250,0,0,0,0,1,0,0,0,0,0,0
dc8cbed3-54a2-4d18-a43c-8d31a8c8faf3,0,0,0,0,0,1,0,0,0,1,0
dcef225e-6b6c-4a3d-8935-eb853db196cb,0,0,0,0,1,0,0,0,0,0,0
dd7ab633-cd66-46d8-9d0f-82c9e8331ad4,0,0,0,0,1,0,0,0,0,0,0
de6fcd98-ce74-4f90-8b10-385372330153,0,0,0,0,0,1,0,0,0,1,0
de98e584-b933-4d1d-9d3a-d96ab2c9d4c8,0,0,0,0,0,0,0,0,0,1,0
deebed44-2bd3-419b-9119-2cf281a32e25,0,0,0,0,1,0,0,0,0,0,0
df3b1621-3f7a-4b2b-96f4-6485307c748a,0,0,0,0,0,0,0,0,0,1,0
df7496fb-3aff-4bb8-a27f-514ccfdd8124,0,0,1,0,0,0,1,1,1,0,0
df8dcbc6-2509-4103-a92b-b8cac57a59b0,0,0,0,0,1,0,0,0,0,0,0
df9140c2-543a-4afc-88b0-f8a176912129,0,0,0,0,0,0,0,0,0,1,0
dfa9f8eb-799c-4db7-9094-9dfa60b30397,0,0,0,0,0,0,0,0,0,1,0
dfb1b5be-58e4-43c7-b3b7-11c94ee2df61,0,0,0,0,1,0,0,0,0,0,0
e0bec0dd-b967-40ff-8aa5-2a14e08de8a4,0,0,0,0,0,0,0,0,0,1,0
e1072764-0525-45b9-92c4-b60b0e4efb86,1,0,0,0,0,0,0,0,0,1,0
e1ab9940-31f3-4ebe-9cc7-ca0656549227,0,0,0,1,0,0,0,0,0,0,0
e1d8b5e8-cffe-463c-9d81-03b5620969a6,1,0,0,0,0,0,0,0,1,0,0
e215a834-caed-4116-a221-1b7e1cfece5e,0,0,0,0,1,0,0,0,0,0,0
e2220a0b-e2d9-4112-84db-4c04cefaa7f5,0,0,0,0,0,1,0,0,0,1,0
e358d6c9-c623-427f-a5be-f08c540a739c,1,0,0,0,0,0,0,0,0,0,0
e3992a45-a717-40a4-a073-c9ac45ff47e8,0,0,1,0,0,0,0,0,0,0,0
e3aed43b-122d-4f57-86fe-80562eed6728,0,0,0,0,0,0,0,0,0,0,1
e43639d0-eb46-45ce-9b74-f721587c90fd,0,0,0,1,0,0,0,0,0,0,0
e44ce7e4-5c64-4f40-a8a0-4ed3ebf9e1fc,0,0,1,0,0,0,1,0,0,0,1
e5502a91-084b-4d5c-9f87-c1feeddfa683,0,0,0,0,1,0,0,0,0,0,0
e5b7b974-598a-47d8-a38a-847ee2033a3b,0,0,0,1,0,0,0,0,1,0,0
e611ef06-ff55-4476-b66d-d6518f5c45bf,0,0,0,0,0,0,1,0,0,0,0
e6657b04-3575-42d3-bd39-71c78cc90265,0,0,0,0,1,0,0,0,0,0,0
e68376ff-af4e-41dc-8f60-8cc16287eb31,0,0,0,0,0,0,1,0,0,0,0
e7039e45-1ab4-492f-b23c-6e0660a4531c,0,0,0,1,1,0,0,0,0,0,0
e80d3fd6-da86-41d8-b7fc-fa7c688daa7a,0,0,0,0,0,0,0,0,0,1,0
e813a0c3-f819-4b2e-ac70-3dd3b93d6811,1,0,0,0,0,0,0,0,0,0,0
e8171333-e465-4069-b5df-40c063a97e76,0,0,0,0,0,0,0,0,0,1,0
e8db683a-1201-4c42-9dd5-03878101dffa,0,0,0,1,0,0,0,0,0,0,0
e96cd82f-3e50-41dd-a527-b148956c1264,0,0,0,0,1,0,0,0,0,0,0
e9913251-bc8b-4a02-bb9e-d107525a28c4,0,0,0,0,1,0,0,0,0,0,0
eaf6a486-f8d3-4b9f-b701-f4e69dcc07a7,1,0,0,0,0,0,0,0,0,0,0
eb5f9397-2d2a-47b0-b6bd-b3cc95702e37,0,0,0,0,0,0,0,0,0,1,0
ec8daad5-73ea-4e85-8480-94e4a90693b9,1,0,0,0,0,0,0,0,0,0,0
ecaa0208-b510-486e-b549-235068d804e9,0,0,0,0,1,0,0,0,0,0,0
ed21e0f8-4fbd-498f-be55-f6a143a7c0a9,0,0,0,1,0,0,0,0,0,0,0
ed222120-c270-4993-a079-50c5ba3ee253,0,0,0,0,0,0,0,0,0,1,0
ed3ef58d-5dbe-461b-9f3d-6b41db45dc3a,1,0,0,0,0,1,0,0,1,0,0
edfd3e56-21c5-4f3f-8a6a-6397289e527d,1,0,0,0,0,0,0,0,1,0,0
ee0548e5-20b1-48f7-bdba-29eb20d14201,0,0,0,0,0,0,0,0,1,0,0
ee2172f3-4e9e-4ca6-9fda-7b9ae461e43d,0,0,0,0,0,0,0,0,0,0,1
ee2c23dd-beef-4ff8-96e4-218bb91a6c0f,0,0,0,0,0,1,0,0,0,0,0
ee4cb627-10ad-445a-9cab-4b3f47a11acb,0,0,0,0,0,0,0,0,0,0,1
eeb7a867-0e5d-49e1-8090-5efa2ab646ab,0,0,1,0,0,0,1,0,0,0,0
ef95f1ff-4a6f-4ed3-b669-ea15a3e51e38,0,0,0,0,0,0,0,0,0,1,0
f08554bd-68a7-43d6-af5a-e0f8bd43e093,0,1,0,0,1,0,0,0,0,0,0
f225fd74-025b-4a54-8810-8fc71b8af482,0,0,0,0,0,0,0,0,0,1,0
f300d5d4-1f94-4bf1-85cf-511721bdfe3f,0,0,0,0,0,0,0,0,1,0,0
f33878f6-5aaa-40d8-a525-336d8b90a3b8,0,0,0,0,1,0,0,0,0,0,0
f396035c-6b40-447f-bb0c-da95de4f3827,0,0,0,1,0,0,0,0,0,0,0
f3d2e90a-e362-4c5e-b3e8-e19161cfd671,0,0,0,0,0,0,0,0,0,1,0
f43862ce-bfac-47ec-a788-1abebe7fed15,0,0,0,1,0,0,0,0,0,0,0
f49c317a-b429-476f-a928-010dce83db7c,0,0,0,0,1,0,0,0,0,0,0
f4e34990-66a7-4527-a2ea-13a4885d948b,0,0,0,0,1,0,0,0,0,0,0
f4e78e59-9495-416b-be05-21f62e2638b3,0,0,0,0,1,1,0,0,0,0,0
f50c22eb-695d-4120-875a-b2e562bd95ee,0,0,0,0,0,0,0,0,0,0,1
f51a3a11-5fb6-4585-87cf-72b62b5e4b7d,0,0,0,1,0,0,0,0,0,0,0
f52eb344-c64d-4ba6-81c0-7634eed27a99,0,0,0,0,1,0,0,0,0,0,0
f5428f2b-b47a-4415-9e34-12624dc593ee,0,0,0,0,0,0,0,0,0,0,1
f63b319e-50b5-4313-b885-568755ebefdd,0,0,0,0,1,0,0,0,0,0,0
f6d3d618-715a-44c5-8e9e-9d576fdd4d7a,0,0,0,1,0,0,0,0,0,0,0
f70c3271-b2cc-471d-9a19-3479e1652d89,0,0,0,0,0,0,0,0,0,1,1
f7428ca3-2743-440c-95ca-6514154adc2e,0,0,0,0,0,0,1,0,1,0,0
f78d318b-6207-4923-9275-b8677888e26e,1,0,0,0,0,0,0,0,1,0,0
f7c6997e-56d2-4ac1-9839-a37ffd241eba,1,0,0,1,0,0,0,0,0,0,0
f7cf3bdf-cc9c-4109-aec6-96b865522a82,0,0,0,0,1,0,0,0,1,0,0
f7dd279a-9454-4bda-944e-d1f0656ea255,0,0,0,0,0,0,0,0,0,0,1
f8425890-05bf-46a0-a196-058d35841071,0,0,0,1,0,0,0,0,0,0,0
f8f1f2f4-c7e4-4512-957b-9e323c80a42e,0,0,0,0,0,0,0,0,0,0,1
f944ddf2-2c15-4967-84b6-c0c48c3c13de,0,0,0,0,0,1,0,0,0,1,0
f9dddeab-324f-45e6-bd8a-f8f9ff17450d,1,0,0,0,0,0,0,0,0,1,0
f9de137d-cdb3-4633-9ab1-496e52ef9599,0,0,0,0,0,1,0,0,0,1,0
fb06ffeb-0de1-4747-a678-4892f3be9b9c,0,0,0,0,1,0,0,0,0,0,0
fb23bf5e-e8ea-45e2-9dd4-39714230f96d,0,0,0,0,0,0,0,0,0,1,0
fb76a8d8-68a7-4ff7-b61a-2d8e16ecc940,1,0,0,0,0,0,0,0,0,0,0
fb9cd73e-58fe-4725-996f-844148af3cf0,1,0,0,0,0,0,0,0,0,0,0
fbafad1d-035c-4cf7-9d3d-ab4d70e9cda1,0,0,0,0,0,1,0,0,0,1,0
fc0bdd4a-d08a-41f6-9be8-135950a18205,0,0,0,0,1,0,0,0,0,1,0
fc134a8b-0e52-4dab-9e29-634f2f4a2b77,0,1,0,0,1,0,0,0,0,0,0
fc8486e2-8ef9-450c-8139-37f198aae05e,0,1,0,0,1,0,0,0,0,0,0
fc8ed8d7-27a0-404c-aee0-f529204f6164,0,0,0,0,1,0,0,0,0,0,0
fce19ec9-843a-4d6a-9e8f-698898bf673b,0,0,0,0,0,0,0,0,1,0,0
fe750d5b-b7ff-49c3-ab68-410ea4d6cf56,0,0,0,0,1,0,0,0,0,0,0
ff411884-cd5a-4d1d-a618-f0f3ec0079f6,1,0,0,1,0,0,0,0,0,0,0


In [ ]:
%%writefile /content/meshqc_code/provenance.json
{
  "vitb336": {
    "source": "vitb336_original_full.ipynb",
    "sha256": "4d8ba1c5fdf055e7731c48e8a8e05b4be8a0f11095195173f1623ab58051a03f",
    "cells": [
      6,
      12,
      17,
      19,
      22,
      24,
      26,
      32,
      34,
      36,
      38
    ],
    "tile": 336
  },
  "v5_224": {
    "source": "v5_224_original_full.ipynb",
    "sha256": "dcaafee00af52235f600365baaedc3285a89c7e691f357becaf2d0ab39521ebe",
    "cells": [
      6,
      10,
      15,
      17,
      20,
      22,
      24,
      25,
      30,
      32,
      34,
      36
    ],
    "tile": 224
  },
  "dino448_s2": {
    "source": "dino448_s2_original.ipynb",
    "sha256": "0e4ceed27d8fb35fedca000414a12c62ee0508c5b48492373f967d942174ed84",
    "cells": [
      5,
      11,
      16,
      18,
      21,
      23,
      25,
      26,
      31,
      33,
      35,
      37
    ],
    "tile": 448
  }
}


In [ ]:
import hashlib
for p in sorted(BUNDLE.iterdir()):
    print(f'  {p.name:<28s} {p.stat().st_size/1024:8.1f} КБ  '
          f'{hashlib.sha256(p.read_bytes()).hexdigest()[:12]}')


In [ ]:
def run_process(argv, log_name):
    log_path = PROJECT_DIR / 'logs' / log_name
    log_path.parent.mkdir(parents=True, exist_ok=True)
    argv = [str(x) for x in argv]
    print('$', ' '.join(argv), flush=True)
    t0 = time.time()
    with open(log_path, 'w', encoding='utf-8') as log:
        proc = subprocess.Popen([sys.executable] + argv, stdout=subprocess.PIPE,
                                stderr=subprocess.STDOUT, text=True, bufsize=1)
        try:
            for line in proc.stdout:
                print(line, end='', flush=True); log.write(line)
        except KeyboardInterrupt:
            proc.send_signal(signal.SIGINT); proc.wait(); raise
    rc = proc.wait()
    print(f'\n[{log_name}] код {rc}, {(time.time()-t0)/60:.1f} мин', flush=True)
    if rc != 0:
        raise RuntimeError(f'{log_name}: код {rc}, лог {log_path}')


def branch(tag, inference_only=False):
    run_process([BUNDLE / 'worker.py', '--branch', tag, '--root', PROJECT_DIR,
                 '--workers', N_WORKERS, '--folds', FOLDS]
                + (['--inference-only'] if inference_only else []),
                f'{tag}_{"infer" if inference_only else "train"}.log')


for t in NAMES:
    run_process([BUNDLE / 'worker.py', '--branch', t, '--root', PROJECT_DIR,
                 '--validate-only'], f'{t}_validate.log')


In [ ]:
branch('vitb336')


In [ ]:
branch('dino448_s2')


In [ ]:
branch('v5_224')


In [ ]:
import numpy as np, pandas as pd, itertools
from sklearn.metrics import f1_score

OUT_DIR = PROJECT_DIR / 'results'; OUT_DIR.mkdir(parents=True, exist_ok=True)


def metric(y, p):
    fq = f1_score(y[:, 10], p[:, 10], zero_division=0)
    fa = f1_score(y[:, :10], p[:, :10], average='weighted', zero_division=0)
    return 10 * fq + 10 * fa, fq, fa


def exact_thr(y, p, plateau=.995):
    """Точный argmax F1 за один проход. Разрез допустим только там, где значение p
    меняется: иначе порог попадает между двумя одинаковыми вероятностями и `p > th`
    выбрасывает оба объекта вместо одного."""
    y = np.asarray(y).astype(np.int32); p = np.asarray(p, np.float64)
    pos = int(y.sum())
    if pos == 0:
        return .5, 0.
    o = np.argsort(-p, kind='stable'); ys, ps = y[o], p[o]
    sc = 2 * np.cumsum(ys) / (np.arange(1, len(y) + 1) + pos)
    cut = np.empty(len(ps), bool); cut[-1] = True; cut[:-1] = ps[:-1] > ps[1:]
    best = float(sc[cut].max())
    idx = np.where((sc >= plateau * best) & cut)[0]
    i = int(idx[len(idx) // 2])
    return float((ps[i] + ps[i+1]) / 2 if i+1 < len(ps) else ps[i] - 1e-6), best


def quick_thr(y, p):
    return np.array([exact_thr(y[:, i], p[:, i])[0] for i in range(11)])


def lmix(arrays, w):
    """Среднее логитов: среднее вероятностей стягивает уверенные предсказания
    к середине, среднее логитов сохраняет уверенность при согласии моделей."""
    w = np.asarray(w, float); w = w / w.sum()
    z = sum(wi * np.log(np.clip(a, 1e-6, 1-1e-6) / (1 - np.clip(a, 1e-6, 1-1e-6)))
            for wi, a in zip(w, arrays))
    return 1 / (1 + np.exp(-z))


def fit_rule(prob, y, q_src):
    th = np.array([exact_thr(y[:, i], prob[:, i])[0] for i in range(10)])
    binary = (prob[:, :10] > th).astype(int)
    soft = np.prod(1 - np.clip(prob[:, :10], 0, 1), 1)
    hard = (binary.sum(1) == 0).astype(float)
    best = (-1, .5, .5, 'mix')
    for a in np.arange(0, 1.01, .05):
        pq = a * q_src + (1 - a) * soft
        t, v = exact_thr(y[:, 10], pq)
        if v > best[0]:
            best = (v, float(a), float(t), 'mix')
    for mode in ('and', 'or'):
        for a in np.arange(0, 1.01, .1):
            pq = a * q_src + (1 - a) * soft
            t, _ = exact_thr(y[:, 10], pq)
            q = (pq > t) * hard if mode == 'and' else np.maximum(pq > t, hard)
            v = f1_score(y[:, 10], q.astype(int), zero_division=0)
            if v > best[0]:
                best = (v, float(a), float(t), mode)
    return dict(th=th, q_alpha=best[1], q_th=best[2], q_mode=best[3])


def apply_rule(prob, q_src, rule):
    dfc = (prob[:, :10] > rule['th']).astype(int)
    soft = np.prod(1 - np.clip(prob[:, :10], 0, 1), 1)
    hard = (dfc.sum(1) == 0).astype(int)
    pq = rule['q_alpha'] * q_src + (1 - rule['q_alpha']) * soft
    qh = (pq > rule['q_th']).astype(int)
    q = {'mix': qh, 'and': qh * hard, 'or': np.maximum(qh, hard)}[rule['q_mode']]
    return np.concatenate([dfc, q[:, None]], 1)


def consistency(pred, prob, th):
    """quality = 1 ровно тогда, когда нет ни одного дефекта. Объекту, названному
    плохим без единого дефекта, дописываем самый вероятный по отношению к порогу."""
    out = pred.copy()
    rows = np.where((out[:, 10] == 0) & (out[:, :10].sum(1) == 0))[0]
    if len(rows):
        ratio = np.nan_to_num(prob[rows, :10] / np.maximum(np.asarray(th)[:10], 1e-6), nan=-1.)
        out[rows, np.argmax(ratio, 1)] = 1
    return out, len(rows)


In [ ]:
base = PROJECT_DIR / 'branches' / NAMES[0]
train = pd.read_csv(base / 'train_manifest.csv').set_index('item_id')
test = pd.read_csv(base / 'test_manifest.csv').set_index('item_id')
y_all = train[TARGETS].to_numpy(int); fold_all = train['fold'].to_numpy()

oof, te, masks = [], [], []
for n in NAMES:
    d = PROJECT_DIR / 'branches' / n
    po = pd.read_csv(d / 'oof.csv').set_index('item_id').loc[train.index, TARGETS].to_numpy(float)
    pt = pd.read_csv(d / 'test_probabilities.csv').set_index('item_id') \
           .loc[test.index, TARGETS].to_numpy(float)
                                                                                 
                                                                           
                                                                 
    nb = int((~np.isfinite(po)).sum())
    if nb:
        print(f'{n}: {nb} нечисловых значений в OOF заменены нулём')
        po = np.nan_to_num(po, nan=0.0, posinf=1.0, neginf=0.0)
    pt = np.nan_to_num(pt, nan=0.0, posinf=1.0, neginf=0.0)
                                                                             
                                
    full = pd.read_csv(d / 'oof.csv').set_index('item_id').loc[train.index, TARGETS].to_numpy(float)
    masks.append(~np.isnan(full).all(1))
    oof.append(po); te.append(pt)

mask = np.logical_and.reduce(masks)
y, folds = y_all[mask], fold_all[mask]
O = [x[mask] for x in oof]
print(f'\nпокрытие: {int(mask.sum())} из {len(train)}')
for n, m in zip(NAMES, masks):
    print(f'  {n:<14s} {int(m.sum())}')


In [ ]:
def sc(p):
    return metric(y, (p > quick_thr(y, p)).astype(int))[0]


solo = {n: sc(o) for n, o in zip(NAMES, O)}
best = (-1, None)
for w in itertools.product((0, .25, .5, .75, 1.), repeat=3):
    if not sum(w):
        continue
    s = sc(lmix(O, w))
    if s > best[0]:
        best = (s, w)
weights = best[1]

po, pt = lmix(O, weights), lmix(te, weights)
qo = lmix([x[:, [10]] for x in O], weights)[:, 0]
qt = lmix([x[:, [10]] for x in te], weights)[:, 0]

rule = fit_rule(po, y, qo)
pred_o, _ = consistency(apply_rule(po, qo, rule), po, rule['th'])
oof_score, fq, fa = metric(y, pred_o)

nested = []
for k in np.unique(folds):
    fit, val = folds != k, folds == k
    rk = fit_rule(po[fit], y[fit], qo[fit])
    pv, _ = consistency(apply_rule(po[val], qo[val], rk), po[val], rk['th'])
    nested.append(metric(y[val], pv)[0])
nested = np.array(nested)

print('одиночные:', {k: round(v, 3) for k, v in solo.items()})
print('веса:     ', dict(zip(NAMES, weights)))
print(f'OOF:       {oof_score:.3f}  (quality {10*fq:.2f} + artefacts {10*fa:.2f})')
print(f'вложенная: {nested.mean():.3f} ± {nested.std():.3f}')


In [ ]:
import hashlib
pred, nfix = consistency(apply_rule(pt, qt, rule), pt, rule['th'])
sub = pd.DataFrame(pred, columns=TARGETS)
sub.insert(0, 'item_id', test.index.to_numpy())

assert len(sub) == len(test) and sub['item_id'].is_unique
assert set(np.unique(sub[TARGETS].to_numpy())) <= {0, 1}
assert not sub.isna().any().any()

path = OUT_DIR / 'submission.csv'
sub.to_csv(path, index=False); sub.to_csv('submission.csv', index=False)
sha = hashlib.sha256(path.read_bytes()).hexdigest().upper()

X = sub[TARGETS].to_numpy(); nd = X[:, :10].sum(1)
report = dict(weights=dict(zip(NAMES, map(float, weights))),
              solo={k: round(v, 4) for k, v in solo.items()},
              coverage=int(mask.sum()), oof=round(float(oof_score), 4),
              nested_mean=round(float(nested.mean()), 4),
              nested_std=round(float(nested.std()), 4),
              consistency_fixes=int(nfix), labels=int(nd.sum()),
              quality_ones=int(X[:, 10].sum()), sha256=sha)
(OUT_DIR / 'report.json').write_text(json.dumps(report, ensure_ascii=False, indent=2),
                                     encoding='utf-8')
print(json.dumps(report, ensure_ascii=False, indent=2))

ref_path = BUNDLE / 'submitted_reference.csv'
if ref_path.exists():
    ref = pd.read_csv(ref_path).set_index('item_id').reindex(sub['item_id'])
    d = X != ref[TARGETS].to_numpy()
    print(f'\nотличий от эталона: {d.sum()} ячеек в {d.any(1).sum()} строках')
    if d.sum():
        print(pd.DataFrame({'расхождений': d.sum(0), 'наш': X.sum(0),
                            'эталон': ref[TARGETS].to_numpy().sum(0)},
                           index=TARGETS).to_string())

display(sub.head())
from google.colab import files
files.download('submission.csv')
